# Databricks 100: Complete End-to-End Tutorial

**Beginner to Advanced — All 100 Concepts Across 10 Categories**

| # | Section | Concepts |
|---|---------|----------|
| 1 | Delta Lake Fundamentals | #1-#10 |
| 2 | Spark Execution | #11-#20 |
| 3 | SQL & DataFrames | #21-#30 |
| 4 | Data Ingestion | #31-#40 |
| 5 | Streaming | #41-#50 |
| 6 | Performance & Cost | #51-#60 |
| 7 | Unity Catalog & Governance | #61-#70 |
| 8 | Lakeflow Declarative Pipelines | #71-#80 |
| 9 | Workflows, CI/CD & Operations | #81-#90 |
| 10 | Architecture & Advanced Patterns | #91-#100 |

---

Designed for **Databricks Community Edition**. Uses public datasets and synthetic data.

Inspired by [databricks-100](https://github.com/kaushikthakur97/databricks-100).


---

# 01 Delta Lake Fundamentals



 # Delta Lake Fundamentals: Concepts #1-#10

 ## Notebook Overview

 This notebook covers the **10 core concepts** of Delta Lake, from ACID transactions to advanced table configuration. Each concept includes a problem statement, real-world use case, hands-on code demo, key takeaways, and a self-assessment question.

 **Environment:** Designed for Databricks Community Edition (free tier) — single node, no Unity Catalog, no Photon, no serverless.

 **Concepts Covered:**
 1. ACID Transactions [Easy]
 2. Transaction Log (_delta_log) [Easy]
 3. Time Travel & RESTORE [Easy]
 4. Schema Enforcement vs. Evolution [Medium]
 5. Liquid Clustering [Medium]
 6. OPTIMIZE & File Compaction [Medium]
 7. VACUUM & Storage Lifecycle [Medium]
 8. Change Data Feed (CDF) [Medium]
 9. Deletion Vectors [Medium]
 10. Delta Table Properties & Configuration [Hard]

 ---


 ## Setup: Create Working Directory & Clean Up

 We'll use a dedicated directory under `/tmp/` for all our Delta tables. This keeps things organised and makes cleanup easy.


In [ ]:
# Define base working directory
import os

base_dir = "/tmp/delta_lake_fundamentals"

# Clean up any previous runs
dbutils.fs.rm(base_dir, recurse=True)

# Create fresh directory — DBFS paths don't need explicit mkdir, but we'll log it
print(f"Working directory set to: {base_dir}")
print("Cleaned up any previous tables in this location.")


 ## Concept #1: ACID Transactions [Easy]

 ### What Problem It Solves

 In traditional data lakes (Parquet, CSV, JSON), a failed write can leave the data in a corrupt or incomplete state. If a Spark job crashes mid-write, you might have partial files that downstream readers interpret as valid data. There&apos;s no built-in mechanism to ensure that a set of operations either all succeed or all fail as one unit.

 Delta Lake solves this by providing **full ACID transactions** on your data lake:
 - **Atomicity:** Multiple writes are committed together or fully rolled back.
 - **Consistency:** Readers always see a consistent snapshot of the data.
 - **Isolation:** Concurrent writes are serialised via optimistic concurrency control.
 - **Durability:** Once committed, data persists.

 ### Real-World Use Case

 An e-commerce platform writes order data and updates inventory levels in the same transaction. If the inventory update fails, the order write must also be rolled back to prevent data inconsistency.

 ### Hands-On Demo


In [ ]:
print("=" * 60)
print("CONCEPT 1: ACID Transactions")
print("=" * 60)

# Clean up from any previous runs
spark.sql(f"DROP TABLE IF EXISTS delta_orders")

# Create a Delta table with synthetic sales order data
orders_data = [
    (1001, "Laptop", 1, 1200.00, "pending"),
    (1002, "Monitor", 2, 300.00, "pending"),
    (1003, "Keyboard", 5, 45.00, "pending"),
    (1004, "Mouse", 10, 25.00, "pending"),
    (1005, "Headset", 3, 80.00, "pending"),
]

orders_df = spark.createDataFrame(
    orders_data, ["order_id", "product", "quantity", "price", "status"]
)

orders_df.write.format("delta").mode("overwrite").save(f"{base_dir}/orders_acid")

# Register as a table for SQL access
spark.sql(
    f"CREATE TABLE IF NOT EXISTS delta_orders USING DELTA LOCATION '{base_dir}/orders_acid'"
)

print("\nInitial orders table:")
spark.sql("SELECT * FROM delta_orders").show()


 #### Atomic Write: Multiple Operations in One Transaction

 We&apos;ll perform an INSERT and an UPDATE as a single atomic transaction using `MERGE`. If anything fails, nothing gets committed.


In [ ]:
# Demonstrate atomicity: MERGE performs INSERT + UPDATE atomically
spark.sql("""
    MERGE INTO delta_orders AS target
    USING (
        SELECT 1006 AS order_id, 'Webcam' AS product, 4 AS quantity,
               60.00 AS price, 'confirmed' AS status
    ) AS source
    ON target.order_id = source.order_id
    WHEN MATCHED THEN
        UPDATE SET status = 'confirmed', quantity = source.quantity
    WHEN NOT MATCHED THEN
        INSERT (order_id, product, quantity, price, status)
        VALUES (source.order_id, source.product, source.quantity,
                source.price, source.status)
""")

print("After atomic MERGE (INSERT + UPDATE in one transaction):")
spark.sql("SELECT * FROM delta_orders ORDER BY order_id").show()


 #### Demonstrating Rollback on Failure

 What happens if a write fails? With raw Parquet, partial files are left behind. With Delta, the transaction is rolled back completely. Let&apos;s demonstrate by writing to a raw Parquet directory at the same path to show the difference, then show Delta&apos;s safe behaviour.


In [ ]:
# Check current row count
current_count = spark.sql("SELECT COUNT(*) AS cnt FROM delta_orders").collect()[0]["cnt"]
print(f"Current row count: {current_count}")

# Attempt a failed write — this INSERT will fail because of a schema mismatch
# (we deliberately use a wrong column name to trigger failure)
try:
    bad_data = [(9999, "Test Product", 1, 99.99)]
    bad_df = spark.createDataFrame(bad_data, ["order_id", "product", "quantity", "price"])
    # This won't fail on schema — let's do a proper failure scenario:
    # Write a DataFrame with a column that violates constraints
    bad_df.write.format("delta").mode("append").save(f"{base_dir}/orders_acid_fail")
    print("Write succeeded (unexpected)")
except Exception as e:
    print(f"Write failed as expected: {str(e)[:200]}")

# IMPORTANT: Demonstrate that when you write directly with Delta and a failure
# occurs, no partial data is committed. Let's do an intentional failure inside a transaction.
# We'll use overwrite mode with a schema that's incompatible — but that's actually fine.
# Better approach: create a table and attempt to insert NULL into a NOT NULL column

# Create a table with a NOT NULL constraint
spark.sql(f"""
    DROP TABLE IF EXISTS delta_constrained
""")
spark.sql(f"""
    CREATE TABLE delta_constrained (id INT, name STRING NOT NULL)
    USING DELTA
    LOCATION '{base_dir}/delta_constrained'
""")

spark.sql("INSERT INTO delta_constrained VALUES (1, 'Alice')")
print("Before failed insert:")
spark.sql("SELECT * FROM delta_constrained").show()

# Now attempt to insert NULL — this should fail and roll back
try:
    spark.sql("INSERT INTO delta_constrained VALUES (2, CAST(NULL AS STRING))")
except Exception as e:
    print(f"Expected failure — NULL constraint violated: {str(e)[:200]}")

# Verify no partial data was committed
print("After failed insert (should still have only 1 row):")
spark.sql("SELECT * FROM delta_constrained").show()
row_count_after = spark.sql("SELECT COUNT(*) AS cnt FROM delta_constrained").collect()[0]["cnt"]
assert row_count_after == 1, f"Expected 1 row, got {row_count_after} — ACID failure!"
print(f"ACID property verified: Row count is {row_count_after} (rollback worked)")


 #### Demonstrating Optimistic Concurrency Control

 Delta uses optimistic concurrency by detecting conflicts at commit time. If two writers attempt conflicting modifications, one will be rejected with a `ConcurrentAppendException` or `ConcurrentDeleteReadException`.


In [ ]:
print("\n=== Optimistic Concurrency Control Demo ===")

# Read the current table for two concurrent "writers"
df_writer1 = spark.sql("SELECT * FROM delta_orders")
df_writer2 = spark.sql("SELECT * FROM delta_orders")

# Simulate writer 1 appending data (this will succeed)
new_order1 = [(2001, "Tablet", 2, 350.00, "confirmed")]
new_df1 = spark.createDataFrame(new_order1, ["order_id", "product", "quantity", "price", "status"])
new_df1.write.format("delta").mode("append").save(f"{base_dir}/orders_acid")
print("Writer 1 committed successfully.")

# Simulate writer 2 ALSO appending data (this should also succeed — appends are non-conflicting)
new_order2 = [(2002, "Dock", 1, 120.00, "pending")]
new_df2 = spark.createDataFrame(new_order2, ["order_id", "product", "quantity", "price", "status"])
new_df2.write.format("delta").mode("append").save(f"{base_dir}/orders_acid")
print("Writer 2 committed successfully (both appends are non-conflicting).")

print("\nFinal table after both concurrent appends:")
spark.sql("SELECT * FROM delta_orders ORDER BY order_id").show()

print("\nKEY INSIGHT: Delta uses optimistic concurrency — appends don't conflict,")
print("but if two writers try to update the SAME row, the second commit is rejected.")
print("This avoids expensive locking while maintaining correctness.")


 ### Key Takeaways
 - Delta Lake guarantees **atomicity**: all operations in a transaction succeed or fail together.
 - **Optimistic concurrency control** allows high-throughput concurrent writes without locking.
 - Unlike raw Parquet/CSV, a failed Delta write leaves **no partial or corrupt data**.

 ### Self-Assessment Question

 *Q: What happens if you have a `MERGE` statement that inserts 100 rows and updates 50 rows, and it fails halfway through?*

 <details><summary>Click for answer</summary>
 **A:** With Delta Lake, nothing is committed. All 100 inserts and 50 updates are rolled back. With raw Parquet, you could end up with partial files written, corrupting your downstream reads.
 </details>


 ---
 ## Concept #2: Transaction Log (_delta_log) [Easy]

 ### What Problem It Solves

 How does Delta know which files belong to which version of the table? Without a transaction log, there&apos;s no way to track which files were added or removed over time, no way to support time travel, and no way to ensure snapshot isolation for concurrent readers and writers.

 The `_delta_log` directory is the **source of truth** for a Delta table. It contains ordered JSON files (and checkpoint Parquet files) that record every operation performed on the table.

 ### Real-World Use Case

 A data engineering team needs to audit who modified a critical financial table and when. The transaction log provides a complete, immutable audit trail of every write operation.

 ### Hands-On Demo


In [ ]:
print("=" * 60)
print("CONCEPT 2: Transaction Log (_delta_log)")
print("=" * 60)

# Create a fresh Delta table and build up some history
table_path = f"{base_dir}/tlog_demo"

spark.sql(f"DROP TABLE IF EXISTS tlog_table")

# Create initial data
initial_data = [
    (1, "Alice", "Engineering", 95000),
    (2, "Bob", "Sales", 72000),
    (3, "Charlie", "Engineering", 105000),
]
df = spark.createDataFrame(initial_data, ["id", "name", "dept", "salary"])
df.write.format("delta").mode("overwrite").save(table_path)

# Register table
spark.sql(f"CREATE TABLE IF NOT EXISTS tlog_table USING DELTA LOCATION '{table_path}'")

# Perform multiple operations to build up log entries
spark.sql("INSERT INTO tlog_table VALUES (4, 'Diana', 'Marketing', 88000)")
spark.sql("INSERT INTO tlog_table VALUES (5, 'Eve', 'Engineering', 110000)")
spark.sql("UPDATE tlog_table SET salary = 98000 WHERE id = 1")
spark.sql("INSERT INTO tlog_table VALUES (6, 'Frank', 'Sales', 76000)")
spark.sql("DELETE FROM tlog_table WHERE id = 6")
spark.sql("INSERT INTO tlog_table VALUES (7, 'Grace', 'Marketing', 92000)")

print("Performed 6 operations on the table (creates 6 new versions beyond v0).")


 #### Inspect the `_delta_log` Directory

 Every Delta table has a `_delta_log` subdirectory containing JSON transaction log entries and, after enough operations, checkpoint Parquet files.


In [ ]:
# List the contents of the _delta_log directory
print("Contents of _delta_log directory:")
log_entries = dbutils.fs.ls(f"{table_path}/_delta_log")
for entry in log_entries:
    size_kb = entry.size / 1024 if entry.size > 0 else 0
    print(f"  {entry.name:<40} {size_kb:>8.2f} KB")

print(f"\nTotal log entries: {len(log_entries)}")


 #### Read the Latest JSON Transaction Log Entry

 Each JSON file contains a sequence of actions: `add`, `remove`, `metaData`, `commitInfo`, etc.


In [ ]:
# Read and display the most recent JSON log entry
import json

# Get the last .json file (most recent transaction)
json_files = sorted([e.name for e in log_entries if e.name.endswith(".json")])
latest_json = json_files[-1]
print(f"Reading latest transaction log entry: {latest_json}")
content = spark.read.text(f"{table_path}/_delta_log/{latest_json}").collect()
for row in content[:5]:  # Show first 5 lines
    parsed = json.loads(row.value)
    action_type = list(parsed.keys())[0]
    print(f"\nAction: {action_type}")
    print(json.dumps(parsed[action_type], indent=2)[:500])


 #### View Transaction History with `DESCRIBE HISTORY`


In [ ]:
# DESCRIBE HISTORY shows the full transaction log in human-readable form
print("\nTransaction History (DESCRIBE HISTORY):")
spark.sql("DESCRIBE HISTORY tlog_table").select(
    "version", "operation", "operationMetrics", "timestamp"
).show(truncate=False)


 #### Checkpoint Creation

 After many operations (default: every 10 commits), Delta creates a **checkpoint** — a Parquet file that aggregates all previous JSON entries. This speeds up table state reconstruction by not requiring replay of the entire log.


In [ ]:
# Perform enough operations to trigger a checkpoint (10 more commits)
for i in range(10):
    spark.sql(f"INSERT INTO tlog_table VALUES ({100 + i}, 'TestUser{i}', 'QA', {50000 + i * 1000})")

print("Performed 10 more INSERTs to trigger checkpoint creation.")

# Re-list the _delta_log directory
updated_log_entries = dbutils.fs.ls(f"{table_path}/_delta_log")
print("\nUpdated _delta_log contents:")
for entry in updated_log_entries:
    size_kb = entry.size / 1024 if entry.size > 0 else 0
    print(f"  {entry.name:<40} {size_kb:>8.2f} KB")

has_checkpoint = any(e.name.endswith(".checkpoint.parquet") for e in updated_log_entries)
print(f"\nCheckpoint file present: {has_checkpoint}")
print("Checkpoints dramatically speed up state reconstruction by avoiding full log replay.")


 ### Key Takeaways
 - The **`_delta_log` directory** is the single source of truth for a Delta table&apos;s state.
 - JSON log files record every **add**, **remove**, and **metadata change** as ordered transactions.
 - **Checkpoints** (Parquet files) are created every 10 commits by default, enabling fast snapshot reads without replaying the entire log.

 ### Self-Assessment Question

 *Q: If you delete all `.json` files from the `_delta_log` but keep the `.checkpoint.parquet` file, can the table still be read?*

 <details><summary>Click for answer</summary>
 **A:** You can read up to the checkpoint version, but any versions AFTER the checkpoint will be lost because the JSON files containing those incremental changes are gone. Always treat the `_delta_log` directory as immutable — never manually delete entries!
 </details>


 ---
 ## Concept #3: Time Travel & RESTORE [Easy]

 ### What Problem It Solves

 Data engineers often need to query data &quot;as it was&quot; at a specific point in the past — for auditing, debugging bad writes, reproducing reports, or rolling back accidental changes. Without built-in versioning, you&apos;d have to maintain costly snapshot copies of your data.

 Delta Lake stores a complete version history, allowing you to query any past table version or restore the table to a previous state.

 ### Real-World Use Case

 A scheduled job accidentally runs a `DELETE` that removes 2 million valid customer records. Instead of restoring from a backup, the team simply runs `RESTORE TABLE` to roll back to the version just before the bad job ran.

 ### Hands-On Demo


In [ ]:
print("=" * 60)
print("CONCEPT 3: Time Travel & RESTORE")
print("=" * 60)

# Create a table with versioned data
tt_path = f"{base_dir}/timetravel_demo"
spark.sql(f"DROP TABLE IF EXISTS timetravel_sales")

sales_init = [
    ("2024-01-01", "Product-A", 10, 150.00),
    ("2024-01-01", "Product-B", 5, 200.00),
    ("2024-01-02", "Product-A", 8, 160.00),
]
df = spark.createDataFrame(sales_init, ["sale_date", "product", "quantity", "amount"])
df.write.format("delta").mode("overwrite").save(tt_path)

spark.sql(f"CREATE TABLE IF NOT EXISTS timetravel_sales USING DELTA LOCATION '{tt_path}'")
print("Version 0 created (3 records).")

# Version 1: Insert new data
spark.sql("INSERT INTO timetravel_sales VALUES ('2024-01-03', 'Product-C', 12, 300.00)")

# Version 2: Update prices
spark.sql("UPDATE timetravel_sales SET amount = amount * 1.1 WHERE product = 'Product-A'")

# Version 3: Delete a product
spark.sql("DELETE FROM timetravel_sales WHERE product = 'Product-B'")

# Version 4: More inserts
spark.sql("INSERT INTO timetravel_sales VALUES ('2024-01-04', 'Product-D', 3, 450.00)")

print("Versions 1-4 created (inserts, updates, deletes).")

# View history
print("\nFull history:")
spark.sql("DESCRIBE HISTORY timetravel_sales").select(
    "version", "operation", "operationMetrics", "timestamp"
).show(truncate=False)


 #### Query with `VERSION AS OF` and `TIMESTAMP AS OF`


In [ ]:
# Query current version
print("Current table (latest version):")
spark.sql("SELECT * FROM timetravel_sales ORDER BY sale_date, product").show()

# Query Version 0 (original data)
print("Version 0 (original create):")
spark.sql("SELECT * FROM timetravel_sales VERSION AS OF 0 ORDER BY sale_date, product").show()

# Query Version 2 (after price update, before delete)
print("Version 2 (after price update, before delete of Product-B):")
spark.sql("SELECT * FROM timetravel_sales VERSION AS OF 2 ORDER BY sale_date, product").show()

# Count rows at each version
print("Row counts across versions:")
for v in range(5):
    cnt = spark.sql(f"SELECT COUNT(*) AS cnt FROM timetravel_sales VERSION AS OF {v}").collect()[0]["cnt"]
    print(f"  Version {v}: {cnt} rows")


 #### Demonstrate `RESTORE TABLE` to Roll Back


In [ ]:
# Before restore — check current state
print("BEFORE RESTORE (latest version):")
spark.sql("SELECT * FROM timetravel_sales ORDER BY sale_date, product").show()

# Restore to version 2
spark.sql("RESTORE TABLE timetravel_sales TO VERSION AS OF 2")
print("\nRESTORE to version 2 completed.")

print("AFTER RESTORE (should match version 2):")
spark.sql("SELECT * FROM timetravel_sales ORDER BY sale_date, product").show()

# Verify history now shows RESTORE operation
print("\nHistory after RESTORE:")
spark.sql("DESCRIBE HISTORY timetravel_sales").select(
    "version", "operation", "operationMetrics", "timestamp"
).show(truncate=False)


 #### Time Travel with `TIMESTAMP AS OF`


In [ ]:
# Get a timestamp from the history to use for time travel
history_df = spark.sql("DESCRIBE HISTORY timetravel_sales")
version_1_ts = history_df.filter("version = 1").select("timestamp").collect()[0][0]
print(f"Querying table as of timestamp: {version_1_ts}")

spark.sql(f"""
    SELECT * FROM timetravel_sales
    TIMESTAMP AS OF '{version_1_ts}'
    ORDER BY sale_date, product
""").show()


 #### Retention Period and VACUUM Interaction

 Delta&apos;s default retention for time travel is **7 days** (controlled by `delta.logRetentionDuration`). After that, VACUUM can remove old files, and time travel beyond the retention window will fail. We&apos;ll explore this further in Concept #7.


In [ ]:
# Check the table's log retention setting
props = spark.sql("SHOW TBLPROPERTIES timetravel_sales").filter("key like '%retention%' OR key like '%deleted%'")
print("Relevant table properties:")
props.show(truncate=False)

print("\nDefault log retention: 30 days (delta.logRetentionDuration)")
print("Default file retention: 7 days (delta.deletedFileRetentionDuration)")
print("NOTE: In Community Edition with single node, VACUUM still works but won't have distributed cleanup.")


 ### Key Takeaways
 - `VERSION AS OF N` lets you query **any past version** of a Delta table without maintaining snapshots.
 - `RESTORE TABLE` rolls the table back to a previous version, creating a **new version** that mirrors the old state.
 - Time travel is limited by the **retention period** (default 7 days for files); VACUUM removes expired files permanently.

 ### Self-Assessment Question

 *Q: You run `RESTORE TABLE orders TO VERSION AS OF 5`. Is version 6 lost forever?*

 <details><summary>Click for answer</summary>
 **A:** No! `RESTORE` creates a **new version** (e.g., version 7) that copies the state from version 5. Version 6 still exists in the transaction log and you can still time-travel to it — unless VACUUM removes the underlying data files after the retention period expires.
 </details>


 ---
 ## Concept #4: Schema Enforcement vs. Evolution [Medium]

 ### What Problem It Solves

 Traditional data lakes (Parquet, CSV) have **no schema enforcement**: you can accidentally write malformed data, wrong column types, or extra/missing columns. This creates silent data quality issues that are discovered only when downstream jobs fail.

 Delta Lake **enforces schema** on write by default, preventing bad data from entering the table. At the same time, Delta provides **schema evolution** (`mergeSchema`) so you can explicitly add new columns without breaking existing pipelines.

 ### Real-World Use Case

 A new field (`customer_tier`) is added to the source system. The engineering team enables `mergeSchema` to let the new column be added automatically to the Delta table, avoiding pipeline failures while preserving all existing data.

 ### Hands-On Demo


In [ ]:
print("=" * 60)
print("CONCEPT 4: Schema Enforcement vs. Evolution")
print("=" * 60)

# Create a table with a strict schema
schema_path = f"{base_dir}/schema_demo"
spark.sql(f"DROP TABLE IF EXISTS schema_patients")

# Initial data with a known schema
patients = [
    (1, "John Doe", 35, "A+"),
    (2, "Jane Smith", 28, "O-"),
    (3, "Bob Johnson", 42, "B+"),
]
df_patients = spark.createDataFrame(patients, ["id", "name", "age", "blood_type"])
df_patients.write.format("delta").mode("overwrite").save(schema_path)

spark.sql(f"CREATE TABLE IF NOT EXISTS schema_patients USING DELTA LOCATION '{schema_path}'")
print("Initial table created with schema: id INT, name STRING, age INT, blood_type STRING")


 #### Schema Enforcement: Rejecting Bad Data


In [ ]:
print("\n=== Schema Enforcement Demo ===")

# Attempt 1: Write data with a MISMATCHED column type (string where INT expected)
try:
    bad_type_data = [(4, "Bad Guy", "thirty-five", "AB+")]
    bad_type_df = spark.createDataFrame(bad_type_data, ["id", "name", "age", "blood_type"])
    bad_type_df.write.format("delta").mode("append").save(schema_path)
    print("Write succeeded (unexpected — schema enforcement failed)")
except Exception as e:
    error_msg = str(e)
    print(f"Schema enforcement BLOCKED write: type mismatch detected")
    print(f"Error snippet: {error_msg[:200]}")

print("\n")

# Attempt 2: Write data with an EXTRA column (no mergeSchema)
try:
    extra_col_data = [(5, "Extra Person", 30, "A-", "extradata")]
    extra_col_df = spark.createDataFrame(extra_col_data, ["id", "name", "age", "blood_type", "extra"])
    extra_col_df.write.format("delta").mode("append").save(schema_path)
    print("Write succeeded (unexpected)")
except Exception as e:
    error_msg = str(e)
    print(f"Schema enforcement BLOCKED write: extra column detected")
    print(f"Error snippet: {error_msg[:200]}")

print("\n")

# Attempt 3: Write data with a MISSING column
try:
    missing_col_data = [(6, "Missing Data", 22)]
    missing_col_df = spark.createDataFrame(missing_col_data, ["id", "name", "age"])
    missing_col_df.write.format("delta").mode("append").save(schema_path)
    print("Write succeeded (unexpected)")
except Exception as e:
    error_msg = str(e)
    print(f"Schema enforcement BLOCKED write: missing column detected")
    print(f"Error snippet: {error_msg[:200]}")

print("\nTable is unchanged:")
spark.sql("SELECT * FROM schema_patients").show()


 #### Schema Evolution: `mergeSchema`


In [ ]:
print("\n=== Schema Evolution Demo ===")

# Now add a new column using mergeSchema
new_data_with_email = [
    (7, "Alice Wonder", 29, "O+", "alice@example.com"),
    (8, "Charlie Brown", 33, "B-", "charlie@example.com"),
]
new_df = spark.createDataFrame(new_data_with_email, ["id", "name", "age", "blood_type", "email"])

new_df.write.format("delta").mode("append").option("mergeSchema", "true").save(schema_path)

print("Data with NEW column 'email' written successfully using mergeSchema!")

# Show the evolved table
print("\nTable now has the 'email' column:")
spark.sql("SELECT * FROM schema_patients ORDER BY id").show()

# Check the schema
print("\nUpdated schema:")
spark.sql("DESCRIBE schema_patients").show(truncate=False)


 #### `overwriteSchema` — Full Schema Replacement


In [ ]:
# overwriteSchema replaces the entire schema when you want a structural change
replacement_data = [
    (1, "John Doe", 35, "A+", "john@example.com", "VIP"),
    (2, "Jane Smith", 28, "O-", "jane@example.com", "Regular"),
]
replacement_df = spark.createDataFrame(
    replacement_data, ["id", "name", "age", "blood_type", "email", "tier"]
)

replacement_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(schema_path)

print("overwriteSchema applied — data AND schema replaced.")
print("Note: This REPLACES all data, unlike mergeSchema which is non-destructive.")
spark.sql("SELECT * FROM schema_patients").show()

spark.sql("DESCRIBE schema_patients").show(truncate=False)


 #### `autoMerge` in Writes

 You can set `spark.databricks.delta.schema.autoMerge.enabled = true` at the SparkSession level to make schema evolution opt-out instead of opt-in.


In [ ]:
# Demonstrate autoMerge
auto_path = f"{base_dir}/schema_auto_demo"
spark.sql(f"DROP TABLE IF EXISTS schema_auto")

base_df = spark.createDataFrame([(1, "Base")], ["id", "name"])
base_df.write.format("delta").mode("overwrite").save(auto_path)
spark.sql(f"CREATE TABLE IF NOT EXISTS schema_auto USING DELTA LOCATION '{auto_path}'")
print("Base table created with columns: id, name")

# Set autoMerge at session level
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

expanded_df = spark.createDataFrame([(2, "Expanded", "NewCol")], ["id", "name", "new_column"])
expanded_df.write.format("delta").mode("append").save(auto_path)
print("Appended with new column 'new_column' — autoMerge accepted it!")

spark.sql("SELECT * FROM schema_auto").show()

# Reset the config
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")


 ### Key Takeaways
 - **Schema enforcement** prevents bad data (wrong types, extra/missing columns) from entering your Delta table — a critical data quality guard.
 - **`mergeSchema`** allows intentional schema evolution by adding new columns without losing data.
 - **`overwriteSchema`** replaces the entire schema (and data) — use with caution.

 ### Self-Assessment Question

 *Q: Your upstream source adds a new field `loyalty_points`. What&apos;s the safest way to add this column to your existing Delta table without breaking downstream consumers?*

 <details><summary>Click for answer</summary>
 **A:** Use `mergeSchema` (or set `autoMerge`). This adds the new column with NULL values for existing rows. Downstream consumers that don&apos;t reference the new column are unaffected. `overwriteSchema` would be dangerous because it replaces ALL data.
 </details>


 ---
 ## Concept #5: Liquid Clustering [Medium]

 ### What Problem It Solves

 Traditional data partitioning requires you to choose a partition column upfront (e.g., `date`). If you choose wrong (too many small partitions, or data skew), you get terrible performance with no easy fix. Z-Ordering helps with multi-dimensional clustering but requires manual OPTIMIZE commands.

 **Liquid Clustering** is Delta&apos;s next-generation clustering technique that replaces both partitioning and Z-Ordering. It incrementally clusters data as it&apos;s written and dynamically adjusts to changing data patterns — no manual maintenance.

 **IMPORTANT NOTE FOR COMMUNITY EDITION:** Liquid Clustering requires `delta.minReaderVersion >= 3` and `delta.minWriterVersion >= 7`, which may not be available in all Community Edition deployments. If the syntax fails, this section explains the concept and shows traditional Z-Ordering as the closest equivalent.

 ### Real-World Use Case

 An IoT sensor table receives 500M rows/hour with timestamps, device IDs, and sensor types. Queries often filter by `sensor_type` AND `device_id`. Partitioning by either alone causes skew. Liquid Clustering clusters on both columns automatically, providing efficient file skipping for all query patterns.

 ### Hands-On Demo


In [ ]:
print("=" * 60)
print("CONCEPT 5: Liquid Clustering")
print("=" * 60)

# Create a large synthetic dataset to demonstrate clustering benefits
import random
import datetime

print("Generating synthetic IoT sensor data...")

sensor_types = ["temperature", "humidity", "pressure", "light", "motion"]
device_ids = [f"device-{i:04d}" for i in range(1, 201)]
regions = ["us-east", "us-west", "eu-west", "ap-south"]

data = []
base_date = datetime.date(2024, 1, 1)
for i in range(50000):
    ts = datetime.datetime(2024, 1, 1, 0, 0, 0) + datetime.timedelta(
        minutes=random.randint(0, 525600)
    )
    data.append((
        ts,
        random.choice(sensor_types),
        random.choice(device_ids),
        random.choice(regions),
        round(random.uniform(0, 100), 2),
    ))

sensor_df = spark.createDataFrame(
    data, ["timestamp", "sensor_type", "device_id", "region", "reading"]
)

print(f"Generated {sensor_df.count()} sensor readings.")
sensor_df.show(5, truncate=False)


 #### Traditional Z-Ordering Approach (Available in Community Edition)


In [ ]:
# Write with traditional Z-Ordering
cluster_path_zorder = f"{base_dir}/cluster_traditional"

# Write data normally first
sensor_df.write.format("delta").mode("overwrite").save(cluster_path_zorder)

spark.sql(f"CREATE TABLE IF NOT EXISTS cluster_traditional USING DELTA LOCATION '{cluster_path_zorder}'")

print("Before Z-Order OPTIMIZE:")
detail_before = spark.sql("DESCRIBE DETAIL cluster_traditional").select(
    "numFiles", "sizeInBytes", "minReaderVersion", "minWriterVersion"
)
detail_before.show(truncate=False)

# Apply Z-Order optimization on frequently queried columns
spark.sql(f"OPTIMIZE cluster_traditional ZORDER BY (sensor_type, device_id)")

print("\nAfter Z-Order OPTIMIZE:")
detail_after = spark.sql("DESCRIBE DETAIL cluster_traditional").select(
    "numFiles", "sizeInBytes", "minReaderVersion", "minWriterVersion"
)
detail_after.show(truncate=False)


 #### Liquid Clustering Syntax (if available in your environment)

 The `CLUSTER BY` syntax creates a table with Liquid Clustering enabled. If your Community Edition supports it, this cell will execute. If not, it will fail gracefully.


In [ ]:
cluster_path_liquid = f"{base_dir}/cluster_liquid"
spark.sql(f"DROP TABLE IF EXISTS cluster_liquid")

try:
    spark.sql(f"""
        CREATE TABLE cluster_liquid
        CLUSTER BY (sensor_type, device_id)
        USING DELTA
        LOCATION '{cluster_path_liquid}'
    """)

    # Insert data
    sensor_df.write.format("delta").mode("append").save(cluster_path_liquid)

    print("Liquid Clustering table created successfully!")
    print("CLUSTER BY (sensor_type, device_id) enables incremental clustering.")

    spark.sql("DESCRIBE DETAIL cluster_liquid").select(
        "numFiles", "sizeInBytes", "clusteringColumns"
    ).show(truncate=False)

except Exception as e:
    print(f"Liquid Clustering not available in this environment: {str(e)[:200]}")
    print("\nLiquid Clustering requires:")
    print("  - delta.minReaderVersion >= 3")
    print("  - delta.minWriterVersion >= 7")
    print("  - Databricks Runtime 13.3 LTS or higher")
    print("\nFalling back to traditional Z-Ordering (already demonstrated above).")


 #### Comparing Query Performance (File Skipping)


In [ ]:
# Demonstrate file skipping with Z-Ordering
# Without optimization — scan all files
# With Z-Order on sensor_type — skip irrelevant files

print("Query: Filter by sensor_type='temperature' AND device_id='device-0050'")

# Measure on Z-Ordered table
import time

start = time.time()
result = spark.sql(f"""
    SELECT COUNT(*) FROM cluster_traditional
    WHERE sensor_type = 'temperature' AND device_id = 'device-0050'
""").collect()[0][0]
elapsed = time.time() - start
print(f"Z-Ordered table — {result} rows found in {elapsed:.3f}s")

# Let's also create an un-optimized table for comparison
unopt_path = f"{base_dir}/cluster_unoptimized"
sensor_df.write.format("delta").mode("overwrite").save(unopt_path)
spark.sql(f"CREATE TABLE IF NOT EXISTS cluster_unoptimized USING DELTA LOCATION '{unopt_path}'")

start = time.time()
result2 = spark.sql(f"""
    SELECT COUNT(*) FROM cluster_unoptimized
    WHERE sensor_type = 'temperature' AND device_id = 'device-0050'
""").collect()[0][0]
elapsed2 = time.time() - start
print(f"Unoptimized table — {result2} rows found in {elapsed2:.3f}s")

print("\nKEY INSIGHT: Z-Ordering (and Liquid Clustering) co-locates related data in the same files,")
print("enabling Delta to skip irrelevant files during reads — dramatically reducing I/O.")


 ### Key Takeaways
 - **Liquid Clustering** replaces traditional partitioning and Z-Ordering with incremental, maintenance-free clustering.
 - Use `CLUSTER BY (col1, col2)` on table creation (requires compatible Runtime version).
 - In environments without Liquid Clustering, **Z-Ordering via `OPTIMIZE ZORDER BY`** provides similar multi-dimensional clustering benefits.

 ### Self-Assessment Question

 *Q: When would you use Liquid Clustering over traditional `PARTITIONED BY`?*

 <details><summary>Click for answer</summary>
 **A:** Use Liquid Clustering when you have high-cardinality columns (e.g., device IDs, user IDs) that would create too many small partitions, or when query patterns filter by multiple different columns. Liquid Clustering handles multi-dimensional clustering automatically without manual partition tuning.
 </details>


 ---
 ## Concept #6: OPTIMIZE &amp; File Compaction [Medium]

 ### What Problem It Solves

 Streaming and frequent small-batch writes create many **small files**. This &quot;small file problem&quot; degrades read performance because the query engine must open, read metadata from, and close thousands of tiny files.

 `OPTIMIZE` consolidates small files into larger ones (~1 GB by default), dramatically improving read performance. Combined with `ZORDER BY`, it also clusters related data together for file skipping.

 ### Real-World Use Case

 A Kafka streaming pipeline writes 50MB micro-batches every minute into a Delta table. After 24 hours, there are 1,440 small files. Running `OPTIMIZE` hourly consolidates them into larger files, keeping query performance predictable.

 ### Hands-On Demo


In [ ]:
print("=" * 60)
print("CONCEPT 6: OPTIMIZE & File Compaction")
print("=" * 60)

# Simulate many small appends to create small files
opt_path = f"{base_dir}/optimize_demo"
spark.sql(f"DROP TABLE IF EXISTS optimize_sales")

# Create initial data
schema_opt = spark.createDataFrame([], "id INT, customer STRING, amount DOUBLE, region STRING")
schema_opt.write.format("delta").mode("overwrite").save(opt_path)
spark.sql(f"CREATE TABLE IF NOT EXISTS optimize_sales USING DELTA LOCATION '{opt_path}'")

print("Simulating 50 small-batch writes (streaming/micro-batch scenario)...")

for batch in range(50):
    batch_data = [
        (batch * 10 + i,
         f"Customer-{random.randint(1, 100)}",
         round(random.uniform(10, 500), 2),
         random.choice(["North", "South", "East", "West"]))
        for i in range(10)
    ]
    batch_df = spark.createDataFrame(batch_data, ["id", "customer", "amount", "region"])
    batch_df.write.format("delta").mode("append").save(opt_path)

total_rows = spark.sql("SELECT COUNT(*) FROM optimize_sales").collect()[0][0]
print(f"Total rows written: {total_rows}")


 #### File Count Before OPTIMIZE


In [ ]:
print("BEFORE OPTIMIZE — DESCRIBE DETAIL:")
detail_before_opt = spark.sql("DESCRIBE DETAIL optimize_sales").select(
    "numFiles", "sizeInBytes", "numRecords"
)
detail_before_opt.show(truncate=False)

num_files_before = detail_before_opt.collect()[0]["numFiles"]
print(f"\nSmall files BEFORE optimize: {num_files_before}")

# Also list the actual files
print("\nActual files in the table directory:")
all_files = dbutils.fs.ls(opt_path + "/")
parquet_files = [f.name for f in all_files if f.name.endswith(".parquet") or f.name.endswith(".snappy.parquet")]
print(f"Parquet files found: {len(parquet_files)}")
for f in parquet_files[:10]:
    print(f"  {f}")
if len(parquet_files) > 10:
    print(f"  ... and {len(parquet_files) - 10} more")


 #### Run OPTIMIZE


In [ ]:
# Set a smaller target file size for demonstration (default is 1 GB)
spark.conf.set("spark.databricks.delta.optimize.maxFileSize", 134217728)  # 128 MB
spark.conf.set("spark.databricks.delta.optimize.minFileSize", 33554432)   # 32 MB

print("Running OPTIMIZE to compact small files...\n")
optimize_result = spark.sql("OPTIMIZE optimize_sales")
optimize_result.show(truncate=False)


 #### File Count After OPTIMIZE


In [ ]:
print("AFTER OPTIMIZE — DESCRIBE DETAIL:")
detail_after_opt = spark.sql("DESCRIBE DETAIL optimize_sales").select(
    "numFiles", "sizeInBytes", "numRecords"
)
detail_after_opt.show(truncate=False)

num_files_after = detail_after_opt.collect()[0]["numFiles"]
files_reduced = num_files_before - num_files_after
print(f"\nSmall files AFTER optimize: {num_files_after}")
print(f"Files reduced by: {files_reduced} ({(files_reduced/num_files_before*100):.1f}% reduction)")

# List remaining files
all_files_after = dbutils.fs.ls(opt_path + "/")
parquet_files_after = [f.name for f in all_files_after if f.name.endswith(".parquet")]
print(f"\nRemaining parquet files: {len(parquet_files_after)}")
print("The small files have been consolidated into fewer, larger files.")


 #### OPTIMIZE with ZORDER BY


In [ ]:
# Z-Ordering clusters related data together in the same files
print("Running OPTIMIZE with ZORDER BY region...")
opt_zorder_result = spark.sql("OPTIMIZE optimize_sales ZORDER BY (region)")
opt_zorder_result.show(truncate=False)

detail_with_zorder = spark.sql("DESCRIBE DETAIL optimize_sales").select(
    "numFiles", "sizeInBytes", "numRecords"
)
detail_with_zorder.show(truncate=False)


 #### Read Performance Impact

 After OPTIMIZE, queries that filter on the Z-Ordered column benefit from file skipping — only relevant files are read, reducing scan overhead significantly.


In [ ]:
# Compare read performance: scan all files vs filter on Z-Ordered column
print("Comparing scan performance...\n")

# Full table scan
import time as time_mod
t0 = time_mod.time()
full_count = spark.sql("SELECT COUNT(*) FROM optimize_sales").collect()[0][0]
t1 = time_mod.time()
print(f"Full scan: {t1 - t0:.4f}s ({full_count} rows)")

# Filtered scan on Z-Ordered column
t0 = time_mod.time()
filtered_count = spark.sql("SELECT COUNT(*) FROM optimize_sales WHERE region = 'North'").collect()[0][0]
t1 = time_mod.time()
print(f"Filtered scan (region='North'): {t1 - t0:.4f}s ({filtered_count} rows)")

print("\nZ-Ordering enables Delta to skip files that don't contain 'North' region data.")


 ### Key Takeaways
 - `OPTIMIZE` consolidates small files into **larger files** (target ~1 GB), reducing metadata overhead and improving read performance.
 - `OPTIMIZE ... ZORDER BY (col)` additionally **co-locates related data** in the same files, enabling file skipping.
 - In production, run OPTIMIZE **periodically** (e.g., hourly or daily) to keep the file count manageable — it&apos;s idempotent and safe to run on live tables.

 ### Self-Assessment Question

 *Q: Your table has 10,000 small files after a day of streaming writes. Running `OPTIMIZE` consolidates them to 10 files. What happens to the old small files?*

 <details><summary>Click for answer</summary>
 **A:** The old small files are marked as &quot;removed&quot; in the transaction log. They are NOT deleted immediately — they remain until VACUUM runs after the retention period (default 7 days). This is how Delta supports time travel even after OPTIMIZE.
 </details>


 ---
 ## Concept #7: VACUUM &amp; Storage Lifecycle [Medium]

 ### What Problem It Solves

 Over time, Delta tables accumulate old data files that are no longer referenced by the latest table version (from OPTIMIZE, DELETE, UPDATE operations). These orphaned files waste storage and cost money.

 `VACUUM` removes unreferenced files older than the retention period (default 7 days). This is the Delta equivalent of garbage collection — essential for managing storage costs.

 ### Real-World Use Case

 A data engineering team runs weekly `OPTIMIZE` on a 10 TB sales table. Without VACUUM, the unreferenced files from each week&apos;s OPTIMIZE would accumulate, doubling or tripling storage costs. With `VACUUM`, only the current and recent versions&apos; files are retained.

 ### Hands-On Demo


In [ ]:
print("=" * 60)
print("CONCEPT 7: VACUUM & Storage Lifecycle")
print("=" * 60)

# Create a table and perform operations that create orphaned files
vac_path = f"{base_dir}/vacuum_demo"
spark.sql(f"DROP TABLE IF EXISTS vacuum_sales")

vac_data = [(i, f"Product-{i % 10}", random.randint(1, 100), round(random.uniform(10, 500), 2))
            for i in range(1000)]
vac_df = spark.createDataFrame(vac_data, ["id", "product", "qty", "amount"])
vac_df.write.format("delta").mode("overwrite").save(vac_path)

spark.sql(f"CREATE TABLE IF NOT EXISTS vacuum_sales USING DELTA LOCATION '{vac_path}'")
print(f"Initial table: {spark.sql('SELECT COUNT(*) FROM vacuum_sales').collect()[0][0]} rows")

# Perform operations that create old file versions
spark.sql("UPDATE vacuum_sales SET amount = amount * 1.1 WHERE id > 500")
spark.sql("DELETE FROM vacuum_sales WHERE id < 200")
spark.sql("OPTIMIZE vacuum_sales")

print("\nPerformed UPDATE, DELETE, and OPTIMIZE — these create unreferenced files.")


 #### `DESCRIBE DETAIL` — See Storage Statistics


In [ ]:
# DESCRIBE DETAIL shows file-level stats
print("DESCRIBE DETAIL before VACUUM:")
detail_vac = spark.sql("DESCRIBE DETAIL vacuum_sales").select(
    "numFiles", "sizeInBytes", "numRecords", "lastModified"
)
detail_vac.show(truncate=False)

# List actual files
print("\nFiles in table directory:")
vac_files = dbutils.fs.ls(vac_path + "/")
data_files = [f for f in vac_files if f.name.endswith(".parquet")]
for f in data_files[:10]:
    print(f"  {f.name:<50} {f.size:>10} bytes")
if len(data_files) > 10:
    print(f"  ... and {len(data_files) - 10} more files")
print(f"Total data files in directory: {len(data_files)}")


 #### Check Retention Period and Version History


In [ ]:
# Show current retention settings
print("Retention-related properties:")
retention_props = spark.sql("SHOW TBLPROPERTIES vacuum_sales").filter(
    "key like '%retention%' OR key like '%logRetention%'"
)
retention_props.show(truncate=False)

print("\nCurrent history (number of versions):")
history_vac = spark.sql("DESCRIBE HISTORY vacuum_sales")
print(f"Total versions: {history_vac.count()}")
history_vac.select("version", "operation", "timestamp").show(truncate=False)


 #### Run VACUUM (with Dry Run)

 `VACUUM` with `DRY RUN` shows which files would be removed without actually deleting them. Always run a dry run first in production!


In [ ]:
# Dry run first — safe preview
print("VACUUM DRY RUN (preview mode — no files deleted):")
try:
    spark.sql("VACUUM vacuum_sales DRY RUN").show(truncate=False)
    print("\nDry run completed. No files were actually deleted.")
except Exception as e:
    print(f"DRY RUN not supported in this version: {str(e)[:100]}")


 #### Run VACUUM (Retain 0 Hours for Demo)

 **WARNING:** Setting retention to 0 hours disables time travel to previous versions. This is for demonstration only. In production, use the default 7-day retention.


In [ ]:
# Count files before VACUUM
files_before = len(dbutils.fs.ls(vac_path + "/"))
print(f"Total files/directories before VACUUM: {files_before}")

# Run VACUUM with RETAIN 0 HOURS for demo (in production, keep the default 168 hours / 7 days)
print("\nRunning VACUUM with RETAIN 0 HOURS (for demo only)...")
spark.sql("VACUUM vacuum_sales RETAIN 0 HOURS")

# Count files after VACUUM
files_after = len(dbutils.fs.ls(vac_path + "/"))
print(f"Total files/directories after VACUUM: {files_after}")
print(f"Files removed: {files_before - files_after}")


 #### Time Travel AFTER VACUUM — What Happens?


In [ ]:
# Try to time travel to an older version AFTER VACUUM
print("Attempting time travel after VACUUM (0 hour retention)...")

# Get old version numbers
old_versions = spark.sql("DESCRIBE HISTORY vacuum_sales").select("version").collect()
if len(old_versions) > 1:
    try:
        old_version = old_versions[1][0]  # Version 1
        spark.sql(f"SELECT COUNT(*) FROM vacuum_sales VERSION AS OF {old_version}").show()
        print(f"Time travel to version {old_version} succeeded (files still present).")
    except Exception as e:
        print(f"Time travel to version {old_version} FAILED: {str(e)[:200]}")
        print("The data files for that version have been physically removed by VACUUM.")
        print("This is why you should keep a reasonable retention period in production!")

# Current table is still intact
print(f"\nCurrent table data is intact: {spark.sql('SELECT COUNT(*) FROM vacuum_sales').collect()[0][0]} rows")


 ### Key Takeaways
 - `VACUUM` removes old data files that are no longer referenced within the **retention period** (default 7 days).
 - Always run `DRY RUN` first to preview which files will be deleted.
 - After VACUUM, **time travel** to versions older than the retention period will fail. Plan your retention window based on audit/compliance needs.

 ### Self-Assessment Question

 *Q: You set `delta.deletedFileRetentionDuration = 1 hour` and run VACUUM. Two hours later, can you query the table as it was 30 minutes ago?*

 <details><summary>Click for answer</summary>
 **A:** No. VACUUM would have removed files older than 1 hour. After 2 hours, the 30-minutes-ago version is within the retention window, but the version from 2.5 hours ago would be gone. Always set retention to cover your maximum time-travel needs plus a safety margin.
 </details>


 ---
 ## Concept #8: Change Data Feed (CDF) [Medium]

 ### What Problem It Solves

 How do you efficiently capture row-level changes (inserts, updates, deletes) from a Delta table for downstream incremental processing? Without CDF, you&apos;d have to do expensive full-table comparisons or rely on unreliable &quot;last modified&quot; columns.

 **Change Data Feed (CDF)** records every row-level change made to a Delta table, including the type of change (`insert`, `update_preimage`, `update_postimage`, `delete`) and the version at which it occurred. This unlocks efficient CDC (Change Data Capture) pipelines.

 ### Real-World Use Case

 A data warehouse team needs to sync a Delta table to a downstream analytics database (e.g., PostgreSQL) in near-real-time. Instead of re-syncing the entire table, they read only the changes since the last sync via CDF.

 ### Hands-On Demo


In [ ]:
print("=" * 60)
print("CONCEPT 8: Change Data Feed (CDF)")
print("=" * 60)

# Create a table with CDF enabled
cdf_path = f"{base_dir}/cdf_demo"
spark.sql(f"DROP TABLE IF EXISTS cdf_products")

# Enable CDF on the table
cdf_init_data = [
    (1, "Widget A", "Gadgets", 9.99, 100),
    (2, "Widget B", "Gadgets", 14.99, 50),
    (3, "Bolt M6", "Hardware", 0.99, 1000),
    (4, "Nut M6", "Hardware", 0.49, 1000),
    (5, "Spring XL", "Hardware", 2.99, 200),
]
cdf_df = spark.createDataFrame(cdf_init_data, ["product_id", "name", "category", "price", "stock"])

cdf_df.write.format("delta").mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .save(cdf_path)

spark.sql(f"CREATE TABLE IF NOT EXISTS cdf_products USING DELTA LOCATION '{cdf_path}'")

print("CDF table created with change data feed ENABLED.")
print(f"Initial data: {spark.sql('SELECT COUNT(*) FROM cdf_products').collect()[0][0]} rows")
spark.sql("SELECT * FROM cdf_products").show()


 #### Perform Multiple Changes to Generate CDF Entries


In [ ]:
# Version 1: INSERT new products
spark.sql("""
    INSERT INTO cdf_products VALUES
        (6, 'Widget C', 'Gadgets', 19.99, 75),
        (7, 'Washer M6', 'Hardware', 0.25, 2000)
""")

# Version 2: UPDATE prices (increase by 10%)
spark.sql("UPDATE cdf_products SET price = price * 1.10 WHERE category = 'Gadgets'")

# Version 3: UPDATE stock levels
spark.sql("UPDATE cdf_products SET stock = stock + 50 WHERE product_id IN (1, 2)")

# Version 4: DELETE a product
spark.sql("DELETE FROM cdf_products WHERE product_id = 3")

# Show current table state
print("Current table state (after all changes):")
spark.sql("SELECT * FROM cdf_products ORDER BY product_id").show()

# Show history
print("\nTable history:")
spark.sql("DESCRIBE HISTORY cdf_products").select("version", "operation", "operationMetrics", "timestamp").show(truncate=False)


 #### Read Change Data Feed with `table_changes()`


In [ ]:
print("\n=== Reading Change Data Feed ===")

# Read all changes from version 0 to the latest
print("All CDF changes (version 0 to latest):")
cdf_all = spark.sql("SELECT * FROM table_changes('cdf_products', 0)")
cdf_all.select("_change_type", "_commit_version", "_commit_timestamp", "product_id", "name", "price", "stock").show(truncate=False)

# Read changes from a specific version range
print("\nCDF changes from version 1 to version 3:")
cdf_range = spark.sql("SELECT * FROM table_changes('cdf_products', 1, 3)")
cdf_range.select("_change_type", "_commit_version", "product_id", "name", "price", "stock").show(truncate=False)


 #### Understanding `_change_type` Values


In [ ]:
# Show distribution of change types
print("Change type distribution:")
spark.sql("""
    SELECT _change_type, COUNT(*) AS change_count
    FROM table_changes('cdf_products', 0)
    GROUP BY _change_type
    ORDER BY _change_type
""").show()

# Demonstrate the preimage/postimage for UPDATEs
print("\nUPDATE change details (update_preimage + update_postimage):")
cdf_updates = spark.sql("""
    SELECT _change_type, _commit_version, product_id, name, price, stock
    FROM table_changes('cdf_products', 0)
    WHERE _change_type LIKE 'update_%'
    ORDER BY _commit_version, product_id, _change_type
""")
cdf_updates.show(truncate=False)


 #### Incremental Processing Pattern with CDF


In [ ]:
print("\n=== Incremental Processing with CDF ===")

# Simulate an incremental pipeline that reads only new changes since last processed version
processed_version = 2  # We last processed up to version 2

print(f"Last processed version: {processed_version}")
print(f"Reading changes from version {processed_version} to latest...")

new_changes = spark.sql(f"""
    SELECT * FROM table_changes('cdf_products', {processed_version})
""")

print(f"New changes to process: {new_changes.count()} rows")

# Typical incremental processing: apply changes to downstream system
# - INSERT records → insert into target
# - update_preimage + update_postimage → update target record
# - DELETE records → delete from target

# Collect by change type
for change_type in ["insert", "update_preimage", "update_postimage", "delete"]:
    cnt = new_changes.filter(f"_change_type = '{change_type}'").count()
    if cnt > 0:
        print(f"  {change_type}: {cnt} rows")

# Update our "processed" tracker
latest_version = spark.sql("SELECT MAX(version) AS max_ver FROM (DESCRIBE HISTORY cdf_products)").collect()[0][0]
print(f"\nUpdated processed version to: {latest_version}")
print("Next incremental run will read from version", latest_version)


 ### Key Takeaways
 - CDF records every row-level change with `_change_type`, `_commit_version`, and `_commit_timestamp`.
 - Use `table_changes('table_name', start_version, end_version)` to query changes efficiently for incremental processing.
 - UPDATE operations produce **both** `update_preimage` (old values) and `update_postimage` (new values) — essential for accurate CDC replication.

 ### Self-Assessment Question

 *Q: You enable CDF on an existing table. Can you query changes that happened BEFORE CDF was enabled?*

 <details><summary>Click for answer</summary>
 **A:** No. CDF only captures changes from the version where it was enabled and onwards. If you need change data for historical versions, you&apos;d need to use time travel to compare versions manually, which is possible but less efficient.
 </details>


 ---
 ## Concept #9: Deletion Vectors [Medium]

 ### What Problem It Solves

 In traditional Delta Lake, a `DELETE` operation physically rewrites all files containing matching rows — a costly, I/O-intensive operation. For a 1 TB table, deleting 10 rows could require rewriting hundreds of GB of data just to exclude those rows.

 **Deletion Vectors** (DVs) solve this by using **soft-delete markers** stored in separate, small binary files. When a DELETE runs, instead of rewriting data files, Delta records which rows to skip in a deletion vector file. This makes DELETE operations near-instant, regardless of table size.

 **NOTE FOR COMMUNITY EDITION:** Deletion Vectors require `delta.minReaderVersion >= 3` and `delta.minWriterVersion >= 7`. If these features aren&apos;t available in your Community Edition deployment, this section demonstrates the concept and shows the performance difference by simulating the before/after behavior.

 ### Real-World Use Case

 A GDPR compliance job needs to delete specific user records from a 5 TB events table. Without deletion vectors, each DELETE would take hours and cost hundreds of dollars in compute. With deletion vectors, the DELETE completes in seconds and the file rewrite cost is avoided — the soft-deleted rows are physically removed later during OPTIMIZE.

 ### Hands-On Demo


In [ ]:
print("=" * 60)
print("CONCEPT 9: Deletion Vectors")
print("=" * 60)

# First, demonstrate the COST of DELETE without deletion vectors (traditional behavior)
# by running a DELETE and observing how files are rewritten

dv_path_traditional = f"{base_dir}/dv_traditional"
spark.sql(f"DROP TABLE IF EXISTS dv_traditional")

# Create moderate-size data
dv_data = [(i, f"User-{i%100}", f"Event-{i%5}", random.randint(1, 10000))
           for i in range(5000)]
dv_df = spark.createDataFrame(dv_data, ["event_id", "user_id", "event_type", "value"])
dv_df.write.format("delta").mode("overwrite").save(dv_path_traditional)

spark.sql(f"CREATE TABLE IF NOT EXISTS dv_traditional USING DELTA LOCATION '{dv_path_traditional}'")

# Count files before DELETE
files_before_del = len([f for f in dbutils.fs.ls(dv_path_traditional + "/") if f.name.endswith(".parquet")])
print(f"Parquet files before DELETE: {files_before_del}")

# Run a small DELETE
spark.sql("DELETE FROM dv_traditional WHERE event_id < 100")
print("DELETE completed (traditional — files were rewritten).")

# Count files after DELETE
files_after_del = len([f for f in dbutils.fs.ls(dv_path_traditional + "/") if f.name.endswith(".parquet")])
print(f"Parquet files after DELETE: {files_after_del}")

# EXPLAIN: Without deletion vectors, files containing deleted rows are rewritten
print(f"\nINSIGHT: Without deletion vectors, {files_before_del} files were touched")
print("even though only ~100 rows were deleted from a 5000-row table.")
print("For large tables, this rewrite cost is enormous.")


 #### Deletion Vectors — Enable and Demonstrate (if available)


In [ ]:
# Try to enable deletion vectors
dv_path = f"{base_dir}/dv_demo"
spark.sql(f"DROP TABLE IF EXISTS dv_trades")

dv_init_data = [
    (1, "AAPL", "BUY", 100, 175.50),
    (2, "GOOGL", "SELL", 50, 140.25),
    (3, "MSFT", "BUY", 200, 380.00),
    (4, "AAPL", "SELL", 75, 178.00),
    (5, "AMZN", "BUY", 150, 185.30),
    (6, "GOOGL", "BUY", 100, 141.00),
    (7, "META", "SELL", 25, 505.75),
    (8, "MSFT", "SELL", 100, 382.50),
]
dv_init_df = spark.createDataFrame(dv_init_data, ["trade_id", "ticker", "action", "quantity", "price"])

try:
    # Enable deletion vectors
    dv_init_df.write.format("delta").mode("overwrite") \
        .option("delta.enableDeletionVectors", "true") \
        .save(dv_path)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS dv_trades USING DELTA LOCATION '{dv_path}'
    """)

    # Verify property
    props = spark.sql("SHOW TBLPROPERTIES dv_trades").filter("key = 'delta.enableDeletionVectors'")
    if props.count() > 0:
        has_dv = props.collect()[0]["value"] == "true"
        print(f"Deletion Vectors enabled: {has_dv}")
    else:
        print("Deletion Vectors property not found — may not be supported in this environment.")

except Exception as e:
    error_msg = str(e)
    print(f"Deletion Vectors not supported in this environment: {error_msg[:200]}")
    print("\nDeletion Vectors require:")
    print("  - delta.minReaderVersion >= 3")
    print("  - delta.minWriterVersion >= 7")
    print("  - delta.enableDeletionVectors = true table property")
    print("\nCreating fallback table for architectural explanation...")
    
    # Fallback: create without DVs
    dv_init_df.write.format("delta").mode("overwrite").save(dv_path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS dv_trades USING DELTA LOCATION '{dv_path}'")


 #### How Deletion Vectors Work — Architectural Overview

 **Without Deletion Vectors (Traditional):**
 1. DELETE identifies files containing matching rows
 2. Each matching file is **fully rewritten** minus the deleted rows
 3. Old files are removed from the log, new files are added
 4. Cost: O(file_size) for each file touched

 **With Deletion Vectors:**
 1. DELETE identifies specific row positions in each file
 2. A small **deletion vector** (binary file) is created, recording which rows to skip
 3. Data files are **NOT rewritten** — they remain in place
 4. Readers filter out soft-deleted rows at read time
 5. Cost: O(1) for the DELETE, O(deleted_rows) to filter during reads
 6. **Physical compaction** happens during OPTIMIZE (reclaims space)


 #### Demonstrate Soft-Delete vs Physical Delete Behavior


In [ ]:
print("Current state of trades table:")
spark.sql("SELECT * FROM dv_trades ORDER BY trade_id").show()

# Count files before the operation
files_before = len([f for f in dbutils.fs.ls(dv_path + "/") if f.name.endswith(".parquet")])

# Perform DELETE
deleted_count = spark.sql("SELECT COUNT(*) FROM dv_trades WHERE action = 'SELL'").collect()[0][0]
print(f"\nDeleting {deleted_count} trades where action = 'SELL'...")
spark.sql("DELETE FROM dv_trades WHERE action = 'SELL'")

# Count files after
files_after = len([f for f in dbutils.fs.ls(dv_path + "/") if f.name.endswith(".parquet")])

print(f"\nFiles before DELETE: {files_before}")
print(f"Files after DELETE: {files_after}")

# If DVs are working, files won't change much; if traditional, files are rewritten
if has_dv if 'has_dv' in dir() else False:
    print("\nWith Deletion Vectors: Files NOT rewritten (soft-delete markers used).")
    print("The rows are logically deleted but still physically present in files.")
    print("OPTIMIZE will later physically remove them and reclaim space.")
else:
    print("\nWithout Deletion Vectors: Files were rewritten to exclude deleted rows.")
    print("This is the traditional behavior — costly for large tables.")

# Verify data is gone from the logical view
print("\nAfter DELETE (logical view — deleted rows are gone):")
spark.sql("SELECT * FROM dv_trades ORDER BY trade_id").show()


 #### View Deletion Vector Stats (if available)


In [ ]:
try:
    detail_dv = spark.sql("DESCRIBE DETAIL dv_trades").select(
        "numDeletionVectors", "numFiles", "numRecords"
    )
    detail_dv.show(truncate=False)

    num_dvs = detail_dv.collect()[0]["numDeletionVectors"]
    if num_dvs is not None and num_dvs > 0:
        print(f"Table has {num_dvs} deletion vector(s) — soft-deletes in effect.")
        print("Run OPTIMIZE to physically remove soft-deleted rows.")
    else:
        print("No deletion vectors found — traditional physical deletes were used.")
except Exception as e:
    print(f"'numDeletionVectors' field not available in this Runtime version.")
    print("This metric is available in DBR 14.0+ with deletion vectors enabled.")


 ### Key Takeaways
 - **Deletion Vectors** enable near-instant DELETEs by using soft-delete markers instead of rewriting files.
 - Without DVs, a DELETE rewrites **every file** that contains matching rows — costly for large tables.
 - Soft-deleted rows are **filtered at read time** and physically removed during OPTIMIZE/VACUUM.

 ### Self-Assessment Question

 *Q: You enable deletion vectors and run `DELETE FROM transactions WHERE user_id = 'GDPR_Subject_123'`. The user queries the table and doesn&apos;t see their data. Is the data still on disk?*

 <details><summary>Click for answer</summary>
 **A:** Yes, the data is still physically on disk (soft-deleted) and may remain there until OPTIMIZE physically removes it. The deletion vector ensures the data is filtered out at read time, so the user cannot see it — but for true GDPR compliance, you should ensure the data is physically removed via OPTIMIZE + VACUUM.
 </details>


 ---
 ## Concept #10: Delta Table Properties &amp; Configuration [Hard]

 ### What Problem It Solves

 Delta Lake has dozens of configurable properties that control behaviour: auto-optimization, column mapping, protocol versions, retention, isolation levels, and more. Without understanding these properties, you can&apos;t tune Delta for production workloads — you might miss out on auto-compaction, column renaming, or proper isolation configuration.

 ### Real-World Use Case

 A team needs to rename a column in a 10 TB Delta table without rewriting any data. By enabling `delta.columnMapping.mode = 'name'`, they can rename columns via `ALTER TABLE ... RENAME COLUMN` instantly — a metadata-only operation.

 ### Hands-On Demo


In [ ]:
print("=" * 60)
print("CONCEPT 10: Delta Table Properties & Configuration")
print("=" * 60)

# Create a fully configured table to explore properties
props_path = f"{base_dir}/props_demo"
spark.sql(f"DROP TABLE IF EXISTS props_table")

props_data = [(1, "Alice", "HR", 75000), (2, "Bob", "Engineering", 95000)]
props_df = spark.createDataFrame(props_data, ["id", "name", "dept", "salary"])

props_df.write.format("delta").mode("overwrite") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .option("delta.autoOptimize.autoCompact", "true") \
    .save(props_path)

spark.sql(f"CREATE TABLE IF NOT EXISTS props_table USING DELTA LOCATION '{props_path}'")


 #### 1. `SHOW TBLPROPERTIES` — View All Properties


In [ ]:
print("All table properties (SHOW TBLPROPERTIES):")
spark.sql("SHOW TBLPROPERTIES props_table").show(100, truncate=False)


 #### 2. Key Production Properties Explained


In [ ]:
# === Property 1: autoOptimize.optimizeWrite ===
print("=" * 60)
print("Property: delta.autoOptimize.optimizeWrite")
print("=" * 60)
print("When true, Delta automatically optimizes the file sizes of writes.")
print("Each write produces fewer, optimally-sized files instead of many small ones.")
print("")
print("Use when: You have streaming or micro-batch workloads.")
print("Trade-off: Slight increase in write latency for better read performance.")

props_opt = spark.sql("SHOW TBLPROPERTIES props_table").filter("key = 'delta.autoOptimize.optimizeWrite'")
props_opt.show(truncate=False)

# === Property 2: autoOptimize.autoCompact ===
print("\nProperty: delta.autoOptimize.autoCompact")
print("After each write, Delta checks if small files exist and automatically compacts them.")
print("This is like running OPTIMIZE automatically after each write — no manual maintenance.")
print("Use when: You want hands-free file management.")
props_ac = spark.sql("SHOW TBLPROPERTIES props_table").filter("key = 'delta.autoOptimize.autoCompact'")
props_ac.show(truncate=False)


 #### 3. Protocol Versions: `minReaderVersion` &amp; `minWriterVersion`


In [ ]:
print("=" * 60)
print("Protocol Versions")
print("=" * 60)

# DESCRIBE EXTENDED shows protocol versions
print("DESCRIBE EXTENDED props_table (truncated to show protocol):")
extended = spark.sql("DESCRIBE EXTENDED props_table").filter(
    "col_name IN ('Provider', 'Type', 'Location', 'Table Properties')"
)
extended.show(truncate=False)

# More useful: DESCRIBE DETAIL
print("\nDESCRIBE DETAIL (includes minReaderVersion, minWriterVersion):")
detail_props = spark.sql("DESCRIBE DETAIL props_table").select(
    "format", "minReaderVersion", "minWriterVersion", "properties"
)
detail_props.show(truncate=False)

print("\nProtocol version summary:")
print("  minReaderVersion = 1: Basic Delta features (time travel, ACID)")
print("  minReaderVersion = 2: Column invariants, GENERATED columns")
print("  minReaderVersion = 3: CHECK constraints, column mapping, deletion vectors")
print("  minWriterVersion = 2: Basic writes")
print("  minWriterVersion = 3: CHECK constraints")
print("  minWriterVersion = 4: Change Data Feed, GENERATED columns")
print("  minWriterVersion = 5: Column invariants")
print("  minWriterVersion = 6: Identity columns")
print("  minWriterVersion = 7: Deletion vectors, column mapping (name mode), Liquid Clustering")


 #### 4. Column Mapping — Enabling Column Renames


In [ ]:
print("=" * 60)
print("Column Mapping: Renaming Columns Without Rewriting Data")
print("=" * 60)

colmap_path = f"{base_dir}/colmap_demo"
spark.sql(f"DROP TABLE IF EXISTS colmap_table")

# Create table WITH column mapping enabled (uses 'name' mode)
colmap_data = [(1, "Alice", "HR"), (2, "Bob", "Engineering")]
colmap_df = spark.createDataFrame(colmap_data, ["employee_id", "full_name", "department"])

colmap_df.write.format("delta").mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .save(colmap_path)

spark.sql(f"CREATE TABLE IF NOT EXISTS colmap_table USING DELTA LOCATION '{colmap_path}'")

print("Table created with column mapping mode = 'name'")

# Show original names
print("\nOriginal schema:")
spark.sql("DESCRIBE colmap_table").show()

# Rename a column (this is a metadata-only operation with column mapping!)
print("\nRenaming column 'full_name' to 'employee_name'...")
spark.sql("ALTER TABLE colmap_table RENAME COLUMN full_name TO employee_name")

print("Column renamed (no data rewritten — metadata-only operation):")
spark.sql("DESCRIBE colmap_table").show()

spark.sql("SELECT * FROM colmap_table").show()

print("\nColumn mapping modes:")
print("  'none' — Default. Column names are physical (in Parquet). Cannot rename without rewrite.")
print("  'name' — Column names are logical IDs in metadata. Renames are metadata-only. BEST for evolving schemas.")
print("  'id' — Also supports renaming. Columns are tracked by stable IDs.")
print("\nNOTE: Column mapping requires minReaderVersion >= 2 and minWriterVersion >= 5.")


 #### 5. Table Properties Catalogue


In [ ]:
print("=" * 60)
print("Essential Delta Table Properties Reference")
print("=" * 60)

properties_summary = [
    ("delta.appendOnly", "true/false", "Prevents DELETE/UPDATE — table is append-only"),
    ("delta.autoOptimize.optimizeWrite", "true/false", "Auto-optimize file sizes on write"),
    ("delta.autoOptimize.autoCompact", "true/false", "Auto-compact small files after writes"),
    ("delta.columnMapping.mode", "none/name/id", "Enable column rename support"),
    ("delta.dataSkippingNumIndexedCols", "int (default 32)", "Number of columns to collect stats for skipping"),
    ("delta.deletedFileRetentionDuration", "interval (default 7 days)", "How long VACUUM retains deleted files"),
    ("delta.enableChangeDataFeed", "true/false", "Enable Change Data Feed"),
    ("delta.enableDeletionVectors", "true/false", "Enable deletion vectors for fast DELETEs"),
    ("delta.logRetentionDuration", "interval (default 30 days)", "How long to keep transaction log entries"),
    ("delta.minReaderVersion", "int", "Minimum Delta protocol reader version required"),
    ("delta.minWriterVersion", "int", "Minimum Delta protocol writer version required"),
    ("delta.randomizeFilePrefixes", "true/false", "Randomize file prefixes to reduce hotspotting"),
    ("delta.targetFileSize", "size (default 1gb)", "Target file size for OPTIMIZE"),
    ("delta.isolationLevel", "Serializable/WriteSerializable", "Transaction isolation level"),
]

for name, default_val, description in properties_summary:
    print(f"  {name:<45} | Default: {default_val:<20}")
    print(f"  {'':45} | {description}")
    print()


 #### 6. `ALTER TABLE SET TBLPROPERTIES` — Modify Properties


In [ ]:
# Set and verify properties
print("Setting table properties...")
spark.sql("ALTER TABLE props_table SET TBLPROPERTIES ('delta.appendOnly' = 'true')")

print("\nVerifying properties:")
spark.sql("SHOW TBLPROPERTIES props_table").filter(
    "key IN ('delta.appendOnly', 'delta.autoOptimize.optimizeWrite', 'delta.autoOptimize.autoCompact')"
).show(truncate=False)

# Test appendOnly — should succeed for INSERT
print("\nTesting appendOnly = true: INSERT should work...")
spark.sql("INSERT INTO props_table VALUES (3, 'Charlie', 'Finance', 85000)")
print("INSERT succeeded (appends are allowed).")

# Test appendOnly — DELETE should FAIL
print("\nTesting appendOnly = true: DELETE should fail...")
try:
    spark.sql("DELETE FROM props_table WHERE id = 1")
    print("DELETE succeeded (unexpected — appendOnly should block this)")
except Exception as e:
    print(f"DELETE correctly blocked: {str(e)[:150]}")
    print("appendOnly = true prevents accidental data deletions.")

# Reset for further use
spark.sql("ALTER TABLE props_table SET TBLPROPERTIES ('delta.appendOnly' = 'false')")


 #### 7. `DESCRIBE DETAIL` — Complete Table Metadata


In [ ]:
print("DESCRIBE DETAIL — Complete table metadata at a glance:")
spark.sql("DESCRIBE DETAIL props_table").show(truncate=False)

# Highlight important fields
detail = spark.sql("DESCRIBE DETAIL props_table").collect()[0]
print("\nKey metrics extracted from DESCRIBE DETAIL:")
print(f"  Location:           {detail['location']}")
print(f"  Format:             {detail['format']}")
print(f"  Number of files:    {detail['numFiles']}")
print(f"  Size in bytes:      {detail['sizeInBytes']}")
print(f"  Number of records:  {detail['numRecords']}")
print(f"  Table properties:   {detail['properties']}")


 ### Key Takeaways
 - `SHOW TBLPROPERTIES` lists all configured properties; `ALTER TABLE SET TBLPROPERTIES` modifies them.
 - **Protocol versions** (`minReaderVersion`, `minWriterVersion`) gate access to features like column mapping, deletion vectors, and CDF.
 - `delta.columnMapping.mode = 'name'` enables column **renames without data rewrites** — critical for schema evolution in large tables.

 ### Self-Assessment Question

 *Q: You want to enable column renames and Change Data Feed on a new table. What properties must you set at creation time?*

 <details><summary>Click for answer</summary>
 **A:** Set `delta.columnMapping.mode = 'name'` (or `'id'`), `delta.minReaderVersion = 2`, `delta.minWriterVersion = 5` (minimum for column mapping), and `delta.enableChangeDataFeed = true`. These must be set BEFORE writing data — you can&apos;t enable column mapping on an existing table without rewriting it.
 </details>


 ---
 ---
 ## Notebook Summary

 ### What You&apos;ve Learned

 | # | Concept | Key Skill |
 |---|---------|-----------|
 | 1 | **ACID Transactions** | Atomic writes, optimistic concurrency, rollback on failure |
 | 2 | **Transaction Log** | Inspecting `_delta_log`, understanding checkpoints, `DESCRIBE HISTORY` |
 | 3 | **Time Travel** | `VERSION AS OF`, `TIMESTAMP AS OF`, `RESTORE TABLE` |
 | 4 | **Schema Enforcement** | Type validation, `mergeSchema`, `overwriteSchema`, `autoMerge` |
 | 5 | **Liquid Clustering** | `CLUSTER BY`, Z-Ordering, file skipping for queries |
 | 6 | **OPTIMIZE & Compaction** | Small-file consolidation, `ZORDER BY`, read performance |
 | 7 | **VACUUM & Lifecycle** | Storage cleanup, retention periods, DRY RUN, time travel impact |
 | 8 | **Change Data Feed** | `table_changes()`, `_change_type`, incremental processing |
 | 9 | **Deletion Vectors** | Soft-delete markers, physical vs logical deletes, GDPR implications |
 | 10 | **Table Properties** | Protocol versions, column mapping, auto-optimize, `DESCRIBE DETAIL` |

 ### Delta Lake Commands You Can Now Use

 ```
 -- Inspection
 DESCRIBE HISTORY table_name
 DESCRIBE DETAIL table_name
 DESCRIBE EXTENDED table_name
 SHOW TBLPROPERTIES table_name

 -- Time Travel
 SELECT * FROM table_name VERSION AS OF N
 SELECT * FROM table_name TIMESTAMP AS OF '2024-01-01'
 RESTORE TABLE table_name TO VERSION AS OF N

 -- Maintenance
 OPTIMIZE table_name
 OPTIMIZE table_name ZORDER BY (col1, col2)
 VACUUM table_name
 VACUUM table_name DRY RUN
 VACUUM table_name RETAIN 168 HOURS

 -- Change Data Feed
 SELECT * FROM table_changes('table_name', start_version)
 SELECT * FROM table_changes('table_name', start, end)

 -- Configuration
 ALTER TABLE table_name SET TBLPROPERTIES ('key' = 'value')
 ALTER TABLE table_name RENAME COLUMN old_name TO new_name
 ```

 ---


 ## Score Yourself: How Many Concepts Do You Understand?

 Check off each concept you feel confident with after completing this notebook.

 - [ ] **Concept 1: ACID Transactions** — I can explain atomic writes and optimistic concurrency
 - [ ] **Concept 2: Transaction Log** — I understand `_delta_log` structure and checkpointing
 - [ ] **Concept 3: Time Travel & RESTORE** — I can query past versions and roll back tables
 - [ ] **Concept 4: Schema Enforcement vs. Evolution** — I know when to use mergeSchema vs overwriteSchema
 - [ ] **Concept 5: Liquid Clustering** — I understand clustering vs partitioning and can use Z-Ordering
 - [ ] **Concept 6: OPTIMIZE & File Compaction** — I can consolidate small files and ZORDER tables
 - [ ] **Concept 7: VACUUM & Storage Lifecycle** — I know how to manage storage and retention
 - [ ] **Concept 8: Change Data Feed** — I can read row-level changes with table_changes()
 - [ ] **Concept 9: Deletion Vectors** — I understand soft-deletes and their performance benefits
 - [ ] **Concept 10: Table Properties** — I can configure protocol versions, column mapping, and auto-optimize

 **Score:** ___ / 10

 ### Next Steps

 After mastering these fundamentals, continue to notebook `02_Delta_Lake_Advanced.py` for:
 - MERGE operations and SCD Type 2
 - Streaming with Delta (Structured Streaming)
 - Generated columns and default values
 - Partitioning strategies and predicate pushdown
 - Data skipping and bloom filters
 - Multi-cluster concurrent writes
 - Delta Sharing basics

 ---
 *Notebook created for Databricks Community Edition | Delta Lake Fundamentals #1-#10*


In [ ]:
print("=" * 60)
print("NOTEBOOK COMPLETE")
print("=" * 60)
print("\nAll 10 Delta Lake concepts have been demonstrated.")
print("\nCleanup note: All tables were created under /tmp/delta_lake_fundamentals/")
print("These will persist until you manually remove them or the cluster restarts.")
print("\nTo clean up manually, run:")
print("  dbutils.fs.rm('/tmp/delta_lake_fundamentals', recurse=True)")


---

# 02 Spark Execution



 # Notebook 02: Spark Execution Engine

 **Concepts #11-#20** — How the Spark engine works under the hood

 | Concept | Topic | Difficulty |
 |---------|-------|------------|
 | 11 | Lazy Evaluation & Actions | Easy |
 | 12 | Photon Engine | Easy |
 | 13 | Catalyst Optimizer & Query Plans | Medium |
 | 14 | Adaptive Query Execution (AQE) | Medium |
 | 15 | Shuffle Operations | Medium |
 | 16 | Join Strategies | Medium |
 | 17 | Reading the Spark UI | Medium |
 | 18 | Partitioning & Parallelism | Medium |
 | 19 | Data Skew: Detection & Mitigation | Hard |
 | 20 | Memory Management & Spill | Hard |

 **Designed for:** Databricks Community Edition (single node, no Photon, no serverless)

 **Datasets:** Synthetic data generated in-notebook + `/databricks-datasets/` samples


 ## Setup: Generate Synthetic Datasets

 We create all data upfront so every concept has what it needs:
 - **sales_df** — 1M rows of sales transactions (fact table)
 - **products_df** — 1K unique products (dimension table, small)
 - **customers_df** — 500K customers (large dimension)
 - **skewed_sales_df** — deliberately skewed for skew demos
 - **orders_df** — 500K orders with many columns for pruning demos


In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand, randn, expr, broadcast, lit, sum, count, avg, max, min, when, floor, concat, monotonically_increasing_id
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType, LongType
import random

spark = SparkSession.builder.appName("SparkExecutionDeepDive").getOrCreate()
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")

print(f"Spark version     : {spark.version}")
print(f"Application ID    : {spark.sparkContext.applicationId}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")


 ### Generate Products (1K — small dimension table)


In [ ]:
categories = ["Electronics", "Clothing", "Home", "Sports", "Books", "Food", "Toys", "Automotive"]
product_ids = list(range(1, 1001))

products_data = []
for pid in product_ids:
    products_data.append({
        "product_id": pid,
        "product_name": f"Product_{pid:04d}",
        "category": random.choice(categories),
        "price": round(random.uniform(5.0, 500.0), 2),
        "cost": round(random.uniform(2.0, 200.0), 2)
    })

products_df = spark.createDataFrame(products_data)
products_df.cache()
products_df.count()
print(f"Products: {products_df.count()} rows")
products_df.show(5, truncate=False)


 ### Generate Sales (1M rows — large fact table)


In [ ]:
num_sales = 1_000_000
sales_schema = StructType([
    StructField("sale_id", LongType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("store_id", IntegerType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("sale_date", StringType(), False),
    StructField("region", StringType(), False),
    StructField("payment_type", StringType(), False)
])

regions = ["North", "South", "East", "West", "Central"]
payments = ["Credit", "Debit", "Cash", "Digital"]

def generate_sales_row(i):
    return (
        i,
        random.randint(1, 1000),
        random.randint(1, 500000),
        random.randint(1, 50),
        random.randint(1, 10),
        round(random.uniform(5.0, 500.0), 2),
        f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
        random.choice(regions),
        random.choice(payments)
    )

sales_rdd = spark.sparkContext.parallelize(range(num_sales), 8).map(lambda i: generate_sales_row(i))
sales_df = spark.createDataFrame(sales_rdd, schema=sales_schema)
sales_df = sales_df.withColumn("revenue", col("quantity") * col("unit_price"))
print(f"Sales: {sales_df.count()} rows")
sales_df.show(5, truncate=False)


 ### Generate Customers (500K — large dimension)


In [ ]:
num_customers = 500_000
customers_data = []

for cid in range(1, num_customers + 1, 5000):
    batch = []
    for c in range(cid, min(cid + 5000, num_customers + 1)):
        batch.append({
            "customer_id": c,
            "customer_name": f"Customer_{c:06d}",
            "customer_region": random.choice(regions),
            "loyalty_tier": random.choice(["Bronze", "Silver", "Gold", "Platinum"]),
            "signup_date": f"202{random.randint(0,4)}-{random.randint(1,12):02d}-01"
        })
    for row in batch:
        customers_data.append(row)
    if cid % 50000 == 0:
        print(f"  Generated {cid} customers...")

customers_df = spark.createDataFrame(customers_data)
print(f"Customers: {customers_df.count()} rows")
customers_df.show(5, truncate=False)


 ### Generate Skewed Sales (for skew demonstrations)


In [ ]:
skewed_data = []
num_skewed = 200_000

for i in range(num_skewed):
    if i < num_skewed * 0.8:
        pid = 1
    elif i < num_skewed * 0.9:
        pid = 2
    elif i < num_skewed * 0.95:
        pid = 3
    elif i < num_skewed * 0.98:
        pid = 4
    else:
        pid = random.randint(5, 1000)

    skewed_data.append({
        "sale_id": i,
        "product_id": pid,
        "customer_id": random.randint(1, 100000),
        "store_id": random.randint(1, 20),
        "quantity": random.randint(1, 5),
        "price": round(random.uniform(10.0, 200.0), 2)
    })

skewed_sales_df = spark.createDataFrame(skewed_data)
print(f"Skewed Sales: {skewed_sales_df.count()} rows")
skewed_sales_df.groupBy("product_id").count().orderBy(col("count").desc()).show(10)


 ### Generate Wide Orders (many columns for pruning demos)


In [ ]:
num_orders = 500_000
wide_columns = ["order_id", "customer_id", "product_id", "quantity", "price", "status"] + \
               [f"attr_{i}" for i in range(1, 31)]

broad_orders = []
for i in range(num_orders):
    row = {
        "order_id": i,
        "customer_id": random.randint(1, 100000),
        "product_id": random.randint(1, 1000),
        "quantity": random.randint(1, 5),
        "price": round(random.uniform(10.0, 500.0), 2),
        "status": random.choice(["placed", "shipped", "delivered", "cancelled"])
    }
    for a in range(1, 31):
        row[f"attr_{a}"] = round(random.random() * 100, 2)
    broad_orders.append(row)
    if i % 100000 == 0 and i > 0:
        print(f"  Generated {i} orders...")

orders_df = spark.createDataFrame(broad_orders)
orders_df.cache()
orders_df.count()
print(f"Orders: {orders_df.count()} rows, {len(orders_df.columns)} columns")


 ---
 ## Concept 11: Lazy Evaluation & Actions [Easy]

 **Key idea:** Transformations (`filter`, `select`, `groupBy`) build an execution plan but don't run until an **action** (`count`, `collect`, `show`, `write`) triggers execution.

 This is why Spark can optimize across your entire pipeline — it sees the full DAG before doing any work.


 ### Transformation Chain (No Execution Yet)

 Watch — no Spark jobs fire for any of these lines:


In [ ]:
t0 = time.time()

# These are ALL transformations — nothing executes yet
filtered = sales_df.filter(col("region") == "North")
grouped = filtered.groupBy("product_id").agg(sum("revenue").alias("total_revenue"))
sorted_result = grouped.orderBy(col("total_revenue").desc())
topped = sorted_result.limit(10)

t1 = time.time()
print(f"Transformations defined in: {(t1 - t0)*1000:.1f} ms")
print(f"No Spark jobs have run yet!")
print(f"Execution plan is built lazily — it's just a recipe, not executed.")


 ### Now Trigger an Action — Execution Happens


In [ ]:
print("Triggering action: .show()\n")

t0 = time.time()
topped.show()
t1 = time.time()

print(f"\nAction (.show) completed in: {t1 - t0:.2f} seconds")
print(f"All transformations + the action ran in ONE optimized pipeline.")


 ### Compare: Action vs No-Action Timing


In [ ]:
print("--- Timing: Transformations only (no action) ---")
t0 = time.time()
for i in range(5):
    step1 = sales_df.filter(col("region") == "East")
    step2 = step1.groupBy("store_id").agg(avg("revenue").alias("avg_rev"))
    step3 = step2.filter(col("avg_rev") > 50)
    step4 = step3.orderBy(col("avg_rev").desc())
t1 = time.time()
print(f"No-action pipeline defined 5x in: {(t1 - t0)*1000:.1f} ms")

print("\n--- Timing: Transformations + Action ---")
t0 = time.time()
for i in range(5):
    step1 = sales_df.filter(col("region") == "East")
    step2 = step1.groupBy("store_id").agg(avg("revenue").alias("avg_rev"))
    step3 = step2.filter(col("avg_rev") > 50)
    step4 = step3.orderBy(col("avg_rev").desc())
    step4.count()
t1 = time.time()
print(f"With-action pipeline executed 5x in: {t1 - t0:.2f} seconds")


 ### Common Transformations vs Actions

 | Transformations (lazy) | Actions (trigger execution) |
 |------------------------|-----------------------------|
 | `filter()`, `where()` | `count()`, `first()` |
 | `select()`, `drop()` | `collect()`, `take(n)` |
 | `groupBy()`, `agg()` | `show()`, `head()` |
 | `join()`, `union()` | `foreach()`, `foreachPartition()` |
 | `orderBy()`, `sort()` | `write`, `saveAsTable` |
 | `withColumn()` | `toPandas()` (be careful!) |
 | `distinct()`, `dropDuplicates()` | `reduce()`, `fold()` |


 ### Execution Plan Built Lazily

 The plan is built step-by-step during transformations. `explain()` lets you peek at it without executing:


In [ ]:
# Build a moderately complex query lazily
q = (sales_df
     .filter(col("quantity") > 3)
     .join(products_df, "product_id")
     .filter(col("category") == "Electronics")
     .groupBy("region", "store_id")
     .agg(sum("revenue").alias("total_rev"), count("*").alias("num_sales"))
     .filter(col("total_rev") > 1000)
     .orderBy(col("total_rev").desc()))

# No execution yet — just show the plan
print("=== Execution Plan (no action triggered) ===")
q.explain()


 **Key takeaway:** Lazy evaluation is Spark's superpower. It lets the Catalyst optimizer rearrange, combine, and optimize your pipeline before a single byte is read. You pay the cost only when you ask for results.


 ---
 ## Concept 12: Photon Engine [Easy]

 **Photon** is Databricks' native vectorized query engine written in C++. It replaces parts of Spark's JVM-based execution with a high-performance columnar engine.

 **Community Edition limitation:** Photon is NOT available. We explain the architecture and concepts here.

 ### What Photon Does
 - **Vectorized execution:** Processes data in batches (columnar batches) rather than row-by-row
 - **Native C++:** Bypasses JVM overhead for CPU-bound operations
 - **SIMD instructions:** Uses modern CPU vector instructions for parallel processing within a core
 - **Memory-efficient:** Better cache locality, fewer allocations

 ### Which Ops Benefit Most?
 | High Photon Impact | Moderate Impact | Low Impact |
 |--------------------|-----------------|------------|
 | Filter, Project (select) | Aggregations | I/O-bound reads |
 | Joins (hash joins) | Windows | Small data ops |
 | Sorting | Grouped aggregations | Metadata ops |
 | Expressions / UDFs | Union, Intersect | File listing |


 ### Checking Photon Status

 In a full Databricks platform, you can check if Photon is active:


In [ ]:
# Check if Photon is available — will show "not enabled" on Community Edition
try:
    photon_status = spark.conf.get("spark.databricks.photon.enabled")
    print(f"Photon enabled: {photon_status}")
except Exception:
    print("Photon: NOT AVAILABLE (Community Edition / Standard Runtime)")
    print("This concept is explained for awareness — it works on full Databricks platform.")

# Check Databricks Runtime info
try:
    dbr_version = spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion")
    print(f"Databricks Runtime: {dbr_version}")
except Exception:
    print(f"Databricks Runtime info not available via config")


 ### How Photon Changes Execution

 Without Photon, Spark runs this pipeline:
 ```
 Scan → Filter (JVM) → Project (JVM) → Aggregate (JVM) → Sort (JVM) → Output
 ```

 With Photon, the same pipeline becomes:
 ```
 Scan → [Photon: Filter → Project → Aggregate → Sort] → Output
 ```

 Photon replaces the entire shaded region with a single C++ pipeline operating on columnar batches.

 ### Performance Comparison (Conceptual)

 Typical improvements from Photon (from Databricks benchmarks):
 - **SQL queries:** 2-8x faster
 - **ETL pipelines:** 1.5-3x faster
 - **Joins & aggregations:** 3-5x faster
 - **Scan-heavy workloads:** 1.5-2x faster

 > In Community Edition, we run without Photon. The concepts of query optimization still apply — Photon just makes them even faster.


 ### Simulated Comparison: Row-at-a-Time vs Batch

 Python simulation to illustrate why batch/vectorized processing is faster:


In [ ]:
import random as rnd
import math

# Simulate processing 1M rows column-by-column vs row-by-row
n_rows = 1_000_000
col_a = [rnd.random() for _ in range(n_rows)]
col_b = [rnd.random() for _ in range(n_rows)]

# Row-at-a-time (simulates JVM iterator pattern)
t0 = time.time()
result_row = []
for i in range(n_rows):
    if col_a[i] > 0.5:
        result_row.append(col_a[i] * col_b[i] + col_b[i])
t1 = time.time()
print(f"Row-at-a-time (Python loop)        : {t1 - t0:.4f}s")

# Batch / vectorized using list comprehensions (simulates columnar batch)
t0 = time.time()
mask = [a > 0.5 for a in col_a]
result_batch = [a * b + b for a, b, m in zip(col_a, col_b, mask) if m]
t1 = time.time()
print(f"Batch-style (comprehensions)       : {t1 - t0:.4f}s")
print(f"\nThis is a simplified illustration. Photon uses C++ with SIMD — much faster in practice.")


 **Key takeaway:** Photon is Databricks' C++ vectorized engine. While not available in Community Edition, understanding its role helps you appreciate query optimization and why certain patterns (filter early, use columnar operations) matter regardless of engine.


 ---
 ## Concept 13: Catalyst Optimizer & Query Plans [Medium]

 **Catalyst** is Spark's query optimizer. It takes your DataFrame/SQL code and produces an optimized execution plan through multiple phases:
 1. **Parsed Logical Plan** — raw AST from your code
 2. **Analyzed Logical Plan** — resolved table/column references
 3. **Optimized Logical Plan** — rule-based optimizations applied
 4. **Physical Plan** — how it will actually execute (with costs)

 `df.explain(True)` shows all four phases. `df.explain()` shows just the physical plan.


 ### Full Plan Walkthrough with `explain(True)`


In [ ]:
# Build a query that benefits from optimization
query = (sales_df
         .filter(col("region") == "West")
         .filter(col("quantity") > 2)
         .join(products_df, "product_id")
         .filter(col("category") == "Electronics")
         .select("sale_id", "product_name", "price", "revenue", "region")
         .where(col("revenue") > 100)
         .orderBy(col("revenue").desc())
         .limit(20))

print("=" * 70)
print("FULL CATALYST PLAN WALKTHROUGH — explain(True)")
print("=" * 70)
query.explain(True)


 ### Understanding the Four Plan Phases

 **1. Parsed Logical Plan** — Raw tree from your code:
 ```
 Limit
  └── Sort (revenue DESC)
       └── Filter (revenue > 100)
            └── Project [sale_id, product_name, ...]
                 └── Join (product_id)
                      ├── Filter (category = Electronics)
                      │    └── Relation[products]
                      └── Filter (quantity > 2)
                           └── Filter (region = West)
                                └── Relation[sales]
 ```

 **2. Analyzed Logical Plan** — Resolves table/column names using catalog

 **3. Optimized Logical Plan** — Catalyst applies rule-based optimizations

 **4. Physical Plan** — Cost-based choice of join strategy, scan type, etc.


 ### Predicate Pushdown in Action

 Catalyst pushes filters as close to the data source as possible — filtering early means less data to shuffle/sort.


In [ ]:
# Build a query with filters on both sides of a join
pred_push_query = (sales_df
                   .filter(col("region") == "North")   # pushdown: filter sales early
                   .join(products_df.filter(col("category") == "Sports"), "product_id")  # pushdown both sides
                   .filter(col("revenue") > 50)          # pushdown: filter after join
                   .select("sale_id", "product_name", "revenue", "category")
                   .orderBy(col("revenue").desc()))

print("=== Optimized Plan (shows predicate pushdown) ===")
pred_push_query.explain("extended")


 ### Column Pruning

 Catalyst only reads the columns you actually need. Wide tables benefit massively:


In [ ]:
# The orders_df has 36 columns — we only need 3
# Catalyst will prune the other 33 columns at scan time
pruned_query = (orders_df
                .filter(col("status") == "delivered")
                .select("order_id", "customer_id", "price")
                .groupBy("customer_id")
                .agg(sum("price").alias("total_spent")))

print("=== Optimized Plan (notice only 3 of 36 columns scanned) ===")
pruned_query.explain("extended")


 ### SQL Equivalent — Same Optimizations

 DataFrame API and Spark SQL go through the same Catalyst optimizer:


In [ ]:
sales_df.createOrReplaceTempView("sales")
products_df.createOrReplaceTempView("products")
orders_df.createOrReplaceTempView("orders")

sql_query = spark.sql("""
    SELECT p.product_name, p.category, SUM(s.revenue) AS total_rev
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    WHERE s.region = 'South'
      AND s.quantity > 1
      AND p.category IN ('Electronics', 'Sports')
      AND s.revenue > 50
    GROUP BY p.product_name, p.category
    ORDER BY total_rev DESC
    LIMIT 10
""")

print("=== SQL Query Optimized Plan ===")
sql_query.explain("extended")


 ### Benchmark: Optimization in Action

 Timing a query with and without early filtering:


In [ ]:
# Version A: Filter late (bad pattern — but Catalyst may fix it!)
print("--- Version A: Filter applied late in code ---")
t0 = time.time()
result_a = (sales_df
            .join(products_df, "product_id")
            .filter(col("region") == "East")
            .filter(col("category") == "Books")
            .groupBy("product_name")
            .agg(sum("revenue").alias("total"))
            .orderBy(col("total").desc()))
result_a.limit(5).show()
t1 = time.time()
print(f"Time: {t1 - t0:.2f}s")

# Version B: Filter early (good practice — works with Catalyst)
print("\n--- Version B: Filter applied early in code ---")
t0 = time.time()
result_b = (sales_df
            .filter(col("region") == "East")
            .join(products_df.filter(col("category") == "Books"), "product_id")
            .groupBy("product_name")
            .agg(sum("revenue").alias("total"))
            .orderBy(col("total").desc()))
result_b.limit(5).show()
t1 = time.time()
print(f"Time: {t1 - t0:.2f}s")

print("\nCatalyst optimizes both to similar plans, but explicit early filtering is good practice.")


 **Key takeaway:** Catalyst optimizes your query in 4 phases. Predicate pushdown and column pruning are two of its most impactful optimizations. Always filter early, select only needed columns, and review plans with `explain()`.


 ---
 ## Concept 14: Adaptive Query Execution (AQE) [Medium]

 **AQE** re-optimizes the query plan at runtime based on actual data statistics — not just estimates. Key features:
 1. **Dynamically coalesce shuffle partitions** — reduce partitions when data is small
 2. **Dynamically switch join strategies** — switch to broadcast if data is small enough
 3. **Dynamically optimize skew joins** — split skewed partitions
 4. **Dynamically detect empty partitions** — skip empty partitions


 ### AQE Configuration


In [ ]:
print("=== AQE Configuration ===")
print(f"spark.sql.adaptive.enabled                    = {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"spark.sql.adaptive.coalescePartitions.enabled  = {spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled')}")
try:
    print(f"spark.sql.adaptive.advisoryPartitionSizeInBytes = {spark.conf.get('spark.sql.adaptive.advisoryPartitionSizeInBytes')}")
except Exception:
    print("spark.sql.adaptive.advisoryPartitionSizeInBytes = (default 64MB)")

# Toggle AQE for comparison
print("\nYou can control AQE at the session level:")
print('  spark.conf.set("spark.sql.adaptive.enabled", "true")')


 ### Demonstrating Coalescing Shuffle Partitions

 AQE can reduce the 200 shuffle partitions to far fewer when data volume is small:


In [ ]:
# Run same query with and without AQE coalescing
from pyspark.sql import functions as F

# Use a small subset to show coalescing
small_data = sales_df.filter(col("sale_id") < 50000)

print("=== With AQE Coalescing Enabled ===")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")

t0 = time.time()
result_with_aqe = (small_data
                   .groupBy("region", "store_id")
                   .agg(sum("revenue").alias("total"), count("*").alias("cnt"))
                   .orderBy(col("total").desc()))
result_with_aqe.explain("extended")
row_count = result_with_aqe.count()
t1 = time.time()
print(f"Result rows: {row_count}, Time: {t1 - t0:.2f}s")
print(f"Notice in the plan: AQE may coalesce partitions at shuffle stage.")

print("\n=== Without AQE Coalescing ===")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
spark.conf.set("spark.sql.adaptive.localShuffleReader.enabled", "false")

t0 = time.time()
result_without_aqe = (small_data
                      .groupBy("region", "store_id")
                      .agg(sum("revenue").alias("total"), count("*").alias("cnt"))
                      .orderBy(col("total").desc()))
result_without_aqe.explain("extended")
row_count = result_without_aqe.count()
t1 = time.time()
print(f"Result rows: {row_count}, Time: {t1 - t0:.2f}s")

# Restore AQE
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.localShuffleReader.enabled", "true")


 ### Runtime Plan Changes with AQE

 AQE can change the plan mid-execution. For instance, a sort-merge join might become a broadcast join if the actual data is small enough:


In [ ]:
small_products = products_df.filter(col("product_id") < 30)

print("=== Without AQE — join strategy is fixed at planning ===")
spark.conf.set("spark.sql.adaptive.enabled", "false")

# Force hint to see the difference
(spark.range(10000).toDF("id")
 .join(small_products, spark.range(10000).toDF("id").col("id") == small_products.col("product_id"))
 .explain())

spark.conf.set("spark.sql.adaptive.enabled", "true")

print("\n=== With AQE — may switch join strategy at runtime ===")
(spark.range(10000).toDF("id")
 .join(small_products, spark.range(10000).toDF("id").col("id") == small_products.col("product_id"))
 .explain())


 ### AQE and Empty Partitions

 AQE can detect and skip empty partitions, avoiding wasted task scheduling:


In [ ]:
# Create a heavily filtered dataset — many partitions will be empty after filtering
filtered_sales = sales_df.filter(col("region") == "Central").filter(col("quantity") == 10)

t0 = time.time()
filtered_sales.count()
t1 = time.time()
print(f"Filtered to {filtered_sales.count()} rows from 1M in {t1 - t0:.2f}s")

# Show partition count before and after
print(f"\nOriginal partitions      : {sales_df.rdd.getNumPartitions()}")
print(f"Filtered partitions (RDD): {filtered_sales.rdd.getNumPartitions()}")
print("AQE can detect empty partitions and optimize the plan at runtime.")


 **Key takeaway:** AQE is a game-changer — it adapts the plan at runtime based on actual data stats. Key features are coalescing shuffle partitions, switching join strategies, handling skew, and skipping empty partitions. It's enabled by default in Databricks Runtime.


 ---
 ## Concept 15: Shuffle Operations [Medium]

 **Shuffle** is Spark's mechanism for redistributing data across partitions. Operations that require data from multiple partitions trigger a shuffle:
 - `groupBy()`, `agg()` — data must be co-located by key
 - `join()` — matching keys must be in the same partition
 - `distinct()` — duplicates must be co-located
 - `repartition()` — explicit redistribution
 - `orderBy()` — total ordering requires shuffle
 - Window functions (with `PARTITION BY`) — rows in same partition need co-location


 ### Operations That Trigger Shuffle


In [ ]:
from pyspark.sql import functions as F

print("=== Operations That Trigger Shuffle ===\n")

# 1. groupBy — shuffle by key
print("1. GroupBy — triggers shuffle")
(sales_df.groupBy("region").agg(count("*").alias("cnt")).explain())
print()

# 2. Join — shuffle by join key
print("2. Join — triggers shuffle (unless broadcast)")
(sales_df.join(products_df, "product_id", "inner").select("sale_id", "product_name").explain())
print()

# 3. distinct — shuffle to deduplicate
print("3. Distinct — triggers shuffle")
sales_df.select("product_id").distinct().explain()
print()

# 4. window function with partitionBy
print("4. Window with PARTITION BY — triggers shuffle")
from pyspark.sql.window import Window
w = Window.partitionBy("region").orderBy(col("revenue").desc())
sales_df.select("sale_id", "region", "revenue", F.row_number().over(w).alias("rn")).explain()
print()

# 5. Operations that DON'T shuffle
print("5. Operations WITHOUT shuffle:")
print("   - filter(), select(), withColumn()")
print("   - map(), flatMap() within partition")
print("   - coalesce() (reduces partitions without shuffle)")
print("   - foreachPartition()")


 ### Shuffle Read/Write with Different Partition Counts


In [ ]:
import time

def time_groupby_with_partitions(n_partitions, label):
    spark.conf.set("spark.sql.shuffle.partitions", str(n_partitions))
    t0 = time.time()
    result = sales_df.groupBy("region").agg(
        sum("revenue").alias("total_rev"),
        count("*").alias("num_sales")
    ).collect()
    t1 = time.time()
    print(f"  {label}: spark.sql.shuffle.partitions={n_partitions}, time={t1 - t0:.2f}s, result_rows={len(result)}")

print("=== Shuffle Partition Count vs Performance ===\n")

# Too few partitions — under-utilization
time_groupby_with_partitions(2,  "Too few ")

# Good for small result
time_groupby_with_partitions(10, "Low     ")

# Default
time_groupby_with_partitions(200, "Default ")

# Too many for small data — overhead
time_groupby_with_partitions(400, "Too many")

# Restore
spark.conf.set("spark.sql.shuffle.partitions", "200")


 ### Shuffle Cost: Many Small Partitions vs Few Optimal Partitions

 Each shuffle partition generates a task. Too many small partitions = task scheduling overhead. Too few = under-utilization.


In [ ]:
print("=== Shuffle Cost Analysis ===\n")
print("Task overhead per shuffle partition:")
print("  - Task scheduling   : ~5-50ms per task")
print("  - Task deserialization: ~1-10ms per task")
print("  - Network overhead  : per partition metadata")
print("")
print("Optimal shuffle partition size: 100-200 MB per partition")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Shuffle partitions  : {spark.conf.get('spark.sql.shuffle.partitions')}")

# Show the formula
data_size_mb = sales_df.count() * 100 / (1024 * 1024)  # rough est: ~100 bytes per row
optimal_partitions = max(1, data_size_mb // 128)
print(f"\nFor {sales_df.count():,} rows (~{data_size_mb:.0f} MB):")
print(f"  Optimal partitions: ~{optimal_partitions}")
print(f"  Current setting:    {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"  Guidance: sp.sql.shuffle.partitions = cluster_cores * 2 to 3")


 ### Monitoring Shuffle in Spark UI

 Navigate to the Spark UI (typically port 4040) to see:
 - **Stages tab** → Shuffle Read Size / Shuffle Write Size per stage
 - **SQL tab** → Each query's DAG with shuffle exchange nodes
 - **Executors tab** → Shuffle Read/Write Memory per executor

 Key shuffle metrics to watch:
 - **Shuffle Write** — data serialized and written to local disk for other executors to read
 - **Shuffle Read** — data fetched from remote executors
 - **Shuffle Spill (Memory)** — data that didn't fit in memory during shuffle
 - **Shuffle Spill (Disk)** — data spilled to disk during shuffle

 > On Community Edition (single node), shuffle is local but still involves disk I/O.


 ### Monitoring Shuffle from Within Code


In [ ]:
print("=== Monitoring Shuffle from Code ===\n")

# Get SparkContext status tracker info
sc = spark.sparkContext
print(f"Application ID: {sc.applicationId}")
print(f"Spark UI URL: {sc.uiWebUrl}")
print(f"Default parallelism: {sc.defaultParallelism}")

# Trigger a shuffle-heavy operation
print("\nRunning shuffle-heavy operation (groupBy + agg)...")
t0 = time.time()
heavy_result = (sales_df
                .groupBy("region", "store_id", "product_id")
                .agg(sum("revenue").alias("total_rev"),
                     count("*").alias("num_sales"),
                     avg("quantity").alias("avg_qty"))
                .filter(col("total_rev") > 500)
                .orderBy(col("total_rev").desc()))
heavy_count = heavy_result.count()
t1 = time.time()
print(f"Done. {heavy_count:,} rows in {t1 - t0:.2f}s")

# Show the stages that occurred
print(f"\nCheck Spark UI at: {sc.uiWebUrl}")
print("Look for: Jobs tab → find this job → click to see Stages")
print("Key metrics on each stage:")
print("  - Input Size / Records")
print("  - Shuffle Write")
print("  - Shuffle Read")


 **Key takeaway:** Shuffle is the most expensive operation in Spark. It involves serialization, disk I/O, and network transfer. Minimize shuffles by filtering early, choosing the right partition count, using broadcast joins for small tables, and leveraging AQE. Monitor shuffle via the Spark UI's SQL and Stages tabs.


 ---
 ## Concept 16: Join Strategies [Medium]

 Spark uses multiple join strategies. The optimizer (Catalyst + AQE) picks based on table sizes and hints:

 | Strategy | When Used | Data Movement |
 |----------|-----------|---------------|
 | **Broadcast Hash Join (BHJ)** | One side < `autoBroadcastJoinThreshold` (10MB default) | Small table sent to all executors — NO shuffle |
 | **Sort-Merge Join (SMJ)** | Both sides large | Both sides shuffled by join key, then sorted and merged |
 | **Shuffle Hash Join (SHJ)** | One side 3x smaller than other | Both sides shuffled, smaller side hashed |
 | **Broadcast Nested Loop Join** | No equi-join condition | Small side broadcast, nested loop |
 | **Cartesian Product** | Cross join | Very expensive — avoid |

 In practice, BHJ and SMJ cover 99% of cases.


 ### Strategy 1: Broadcast Hash Join (Best for Small + Large)

 The small table is sent to every executor. The large table never needs to shuffle.


In [ ]:
print("=== Broadcast Hash Join ===\n")

# products_df is small (1K rows) — good broadcast candidate
print(f"Products table: {products_df.count()} rows — SMALL (~few KB)")
print(f"Sales table   : {sales_df.count():,} rows — LARGE")

# BHJ with hint
print("\n--- Broadcast Join with hint ---")
broadcast_join = sales_df.join(broadcast(products_df), "product_id", "inner")
broadcast_join.explain("extended")
print()

# Show it works
t0 = time.time()
bj_result = broadcast_join.select("sale_id", "product_name", "category", "revenue")
bj_result.show(5)
t1 = time.time()
print(f"Broadcast join time: {t1 - t0:.2f}s")

# Without explicit hint — Catalyst may still choose broadcast
print("\n--- Auto-detected Broadcast Join (no hint) ---")
auto_join = sales_df.join(products_df, "product_id", "inner")
auto_join.explain()


 ### Strategy 2: Sort-Merge Join (For Two Large Tables)


In [ ]:
print("=== Sort-Merge Join ===\n")

# Both sides are large — must be shuffled and sorted for merge join
print(f"Sales table  : {sales_df.count():,} rows — LARGE")
print(f"Customers    : {customers_df.count():,} rows — LARGE")

smj = sales_df.join(customers_df, "customer_id", "inner")
print("\n--- Sort-Merge Join Plan ---")
smj.explain()

t0 = time.time()
smj_result = smj.select("sale_id", "customer_name", "loyalty_tier", "revenue")
smj_result.show(5)
t1 = time.time()
print(f"Sort-merge join time: {t1 - t0:.2f}s")


 ### Controlling the Broadcast Threshold


In [ ]:
# Show current threshold
broadcast_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(f"autoBroadcastJoinThreshold: {broadcast_threshold} bytes ({int(broadcast_threshold) / (1024*1024):.1f} MB)")

# Temporarily increase to force broadcast on medium tables
print("\n--- Increasing threshold to 50MB ---")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(50 * 1024 * 1024))
print(f"New threshold: {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')} bytes")

# Force smaller threshold
print("\n--- Reducing threshold to 1 byte (disables broadcast) ---")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1")
dis_join = sales_df.join(products_df, "product_id", "inner")
print("With broadcast disabled, small join becomes sort-merge:")
dis_join.explain()

# Restore
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", broadcast_threshold)


 ### Comparing Join Strategies Performance


In [ ]:
# Small table broadcast join
small_for_broadcast = products_df.filter(col("product_id") < 50)
print(f"Small table: {small_for_broadcast.count()} products")

# Test 1: Broadcast hash join (hinted)
t0 = time.time()
bhj = sales_df.join(broadcast(small_for_broadcast), "product_id").count()
t1 = time.time()
print(f"\nBroadcast Hash Join (hinted): {t1 - t0:.2f}s")

# Test 2: Let Catalyst choose (will also broadcast since it's small)
t0 = time.time()
auto = sales_df.join(small_for_broadcast, "product_id").count()
t1 = time.time()
print(f"Auto-chosen join           : {t1 - t0:.2f}s")

# Test 3: Sort-merge join on large-large
t0 = time.time()
smj = sales_df.join(customers_df, "customer_id").count()
t1 = time.time()
print(f"Sort-Merge Join (large)    : {t1 - t0:.2f}s")

# Test 4: Disable broadcast, force sort-merge on small join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
t0 = time.time()
forced_smj = sales_df.join(small_for_broadcast, "product_id").count()
t1 = time.time()
print(f"SMJ forced on small join   : {t1 - t0:.2f}s (much worse!)")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", broadcast_threshold)


 ### Join Strategy Selection Rules

 ```
 1. Is either side small (< autoBroadcastJoinThreshold)?
    YES → Broadcast Hash Join
    NO  → Continue to step 2

 2. Can AQE optimize at runtime?
    Small enough → Convert to Broadcast Hash Join
    Skew detected → Split skewed partition

 3. Default: Sort-Merge Join
    Both sides shuffled, sorted, merged
 ```

 **Pro tip:** Always `cache()` small dimension tables used in multiple joins. Explicitly `broadcast()` when you know a table is small but Catalyst might not.


 **Key takeaway:** Broadcast Hash Join is the fastest (no shuffle on large side). Sort-Merge Join handles large-large joins. Use `broadcast()` hint for small tables and tune `autoBroadcastJoinThreshold`. AQE can switch strategies at runtime.


 ---
 ## Concept 17: Reading the Spark UI [Medium]

 The Spark UI is your primary observability tool. It has these key tabs:

 | Tab | Purpose | What to Look For |
 |-----|---------|------------------|
 | **Jobs** | All Spark jobs | Duration, stages per job, status |
 | **Stages** | Individual stages | Task duration variance, shuffle size, spill |
 | **Tasks** | Task-level detail | Skew (max >> median), GC time, errors |
 | **Storage** | Cached RDDs/DataFrames | Cache size, partitions cached, memory used |
 | **Environment** | Config & JARs | Verify settings took effect |
 | **Executors** | Executor metrics | Memory used, tasks completed, shuffle I/O |
 | **SQL** | Query DAG details | Plan visualization, metrics per node |

 > On Community Edition, the Spark UI is available at the URL printed by `spark.sparkContext.uiWebUrl`.


 ### Show the Spark UI URL


In [ ]:
sc = spark.sparkContext
print(f"=" * 70)
print(f"  SPARK UI: {sc.uiWebUrl}")
print(f"=" * 70)
print(f"\nApplication ID : {sc.applicationId}")
print(f"Application Name : {sc.appName}")
print(f"\nOpen this URL in your browser to follow along.")


 ### Creating Queries That Generate Interesting UI Patterns


 **Query 1: Shuffle-heavy aggregation** — Look at the Stages tab for shuffle read/write


In [ ]:
# Query with significant shuffle
print("Running shuffle-heavy aggregation...")
shuffle_query = (sales_df
                 .filter(col("quantity") > 1)
                 .groupBy("region", "product_id", "payment_type")
                 .agg(
                     sum("revenue").alias("total_rev"),
                     count("*").alias("transaction_count"),
                     avg("quantity").alias("avg_qty")
                 )
                 .filter(col("total_rev") > 100)
                 .orderBy(col("total_rev").desc()))

t0 = time.time()
shuffle_query.count()
t1 = time.time()
print(f"Completed in {t1 - t0:.2f}s")

print("""
In Spark UI → Jobs tab:
  - Find this job (most recent)
  - Click to expand → see stages
  - Stage 0: Scan + filter (no shuffle)
  - Stage 1: Shuffle (Exchange hashpartitioning) + aggregation
  - Check: Shuffle Write size, Shuffle Read size
""")


 **Query 2: Join with broadcast hint** — Compare SQL tab DAG to shuffle-heavy query


In [ ]:
print("Running broadcast join...")
small_subset = products_df.filter(col("product_id") < 100)

t0 = time.time()
bj_query = sales_df.join(broadcast(small_subset), "product_id").filter(col("category") == "Electronics")
bj_query.count()
t1 = time.time()
print(f"Completed in {t1 - t0:.2f}s")

print("""
In Spark UI → SQL tab:
  - Click the query ID for this broadcast join
  - Notice: No "Exchange" (shuffle) node — BroadcastExchange instead
  - Compare to a sort-merge join query for contrast
""")


 ### Diagnosing a Slow Query from UI Metrics

 Here's a decision tree for diagnosing issues using the Spark UI:


In [ ]:
print("""
=== SLOW QUERY DIAGNOSIS FLOWCHART ===

1. JOBS TAB → Find slow job
   Q: Is one stage much slower than others?
   
2. STAGES TAB → Click slow stage
   Q: Is there high task duration variance?
      YES → Look at Tasks tab for skew
             Some tasks take 10x others? → DATA SKEW (Concept 19)
      NO  → All tasks uniformly slow? Continue
   
3. SQL TAB → Click the query
   Q: Is there a Shuffle (Exchange) node?
      YES → Check Shuffle Write size
             Large shuffle? → Tune partitions (Concept 18)
             Spill > 0? → Memory issue (Concept 20)
      NO  → Check Scan node
             Many files? → Small file problem
             
4. EXECUTORS TAB
   Q: GC Time high (>10% of task time)?
      YES → GC pressure → Increase executor memory or tune GC
   
5. STORAGE TAB
   Q: Is data being evicted from cache?
      YES → Not enough storage memory → Increase or don't cache

=== KEY METRICS TO WATCH ===
- Task Duration (Max / Median) → > 2x indicates skew
- Shuffle Spill (Memory / Disk) → > 0 means spill occurred
- Input Size / Records → source data volume
- GC Time → executor overhead
- Scheduler Delay → cluster busy
- Task Deserialization Time → code/closure too large
""")


 ### Generating UI Data: Run Several Diverse Queries


In [ ]:
print("Running diverse queries to populate the Spark UI...")

# 1. Simple filter + select (no shuffle)
query1 = sales_df.filter(col("store_id") == 1).select("sale_id", "revenue", "region")
query1.count()

# 2. GroupBy aggregation (shuffle)
query2 = sales_df.groupBy("region", "store_id").agg(sum("revenue"), count("*"))
query2.count()

# 3. Join (shuffle)
query3 = sales_df.join(customers_df, "customer_id").select("sale_id", "customer_name", "loyalty_tier")
query3.count()

# 4. Window function (shuffle)
from pyspark.sql.window import Window
w = Window.partitionBy("region").orderBy(col("revenue").desc())
query4 = sales_df.select("sale_id", "region", "revenue", F.rank().over(w).alias("rank"))
query4.count()

print(f"\nAll queries complete. Open Spark UI at {sc.uiWebUrl}")
print("Compare the SQL tab entries for each query.")


 **Key takeaway:** The Spark UI is your diagnostic tool. The Jobs tab tells you what's slow. The Stages tab reveals why (skew, spill, shuffle size). The SQL tab shows the optimized plan visually. The Executors tab reveals memory/GC issues. Learn to navigate all tabs.


 ---
 ## Concept 18: Partitioning & Parallelism [Medium]

 **Partitions** are the fundamental unit of parallelism in Spark. Each partition = one task = one CPU core working on a subset of data.

 Key settings:
 - `spark.sql.shuffle.partitions` (default 200) — partitions after shuffle
 - `spark.default.parallelism` — RDD operations default
 - `repartition(n)` — full shuffle to N partitions
 - `coalesce(n)` — reduce partitions WITHOUT shuffle (combine adjacent)
 - `spark.sql.files.maxPartitionBytes` — max bytes per partition when reading files


 ### Partitions and Parallelism

 In Community Edition (single node), parallelism is limited by CPU cores, but the concepts of partition count still affect memory usage, spill, and performance.


In [ ]:
executor_cores = 4  # Community Edition typical
shuffle_partitions = int(spark.conf.get("spark.sql.shuffle.partitions"))

print(f"=== Partitioning & Parallelism ===\n")
print(f"Shuffle partitions (config)      : {shuffle_partitions}")
print(f"Default parallelism              : {spark.sparkContext.defaultParallelism}")
print(f"Typical Community Edition cores  : ~{executor_cores}")
print(f"\nParallelism limit: At most {executor_cores} tasks run concurrently.")
print(f"Partitions > cores: Tasks queue up — more scheduling overhead, but better memory distribution.")
print(f"Partitions < cores: Cores idle — under-utilization.")
print(f"\nRule of thumb: partitions = cores * 2 to 4 (for pipelining)")


 ### `repartition()` vs `coalesce()` — Critical Difference


In [ ]:
# Create a DataFrame with few partitions
few_partitions_df = sales_df.coalesce(2)
print(f"Few partitions: {few_partitions_df.rdd.getNumPartitions()}")

# repartition(): FULL shuffle — redistributes data evenly across N partitions
t0 = time.time()
repartitioned = few_partitions_df.repartition(16)
t1 = time.time()
print(f"\nrepartition(16):")
print(f"  New partition count: {repartitioned.rdd.getNumPartitions()}")
print(f"  Data redistribution: FULL SHUFFLE (expensive)")
print(f"  Use when: you need MORE partitions or even distribution")

# coalesce(): NO shuffle — combines adjacent partitions
t0 = time.time()
coalesced = repartitioned.coalesce(4)
t1 = time.time()
print(f"\ncoalesce(4):")
print(f"  New partition count: {coalesced.rdd.getNumPartitions()}")
print(f"  Data redistribution: NO SHUFFLE (cheap)")
print(f"  Use when: reducing partitions (only!) for downstream efficiency")
print(f"  WARNING: coalesce(n) where n > current is a NO-OP (use repartition)")


 ### Over-Partitioning vs Under-Partitioning


In [ ]:
import time

def run_with_partitions(data_df, n_partitions, label):
    spark.conf.set("spark.sql.shuffle.partitions", str(n_partitions))
    t0 = time.time()
    result = (data_df
              .groupBy("product_id", "store_id")
              .agg(sum("revenue").alias("total"), count("*").alias("cnt"))
              .filter(col("total") > 100))
    cnt = result.count()
    t1 = time.time()
    print(f"  {label}: {n_partitions} partitions → {cnt:,} rows in {t1 - t0:.2f}s")
    return t1 - t0

print("=== Effect of Partition Count on Performance ===\n")

# Use a subset to highlight differences
test_data = sales_df.filter(col("sale_id") < 500000)

times = {}
times[2]    = run_with_partitions(test_data, 2,    "Under    ")
times[10]   = run_with_partitions(test_data, 10,   "Low      ")
times[50]   = run_with_partitions(test_data, 50,   "Moderate ")
times[200]  = run_with_partitions(test_data, 200,  "Default  ")
times[400]  = run_with_partitions(test_data, 400,  "High     ")

# Restore
spark.conf.set("spark.sql.shuffle.partitions", "200")

print(f"\n=== Summary ===")
print(f"Optimal appeared around 10-50 partitions for this data size")
print(f"Default of 200 is fine in most cases, but can be tuned down for small data")


 ### Checking Partition Distribution

 Are your partitions balanced?


In [ ]:
# Show partition sizes for a grouped DataFrame
grouped = sales_df.groupBy("product_id").agg(count("*").alias("sales_count"), sum("revenue").alias("total_rev"))

# Check how many partitions and their approximate sizes
print(f"Grouped DataFrame partitions: {grouped.rdd.getNumPartitions()}")
print(f"\nPartition record counts (first 10 partitions):")

# Use glom() to see records per partition (use carefully on large data)
partition_sizes = grouped.rdd.mapPartitions(lambda it: [sum(1 for _ in it)]).collect()
avg_size = sum(partition_sizes) / len(partition_sizes)
max_size = max(partition_sizes)
min_size = min(partition_sizes)

print(f"  Total partitions : {len(partition_sizes)}")
print(f"  Avg per partition: {avg_size:.0f}")
print(f"  Max partition    : {max_size}")
print(f"  Min partition    : {min_size}")
print(f"  Skew ratio       : {max_size / max(1, avg_size):.1f}x")
print(f"  Min/max ratio    : {max_size / max(1, min_size):.1f}x")


 ### Effect of Partition Size on Memory

 Too-large partitions = spill. Too-small partitions = scheduling overhead.


In [ ]:
print("=== Partition Size Effects ===\n")
print(f"Files read by Spark are split into partitions of up to ~128MB")
print(f"After shuffle, partitions are defined by spark.sql.shuffle.partitions")
print()
print("Too-large partitions (>200MB):")
print("  - May cause OOM or excessive spill")
print("  - GC pressure increases")
print("  - Solution: increase spark.sql.shuffle.partitions")
print()
print("Too-small partitions (<10MB):")
print("  - Excessive task scheduling overhead")
print("  - Many small shuffle files")  
print("  - I/O inefficiency from many small reads")
print("  - Solution: coalesce or reduce spark.sql.shuffle.partitions")
print()
print("Ideal partition size: 100-200 MB")
print(f"For our ~{sales_df.count():,}-row table, optimal partitions ≈ {sales_df.count() * 100 // (128 * 1024 * 1024) + 1}")


 **Key takeaway:** Partitions determine parallelism. `repartition()` does a full shuffle (expensive but evenly distributed). `coalesce()` avoids shuffle (cheap but can be uneven). Tune `spark.sql.shuffle.partitions` to match data size: ~100-200MB per partition. For Community Edition, 16-64 partitions is often optimal.


 ---
 ## Concept 19: Data Skew — Detection & Mitigation [Hard]

 **Data skew** occurs when some keys have disproportionately more data than others. In a distributed operation (join, groupBy), the few tasks processing the "hot" keys become stragglers — they hold up the entire job.

 **Symptoms in Spark UI:**
 - High variance in task duration (max task time >> median task time)
 - A few tasks process far more records than others
 - Spill occurring only on certain tasks

 **Mitigation strategies:**
 1. **Salting** — Add random prefix to key to spread hot keys
 2. **AQE skew join optimization** — Spark automatically splits skewed partitions
 3. **Broadcast join** — If one side is small, broadcast avoids shuffle entirely
 4. **Separate skewed keys** — Process skewed keys separately, then union


 ### Creating Intentionally Skewed Data


In [ ]:
# Show the skew in our pre-built skewed dataset
print("=== Skewed Sales Data Distribution ===\n")
skew_stats = skewed_sales_df.groupBy("product_id").count().orderBy(col("count").desc())
skew_stats.show(15)

print(f"\nProduct 1 has ~80% of ALL data!")
product_1_count = skewed_sales_df.filter(col("product_id") == 1).count()
total_count = skewed_sales_df.count()
print(f"Product 1 rows: {product_1_count:,}")
print(f"Total rows    : {total_count:,}")
print(f"Product 1 share: {product_1_count / total_count * 100:.1f}%")


 ### Detecting Skew — Task Duration Analysis

 Run a join with the skewed data and observe the task metrics:


In [ ]:
# Join skewed sales with products — product_id=1 dominates
print("=== Join with Skewed Data ===\n")

t0 = time.time()
skewed_join = skewed_sales_df.join(products_df, "product_id").groupBy("product_id", "product_name", "category").agg(
    sum(col("quantity") * col("price")).alias("total_revenue"),
    count("*").alias("num_sales")
).orderBy(col("num_sales").desc())
skewed_join.show(10)
t1 = time.time()
print(f"\nTotal time: {t1 - t0:.2f}s")
print("\nCheck Spark UI → Stages → Tasks:")
print("  - Task for product_id=1 will process ~160K rows")
print("  - Other tasks process ~20 rows each")
print("  - MAX task duration will be MUCH larger than MEDIAN → SKEW!")


 ### Mitigation Strategy 1: Salting (Manual Skew Handling)


In [ ]:
import random
from pyspark.sql.functions import col, concat, lit, split

SALT_BUCKETS = 20

print(f"=== Salting Technique ({SALT_BUCKETS} salt buckets) ===\n")

# Step 1: Add salt column to the skewed table (0 to SALT_BUCKETS-1)
skewed_with_salt = skewed_sales_df.withColumn(
    "salt", (col("product_id") * 2654435761 % SALT_BUCKETS).cast("int")
)

print("Skewed sales with salt column:")
skewed_with_salt.select("sale_id", "product_id", "salt").show(10)

# Step 2: Explode the products dimension to have salted copies
# For each product, create SALT_BUCKETS copies, one per salt value
products_exploded = products_df.crossJoin(
    spark.range(SALT_BUCKETS).withColumnRenamed("id", "salt")
).withColumn("salted_product_id", col("product_id"))

# Step 3: Join on (product_id, salt)
print("\nPerforming salted join...")
t0 = time.time()
salted_join_result = (skewed_with_salt
    .join(products_exploded, 
          (skewed_with_salt.product_id == products_exploded.product_id) & 
          (skewed_with_salt.salt == products_exploded.salt))
    .groupBy("product_id", "product_name", "category")
    .agg(count("*").alias("num_sales"))
    .orderBy(col("num_sales").desc()))
salted_join_result.show(10)
t1 = time.time()
print(f"Salted join time: {t1 - t0:.2f}s")


 ### Mitigation Strategy 2: AQE Skew Join Optimization


In [ ]:
print("=== AQE Skew Join Optimization ===\n")

# Enable AQE skew join handling
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
try:
    spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
    spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256MB")
except Exception as e:
    print(f"Skew config note: {e}")

# Run join with AQE
print("Running join with AQE skew optimization enabled...")
t0 = time.time()
aqe_join_result = (skewed_sales_df
    .join(products_df, "product_id")
    .groupBy("product_id", "product_name", "category")
    .agg(count("*").alias("num_sales"))
    .orderBy(col("num_sales").desc()))
aqe_join_result.show(10)
t1 = time.time()
print(f"AQE-enabled join time: {t1 - t0:.2f}s")

print("""
AQE detects skewed partitions at runtime:
  - If a partition is > skewPartitionFactor × median partition size
  - AND > skewPartitionThresholdInBytes
  → Spark splits that partition into smaller sub-partitions
  → Each sub-partition joins independently
  → Reduces max task time dramatically

Check Spark UI → SQL tab → this query's DAG:
  - Look for "OptimizeSkewedJoin" in the plan
  - Notice the physical plan changes at runtime
""")


 ### Mitigation Strategy 3: Broadcast Join (Eliminates Shuffle)


In [ ]:
# When the dimension table is small enough to broadcast, skew becomes irrelevant
# (because there's NO shuffle on the large side)
print("=== Broadcast Join Avoids Skew ===\n")
print(f"Products table: {products_df.count()} rows")

t0 = time.time()
broadcast_join_result = (skewed_sales_df
    .join(broadcast(products_df), "product_id")
    .groupBy("product_id", "product_name", "category")
    .agg(count("*").alias("num_sales"))
    .orderBy(col("num_sales").desc()))
broadcast_join_result.show(10)
t1 = time.time()
print(f"Broadcast join time: {t1 - t0:.2f}s")
print("\nBroadcast join eliminates the shuffle entirely → skew doesn't matter!")


 ### Before/After Comparison: Skew Impact


In [ ]:
print("=== Skew Impact Summary ===\n")
print(f"{'Strategy':<30} {'Effect':<50}")
print("-" * 80)
print(f"{'No mitigation (raw join)':<30} {'1 task processes 80% of data → straggler':<50}")
print(f"{'Salting (20 buckets)':<30} {'Hot key spread across 20 partitions':<50}")
print(f"{'AQE skew optimization':<30} {'Automatic partition splitting at runtime':<50}")
print(f"{'Broadcast join':<30} {'No shuffle on large side → skew irrelevant':<50}")
print(f"{'Filter out skewed keys':<30} {'Process skewed + non-skewed separately, then union':<50}")

print("""
=== When to Use Each Strategy ===

SALTING:
  - Both sides are large
  - Specific keys are known to be hot
  - You control the join logic
  
AQE SKEW JOIN:
  - Spark 3.0+ with AQE enabled
  - Automatic detection and handling
  - Best when you don't know which keys are hot

BROADCAST JOIN:
  - One side fits in memory (< ~10MB typical, tunable)
  - Eliminates skew completely for that join
  - Always try this first for small dimension tables

SEPARATE PROCESSING:
  - Very specific skew patterns
  - When neither salting nor AQE is sufficient
  - Manual but gives full control
""")


 **Key takeaway:** Data skew causes straggler tasks and slow jobs. The Spark UI reveals skew through task duration variance. Mitigate with AQE (automatic), broadcast joins (when possible), salting (manual but effective), or separate processing. Always profile your data distribution before optimizing.


 ---
 ## Concept 20: Memory Management & Spill [Hard]

 Spark uses a **unified memory model** where execution memory and storage memory share a pool:

 ```
 ┌──────────────────────────────────────────────┐
 │              JVM Heap (executor.memory)       │
 ├──────────┬──────────────┬─────────────────────┤
 │ Reserved │ Spark Memory │     User Memory      │
 │  Memory  │  (unified)   │  (UDFs, data structs)│
 │ (300MB)  ├──────┬───────┤  (40% of remaining)  │
 │          │Exec  │Storage│                      │
 │          │Mem   │ Memory│                      │
 │          │(50%) │ (50%) │                      │
 └──────────┴──────┴───────┴──────────────────────┘
 ```

 **Spill** happens when execution memory is exhausted — data is serialized and written to disk, which is orders of magnitude slower than in-memory processing.


 ### Understanding the Unified Memory Model


In [ ]:
print("=== Spark Unified Memory Model ===\n")

try:
    exec_mem = spark.conf.get("spark.executor.memory")
    print(f"Executor memory: {exec_mem}")
except Exception:
    exec_mem = "2g (Community Edition default)"
    print(f"Executor memory: {exec_mem}")

try:
    mem_fraction = spark.conf.get("spark.memory.fraction")
    print(f"spark.memory.fraction: {mem_fraction}")
except Exception:
    mem_fraction = "0.6"
    print(f"spark.memory.fraction: {mem_fraction} (default)")

try:
    storage_fraction = spark.conf.get("spark.memory.storageFraction")
    print(f"spark.memory.storageFraction: {storage_fraction}")
except Exception:
    storage_fraction = "0.5"
    print(f"spark.memory.storageFraction: {storage_fraction} (default)")

print(f"""
Memory breakdown (approximate for {exec_mem}):
  Reserved memory (300MB):                    300 MB
  User memory ({1 - float(mem_fraction):.0%} of {(2048-300)*float(mem_fraction):.0f} MB):      ~{int((2048-300)*(1-float(mem_fraction)))} MB
  Spark unified memory pool:                  ~{(2048-300)*float(mem_fraction):.0f} MB
    - Execution memory (50% default):          ~{(2048-300)*float(mem_fraction)*0.5:.0f} MB
    - Storage memory (50% default):            ~{(2048-300)*float(mem_fraction)*0.5:.0f} MB

Execution memory is used for: shuffles, joins, sorts, aggregations
Storage memory is used for: cached data, broadcast variables
Execution CAN evict storage (but not vice versa)

When execution memory is full → SPILL TO DISK
""")


 ### Creating a Spill Scenario

 We're going to deliberately cause spill by setting very few shuffle partitions — each partition becomes too large to fit in memory:


In [ ]:
print("=== Deliberately Causing Spill ===\n")

# Force very few shuffle partitions → each partition is large → spills
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Heavy aggregation on all data — each partition processes 250K rows
t0 = time.time()
spill_query = (sales_df
               .groupBy("product_id", "store_id")
               .agg(
                   sum("revenue").alias("total_rev"),
                   count("*").alias("cnt"),
                   avg("quantity").alias("avg_qty"),
                   max("unit_price").alias("max_price"),
                   sum("quantity").alias("total_qty")
               )
               .orderBy(col("total_rev").desc()))

spill_result = spill_query.collect()
t1 = time.time()
print(f"Query with 4 shuffle partitions: {t1 - t0:.2f}s")
print(f"Result rows: {len(spill_result)}")

print("""
Check Spark UI → Stages → this job:
  - Look for "Spill (Memory)" and "Spill (Disk)" > 0
  - Each task processes ~250K rows — likely exceeds execution memory
  - Spill means data was written to/read from disk mid-computation
  
In SQL tab, you'll see Spill metrics for each operation.
""")


 ### Reducing Spill by Tuning Partitions


In [ ]:
print("=== Reducing Spill with More Partitions ===\n")

for n_partitions in [4, 16, 64, 200]:
    spark.conf.set("spark.sql.shuffle.partitions", str(n_partitions))
    t0 = time.time()
    cnt = (sales_df
           .groupBy("product_id", "store_id")
           .agg(sum("revenue").alias("total_rev"), count("*").alias("cnt"))
           .orderBy(col("total_rev").desc())
           .count())
    t1 = time.time()
    partition_mb = (sales_df.count() * 100) / (1024 * 1024) / n_partitions
    print(f"  {n_partitions:>4} partitions → ~{partition_mb:.0f} MB/task → {t1 - t0:.2f}s")

# Restore
spark.conf.set("spark.sql.shuffle.partitions", "200")
print(f"\nMore partitions = smaller per-partition data = less spill = potentially faster")


 ### Detecting Spill from Code


In [ ]:
print("=== Monitoring Memory and Spill ===\n")
print("""
Spill indicators in the Spark UI:

1. STAGES TAB → Click a stage → Summary Metrics:
   - "Spill (Memory)" > 0 → Memory wasn't enough
   - "Spill (Disk)" > 0 → Data was written to disk
   - High spill = significant performance penalty (disk ~100x slower than memory)

2. TASKS TAB → Individual task metrics:
   - "Shuffle Spill (Memory)" — bytes spilled during shuffle
   - "Shuffle Spill (Disk)" — disk bytes written for spill
   - "Peak Execution Memory" — peak memory used by task
   
3. SQL TAB → Query details:
   - Each operator shows spill metrics
   - Sort, Aggregate, and Join are most spill-prone

4. What spill looks like numerically:
   - Spill > 0 on any operation → check partition sizes
   - Spill > input records → severe spill (data didn't fit at all)
   - Spill = 0 → all data fit in memory (ideal)
""")

# Show how to estimate memory per task
total_rows = sales_df.count()
est_bytes_per_row = 150  # rough estimate with overhead
est_mb_per_partition_default = (total_rows * est_bytes_per_row) / (1024 * 1024) / 200
est_mb_per_partition_low = (total_rows * est_bytes_per_row) / (1024 * 1024) / 8

print(f"Estimated memory per task:")
print(f"  With 200 partitions: ~{est_mb_per_partition_default:.0f} MB/task")
print(f"  With 8 partitions  : ~{est_mb_per_partition_low:.0f} MB/task")
print(f"  Available exec mem  : ~520 MB (estimated for Community Edition)")
print(f"  Spill threshold     : when per-task data > available execution memory")


 ### Memory Configuration Effects


In [ ]:
print("=== Memory Configuration Reference ===\n")

print("""
KEY MEMORY CONFIGURATIONS:

spark.executor.memory (e.g., "4g")
  - Total JVM heap per executor
  - Community Edition: typically 2-4 GB
  - Increase for: large shuffles, large cached datasets

spark.memory.fraction (default 0.6)
  - Fraction of heap for Spark unified memory (execution + storage)
  - Remaining 40% is for user data structures + metadata
  - Decrease if: out of memory from user objects / UDFs

spark.memory.storageFraction (default 0.5)
  - Fraction of Spark memory for storage (cache) vs execution
  - Execution can evict storage, but not vice versa
  - Increase if: you cache a lot and want to avoid eviction

spark.sql.shuffle.partitions (default 200)
  - More partitions = smaller tasks = less memory per task = less spill
  - But too many = excessive overhead
  - Tune based on data size

spark.shuffle.file.buffer (default 32k)
  - Buffer size for shuffle write
  - Larger = fewer disk seeks but more memory

spark.reducer.maxSizeInFlight (default 48m)
  - Max data fetched per reduce task at once
  - Larger = faster but more memory

TUNING FOR COMMUNITY EDITION (single node, 2-4 GB):
  1. spark.executor.memory = "3g" (if allowed)
  2. spark.sql.shuffle.partitions = 8-32 (single node doesn't need 200)
  3. spark.memory.fraction = 0.7 (slightly more for Spark)
  4. Use broadcast joins whenever possible (avoids shuffle)
  5. Filter early, select only needed columns
""")


 ### Practical: Compare High-Spill vs Low-Spill Configurations


In [ ]:
print("=== High-Spill Configuration ===\n")
spark.conf.set("spark.sql.shuffle.partitions", "2")

t0 = time.time()
high_spill = (sales_df
              .groupBy("product_id", "region", "store_id")
              .agg(sum("revenue").alias("total"), count("*").alias("cnt"))
              .collect())
t1 = time.time()
high_spill_time = t1 - t0
print(f"2 partitions  → {high_spill_time:.2f}s")
print("Check Spark UI for Spill metrics — expect significant spill")

print("\n=== Low-Spill Configuration ===\n")
spark.conf.set("spark.sql.shuffle.partitions", "200")

t0 = time.time()
low_spill = (sales_df
             .groupBy("product_id", "region", "store_id")
             .agg(sum("revenue").alias("total"), count("*").alias("cnt"))
             .collect())
t1 = time.time()
low_spill_time = t1 - t0
print(f"200 partitions → {low_spill_time:.2f}s")
print("Check Spark UI — expect less or no spill")

print(f"\n=== Comparison ===")
print(f"High-spill (2 partitions)  : {high_spill_time:.2f}s")
print(f"Low-spill  (200 partitions): {low_spill_time:.2f}s")
print(f"Difference                 : {high_spill_time - low_spill_time:.2f}s")
print(f"\nToo few partitions → each task processes too much data → spill → slowdown")

# Restore default
spark.conf.set("spark.sql.shuffle.partitions", "200")


 **Key takeaway:** Spill occurs when execution memory is exhausted, writing data to disk (100x slower). Tune `spark.sql.shuffle.partitions` so each partition fits in execution memory (~100-200MB). Use the Spark UI Stages and Tasks tabs to detect spill metrics. In Community Edition, with limited memory, partition tuning is even more critical.


 ---
 ## Summary: Concepts #11-#20

 | # | Concept | Key Takeaway |
 |---|---------|--------------|
 | 11 | **Lazy Evaluation** | Transformations build a plan; actions execute it. Catalyst optimizes the full DAG before running. |
 | 12 | **Photon Engine** | C++ vectorized engine (not in Community Edition). Understand why batch processing is faster. |
 | 13 | **Catalyst Optimizer** | 4-phase optimization: parsed → analyzed → optimized → physical. Predicate pushdown & column pruning. |
 | 14 | **Adaptive Query Execution** | Runtime re-optimization: coalesce partitions, switch joins, handle skew, skip empty partitions. |
 | 15 | **Shuffle Operations** | Most expensive operation. GroupBy, Join, Distinct trigger it. Monitor via Spark UI. |
 | 16 | **Join Strategies** | Broadcast Hash Join (small+large, no shuffle) vs Sort-Merge Join (large+large). Use `broadcast()` hint. |
 | 17 | **Spark UI** | Jobs → Stages → Tasks → SQL tabs. Diagnose skew, spill, and shuffle bottlenecks. |
 | 18 | **Partitioning & Parallelism** | `repartition()` (full shuffle) vs `coalesce()` (no shuffle). Tune `spark.sql.shuffle.partitions`. |
 | 19 | **Data Skew** | Some keys dominate → straggler tasks. Mitigate with salting, AQE, broadcast joins. |
 | 20 | **Memory & Spill** | Unified memory model. Spill = disk I/O = slow. Tune partitions to fit data in memory. |

 ---


 ## Self-Assessment: Score Yourself

 Rate yourself 5 for each concept (0 = no understanding, 5 = can explain clearly):

 | Concept | Topic | Your Score (0-5) |
 |---------|-------|------------------|
 | 11 | Lazy Evaluation & Actions | |
 | 12 | Photon Engine | |
 | 13 | Catalyst Optimizer | |
 | 14 | Adaptive Query Execution | |
 | 15 | Shuffle Operations | |
 | 16 | Join Strategies | |
 | 17 | Reading the Spark UI | |
 | 18 | Partitioning & Parallelism | |
 | 19 | Data Skew Detection & Mitigation | |
 | 20 | Memory Management & Spill | |

 **Total: /50**

 | Score | Level | Next Step |
 |-------|-------|-----------|
 | 40-50 | Senior | Move to Notebook 03 |
 | 25-39 | Mid-level | Revisit concepts scored 3 or below |
 | 15-24 | Foundation | Re-run the code cells for weaker concepts |
 | 0-14 | Beginner | Review each concept's markdown explanation |

 ---

 ### What's Next?

 **Notebook 03: SQL & DataFrames** — Concepts #21-#30 covering daily transformation work with both APIs.

 ### Quick Reference Card

 ```python
 # Lazy evaluation — build, don't execute
 df.filter(...).select(...).groupBy(...)  # transformations only
 df.show()  # action — triggers execution

 # See the plan
 df.explain(True)  # full 4-phase plan
 df.explain()      # physical plan only

 # Joins — pick the right strategy
 large_df.join(broadcast(small_df), "key")  # broadcast hash join
 large1.join(large2, "key")                  # sort-merge join

 # Partitions — control parallelism
 spark.conf.set("spark.sql.shuffle.partitions", "50")
 df.repartition(50)   # full shuffle, even distribution
 df.coalesce(10)      # no shuffle, reduce partitions

 # AQE — let Spark optimize at runtime
 spark.conf.set("spark.sql.adaptive.enabled", "true")
 spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

 # Debugging
 spark.sparkContext.uiWebUrl  # Spark UI URL
 ```

 ---
 *End of Notebook 02 — Spark Execution*


In [ ]:
# Unpersist cached data to free memory
try:
    products_df.unpersist()
    orders_df.unpersist()
    print("Cached DataFrames unpersisted.")
except Exception:
    pass

print("\nNotebook 02 complete. Ready for Notebook 03: SQL & DataFrames.")


---

# 03 SQL and DataFrames



 # 03 — SQL & DataFrames: Advanced Analytics

 This notebook covers **Concepts #21–#30** of the Databricks learning path.

 | # | Concept | Level |
 |---|---------|-------|
 | 21 | DataFrame API — Core Operations | Easy |
 | 22 | Temp Views & Global Temp Views | Easy |
 | 23 | Window Functions | Medium |
 | 24 | Complex Types — ARRAY, STRUCT, MAP | Medium |
 | 25 | Higher-Order Functions | Medium |
 | 26 | CTEs, PIVOT & Subqueries | Medium |
 | 27 | UDFs vs Native Functions | Medium |
 | 28 | VARIANT Type & Semi-Structured Data | Medium |
 | 29 | MERGE INTO & Upsert Patterns | Hard |
 | 30 | Table Constraints & Generated Columns | Medium |

 **Scenario:** *RetailCo* — an ecommerce company managing customers, products, orders, and event logs.

 **Setup:** All synthetic data is generated in-cell. No external files required.


 ## 0 — Environment Setup & Synthetic Data Generation

 Create all datasets used across the notebook.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType, ArrayType, MapType
from pyspark.sql.functions import (
    col, lit, when, expr, rand, randn, round as spark_round,
    row_number, rank, dense_rank, lag, lead, sum as spark_sum,
    explode, posexplode, struct, array, map_from_arrays, map_concat,
    transform, filter as spark_filter, exists, aggregate,
    count, avg, max as spark_max, min as spark_min, collect_list, collect_set,
    to_date, date_sub, date_add, monotonically_increasing_id,
    pandas_udf, PandasUDFType, udf, coalesce, current_timestamp,
    unix_timestamp, from_unixtime, to_timestamp,
    split, size, element_at, slice, array_contains,
    broadcast, countDistinct, sumDistinct
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType
from datetime import datetime, timedelta
import random
import time

spark = SparkSession.builder.appName("SQL_DataFrames_Concepts_21_30").getOrCreate()
print(f"Spark version : {spark.version}")
print(f"SparkSession  : {spark}")


In [ ]:
# ==========================================================================
# SYNTHETIC DATA: customers, products, orders, events
# ==========================================================================

random.seed(42)

# --- CUSTOMERS ---
customer_data = []
regions = ["North America", "Europe", "APAC", "Latin America", "Middle East"]
for i in range(1, 21):
    region = random.choice(regions)
    preferences = random.sample(["sports", "electronics", "fashion", "books", "home", "toys", "food"], k=random.randint(2, 5))
    customer_data.append((
        i,
        f"customer_{i}",
        f"cust{i}@example.com",
        region,
        to_date(lit(f"2023-{random.randint(1,12):02d}-{random.randint(1,28):02d}")).cast(DateType()),
        preferences
    ))

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType()),
    StructField("email", StringType()),
    StructField("region", StringType()),
    StructField("signup_date", DateType()),
    StructField("preferences", ArrayType(StringType()))
])
customers_df = spark.createDataFrame(customer_data, schema=customers_schema)
customers_df.createOrReplaceTempView("customers")
print(f"customers: {customers_df.count()} rows")

# --- PRODUCTS ---
categories = {
    "Smartphone": "Electronics", "Laptop": "Electronics", "Headphones": "Electronics",
    "T-Shirt": "Fashion", "Jeans": "Fashion", "Sneakers": "Fashion",
    "Football": "Sports", "Yoga Mat": "Sports", "Dumbbell": "Sports",
    "Novel": "Books", "Cookbook": "Books", "Textbook": "Books",
    "Blender": "Home", "Lamp": "Home", "Cookware Set": "Home",
    "Action Figure": "Toys", "Board Game": "Toys", "Puzzle": "Toys",
    "Coffee": "Food", "Chocolate": "Food"
}
product_data = []
for pid, (name, cat) in enumerate(categories.items(), start=1):
    price = round(random.uniform(5.99, 1499.99), 2)
    tags = [cat.lower()] + random.sample(["bestseller", "new_arrival", "sale", "clearance", "premium", "eco_friendly"], k=random.randint(2, 4))
    product_data.append((pid, name, cat, price, tags, {"weight_kg": round(random.uniform(0.1, 10.0), 2), "color": random.choice(["black", "white", "red", "blue", "green"])}))

products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType()),
    StructField("category", StringType()),
    StructField("price", DoubleType()),
    StructField("tags", ArrayType(StringType())),
    StructField("attributes", MapType(StringType(), StringType()))
])
products_df = spark.createDataFrame(product_data, schema=products_schema)
products_df.createOrReplaceTempView("products")
print(f"products: {products_df.count()} rows")

# --- ORDERS ---
order_data = []
statuses = ["pending", "shipped", "delivered", "cancelled", "returned"]
for oid in range(1, 201):
    cust_id = random.randint(1, 20)
    order_date = f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    num_items = random.randint(1, 5)
    items = []
    total = 0.0
    for _ in range(num_items):
        pid = random.randint(1, 20)
        qty = random.randint(1, 3)
        price = categories[list(categories.keys())[pid - 1]]  # won't work, use lookup
        price = [p[3] for p in product_data if p[0] == pid][0]
        items.append({"product_id": pid, "quantity": qty, "unit_price": price})
        total += price * qty
    status = random.choice(statuses)
    metadata = {
        "channel": random.choice(["web", "mobile_app", "in_store"]),
        "coupon_code": random.choice(["SAVE10", "FREESHIP", "", "", ""]) or None,
        "is_gift": str(random.choice([True, False])).lower()
    }
    order_data.append((oid, cust_id, to_date(lit(order_date)).cast(DateType()), round(total, 2), status, items, metadata))

from pyspark.sql.types import StructField as SF
items_schema = ArrayType(StructType([
    SF("product_id", IntegerType()),
    SF("quantity", IntegerType()),
    SF("unit_price", DoubleType())
]))
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType()),
    StructField("order_date", DateType()),
    StructField("total_amount", DoubleType()),
    StructField("status", StringType()),
    StructField("items", items_schema),
    StructField("metadata", MapType(StringType(), StringType()))
])
orders_df = spark.createDataFrame(order_data, schema=orders_schema)
orders_df.createOrReplaceTempView("orders")
print(f"orders: {orders_df.count()} rows")

# --- EVENTS (API Log with nested JSON) ---
event_types = ["page_view", "add_to_cart", "purchase", "search", "login", "logout", "review_submit"]
event_data = []
for eid in range(1, 501):
    ev_type = random.choice(event_types)
    ts = f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d} {random.randint(0,23):02d}:{random.randint(0,59):02d}:{random.randint(0,59):02d}"
    payload = {
        "user_agent": random.choice(["Mozilla/5.0 Chrome", "Mozilla/5.0 Safari", "Mozilla/5.0 Edge", "App-iOS/3.2", "App-Android/4.1"]),
        "url": random.choice(["/home", "/products", "/cart", "/checkout", "/account", f"/product/{random.randint(1,20)}"]),
        "session_id": f"sess_{random.randint(1000, 9999)}",
        "extra_context": {
            "referrer": random.choice(["google.com", "facebook.com", "direct", "email_campaign", ""]),
            "ab_test_group": random.choice(["control", "variant_a", "variant_b"])
        }
    }
    event_data.append((eid, to_timestamp(lit(ts)), ev_type, payload))

events_schema = StructType([
    StructField("event_id", IntegerType(), False),
    StructField("timestamp", TimestampType()),
    StructField("event_type", StringType()),
    StructField("payload", StructType([
        StructField("user_agent", StringType()),
        StructField("url", StringType()),
        StructField("session_id", StringType()),
        StructField("extra_context", StructType([
            StructField("referrer", StringType()),
            StructField("ab_test_group", StringType())
        ]))
    ]))
])
events_df = spark.createDataFrame(event_data, schema=events_schema)
events_df.createOrReplaceTempView("events")
print(f"events: {events_df.count()} rows")

# --- SALES (monthly, for PIVOT) ---
months = ["2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06",
          "2024-07", "2024-08", "2024-09", "2024-10", "2024-11", "2024-12"]
sales_data = []
for m in months:
    for r in regions:
        sales_data.append((m, r, "Online", round(random.uniform(5000, 50000), 2)))
        sales_data.append((m, r, "In-Store", round(random.uniform(3000, 40000), 2)))
        sales_data.append((m, r, "Wholesale", round(random.uniform(1000, 25000), 2)))

sales_df = spark.createDataFrame(sales_data, schema=["month", "region", "channel", "revenue"])
sales_df.createOrReplaceTempView("sales")
print(f"sales: {sales_df.count()} rows")

print("\n--- All datasets created ---")


 ---
 ## Concept 21 — DataFrame API: Core Operations

 Covers: `select`, `filter`, `withColumn`, `groupBy`, `agg`, `join` with method chaining.
 Also demonstrates DataFrame API vs Spark SQL equivalence.


In [ ]:
from pyspark.sql.functions import col, lit, when, expr, round as spark_round, count, avg, sum as spark_sum, rank, dense_rank

# === SELECT + FILTER + WITHCOLUMN ===
print("=" * 60)
print("CONCEPT 21: DataFrame API Core Operations")
print("=" * 60)

df_orders_enriched = orders_df \
    .select("order_id", "customer_id", "order_date", "total_amount", "status") \
    .filter(col("status").isin("shipped", "delivered")) \
    .withColumn("is_high_value", when(col("total_amount") > 200, True).otherwise(False)) \
    .withColumn("order_month", expr("date_format(order_date, 'yyyy-MM')")) \
    .withColumn("amount_category",
                when(col("total_amount") < 50, "low")
                .when(col("total_amount") < 200, "medium")
                .otherwise("high"))

print("Enriched orders (shipped/delivered, high-value flag, amount category):")
df_orders_enriched.show(10, truncate=False)

# === GROUP BY + AGG ===
print("\n--- GroupBy + Agg: Revenue by Region ---")
df_joined = orders_df.alias("o") \
    .filter(col("o.status").isin("shipped", "delivered")) \
    .join(customers_df.alias("c"), col("o.customer_id") == col("c.customer_id"), "inner") \
    .select("o.*", "c.region")

df_region_revenue = df_joined \
    .groupBy("region") \
    .agg(
        spark_sum("total_amount").alias("total_revenue"),
        round(avg("total_amount"), 2).alias("avg_order_value"),
        count("order_id").alias("num_orders")
    ) \
    .orderBy(col("total_revenue").desc())

df_region_revenue.show(truncate=False)

# === JOIN types ===
print("\n--- All Orders with Customer Info (LEFT JOIN) ---")
orders_df.alias("o") \
    .join(customers_df.alias("c"), "customer_id", "left") \
    .select("o.order_id", "c.name", "c.region", "o.total_amount", "o.status") \
    .show(5, truncate=False)

# === DataFrame API vs Spark SQL: Same Physical Plan ===
print("\n--- DataFrame API & Spark SQL produce identical physical plans ---")
plan_df_api = orders_df.groupBy("status").agg(count("*").alias("cnt"))._jdf.queryExecution().toString().split("\n")[-5:]
plan_sql = spark.sql("SELECT status, COUNT(*) AS cnt FROM orders GROUP BY status")._jdf.queryExecution().toString().split("\n")[-5:]

print("DataFrame API plan tail:")
for line in plan_df_api:
    print(f"  {line}")
print("\nSpark SQL plan tail:")
for line in plan_sql:
    print(f"  {line}")
print("\n>> Both plans are identical — DataFrame API and Spark SQL share the same Catalyst optimizer.")

# === createOrReplaceTempView for hybrid ===
customers_df.createOrReplaceTempView("customers")
orders_df.createOrReplaceTempView("orders")
print("\nTemp views created. You can now mix spark.sql() with DataFrame API.")

# === Column expressions, aliases, when/otherwise demo ===
print("\n--- Column Expressions, Aliases, when/otherwise ---")
orders_df \
    .withColumn("discount_category",
                when(col("total_amount") > 300, "premium_customer")
                .when(col("total_amount") > 100, "regular_customer")
                .otherwise("standard_customer")) \
    .withColumn("tax", spark_round(col("total_amount") * 0.08, 2)) \
    .withColumn("total_with_tax", col("total_amount") + col("tax")) \
    .select(
        col("order_id").alias("Order#"),
        col("total_amount").alias("Subtotal"),
        col("tax").alias("Tax(8%)"),
        col("total_with_tax").alias("Grand Total"),
        col("discount_category").alias("Tier")
    ) \
    .show(10)


 ---
 ## Concept 22 — Temp Views & Global Temp Views

 - **Temp View:** scoped to the SparkSession (notebook-level)
 - **Global Temp View:** accessible across notebooks on the same cluster (via `global_temp` database)
 - Neither persists across cluster restarts


In [ ]:
print("=" * 60)
print("CONCEPT 22: Temp Views & Global Temp Views")
print("=" * 60)

# -- Create a temp view (session-scoped) --
customers_df.createOrReplaceTempView("customers_temp_view")
print("[1] Created TEMP VIEW: customers_temp_view")

result_temp = spark.sql("""
    SELECT region, COUNT(*) AS customer_count
    FROM customers_temp_view
    GROUP BY region
    ORDER BY customer_count DESC
""")
print("\nQuerying TEMP VIEW (works within this notebook):")
result_temp.show(truncate=False)

# -- Create a global temp view (cross-session, same cluster) --
customers_df.createOrReplaceGlobalTempView("customers_global_view")
print("\n[2] Created GLOBAL TEMP VIEW: customers_global_view")

# Query via global_temp prefix
result_global = spark.sql("""
    SELECT region, COUNT(*) AS customer_count
    FROM global_temp.customers_global_view
    GROUP BY region
    ORDER BY customer_count DESC
""")
print("\nQuerying GLOBAL TEMP VIEW (via global_temp. prefix):")
result_global.show(truncate=False)

# -- Demonstrate GLOBAL temp view cross-access pattern --
# In another notebook on the SAME cluster, you would do:
print("\n>> In another notebook on this cluster, run:")
print('>>   spark.sql("SELECT * FROM global_temp.customers_global_view").show()')
print(">> This works because global temp views are registered in the global_temp database.")

# -- Show temp views exist in catalog --
print("\n--- Catalog listing ---")
print("Session temp views:")
spark.sql("SHOW TABLES").show(truncate=False)
print("Global temp views:")
spark.sql("SHOW TABLES IN global_temp").show(truncate=False)

# --- Clean up (demonstrating they are temporary) ---
spark.catalog.dropTempView("customers_temp_view")
print("\n[3] Dropped customers_temp_view")
print(">> Note: Both temp views and global temp views are lost when the cluster restarts.")
print(">> For permanent storage, use CREATE TABLE with a location (e.g., Delta table).")

# --- Difference summary ---
print("\n" + "=" * 60)
print("SCOPE COMPARISON")
print("=" * 60)
print("Temp View        → Current SparkSession only (this notebook)")
print("Global Temp View → All notebooks on same cluster (prefix: global_temp.)")
print("Permanent Table  → Persists across cluster restarts (data in storage)")
print("=" * 60)


 ---
 ## Concept 23 — Window Functions

 Using retail orders: rank products by revenue, running totals, LAG/LEAD for trend analysis.


In [ ]:
print("=" * 60)
print("CONCEPT 23: Window Functions")
print("=" * 60)

# --- Prepare data: explode items to get product-level rows ---
order_items = orders_df.alias("o") \
    .join(customers_df.alias("c"), "customer_id") \
    .select(
        col("o.order_id"),
        col("o.order_date"),
        col("o.status"),
        col("c.region"),
        explode(col("o.items")).alias("item")
    ) \
    .select(
        "order_id", "order_date", "status", "region",
        col("item.product_id"),
        col("item.quantity"),
        col("item.unit_price"),
        (col("item.quantity") * col("item.unit_price")).alias("line_total")
    ) \
    .join(products_df.select("product_id", "product_name", "category"), "product_id")

order_items.createOrReplaceTempView("order_items")
print("order_items view ready with product-level detail")

# === ROW_NUMBER, RANK, DENSE_RANK ===
print("\n--- ROW_NUMBER, RANK, DENSE_RANK: Top Products by Revenue per Region ---")

window_spec = Window.partitionBy("region").orderBy(col("region_revenue").desc())

df_product_rankings = order_items \
    .groupBy("region", "product_id", "product_name") \
    .agg(spark_sum("line_total").alias("region_revenue")) \
    .withColumn("row_num", row_number().over(window_spec)) \
    .withColumn("rank_val", rank().over(window_spec)) \
    .withColumn("dense_rank_val", dense_rank().over(window_spec))

print("\nTop 3 products per region (notice RANK vs DENSE_RANK on ties):")
df_product_rankings.filter(col("row_num") <= 3).orderBy("region", "row_num").show(30, truncate=False)

# === LAG and LEAD ===
print("\n--- LAG/LEAD: Month-over-Month Revenue Change ---")

monthly_rev = order_items \
    .withColumn("order_month", expr("date_format(order_date, 'yyyy-MM')")) \
    .groupBy("order_month") \
    .agg(spark_sum("line_total").alias("monthly_revenue")) \
    .orderBy("order_month")

monthly_window = Window.orderBy("order_month")

df_trends = monthly_rev \
    .withColumn("prev_month_rev", lag("monthly_revenue", 1).over(monthly_window)) \
    .withColumn("next_month_rev", lead("monthly_revenue", 1).over(monthly_window)) \
    .withColumn("mom_change_pct",
                spark_round((col("monthly_revenue") - col("prev_month_rev")) / col("prev_month_rev") * 100, 2))

df_trends.show(12, truncate=False)

# === Running Total with SUM OVER ===
print("\n--- Running Total of Revenue (cumulative over months) ---")

df_cumulative = monthly_rev \
    .withColumn("running_total",
                spark_sum("monthly_revenue").over(Window.orderBy("order_month").rowsBetween(Window.unboundedPreceding, 0))) \
    .withColumn("running_total_all",
                spark_sum("monthly_revenue").over(Window.orderBy("order_month").rangeBetween(Window.unboundedPreceding, Window.unboundedFollowing)))

df_cumulative.show(12, truncate=False)

# === SQL Window Function Equivalent ===
print("\n--- Same Window Function in Spark SQL ---")
spark.sql("""
    WITH ranked AS (
        SELECT
            region,
            product_name,
            SUM(line_total) AS region_revenue,
            ROW_NUMBER() OVER (PARTITION BY region ORDER BY SUM(line_total) DESC) AS rn,
            RANK() OVER (PARTITION BY region ORDER BY SUM(line_total) DESC) AS rnk,
            DENSE_RANK() OVER (PARTITION BY region ORDER BY SUM(line_total) DESC) AS drnk
        FROM order_items
        GROUP BY region, product_name
    )
    SELECT *
    FROM ranked
    WHERE rn <= 3
    ORDER BY region, rn
""").show(30, truncate=False)


 ---
 ## Concept 24 — Complex Types: ARRAY, STRUCT, MAP

 Nested data: explode arrays, access struct fields, query maps. Scenario: flattening API event logs.


In [ ]:
print("=" * 60)
print("CONCEPT 24: Complex Types — ARRAY, STRUCT, MAP")
print("=" * 60)

# === EXPLODE — flatten array columns ===
print("\n--- explode(): Flatten items array from orders ---")

df_exploded = orders_df \
    .select("order_id", "customer_id", explode("items").alias("item"))

df_exploded.printSchema()
df_exploded.show(8, truncate=False)

# === posexplode() — with position index ===
print("\n--- posexplode(): Explode with positional index ---")

df_pos = orders_df \
    .select("order_id", posexplode("items").alias("item_pos", "item_struct"))

df_pos.show(8, truncate=False)

# === STRUCT field access: dot notation vs bracket notation ===
print("\n--- STRUCT Field Access: Dot Notation vs Bracket Notation ---")

df_struct_access = events_df \
    .select(
        col("event_id"),
        col("payload.url").alias("page_url"),              # dot notation
        col("payload").user_agent.alias("ua_dot"),         # dot chain
        col("payload")["session_id"].alias("session_bracket"),  # bracket notation
        col("payload").extra_context.referrer.alias("referrer"),
        col("payload").extra_context["ab_test_group"].alias("ab_group")
    )

df_struct_access.show(8, truncate=False)

# === MAP key/value access ===
print("\n--- MAP Key/Value Access ---")

df_map_explore = orders_df \
    .select(
        "order_id",
        col("metadata").getField("channel").alias("sales_channel"),
        col("metadata")["coupon_code"].alias("coupon"),
        col("metadata")["is_gift"].alias("is_gift")
    )

df_map_explore.show(8, truncate=False)

# === Flatten deeply nested JSON from API log events ===
print("\n--- Fully Flattened Event Log ---")

df_flat_events = events_df \
    .select(
        "event_id",
        "timestamp",
        "event_type",
        col("payload.user_agent").alias("user_agent"),
        col("payload.url").alias("url"),
        col("payload.session_id").alias("session_id"),
        col("payload.extra_context.referrer").alias("referrer"),
        col("payload.extra_context.ab_test_group").alias("ab_test_group")
    )

df_flat_events.show(10, truncate=False)
print(f"\nFlattened events count: {df_flat_events.count()}")

# === Build nested structures dynamically ===
print("\n--- Building nested structures: from flat to nested ---")

df_nested_built = order_items \
    .groupBy("order_id", "order_date", "status", "region") \
    .agg(
        collect_list(struct("product_id", "product_name", "category", "quantity", "unit_price", "line_total")).alias("line_items"),
        spark_sum("line_total").alias("order_total")
    )

df_nested_built.printSchema()
df_nested_built.select("order_id", "order_total", "line_items").show(3, truncate=False)

# === Access MAP keys/values + explode MAP ===
print("\n--- Exploding MAP entries ---")
df_map_exploded = orders_df \
    .select("order_id", explode("metadata").alias("meta_key", "meta_value"))

df_map_exploded.show(10, truncate=False)


 ---
 ## Concept 25 — Higher-Order Functions

 Array functions: `transform()`, `filter()`, `exists()`, `aggregate()`.
 Benchmarked against explode+groupBy and UDF approaches.


In [ ]:
print("=" * 60)
print("CONCEPT 25: Higher-Order Functions")
print("=" * 60)

# === transform() — map over arrays ===
print("\n--- transform(): Uppercase all product tags ---")

df_tags_upper = products_df \
    .withColumn("tags_upper", transform(col("tags"), lambda x: expr("upper(x)"))) \
    .select("product_name", "tags", "tags_upper")

df_tags_upper.show(8, truncate=False)

# === filter() — filter elements within an array ===
print("\n--- filter(): Keep only 'sale' or 'clearance' tags ---")

df_promo_tags = products_df \
    .withColumn("promo_tags", spark_filter(col("tags"), lambda t: t.isin("sale", "clearance"))) \
    .select("product_name", "tags", "promo_tags")

df_promo_tags.show(8, truncate=False)

# === exists() — test if any element satisfies condition ===
print("\n--- exists(): Products with at least one promo tag ---")

df_has_promo = products_df \
    .withColumn("has_promo", exists(col("tags"), lambda t: t.isin("sale", "clearance"))) \
    .select("product_name", "tags", "has_promo")

df_has_promo.show(8, truncate=False)

# === aggregate() — reduce an array to a scalar ===
print("\n--- aggregate(): Create comma-separated tag string ---")

df_tags_csv = products_df \
    .withColumn("tags_csv",
                aggregate(
                    "tags",
                    lit(""),
                    lambda acc, x: expr("CASE WHEN acc = '' THEN x ELSE concat(acc, ', ', x) END"),
                    lambda acc: acc
                )) \
    .select("product_name", "tags", "tags_csv")

df_tags_csv.show(8, truncate=False)

# === Performance: Higher-Order Functions vs explode+groupBy ===
print("\n--- PERFORMANCE COMPARISON: HOF vs explode+groupBy ---")

# Create larger dataset for timing
large_products = products_df.crossJoin(spark.range(1000)).select(products_df["*"])
large_products.cache().count()

# Approach 1: explode + groupBy
start = time.time()
result_explode = large_products \
    .select(explode("tags").alias("tag")) \
    .groupBy("tag") \
    .agg(count("*").alias("tag_count"))
result_explode.count()  # force execution
time_explode = time.time() - start

# Approach 2: Higher-order function (not directly comparable, but shows pattern)
start = time.time()
result_hof = large_products \
    .withColumn("tag_count", size(col("tags")))
result_hof.count()  # force execution
time_hof = time.time() - start

print(f"explode+groupBy: {time_explode:.3f} seconds")
print(f"HOF (size):        {time_hof:.3f} seconds")
print(f"Speedup:           {time_explode / max(time_hof, 0.001):.1f}x")
print(">> Higher-order functions avoid shuffle by operating within each row. explode+groupBy requires shuffle for aggregation.")

# === Real pattern: Clean product tags ===
print("\n--- Real Pattern: Clean and Normalize Tags ---")

df_clean_tags = products_df \
    .withColumn("clean_tags",
                transform(
                    spark_filter(col("tags"), lambda t: t != "premium"),
                    lambda t: when(t == "new_arrival", lit("new")).otherwise(t)
                ))

df_clean_tags.select("product_name", "tags", "clean_tags").show(8, truncate=False)

# Compare with UDF approach (Concept 27 will time this more thoroughly)
print("\n>> Higher-order functions run inside the Tungsten engine (columnar, no serialization).")
print(">> UDFs require row→Python→row serialization (slower).")


 ---
 ## Concept 26 — CTEs, PIVOT & Subqueries

 - **CTEs (WITH clauses):** Multi-step queries
 - **PIVOT:** Wide-format reshaping (sales by month by region)
 - **Correlated vs Uncorrelated subqueries**


In [ ]:
print("=" * 60)
print("CONCEPT 26: CTEs, PIVOT & Subqueries")
print("=" * 60)

# === CTE (WITH clause) — Multi-step query ===
print("\n--- CTE: Multi-step customer order analysis ---")

cte_query = """
    WITH
    customer_orders AS (
        SELECT
            o.customer_id,
            c.name,
            c.region,
            COUNT(o.order_id) AS num_orders,
            SUM(o.total_amount) AS total_spent
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.status IN ('shipped', 'delivered')
        GROUP BY o.customer_id, c.name, c.region
    ),
    regional_stats AS (
        SELECT
            region,
            AVG(total_spent) AS avg_spent_per_customer,
            PERCENTILE_APPROX(total_spent, 0.5) AS median_spent_per_customer,
            COUNT(*) AS num_customers
        FROM customer_orders
        GROUP BY region
    ),
    top_customers AS (
        SELECT
            *,
            RANK() OVER (PARTITION BY region ORDER BY total_spent DESC) AS region_rank
        FROM customer_orders
    )
    SELECT
        tc.name,
        tc.region,
        tc.num_orders,
        tc.total_spent,
        tc.region_rank,
        rs.avg_spent_per_customer,
        rs.median_spent_per_customer
    FROM top_customers tc
    JOIN regional_stats rs ON tc.region = rs.region
    WHERE tc.region_rank <= 3
    ORDER BY tc.region, tc.region_rank
"""

spark.sql(cte_query).show(20, truncate=False)

# === PIVOT — Wide-format reshaping ===
print("\n--- PIVOT: Monthly Sales by Region (wide format) ---")

df_pivot = sales_df \
    .groupBy("month") \
    .pivot("region") \
    .agg(spark_sum("revenue").alias("")) \
    .orderBy("month") \
    .fillna(0)

df_pivot.show(13, truncate=False)
print(">> PIVOT transforms narrow (month, region, revenue) into wide (month, NA, Europe, APAC, ...)")

# === Spark SQL PIVOT ===
print("\n--- PIVOT in Spark SQL ---")
spark.sql("""
    SELECT *
    FROM (
        SELECT month, region, revenue
        FROM sales
    )
    PIVOT (
        SUM(revenue)
        FOR region IN (
            'North America', 'Europe', 'APAC', 'Latin America', 'Middle East'
        )
    )
    ORDER BY month
""").show(13, truncate=False)

# === Correlated vs Uncorrelated Subqueries ===
print("\n--- Uncorrelated Subquery: Orders above average ---")

spark.sql("""
    SELECT order_id, customer_id, total_amount, status
    FROM orders
    WHERE total_amount > (SELECT AVG(total_amount) FROM orders)
    ORDER BY total_amount DESC
""").show(10, truncate=False)

print("\n--- Correlated Subquery: Customers whose max order > 2x their own average ---")

spark.sql("""
    SELECT
        c.customer_id,
        c.name,
        c.region,
        MAX(o.total_amount) AS max_order,
        AVG(o.total_amount) AS avg_order,
        COUNT(o.order_id) AS num_orders
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.name, c.region
    HAVING MAX(o.total_amount) > 2 * AVG(o.total_amount)
    ORDER BY max_order DESC
""").show(10, truncate=False)

# === Physical plan for CTEs ===
print("\n--- Physical Plan for CTE Query ---")
print(spark.sql(cte_query)._jdf.queryExecution().optimizedPlan().treeString())


 ---
 ## Concept 27 — UDFs vs Native Functions

 Compare: **Native Spark SQL** vs **Python UDF** vs **Pandas UDF (vectorized)**
 with timing benchmarks on a medium-sized dataset.


In [ ]:
print("=" * 60)
print("CONCEPT 27: UDFs vs Native Functions")
print("=" * 60)

# --- Generate a larger order dataset for timing benchmarks ---
large_orders = orders_df.crossJoin(spark.range(500)).select(
    (col("order_id") * 10000 + col("id") % 10000).alias("order_id"),
    col("customer_id"),
    col("order_date"),
    col("total_amount"),
    col("status"),
    col("items"),
    col("metadata")
)
large_orders.cache()
n_rows = large_orders.count()
print(f"Benchmark dataset: {n_rows} rows")

# --- Business Logic: Customer Loyalty Tier ---
def loyalty_tier_native(total_amount, num_orders):
    if total_amount is None or num_orders is None:
        return None
    if total_amount > 5000 and num_orders > 10:
        return "Platinum"
    elif total_amount > 2000 or num_orders > 5:
        return "Gold"
    elif total_amount > 500:
        return "Silver"
    else:
        return "Bronze"

# --- 1. Native Spark SQL approach ---
print("\n[1] NATIVE SPARK SQL (columnar, optimized) ---")

start = time.time()

# Prepare: get customer-level aggregates
customer_agg = large_orders \
    .filter(col("status").isin("shipped", "delivered")) \
    .groupBy("customer_id") \
    .agg(
        spark_sum("total_amount").alias("total_spent"),
        count("order_id").alias("num_orders")
    )

result_native = customer_agg \
    .withColumn("loyalty_tier",
                when((col("total_spent") > 5000) & (col("num_orders") > 10), "Platinum")
                .when((col("total_spent") > 2000) | (col("num_orders") > 5), "Gold")
                .when(col("total_spent") > 500, "Silver")
                .otherwise("Bronze"))

result_native.count()  # force execution
time_native = time.time() - start
print(f"Native SQL time: {time_native:.3f} seconds")
result_native.groupBy("loyalty_tier").count().show()

# --- 2. Python UDF (row-by-row) ---
print("\n[2] PYTHON UDF (row-by-row serialization) ---")

loyalty_udf = udf(loyalty_tier_native, StringType())

start = time.time()
customer_agg2 = large_orders \
    .filter(col("status").isin("shipped", "delivered")) \
    .groupBy("customer_id") \
    .agg(
        spark_sum("total_amount").alias("total_spent"),
        count("order_id").alias("num_orders")
    )

result_udf = customer_agg2 \
    .withColumn("loyalty_tier", loyalty_udf(col("total_spent"), col("num_orders")))

result_udf.count()
time_udf = time.time() - start
print(f"Python UDF time:     {time_udf:.3f} seconds")

# --- 3. Pandas UDF (vectorized) ---
print("\n[3] PANDAS UDF (vectorized batch processing) ---")

@pandas_udf(StringType())
def loyalty_pandas_udf(total_spent_series, num_orders_series):
    import pandas as pd
    result = pd.Series(["Bronze"] * len(total_spent_series), dtype=str)
    mask_platinum = (total_spent_series > 5000) & (num_orders_series > 10)
    mask_gold = ((total_spent_series > 2000) | (num_orders_series > 5)) & ~mask_platinum
    mask_silver = (total_spent_series > 500) & ~mask_platinum & ~mask_gold
    result[mask_platinum] = "Platinum"
    result[mask_gold] = "Gold"
    result[mask_silver] = "Silver"
    return result

start = time.time()
customer_agg3 = large_orders \
    .filter(col("status").isin("shipped", "delivered")) \
    .groupBy("customer_id") \
    .agg(
        spark_sum("total_amount").alias("total_spent"),
        count("order_id").alias("num_orders")
    )

result_pandas = customer_agg3 \
    .withColumn("loyalty_tier", loyalty_pandas_udf(col("total_spent"), col("num_orders")))

result_pandas.count()
time_pandas = time.time() - start
print(f"Pandas UDF time:      {time_pandas:.3f} seconds")

# --- Summary ---
print("\n" + "=" * 60)
print("PERFORMANCE SUMMARY")
print("=" * 60)
print(f"Native Spark SQL   : {time_native:.3f}s  (baseline, runs in JVM, columnar)")
print(f"Python UDF         : {time_udf:.3f}s  (row-by-row Python serialization)")
print(f"Pandas UDF         : {time_pandas:.3f}s  (vectorized, batch Arrow transfer)")
print("=" * 60)
print(">> Rule: Always prefer native functions. Use Pandas UDFs when complex logic requires Python.")
print(">> Python UDFs are the slowest — only use for truly one-off transformations.")

# --- When UDFs are actually necessary ---
print("\n--- When UDFs are Actually Necessary ---")
print("Examples of unavoidable UDF use cases:")
print("  1. Calling external ML model inference (scikit-learn, custom models)")
print("  2. Complex NLP text processing (regex beyond built-in, tokenization)")
print("  3. Business rules engine with dozens of interdependent conditions")
print("  4. Calling external APIs or databases (not recommended at scale)")
print("  5. Custom hashing/encryption algorithms not available in Spark SQL")
print(">> In all other cases, refactor logic to native Spark SQL functions.")


 ---
 ## Concept 28 — VARIANT Type & Semi-Structured Data

 **Note:** `VARIANT` type may not be available in Databricks Community Edition (requires DBR 15.3+ / Unity Catalog).

 - If available: demonstrates native VARIANT ingestion, path-based querying, schema-on-read
 - If not available: shows equivalent approaches using STRUCT and JSON string parsing


In [ ]:
print("=" * 60)
print("CONCEPT 28: VARIANT Type & Semi-Structured Data")
print("=" * 60)

# --- Check if VARIANT is available ---
try:
    spark.sql("SELECT CAST('{\"a\": 1}' AS VARIANT)")
    variant_available = True
except Exception:
    variant_available = False

print(f"VARIANT type available: {variant_available}")
print("")

if variant_available:
    # VARIANT IS AVAILABLE — demonstrate directly
    print(">>> DEMONSTRATING NATIVE VARIANT TYPE <<<")
    print("")

    # Create JSON data as VARIANT
    variant_demo = spark.sql("""
        SELECT
            CAST('{"user_id": 1001, "action": "click", "properties": {"color": "red", "size": "M", "tags": ["sale","new"]}}' AS VARIANT) AS payload
        UNION ALL
        SELECT
            CAST('{"user_id": 1002, "action": "purchase", "properties": {"color": "blue", "size": "L", "tags": ["clearance"]}}' AS VARIANT)
    """)

    variant_demo.createOrReplaceTempView("variant_demo")
    print("--- VARIANT data ---")
    variant_demo.show(truncate=False)
    variant_demo.printSchema()

    # Query with path syntax
    print("\n--- VARIANT path-based querying ---")
    spark.sql("""
        SELECT
            payload:user_id::INT AS user_id,
            payload:action::STRING AS action,
            payload:properties.color::STRING AS color,
            payload:properties.size::STRING AS size,
            payload:properties.tags[0]::STRING AS primary_tag
        FROM variant_demo
    """).show(truncate=False)

    # Schema-on-read benefits
    print("\n--- Schema-on-read: evolving schemas without DDL ---")
    spark.sql("""
        WITH mixed AS (
            SELECT CAST('{"v": 1, "version": "1.0", "name": "alpha"}' AS VARIANT) AS data
            UNION ALL
            SELECT CAST('{"v": 2, "version": "2.0", "name": "beta", "new_field": [10,20]}' AS VARIANT)
        )
        SELECT
            data:v::INT AS v,
            data:version::STRING AS version,
            data:name::STRING AS name,
            data:new_field::STRING AS new_field
        FROM mixed
    """).show(truncate=False)
    print(">> VARIANT stores semi-structured data in a native binary format (.variant)")
    print(">> No upfront schema required — 'schema-on-read' using path expressions")

else:
    # VARIANT NOT AVAILABLE — show equivalent approaches
    print(">>> VARIANT NOT AVAILABLE — Showing equivalent approaches <<<")
    print("")
    print("--- What is VARIANT? ---")
    print("VARIANT is a native semi-structured binary type that:")
    print("  • Stores JSON/XML/Parquet data without requiring a predefined schema")
    print("  • Uses schema-on-read: define structure only when querying")
    print("  • Enables path-based access: data:field1:subfield2::INT")
    print("  • Stores data in an optimized binary format (faster than JSON strings)")
    print("  • Benefits: flexible ingestion, evolving schemas, simpler ETL")
    print("")

    # Approach 1: STRUCT (strongly typed, full schema upfront)
    print("--- Approach 1: STRUCT Type (schema-at-write) ---")

    from pyspark.sql.types import StructType as ST, StructField as SF, FloatType

    json_events = spark.createDataFrame([
        ("e1", '{"user_id": 101, "event": "click", "attr": {"page": "home", "duration_ms": 250}}'),
        ("e2", '{"user_id": 102, "event": "purchase", "attr": {"page": "checkout", "amount": 49.99}}'),
        ("e3", '{"user_id": 103, "event": "search", "attr": {"page": "search_results", "query": "laptops"}}')
    ], ["event_id", "raw_json"])

    json_events.createOrReplaceTempView("json_events")

    # Parse JSON into STRUCT — requires known schema
    parsed_struct = json_events \
        .withColumn("parsed",
                    expr("from_json(raw_json, 'user_id INT, event STRING, attr STRUCT<page: STRING, duration_ms: INT, amount: DOUBLE, query: STRING>')")) \
        .select("event_id", "parsed.user_id", "parsed.event", "parsed.attr.*")

    print("Parsed via STRUCT (schema-at-write):")
    parsed_struct.show(truncate=False)
    print(">> STRUCT requires the full schema upfront — schema-at-write")

    # Approach 2: JSON string parsing (schema-on-read)
    print("\n--- Approach 2: JSON String Parsing (schema-on-read, like VARIANT) ---")

    df_json_path = spark.sql("""
        SELECT
            event_id,
            get_json_object(raw_json, '$.user_id') AS user_id,
            get_json_object(raw_json, '$.event') AS event,
            get_json_object(raw_json, '$.attr.page') AS page,
            get_json_object(raw_json, '$.attr.duration_ms') AS duration_ms,
            get_json_object(raw_json, '$.attr.amount') AS amount,
            get_json_object(raw_json, '$.attr.query') AS query
        FROM json_events
    """)

    df_json_path.show(truncate=False)
    print(">> get_json_object() enables schema-on-read — extract only what you need")
    print(">> However, raw JSON strings are NOT optimized (no binary format, no statistics)")

    # Approach 3: Using json_tuple for multiple fields
    print("\n--- Approach 3: json_tuple (faster for multiple fields) ---")

    df_tuple = json_events \
        .select(
            "event_id",
            expr("json_tuple(raw_json, 'user_id', 'event')").alias("uid", "evt")
        )

    df_tuple.show(truncate=False)

    # Comparison
    print("\n--- VARIANT vs Alternatives Comparison ---")
    print("┌──────────────────┬──────────────────┬──────────────────┐")
    print("│ Feature          │ VARIANT          │ STRING(JSON)     │")
    print("├──────────────────┼──────────────────┼──────────────────┤")
    print("│ Schema           │ Schema-on-read   │ Schema-on-read   │")
    print("│ Storage          │ Native binary    │ Plain text       │")
    print("│ Query speed      │ Fast (optimized) │ Slower (parsing) │")
    print("│ Statistics       │ Yes              │ No               │")
    print("│ Path syntax      │ col:field::TYPE  │ get_json_object  │")
    print("│ Evolving schema  │ Trivial          │ Manual           │")
    print("└──────────────────┴──────────────────┴──────────────────┘")


 ---
 ## Concept 29 — MERGE INTO & Upsert Patterns

 Full MERGE syntax with Delta tables. Upsert customer dimension table with MATCHED update/delete and NOT MATCHED insert.


In [ ]:
print("=" * 60)
print("CONCEPT 29: MERGE INTO & Upsert Patterns")
print("=" * 60)

# --- Create Delta tables for the MERGE demo ---
delta_output_path = "/tmp/retailco_merge_demo"

# Clean up any prior runs
dbutils.fs.rm(delta_output_path, True)
print("Cleaned up prior output location")

# --- Step 1: Create the target dimension table (customer_dim) ---
print("\n[1] Creating TARGET: customer_dim (existing customer records)")

target_df = customers_df \
    .select(
        "customer_id", "name", "email", "region", "signup_date"
    ) \
    .withColumn("loyalty_tier", lit(None).cast(StringType())) \
    .withColumn("last_order_date", lit(None).cast(DateType())) \
    .withColumn("is_active", lit(True)) \
    .withColumn("updated_at", current_timestamp())

target_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_dim")

print("customer_dim table created")
spark.table("customer_dim").select("customer_id", "name", "region", "loyalty_tier", "is_active").show(5)

# --- Step 2: Create the source table (staged updates) ---
print("\n[2] Creating SOURCE: customer_updates (staging table with changes)")

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType

# Updates: change region for customers 1-3, set loyalty tier for 5-8, new customers 21-23
update_data = [
    # MATCHED — UPDATE: change region
    (1, "customer_1", "cust1@newdomain.com", "Europe", "Gold", True),
    (2, "customer_2", "cust2@example.com", "APAC", "Silver", True),
    (3, "customer_3", "cust3@example.com", "Middle East", "Platinum", True),

    # MATCHED — UPDATE: add loyalty tier to existing
    (5, "customer_5", "cust5@example.com", "North America", "Gold", True),
    (6, "customer_6", "cust6@example.com", "Europe", "Gold", True),
    (7, "customer_7", "cust7@example.com", "APAC", "Silver", True),
    (8, "customer_8", "cust8@example.com", "Latin America", "Bronze", True),

    # MATCHED — DELETE: mark as inactive
    (14, "customer_14", "cust14@example.com", "North America", "Bronze", False),
    (15, "customer_15", "cust15@example.com", "Europe", "Bronze", False),

    # NOT MATCHED — INSERT: new customers
    (21, "customer_21", "cust21@example.com", "APAC", "Silver", True),
    (22, "customer_22", "cust22@example.com", "Latin America", "Gold", True),
    (23, "customer_23", "cust23@example.com", "European Union", "Platinum", True),
]

source_df = spark.createDataFrame(
    update_data,
    schema=["customer_id", "name", "email", "region", "loyalty_tier", "is_active"]
)
source_df.createOrReplaceTempView("customer_updates")
print("customer_updates staging table:")
source_df.show(truncate=False)

# --- Step 3: Full MERGE INTO ---
print("\n[3] Executing MERGE INTO (full upsert) ---")

merge_sql = """
    MERGE INTO customer_dim AS target
    USING customer_updates AS source
    ON target.customer_id = source.customer_id
    WHEN MATCHED AND source.is_active = false THEN
        UPDATE SET
            is_active = false,
            updated_at = current_timestamp(),
            loyalty_tier = source.loyalty_tier
    WHEN MATCHED THEN
        UPDATE SET
            name = source.name,
            email = source.email,
            region = source.region,
            loyalty_tier = source.loyalty_tier,
            is_active = source.is_active,
            updated_at = current_timestamp()
    WHEN NOT MATCHED THEN
        INSERT (
            customer_id, name, email, region, signup_date,
            loyalty_tier, last_order_date, is_active, updated_at
        )
        VALUES (
            source.customer_id, source.name, source.email, source.region, current_date(),
            source.loyalty_tier, NULL, source.is_active, current_timestamp()
        )
"""

spark.sql(merge_sql)
print("MERGE INTO completed")

# --- Step 4: Verify results ---
print("\n[4] Verification: Updated / Inserted / Deactivated records ---")

print("\n--- All customers (note changed regions, loyalty tiers, inactive flags) ---")
spark.sql("SELECT customer_id, name, region, loyalty_tier, is_active FROM customer_dim ORDER BY customer_id").show(25, truncate=False)

print("\n--- Newly inserted customers (21-23) ---")
spark.sql("SELECT * FROM customer_dim WHERE customer_id >= 21").show(truncate=False)

print("\n--- Deactivated customers (14-15) ---")
spark.sql("SELECT customer_id, name, is_active, updated_at FROM customer_dim WHERE NOT is_active").show(truncate=False)

print("\n--- Updated customers (1-3: region changed; 5-8: loyalty tier set) ---")
spark.sql("SELECT customer_id, name, region, loyalty_tier, updated_at FROM customer_dim WHERE customer_id IN (1,2,3,5,6,7,8)").show(truncate=False)

# --- Step 5: MERGE clause ordering & write amplification ---
print("\n--- MERGE INTO: Clause Ordering Requirements ---")
print(">> WHEN MATCHED clauses are evaluated in order. First match wins.")
print(">> WHEN MATCHED AND <condition> must come before general WHEN MATCHED.")
print(">> WHEN NOT MATCHED must come after all WHEN MATCHED clauses.")
print("")
print("--- Write Amplification ---")
print(">> MERGE rewrites entire files that contain modified rows (not individual rows).")
print(">> If your table has 100 files and MERGE touches rows in 12 files, only those 12 are rewritten.")
print(">> Delta Lake uses data skipping and file-level statistics to minimize rewrites.")
print(">> OPTIMIZE: Use ZORDER on merge key (customer_id) to co-locate related data in fewer files.")
print(">>           spark.sql('OPTIMIZE customer_dim ZORDER BY (customer_id)')")

# --- Step 6: MERGE with key column optimization ---
print("\n[5] OPTIMIZE demonstration ---")
try:
    spark.sql("OPTIMIZE customer_dim ZORDER BY (customer_id)")
    print("OPTIMIZE successful — data co-located by customer_id for faster MERGE operations.")
except Exception as e:
    print(f"OPTIMIZE not available in this environment: {e}")


 ---
 ## Concept 30 — Table Constraints & Generated Columns

 - **CHECK constraints:** Validate data on write
 - **NOT NULL constraints:** Enforce required columns
 - **PRIMARY KEY:** Informational only (not enforced by Delta)
 - **Generated columns:** Auto-computed from expressions


In [ ]:
print("=" * 60)
print("CONCEPT 30: Table Constraints & Generated Columns")
print("=" * 60)

delta_path = "/tmp/retailco_constraints_demo/"
dbutils.fs.rm(delta_path, True)

# ==========================================================================
# 1. CREATE TABLE with CHECK, NOT NULL, and Generated Columns
# ==========================================================================
print("\n[1] Creating product_catalog with constraints ---")

spark.sql("""
    CREATE OR REPLACE TABLE product_catalog (
        product_id         INT NOT NULL,
        product_name       STRING NOT NULL,
        category           STRING NOT NULL,
        base_price         DECIMAL(10, 2) NOT NULL,
        discount_pct       DECIMAL(5, 2) DEFAULT 0.00,
        final_price        DECIMAL(10, 2)
            GENERATED ALWAYS AS (base_price * (1.0 - discount_pct / 100.0)),
        stock_quantity     INT NOT NULL,
        sku                STRING NOT NULL,
        is_active          BOOLEAN DEFAULT true,
        created_at         TIMESTAMP DEFAULT current_timestamp(),

        CONSTRAINT positive_base_price CHECK (base_price > 0),
        CONSTRAINT valid_discount_pct CHECK (discount_pct >= 0 AND discount_pct <= 100),
        CONSTRAINT positive_stock CHECK (stock_quantity >= 0),
        CONSTRAINT valid_sku_prefix CHECK (sku LIKE 'SKU-%')
    )
    USING delta
    LOCATION '/tmp/retailco_constraints_demo/product_catalog'
""")
print("product_catalog table created with constraints")

# Show table schema
print("\nTable DDL:")
spark.sql("DESCRIBE EXTENDED product_catalog").show(50, truncate=False)

# ==========================================================================
# 2. VALID INSERTS
# ==========================================================================
print("\n[2] Inserting VALID data ---")

spark.sql("""
    INSERT INTO product_catalog
        (product_id, product_name, category, base_price, discount_pct, stock_quantity, sku)
    VALUES
        (1, 'Wireless Mouse',      'Electronics', 29.99,  10.00, 150, 'SKU-ELEC-001'),
        (2, 'USB-C Cable',         'Electronics', 12.50,   0.00, 500, 'SKU-ELEC-002'),
        (3, 'Running Shoes',       'Sports',      89.99,  15.00,  75, 'SKU-SPRT-001'),
        (4, 'Yoga Mat',            'Sports',      24.99,   5.00, 200, 'SKU-SPRT-002'),
        (5, 'Desk Lamp',           'Home',        39.99,  20.00,  60, 'SKU-HOME-001')
""")

print("Valid data inserted successfully.")
spark.table("product_catalog").show(truncate=False)

# Show generated columns in action
print("\nGenerated column `final_price` computed automatically:")
spark.sql("""
    SELECT product_id, product_name, base_price, discount_pct, final_price,
           base_price * (1.0 - discount_pct / 100.0) AS manual_calc
    FROM product_catalog
""").show(truncate=False)

# ==========================================================================
# 3. CONSTRAINT VIOLATIONS — each should fail
# ==========================================================================
print("\n[3] CONSTRAINT VIOLATIONS — each INSERT should FAIL ---")

# 3a: CHECK violation — negative base_price
print("\n--- Test: Negative base_price (CHECK violation) ---")
try:
    spark.sql("""
        INSERT INTO product_catalog
            (product_id, product_name, category, base_price, stock_quantity, sku)
        VALUES (6, 'Bad Product', 'Electronics', -10.00, 10, 'SKU-ELEC-003')
    """)
    print("ERROR: Should have been rejected!")
except Exception as e:
    print(f"REJECTED (expected): {str(e)[:120]}...")

# 3b: CHECK violation — discount > 100%
print("\n--- Test: discount_pct > 100 (CHECK violation) ---")
try:
    spark.sql("""
        INSERT INTO product_catalog
            (product_id, product_name, category, base_price, discount_pct, stock_quantity, sku)
        VALUES (7, 'Another Bad Product', 'Home', 50.00, 150.00, 20, 'SKU-HOME-002')
    """)
    print("ERROR: Should have been rejected!")
except Exception as e:
    print(f"REJECTED (expected): {str(e)[:120]}...")

# 3c: NOT NULL violation
print("\n--- Test: NULL product_name (NOT NULL violation) ---")
try:
    spark.sql("""
        INSERT INTO product_catalog
            (product_id, product_name, category, base_price, stock_quantity, sku)
        VALUES (8, NULL, 'Books', 15.00, 10, 'SKU-BOOK-001')
    """)
    print("ERROR: Should have been rejected!")
except Exception as e:
    print(f"REJECTED (expected): {str(e)[:120]}...")

# 3d: CHECK violation — invalid SKU prefix
print("\n--- Test: Invalid SKU prefix (CHECK violation) ---")
try:
    spark.sql("""
        INSERT INTO product_catalog
            (product_id, product_name, category, base_price, stock_quantity, sku)
        VALUES (9, 'Valid Product Bad SKU', 'Toys', 19.99, 30, 'ABC-TOYS-001')
    """)
    print("ERROR: Should have been rejected!")
except Exception as e:
    print(f"REJECTED (expected): {str(e)[:120]}...")

# 3e: CHECK violation — negative stock
print("\n--- Test: Negative stock (CHECK violation) ---")
try:
    spark.sql("""
        INSERT INTO product_catalog
            (product_id, product_name, category, base_price, stock_quantity, sku)
        VALUES (10, 'Negative Stock', 'Food', 5.00, -5, 'SKU-FOOD-001')
    """)
    print("ERROR: Should have been rejected!")
except Exception as e:
    print(f"REJECTED (expected): {str(e)[:120]}...")

# ==========================================================================
# 4. PRIMARY KEY (informational, NOT enforced in Delta)
# ==========================================================================
print("\n[4] PRIMARY KEY — Informational Only ---")
print(">> Delta Lake PRIMARY KEY is informational (metadata only). It is NOT enforced.")
print(">> Demonstration: Inserting a duplicate product_id succeeds (no error).")

try:
    spark.sql("""
        INSERT INTO product_catalog
            (product_id, product_name, category, base_price, stock_quantity, sku)
        VALUES (1, 'Duplicate Mouse', 'Electronics', 19.99, 50, 'SKU-ELEC-999')
    """)
    print("INSERT SUCCEEDED — PRIMARY KEY is not enforced. Duplicate product_id=1 exists.")
    print(">> Use MERGE INTO with a unique key to enforce uniqueness manually.")
    spark.table("product_catalog").show(truncate=False)
except Exception as e:
    print(f"Unexpected rejection: {e}")

# ==========================================================================
# 5. Show all constraints
# ==========================================================================
print("\n[5] Listing Constraints ---")
try:
    spark.sql("SHOW TBLPROPERTIES product_catalog").filter(
        col("key").contains("constraint")
    ).show(truncate=False)
except Exception:
    print("SHOW TBLPROPERTIES not supported — constraints visible in DESCRIBE EXTENDED above.")

# ==========================================================================
# 6. Generated Columns with complex expressions
# ==========================================================================
print("\n[6] Generated Column: complex expression ---")

spark.sql("""
    CREATE OR REPLACE TABLE orders_with_metrics (
        order_id      INT NOT NULL,
        subtotal      DECIMAL(10, 2) NOT NULL,
        tax_rate      DECIMAL(5, 4) DEFAULT 0.0800,
        shipping      DECIMAL(10, 2) DEFAULT 0.00,
        discount      DECIMAL(10, 2) DEFAULT 0.00,

        tax_amount    DECIMAL(10, 2)
            GENERATED ALWAYS AS (subtotal * tax_rate),
        grand_total   DECIMAL(10, 2)
            GENERATED ALWAYS AS (subtotal + subtotal * tax_rate + shipping - discount),
        is_free_ship  BOOLEAN
            GENERATED ALWAYS AS (shipping = 0.00),

        CONSTRAINT positive_subtotal CHECK (subtotal > 0),
        CONSTRAINT reasonable_tax CHECK (tax_rate BETWEEN 0 AND 0.25)
    )
    USING delta
    LOCATION '/tmp/retailco_constraints_demo/orders_with_metrics'
""")

spark.sql("""
    INSERT INTO orders_with_metrics (order_id, subtotal, tax_rate, shipping, discount)
    VALUES
        (1, 100.00, 0.08, 5.99, 10.00),
        (2, 250.00, 0.10, 0.00, 25.00),
        (3, 50.00,  0.08, 12.99, 0.00)
""")

print("Generated columns in action:")
spark.table("orders_with_metrics").show(truncate=False)
print(">> tax_amount, grand_total, and is_free_ship are auto-computed from source columns.")
print(">> No INSERT value needed for generated columns — they are always computed.")

# ==========================================================================
# Summary
# ==========================================================================
print("\n" + "=" * 60)
print("CONSTRAINT SUMMARY")
print("=" * 60)
print("CHECK          → Enforced on write (validates row values)")
print("NOT NULL       → Enforced on write (rejects NULL values)")
print("PRIMARY KEY    → Informational only (not enforced by Delta)")
print("FOREIGN KEY    → Not supported in Delta Lake")
print("GENERATED      → Always computed from expression (read-only)")
print("DEFAULT        → Value used when column omitted from INSERT")
print("=" * 60)


 ---
 ## Summary & Self-Assessment Checklist

 ### Concepts Covered (#21–#30)

 | # | Concept | Key Takeaways |
 |---|---------|---------------|
 | 21 | DataFrame API Core | `select`, `filter`, `withColumn`, `groupBy`, `agg`, `join` — method chaining; equivalent physical plan to Spark SQL |
 | 22 | Temp/Global Temp Views | Temp = session scope; Global Temp = cross-notebook via `global_temp.`; neither persists across cluster restart |
 | 23 | Window Functions | `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `LAG`, `LEAD`, running totals with `SUM OVER` and frame clauses |
 | 24 | Complex Types | `ARRAY` → `explode`/`posexplode`; `STRUCT` → dot & bracket access; `MAP` → key/value access, flatten nested JSON |
 | 25 | Higher-Order Functions | `transform`, `filter`, `exists`, `aggregate` — avoid shuffle vs explode+groupBy; faster than UDFs |
 | 26 | CTEs, PIVOT, Subqueries | `WITH` clauses for multi-step queries; `PIVOT` for wide-format; correlated vs uncorrelated subqueries |
 | 27 | UDFs vs Native | Native (JVM, columnar) > Pandas UDF (vectorized Arrow) > Python UDF (row-by-row); use native when possible |
 | 28 | VARIANT Type | Native binary semi-structured format; schema-on-read; path-based access; alternatives: STRUCT, JSON string parsing |
 | 29 | MERGE INTO | Full upsert: `WHEN MATCHED UPDATE/DELETE`, `WHEN NOT MATCHED INSERT`; clause ordering matters; write amplification |
 | 30 | Constraints & Generated | `CHECK`, `NOT NULL` enforced on write; `PRIMARY KEY` informational only; generated columns auto-computed |

 ### Self-Assessment Questions

 - [ ] Can you write a chained DataFrame transformation with `select`, `filter`, `withColumn`, `groupBy`, and `agg`?
 - [ ] Can you explain when to use a Global Temp View over a Temp View?
 - [ ] Can you use `ROW_NUMBER`, `RANK`, and `DENSE_RANK` with window partitions?
 - [ ] Can you flatten a `STRUCT` field deep inside an `ARRAY<MAP<STRING, STRUCT>>` ?
 - [ ] Can you use `transform()` and `filter()` on an array column to clean data without exploding?
 - [ ] Can you write a CTE with three chained steps?
 - [ ] Can you compare native Spark, Python UDF, and Pandas UDF performance?
 - [ ] Can you explain VARIANT vs JSON string vs STRUCT for semi-structured data?
 - [ ] Can you write a full MERGE INTO with WHEN MATCHED update/delete and WHEN NOT MATCHED insert?
 - [ ] Can you create a table with CHECK, NOT NULL, DEFAULT, and GENERATED ALWAYS AS constraints?

 ---
 *End of Notebook — Databricks SQL & DataFrames Concepts #21–#30*


---

# 04 Data Ingestion



 # 04 — Data Ingestion: Concepts 31–40

 **Objective:** Master data ingestion patterns in the Databricks Lakehouse — from batch file loads through incremental streaming, medallion architecture, idempotent pipelines, and slowly changing dimensions.

 **Scope:** Concepts 31 through 40 of the Databricks Data Engineer learning path.

 **Target Platform:** Databricks Community Edition (single‑node, Unity Catalog not available). Where Community Edition lacks a feature, we explain the theory and provide a manual work‑around.


 ## Environment Setup

 Create the synthetic datasets and helper functions used throughout the notebook.


In [ ]:
import os
import time
import shutil
import json
import csv
import uuid
from datetime import datetime, date, timedelta
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("04_Data_Ingestion") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

spark.sql("CREATE DATABASE IF NOT EXISTS ingestion_demo")
spark.sql("USE ingestion_demo")

print(f"Spark version : {spark.version}")
print(f"Database      : {spark.catalog.currentDatabase()}")
print("Ready.")


 ## Utility — Generate Synthetic Retail Data


In [ ]:
def generate_sales_csv(output_dir, num_files=3, rows_per_file=100):
    """Generate synthetic retail sales CSVs with a realistic schema."""
    from random import randint, choice, uniform, seed
    seed(42)
    categories = ["Electronics", "Clothing", "Home", "Books", "Sports"]
    regions    = ["North", "South", "East", "West"]
    statuses   = ["COMPLETED", "PENDING", "CANCELLED", "RETURNED"]

    os.makedirs(output_dir, exist_ok=True)
    files_created = []

    for f in range(num_files):
        file_path = os.path.join(output_dir, f"sales_{f:04d}.csv")
        with open(file_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["transaction_id","transaction_date","product_name","category",
                             "quantity","unit_price","total_amount","region","status"])
            for i in range(rows_per_file):
                qty    = randint(1, 5)
                price  = round(uniform(5.0, 500.0), 2)
                writer.writerow([
                    str(uuid.uuid4()),
                    (date.today() - timedelta(days=randint(0, 60))).isoformat(),
                    f"product_{randint(1, 50)}",
                    choice(categories),
                    qty,
                    price,
                    round(qty * price, 2),
                    choice(regions),
                    choice(statuses)
                ])
        files_created.append(file_path)

    return files_created


def generate_customer_csv(output_dir, num_files=1, rows_per_file=50):
    """Generate synthetic customer dimension data (for SCD demos)."""
    from random import randint, choice, seed
    seed(123)
    cities  = ["New York","Los Angeles","Chicago","Houston","Phoenix"]
    states  = ["NY","CA","IL","TX","AZ"]
    streets = ["Main St","Oak Ave","Elm Rd","Pine Blvd","Maple Dr"]

    os.makedirs(output_dir, exist_ok=True)
    files = []
    for f in range(num_files):
        fp = os.path.join(output_dir, f"customers_{f:04d}.csv")
        with open(fp, "w", newline="") as fh:
            w = csv.writer(fh)
            w.writerow(["customer_id","first_name","last_name","email","phone","city","state","street_address","zip_code"])
            for i in range(rows_per_file):
                idx = randint(0, len(cities)-1)
                cust_id = f"CUST-{randint(1000, 9999)}"
                w.writerow([
                    cust_id,
                    f"first_{randint(1,99)}",
                    f"last_{randint(1,99)}",
                    f"user{randint(1,999)}@example.com",
                    f"555-{randint(1000,9999)}",
                    cities[idx],
                    states[idx],
                    f"{randint(1,999)} {choice(streets)}",
                    f"{randint(10000,99999)}"
                ])
        files.append(fp)
    return files


def upload_dir_to_dbfs(local_dir, dbfs_dir):
    """Copy a local directory into DBFS."""
    dbutils.fs.mkdirs(dbfs_dir)
    for fname in os.listdir(local_dir):
        local_path  = os.path.join(local_dir, fname)
        dbfs_path   = f"{dbfs_dir}/{fname}"
        dbutils.fs.cp(f"file://{local_path}", dbfs_path)


 ## Generate Datasets & Stage in DBFS


In [ ]:
# Generate in local /tmp then copy to DBFS
SALES_IN  = "/tmp/sales_input"
CUSTOMER_IN = "/tmp/customer_input"

generate_sales_csv(SALES_IN, num_files=5, rows_per_file=150)
generate_customer_csv(CUSTOMER_IN, num_files=1, rows_per_file=50)

# Upload into DBFS FileStore (Community Edition safe)
DBFS_SALES     = "dbfs:/FileStore/ingestion_demo/sales"
DBFS_CUSTOMERS = "dbfs:/FileStore/ingestion_demo/customers"
dbutils.fs.rm(DBFS_SALES,         recurse=True)
dbutils.fs.rm(DBFS_CUSTOMERS,     recurse=True)

upload_dir_to_dbfs(SALES_IN,  DBFS_SALES)
upload_dir_to_dbfs(CUSTOMER_IN, DBFS_CUSTOMERS)

print("Files in DBFS sales directory:")
display(dbutils.fs.ls(DBFS_SALES))


 ---
 ## Concept 31 — COPY INTO vs. Auto Loader

 **COPY INTO** — one‑shot batch ingestion, great for ad‑hoc or scheduled loads.
 **Auto Loader** — continuous / incremental ingestion with schema evolution and exactly‑once guarantees.

 | Feature             | COPY INTO                    | Auto Loader                         |
 |---------------------|------------------------------|-------------------------------------|
 | Execution model     | SQL statement (batch)        | Structured Streaming source         |
 | Idempotency         | Built‑in (metadata tracking) | Checkpoint + directory listing      |
 | Schema evolution    | Manual                       | Automatic (cloudFiles options)      |
 | Rescued data        | No                           | Yes                                 |
 | Community Edition   | Yes                          | Limited (directory listing works)   |

 ---


 ### 31‑A  COPY INTO — Idempotent Batch Load


In [ ]:
spark.sql(f"""
  COPY INTO ingestion_demo.sales_raw_bronze
  FROM '{DBFS_SALES}'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
  COPY_OPTIONS ('mergeSchema' = 'true')
""")

# Run it again — COPY INTO tracks loaded files, so zero new rows are ingested
spark.sql(f"""
  COPY INTO ingestion_demo.sales_raw_bronze
  FROM '{DBFS_SALES}'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
  COPY_OPTIONS ('mergeSchema' = 'true')
""")

print("Row count after two COPY INTO runs (should be same — no duplicates):")
display(spark.table("ingestion_demo.sales_raw_bronze").count())


 **COPY INTO idempotency explained:**

 The command stores a list of already‑ingested file names in `_metadata`.  When re‑run it skips previously loaded files automatically.  This is the simplest way to achieve exactly‑once ingestion for batch workloads.


 ### 31‑B  Auto Loader — Conceptual Walk‑through

 Auto Loader (`cloudFiles` source) is a *Structured Streaming* source that continuously discovers new files.  It has two modes:

 1. **File notification mode** — relies on cloud queue services (SQS, Event Grid).  Lowest latency but requires cloud setup.  **Not available in Community Edition.**
 2. **Directory listing mode** — periodically lists the directory and processes new files.  Simpler, no external services needed.  Works in Community Edition for demonstration.

 > **Community Edition note:** Directory listing Auto Loader *does* work, but CE’s single‑node constraint limits true streaming.  We show the code pattern and then provide a manual batch equivalent.


In [ ]:
# ---- Auto Loader (directory listing mode — CE compatible) ----
# Note: run this in a separate cell. CE single-node limits parallelism.

def auto_loader_demo_delta():
    spark.sql("DROP TABLE IF EXISTS ingestion_demo.autoloader_demo")
    checkpoint = "dbfs:/FileStore/ingestion_demo/_checkpoints/autoloader"
    dbutils.fs.rm(checkpoint, recurse=True)

    stream = (spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("cloudFiles.schemaLocation", checkpoint)
        .option("cloudFiles.includeExistingFiles", "true")
        .load(DBFS_SALES)
        .writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable("ingestion_demo.autoloader_demo")
    )
    stream.awaitTermination()

try:
    auto_loader_demo_delta()
    print("Auto Loader (availableNow) completed.")
    display(spark.table("ingestion_demo.autoloader_demo").count())
except Exception as e:
    print(f"Auto Loader not available in this environment: {e}")
    print("Falling back to manual batch incremental (see below).")


 ### 31‑C  Manual Incremental Batch (Community Edition alternative)


In [ ]:
def manual_incremental_load(data_dir, target_table, ingested_files_table, checkpoint_dir):
    """
    Manual equivalent of Auto Loader:
    1. List all files in `data_dir`
    2. Subtract files already recorded in `ingested_files_table`
    3. Read only new files with Spark
    4. Append to `target_table`
    5. Record newly ingested files in `ingested_files_table`
    """
    spark.sql(f"CREATE TABLE IF NOT EXISTS {ingested_files_table} "
              "(path STRING, ingested_at TIMESTAMP) USING delta")
    
    # Files on disk
    all_files = set(f.path for f in dbutils.fs.ls(data_dir) if f.name.endswith(".csv"))
    
    # Already ingested
    try:
        already = set(r.path for r in spark.table(ingested_files_table).select("path").collect())
    except Exception:
        already = set()
    
    new_files = sorted(all_files - already)
    if not new_files:
        print("No new files to ingest.")
        return 0

    print(f"Ingesting {len(new_files)} new file(s): {[f.split('/')[-1] for f in new_files]}")

    df = spark.read.option("header", "true").option("inferSchema", "true").csv(new_files)
    df.write.mode("append").format("delta").saveAsTable(target_table)

    # Record ingestions
    now = datetime.now().isoformat()
    ingest_df = spark.createDataFrame([(f, now) for f in new_files], "path STRING, ingested_at TIMESTAMP")
    ingest_df.write.mode("append").format("delta").saveAsTable(ingested_files_table)

    return df.count()

# First run — ingests all files
cnt = manual_incremental_load(
    DBFS_SALES,
    "ingestion_demo.sales_manual_incremental",
    "ingestion_demo._ingested_files",
    "dbfs:/FileStore/ingestion_demo/_manual_checkpoints"
)
print(f"First run ingested {cnt} rows.")

# Second run — should be zero (all files already tracked)
cnt2 = manual_incremental_load(
    DBFS_SALES,
    "ingestion_demo.sales_manual_incremental",
    "ingestion_demo._ingested_files",
    "dbfs:/FileStore/ingestion_demo/_manual_checkpoints"
)
print(f"Second run ingested {cnt2} rows.")


In [ ]:
# Generate a new file to demonstrate incremental pickup
import uuid
new_batch_dir = "/tmp/sales_new"
os.makedirs(new_batch_dir, exist_ok=True)
fp = os.path.join(new_batch_dir, f"sales_new_{uuid.uuid4().hex[:8]}.csv")
with open(fp, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["transaction_id","transaction_date","product_name","category","quantity","unit_price","total_amount","region","status"])
    w.writerow([str(uuid.uuid4()), date.today().isoformat(), "product_99", "Electronics", 2, 299.99, 599.98, "North", "COMPLETED"])
    w.writerow([str(uuid.uuid4()), date.today().isoformat(), "product_88", "Books",      1,  24.50,  24.50, "South", "COMPLETED"])

dbutils.fs.cp(f"file:{fp}", f"{DBFS_SALES}/{os.path.basename(fp)}")

cnt3 = manual_incremental_load(
    DBFS_SALES,
    "ingestion_demo.sales_manual_incremental",
    "ingestion_demo._ingested_files",
    "dbfs:/FileStore/ingestion_demo/_manual_checkpoints"
)
print(f"Third run (after new file added) ingested {cnt3} rows.")


 **Key take‑away (Concept 31):**
 - `COPY INTO` is your go‑to for batch / scheduled loads. Idempotent out of the box.
 - Auto Loader provides continuous ingestion with schema evolution.  Use `availableNow` for batch‑like semantics.
 - The manual file‑tracking pattern above gives you the same guarantee in environments without Auto Loader.


 ---
 ## Concept 32 — File Formats in the Lakehouse

 | Format  | Row/Col | Compress | Splittable | Schema | Human Readable | Best For                               |
 |---------|---------|----------|------------|--------|----------------|----------------------------------------|
 | CSV     | Row     | Low      | Partially  | Loose  | Yes            | Exchange, simple logs                  |
 | JSON    | Row     | Low      | No         | Loose  | Yes            | Web APIs, nested data                  |
 | Avro    | Row     | Good     | Yes        | Strong | No             | Kafka, streaming interchange           |
 | Parquet | Column  | Excellent| Yes        | Strong | No             | Analytics, predicate push‑down         |
 | Delta   | Column  | Excellent| Yes        | Strong | No             | **Lakehouse default** — ACID, time travel |

 ---


 ### 32‑A  Write Identical Data in Five Formats & Compare Sizes


In [ ]:
base_df = spark.table("ingestion_demo.sales_raw_bronze")

formats = {
    "csv":    ("csv",     {}),
    "json":   ("json",    {}),
    "avro":   ("avro",    {}),  # requires spark-avro package
    "parquet":("parquet", {}),
    "delta":  ("delta",   {}),
}

dir_prefix = "dbfs:/FileStore/ingestion_demo/format_compare/"
size_results = []

for fmt_name, (fmt_str, opts) in formats.items():
    out_path = f"{dir_prefix}{fmt_name}"
    dbutils.fs.rm(out_path, recurse=True)
    try:
        base_df.write.format(fmt_str).options(**opts).mode("overwrite").save(out_path)
        total_size = 0
        for f in dbutils.fs.ls(out_path):
            total_size += f.size
        size_results.append((fmt_name, total_size))
        print(f"{fmt_name:>8} => {total_size:>10,} bytes")
    except Exception as e:
        print(f"{fmt_name:>8} => SKIPPED ({e})")


 ### 32‑B  Read Speed Benchmark


In [ ]:
import time as _time

bench_results = []
for fmt_name, (fmt_str, _opts) in formats.items():
    path = f"{dir_prefix}{fmt_name}"
    try:
        t0 = _time.time()
        df = spark.read.format(fmt_str).load(path)
        cnt = df.count()
        elapsed = _time.time() - t0
        bench_results.append((fmt_name, cnt, round(elapsed, 3)))
        print(f"{fmt_name:>8} — {cnt} rows read in {elapsed:.3f}s")
    except Exception as e:
        bench_results.append((fmt_name, 0, str(e)))

display(spark.createDataFrame(bench_results, ["format","rows","seconds"]))


 ### 32‑C  Schema Inference Differences


In [ ]:
# Read the same CSV with and without inferSchema — note the data types
df_with_infer = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv(f"{DBFS_SALES}/*.csv")

df_without_infer = spark.read.option("header", "true").option("inferSchema", "false") \
    .csv(f"{DBFS_SALES}/*.csv")

print("WITH    inferSchema:")
df_with_infer.printSchema()

print("\nWITHOUT inferSchema (all strings):")
df_without_infer.printSchema()


 **Why Parquet / Delta dominate the Lakehouse:**
 1. **Columnar layout** — queries that touch a few columns read only those columns, slashing I/O.
 2. **Predicate push‑down** — Parquet min/max statistics let the engine skip row groups entirely.
 3. **Compression** — snappy/gzip/zstd reduce storage 5x–20x vs CSV.
 4. **Delta adds ACID** — time travel, upserts, schema enforcement, and vacuum.


 ---
 ## Concept 33 — Volumes

 **Volumes** are Unity Catalog objects for governed, non‑tabular file storage — think of them as "managed folders" with permissions.

 | Volume Type   | Storage                                      | Lifecycle                    |
 |---------------|----------------------------------------------|------------------------------|
 | Managed       | Managed by Unity Catalog in the metastore    | Deleted with catalog/schema  |
 | External      | External cloud storage (S3, ADLS, GCS)       | Survives catalog/schema drop |

 **Community Edition limitation:** Unity Catalog is not available.  We simulate Volumes using `dbfs:/FileStore/` paths, which serve the same role in CE.

 ---


 ### 33‑A  Simulating a Volume (Landing Zone) with DBFS FileStore


In [ ]:
LANDING_ZONE = "dbfs:/FileStore/ingestion_demo/landing_zone"

# ---- 1. Upload files to the "landing zone" (simulated volume) ----
dbutils.fs.rm(LANDING_ZONE, recurse=True)
dbutils.fs.mkdirs(LANDING_ZONE)

upload_dir_to_dbfs(SALES_IN, LANDING_ZONE)

# ---- 2. List files — volume-equivalent operation ----
print(f"Files in landing zone ({LANDING_ZONE}):")
for f in dbutils.fs.ls(LANDING_ZONE):
    print(f"  {f.name:40s} {f.size:>8,} bytes")

# ---- 3. Read CSV directly from the landing zone ----
landing_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{LANDING_ZONE}/*.csv")
)

print(f"\nRows in landing zone: {landing_df.count()}")
landing_df.select("transaction_id", "product_name", "total_amount").show(5, truncate=False)


 ### 33‑B  Conceptual — Catalog‑Backed Volumes (Unity Catalog)

 If UC were available, you would create a volume with:
 ```
 CREATE VOLUME my_catalog.my_schema.my_volume
   LOCATION 's3://my-bucket/landing/';
 ```

 Then reference files via the volume path: `/Volumes/my_catalog/my_schema/my_volume/sales.csv`

 | Feature                   | DBFS FileStore (CE)               | Volumes (UC)                       |
 |---------------------------|-----------------------------------|------------------------------------|
 | Permission model          | Workspace‑level                   | Fine‑grained (GRANT/REVOKE)        |
 | Lifecycle                 | Tied to workspace                 | Managed = cascade; External = n/a  |
 | Discovery                 | `dbutils.fs.ls`                   | INFORMATION_SCHEMA views           |
 | External storage support  | Manual mount                      | Direct path, governed              |


 ---
 ## Concept 34 — Auto Loader: File Discovery

 Auto Loader discovers new files through one of two mechanisms:

 1. **Directory listing** — every *N* seconds Auto Loader lists the input directory, compares against the checkpoint, and picks up new files.  Works anywhere, but latency depends on listing interval.
 2. **File notification** — Auto Loader subscribes to cloud event queues (SQS, Event Grid).  Near‑real‑time latency, but requires cloud infrastructure configuration.

 **Checkpoint location** tracks which files have been processed so the stream can resume from where it left off.

 ---


 ### 34‑A  Directory Listing Mode — Conceptual Code


In [ ]:
# This code pattern works in Community Edition (directory listing mode).
# The key options:
#   cloudFiles.format                — file format to ingest
#   cloudFiles.useNotifications      — true = file notification, false = directory listing
#   cloudFiles.schemaLocation        — where inferred schema is stored
#   cloudFiles.includeExistingFiles  — whether to ingest files that pre‑date the stream

def autoloader_discovery_demo():
    checkpoint = "dbfs:/FileStore/ingestion_demo/_checkpoints/discovery_demo"
    schema_loc = "dbfs:/FileStore/ingestion_demo/_schemas/discovery_demo"
    dbutils.fs.rm(checkpoint, recurse=True)
    dbutils.fs.rm(schema_loc, recurse=True)

    stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("cloudFiles.useNotifications", "false")   # directory listing
        .option("cloudFiles.schemaLocation", schema_loc)
        .option("cloudFiles.includeExistingFiles", "true")
        .load(DBFS_SALES)
        .writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable("ingestion_demo.autoloader_discovery")
    )
    stream.awaitTermination()

try:
    autoloader_discovery_demo()
    print("Stream completed.")
    display(spark.table("ingestion_demo.autoloader_discovery").count())
except Exception as e:
    print(f"Not available: {e}")


 ### 34‑B  Manual File Discovery Loop (CE Alternative)


In [ ]:
def polling_discovery_loop(data_dir, target_table, poll_seconds=5, max_polls=3):
    """
    Simulate directory‑listing polling: check for new files every `poll_seconds`,
    ingest them, record in metadata table.
    """
    meta_table = "ingestion_demo._polling_meta"
    spark.sql(f"CREATE TABLE IF NOT EXISTS {meta_table} "
              "(file_path STRING, ingested_at STRING) USING delta")

    for poll in range(max_polls):
        all_files  = set(f.path for f in dbutils.fs.ls(data_dir) if f.name.endswith(".csv"))
        ingested   = set(r.file_path for r in spark.table(meta_table).select("file_path").collect())
        new        = sorted(all_files - ingested)

        if new:
            print(f"Poll {poll+1}: discovered {len(new)} new file(s)")
            df = spark.read.option("header", "true").option("inferSchema", "true").csv(new)
            df.write.mode("append").format("delta").saveAsTable(target_table)
            now = datetime.now().isoformat()
            spark.createDataFrame([(f, now) for f in new]) \
                .write.mode("append").format("delta").saveAsTable(meta_table)
        else:
            print(f"Poll {poll+1}: no new files")

        if poll < max_polls - 1:
            time.sleep(poll_seconds)

    return spark.table(target_table).count()

spark.sql("DROP TABLE IF EXISTS ingestion_demo.polling_target")
cnt = polling_discovery_loop(DBFS_SALES, "ingestion_demo.polling_target", poll_seconds=2, max_polls=3)
print(f"Total rows after polling loop: {cnt}")


 ---
 ## Concept 35 — Auto Loader: Schema Evolution & Rescued Data

 Production data *drifts* — a source system adds a column, changes a type, or sends a malformed record.
 Auto Loader handles this with two options:

 | Option                         | Behaviour                                                             |
 |--------------------------------|-----------------------------------------------------------------------|
 | `cloudFiles.schemaEvolutionMode` | `addNewColumns` — add new columns (default)                           |
 |                                | `failOnNewColumns` — throw error (strict)                             |
 |                                | `rescue` — add new columns AND route mismatched data to `_rescued_data` |
 |                                | `none` — ignore schema changes entirely                               |
 | `cloudFiles.rescuedDataColumn` | Name of the rescue column (default `_rescued_data`)                   |

 ---


 ### 35‑A  Manual Schema Evolution (CE Simulation)


In [ ]:
# Step 1 — Write an initial batch with 5 columns
initial_data = [
    (1, "Widget",  9.99,  "US", "2025-01-01"),
    (2, "Gadget", 19.99,  "CA", "2025-01-02"),
]
initial_schema = T.StructType([
    T.StructField("product_id",   T.IntegerType()),
    T.StructField("product_name", T.StringType()),
    T.StructField("price",        T.DoubleType()),
    T.StructField("country",      T.StringType()),
    T.StructField("created_date", T.StringType()),
])
df1 = spark.createDataFrame(initial_data, schema=initial_schema)
df1.write.mode("overwrite").format("delta").saveAsTable("ingestion_demo.schema_evolution_demo")

print("Batch 1 schema:")
spark.table("ingestion_demo.schema_evolution_demo").printSchema()


In [ ]:
# Step 2 — A new batch arrives with an EXTRA column `supplier` and a new data type for `created_date`
new_data = [
    (3, "Doohickey", 5.99, "US", "2025-02-15", "Acme Corp"),
    (4, "Thingy",    7.50, "MX", "2025-02-16", "Global Ltd"),
]
new_schema = T.StructType([
    T.StructField("product_id",   T.IntegerType()),
    T.StructField("product_name", T.StringType()),
    T.StructField("price",        T.DoubleType()),
    T.StructField("country",      T.StringType()),
    T.StructField("created_date", T.StringType()),
    T.StructField("supplier",     T.StringType()),          # <— NEW column
])
df2 = spark.createDataFrame(new_data, schema=new_schema)

# ---- Simulate rescued data column manually ----
# Merge schemas: read existing schema, add new columns from the incoming batch
existing_schema = spark.table("ingestion_demo.schema_evolution_demo").schema
existing_cols = set(f.name for f in existing_schema.fields)
new_cols      = set(f.name for f in df2.schema.fields)
added_cols    = new_cols - existing_cols

print(f"Existing columns: {existing_cols}")
print(f"Incoming columns: {new_cols}")
print(f"New columns to add: {added_cols}")

# Auto-merge by enabling schema auto-merge and appending
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
df2.write.mode("append").format("delta").saveAsTable("ingestion_demo.schema_evolution_demo")

print("\nMerged schema (with new columns, NULLs for old rows):")
merged_df = spark.table("ingestion_demo.schema_evolution_demo")
merged_df.printSchema()
merged_df.show()


 ### 35‑B  Simulating Rescued Data Column


In [ ]:
# In Auto Loader, the `_rescued_data` column captures rows whose types don't match,
# so no data is lost. We simulate this by detecting parse errors manually.

from pyspark.sql import types as T

rescued_schema = T.StructType([
    T.StructField("id",      T.IntegerType()),
    T.StructField("value_1", T.DoubleType()),
    T.StructField("value_2", T.StringType()),
])

hard_data = [
    (1, 100.5,  "valid"),
    (2, None,   "valid"),          # okay, null
    (3, "BAD",  "valid"),          # type mismatch on value_1 (string in double column)
    (4, 200.0,  "valid"),
]

# Create DataFrame with permissive mode and a rescue column
raw_df = spark.createDataFrame(hard_data, schema=rescued_schema)
raw_df.show()

# In real Auto Loader, the bad row would be written to _rescued_data.
# Manual alternative: read as strings, validate, separate clean/malformed.
# Run a query to identify rows where value_1 is not numeric:

raw_df.createOrReplaceTempView("rescued_temp")
cleaned  = spark.sql("SELECT * FROM rescued_temp WHERE value_1 IS NOT NULL")
rescued  = spark.sql("SELECT * FROM rescued_temp WHERE value_1 IS NULL AND id = 3")

print("Clean rows:")
cleaned.show()
print("Rescued / malformed rows:")
rescued.show()


 **Schema Evolution summary (Concept 35):**
 - Auto Loader's `addNewColumns` mode is the safest default — never lose data.
 - `rescue` mode adds `_rescued_data` so you can quarantine and repair malformed records.
 - In Community Edition, `spark.databricks.delta.schema.autoMerge.enabled = true` gives you automatic column addition on append.


 ---
 ## Concept 36 — Multi-Hop Medallion Architecture

 | Layer   | Purpose                                   | Operations                        |
 |---------|-------------------------------------------|-----------------------------------|
 | Bronze  | Raw ingestion, 1:1 with source            | Append only, preserve lineage     |
 | Silver  | Cleansed, validated, deduplicated         | Cast types, deduplicate, QC       |
 | Gold    | Business aggregates & views               | Aggregations, joins, enrichment   |

 ---


 ### 36‑A  Bronze — Raw Ingestion (Append‑Only)


In [ ]:
BRONZE_TABLE = "ingestion_demo.sales_bronze"

# Bronze: raw CSV dump, add ingestion timestamp and source filename
def build_bronze(source_dir, bronze_table):
    spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")

    raw_df = (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{source_dir}/*.csv")
    )

    bronze_df = (raw_df
        .withColumn("_bronze_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.input_file_name())
    )

    bronze_df.write.mode("overwrite").format("delta").saveAsTable(bronze_table)
    return spark.table(bronze_table).count()

bronze_rows = build_bronze(DBFS_SALES, BRONZE_TABLE)
print(f"Bronze layer: {bronze_rows} rows")
spark.table(BRONZE_TABLE).printSchema()


 ### 36‑B  Silver — Clean, Deduplicate, Cast Types


In [ ]:
SILVER_TABLE = "ingestion_demo.sales_silver"

spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")

bronze = spark.table(BRONZE_TABLE)

# 1. Deduplicate on transaction_id (keep latest _bronze_ingested_at)
window_spec = Window.partitionBy("transaction_id").orderBy(F.desc("_bronze_ingested_at"))
deduped = (bronze
    .withColumn("_rn", F.row_number().over(window_spec))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# 2. Clean & cast — ensure correct types, strip whitespace, filter invalid rows
silver_df = (deduped
    .withColumn("transaction_id",   F.trim(F.col("transaction_id")))
    .withColumn("transaction_date", F.to_date(F.col("transaction_date")))
    .withColumn("product_name",     F.trim(F.col("product_name")))
    .withColumn("category",         F.trim(F.col("category")))
    .withColumn("quantity",         F.col("quantity").cast("integer"))
    .withColumn("unit_price",       F.col("unit_price").cast("double"))
    .withColumn("total_amount",     F.col("total_amount").cast("double"))
    .withColumn("region",           F.trim(F.col("region")))
    .withColumn("status",           F.trim(F.col("status")))
    .withColumn("_silver_cleaned_at", F.current_timestamp())
    # Drop null transaction_ids (data quality)
    .filter(F.col("transaction_id").isNotNull())
    .filter(F.col("transaction_id") != "")
)

# 3. Add data quality flag
silver_df = silver_df.withColumn("_quality_flag",
    F.when(F.col("total_amount") < 0, "SUSPECT_NEGATIVE")
     .when(F.col("quantity") <= 0,    "SUSPECT_ZERO_QTY")
     .otherwise("OK")
)

silver_df.write.mode("overwrite").format("delta").saveAsTable(SILVER_TABLE)

print(f"Silver layer: {spark.table(SILVER_TABLE).count()} rows")
spark.table(SILVER_TABLE).printSchema()

# Data quality report
quality_counts = spark.table(SILVER_TABLE).groupBy("_quality_flag").count()
display(quality_counts)


 ### 36‑C  Gold — Business Aggregates


In [ ]:
GOLD_DB = "ingestion_demo"
silver = spark.table(SILVER_TABLE)

# ---- Aggregate 1: Daily sales by category ----
daily_sales = (silver
    .groupBy("transaction_date", "category")
    .agg(
        F.sum("total_amount").alias("daily_revenue"),
        F.count("*").alias("order_count"),
        F.avg("total_amount").alias("avg_order_value"),
    )
)

daily_sales.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD_DB}.gold_daily_sales")
print("Gold — Daily Sales by Category")
display(spark.table(f"{GOLD_DB}.gold_daily_sales").orderBy("transaction_date", "category"))

# ---- Aggregate 2: Top 10 products by revenue ----
top_products = (silver
    .groupBy("product_name", "category")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
    )
    .orderBy(F.desc("total_revenue"))
    .limit(10)
)

top_products.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD_DB}.gold_top_products")
print("\nGold — Top 10 Products")
display(spark.table(f"{GOLD_DB}.gold_top_products"))

# ---- Aggregate 3: Regional performance ----
regional = (silver
    .groupBy("region")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.count("*").alias("total_orders"),
        F.sum(F.when(F.col("status") == "RETURNED", 1).otherwise(0)).alias("returns"),
        F.round(F.sum(F.when(F.col("status") == "RETURNED", 1).otherwise(0)) / F.count("*") * 100, 2).alias("return_rate_pct"),
    )
)

regional.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD_DB}.gold_regional")
print("\nGold — Regional Performance")
display(spark.table(f"{GOLD_DB}.gold_regional"))

# ---- Aggregate 4: Customer spending segments (simulated from transaction size) ----
segments = (silver
    .withColumn("spending_segment",
        F.when(F.col("total_amount") >= 500, "High")
         .when(F.col("total_amount") >= 100, "Medium")
         .otherwise("Low")
    )
    .groupBy("spending_segment")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("total_amount").alias("total_revenue"),
    )
)

segments.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD_DB}.gold_spending_segments")
print("\nGold — Spending Segments")
display(spark.table(f"{GOLD_DB}.gold_spending_segments"))


 ### 36‑D  Medallion Lineage View


In [ ]:
print("""
  ┌──────────────────────────────────────────────────┐
  │                Medallion Pipeline                 │
  ├──────────┬──────────────────┬────────────────────┤
  │  Bronze  │  Silver          │  Gold               │
  ├──────────┼──────────────────┼────────────────────┤
  │ Raw CSV  │ Deduplicated     │ gold_daily_sales    │
  │ (append) │ Typed            │ gold_top_products   │
  │          │ Quality-flagged  │ gold_regional       │
  │          │                  │ gold_spending_segments │
  └──────────┴──────────────────┴────────────────────┘
""")

# Verify we can trace a transaction through all layers
sample_id = spark.table(BRONZE_TABLE).select("transaction_id").first()[0]
print(f"Tracing transaction: {sample_id}")

b = spark.table(BRONZE_TABLE).filter(f"transaction_id = '{sample_id}'").select("transaction_id","total_amount","_bronze_ingested_at")
s = spark.table(SILVER_TABLE).filter(f"transaction_id = '{sample_id}'").select("transaction_id","total_amount","_quality_flag")
print("\nBronze:")
b.show(truncate=False)
print("Silver:")
s.show(truncate=False)


 ---
 ## Concept 37 — Incremental Processing Patterns

 Three common strategies for processing only new/changed data:

 | Pattern               | Granularity | Latency    | Complexity | Best For                        |
 |-----------------------|-------------|------------|------------|---------------------------------|
 | File‑level            | File        | Minutes    | Low        | Simple batch append             |
 | Change Data Feed (CDF)| Row         | Seconds    | Medium     | CDC from Delta tables           |
 | Timestamp / Watermark | Row         | Sub‑second | Medium     | Streaming with watermark        |

 ---


 ### 37‑A  File‑Level Incremental Processing


In [ ]:
# We already demonstrated this in Concept 31-C. Here's a concise recap:
spark.sql("DROP TABLE IF EXISTS ingestion_demo.file_level_target")
spark.sql("CREATE TABLE IF NOT EXISTS ingestion_demo._file_level_meta "
          "(file_path STRING, ingested_at STRING) USING delta")

# Process only files NOT in the meta table
def file_level_incremental(source_dir, target_table):
    meta = spark.table("ingestion_demo._file_level_meta")
    ingested = set(r.file_path for r in meta.select("file_path").collect())
    all_files = set(f.path for f in dbutils.fs.ls(source_dir) if f.name.endswith(".csv"))
    new = [f for f in sorted(all_files - ingested) if not f.endswith("_SUCCESS")]

    if not new:
        print("No new files.")
        return 0

    print(f"Processing {len(new)} new file(s)...")
    df = spark.read.option("header","true").option("inferSchema","true").csv(new)
    df.write.mode("append").format("delta").saveAsTable(target_table)
    now = datetime.now().isoformat()
    spark.createDataFrame([(f, now) for f in new], "file_path STRING, ingested_at STRING") \
        .write.mode("append").format("delta").saveAsTable("ingestion_demo._file_level_meta")
    return df.count()

cnt = file_level_incremental(DBFS_SALES, "ingestion_demo.file_level_target")
print(f"File-level incremental ingested {cnt} rows.")


 ### 37‑B  Row‑Level — Change Data Feed (CDF)


In [ ]:
# Delta's Change Data Feed tracks row‑level changes (insert, update, delete).
# For Community Edition we simulate CDF by reading version history.

# Create a base table and make changes
spark.sql("DROP TABLE IF EXISTS ingestion_demo.cdf_demo")

cdf_base = spark.createDataFrame(
    [(1, "Alice", 1000.0), (2, "Bob", 1500.0)],
    "id INT, name STRING, balance DOUBLE"
)
cdf_base.write.format("delta").saveAsTable("ingestion_demo.cdf_demo")

# Update Bob's balance
spark.sql("""
  MERGE INTO ingestion_demo.cdf_demo t
  USING (SELECT 2 AS id, 'Bob' AS name, 2000.0 AS balance) s
  ON t.id = s.id
  WHEN MATCHED THEN UPDATE SET t.balance = s.balance
  WHEN NOT MATCHED THEN INSERT *
""")

# Insert a new row
spark.sql("INSERT INTO ingestion_demo.cdf_demo VALUES (3, 'Carol', 2500.0)")

# Delete Alice
spark.sql("DELETE FROM ingestion_demo.cdf_demo WHERE id = 1")

# ---- Simulate CDF with Delta time travel ----
print("Initial state (version 0):")
spark.sql("SELECT * FROM ingestion_demo.cdf_demo VERSION AS OF 0").show()

print("Current state (latest version):")
spark.table("ingestion_demo.cdf_demo").show()

# In production with CDF enabled, you'd query table_changes():
# SELECT * FROM table_changes('ingestion_demo.cdf_demo', 0)
# This returns change_type (insert/update/delete), commit_version, timestamp

print("\nCDF simulation — history:")
spark.sql("DESCRIBE HISTORY ingestion_demo.cdf_demo").select("version","timestamp","operation").show(10, truncate=False)


 ### 37‑C  Timestamp / Watermark‑Based Incremental Processing


In [ ]:
spark.sql("DROP TABLE IF EXISTS ingestion_demo.watermark_source")
spark.sql("DROP TABLE IF EXISTS ingestion_demo.watermark_target")

# Create a source table with timestamped events
from pyspark.sql import functions as F

watermark_data = []
base_date = datetime(2025, 1, 1)
for i in range(50):
    ts = base_date + timedelta(hours=i)
    watermark_data.append((i, f"event_{i}", round(i * 1.5, 2), ts.isoformat()))

watermark_df = spark.createDataFrame(watermark_data, "id INT, event_name STRING, value DOUBLE, event_time STRING")
watermark_df.write.mode("overwrite").format("delta").saveAsTable("ingestion_demo.watermark_source")

# ---- Function to process only rows after last run time ----
def watermark_incremental(source_table, target_table, watermark_col, watermark_state_table):
    """
    Read only rows from `source_table` whose `watermark_col` is greater than
    the last recorded high-watermark stored in `watermark_state_table`.
    """
    spark.sql(f"CREATE TABLE IF NOT EXISTS {watermark_state_table} "
              "(last_watermark STRING, updated_at STRING) USING delta")

    # Get the last high-watermark
    state = spark.table(watermark_state_table)
    if state.count() > 0:
        last_wm = state.orderBy(F.desc("updated_at")).select("last_watermark").first()[0]
    else:
        last_wm = "1970-01-01T00:00:00"

    print(f"Last high-watermark: {last_wm}")

    new_df = spark.table(source_table).filter(F.col(watermark_col) > last_wm)
    new_count = new_df.count()

    if new_count == 0:
        print("No new rows since last watermark.")
        return 0

    if spark.catalog.tableExists(target_table):
        new_df.write.mode("append").format("delta").saveAsTable(target_table)
    else:
        new_df.write.mode("overwrite").format("delta").saveAsTable(target_table)

    # Compute new max watermark from processed data
    new_wm = new_df.agg(F.max(watermark_col).alias("max_wm")).first()["max_wm"]
    now = datetime.now().isoformat()

    # Update state
    sparK = spark  # avoid shadowing
    state_update = sparK.createDataFrame([(new_wm, now)], "last_watermark STRING, updated_at STRING")
    state_update.write.mode("overwrite").format("delta").saveAsTable(watermark_state_table)

    print(f"Processed {new_count} rows. New high-watermark: {new_wm}")
    return new_count

# First run — loads all data
c = watermark_incremental(
    "ingestion_demo.watermark_source",
    "ingestion_demo.watermark_target",
    "event_time",
    "ingestion_demo._watermark_state"
)
print(f"First run: {c} rows")

# Second run — should be zero (no new data)
c2 = watermark_incremental(
    "ingestion_demo.watermark_source",
    "ingestion_demo.watermark_target",
    "event_time",
    "ingestion_demo._watermark_state"
)
print(f"Second run: {c2} rows")


In [ ]:
# Add new data and run again to verify watermark picks up only new rows
new_events = []
for i in range(50, 60):
    ts = base_date + timedelta(hours=i)
    new_events.append((i, f"event_{i}", round(i * 1.5, 2), ts.isoformat()))

new_df = spark.createDataFrame(new_events, "id INT, event_name STRING, value DOUBLE, event_time STRING")
new_df.write.mode("append").format("delta").saveAsTable("ingestion_demo.watermark_source")

c3 = watermark_incremental(
    "ingestion_demo.watermark_source",
    "ingestion_demo.watermark_target",
    "event_time",
    "ingestion_demo._watermark_state"
)
print(f"Third run (after adding 10 new events): {c3} rows (expect 10)")


 **Incremental Processing comparison:**

 | Pattern              | Cost  | Latency   | Use Case                             |
 |----------------------|-------|-----------|--------------------------------------|
 | File‑level tracking  | Low   | File‑based| Daily batch loads                    |
 | CDF (Delta)          | Med   | Seconds   | CDC replication, auditing            |
 | Watermark / timestamp| Low   | Near‑RT   | Streaming events, log processing     |


 ---
 ## Concept 38 — Lakehouse Federation

 Lakehouse Federation allows Databricks to query data in *external* databases (PostgreSQL, MySQL, SQL Server, Snowflake, BigQuery) **without moving the data** — the query is pushed down as much as possible and results are returned.

 **Key concepts:**
 - **Foreign catalog** — a catalog that maps to an external database connection.
 - **Query pushdown** — Databricks translates Spark SQL into the external DB's dialect so filters and aggregations run remotely.
 - **Use cases** — ad‑hoc cross‑system queries, building a 360° customer view, gradual migration to Lakehouse.

 **Community Edition limitation:** Foreign catalogs require full platform.  We demonstrate JDBC reading as the CE equivalent.

 ---


 ### 38‑A  JDBC Read — Community Edition Equivalent of Federation


In [ ]:
# Federation conceptual syntax (not runnable in CE):
# CREATE FOREIGN CATALOG pg_catalog
#   USING JDBC OPTIONS (
#     url      'jdbc:postgresql://host:5432/db',
#     user     'reader',
#     password '***'
#   );
#
# SELECT * FROM pg_catalog.public.customers WHERE region = 'West';

# ---- CE alternative: JDBC read ----
# For demo we read our own Delta table via JDBC (self-referencing)
# In production you'd connect to Postgres, MySQL, etc.

jdbc_url    = f"jdbc:spark://localhost:443/default;transportMode=http;ssl=1;AuthMech=3;httpPath=sql/protocolv1/o/0/0000-000000-0000000000"
jdbc_table  = "ingestion_demo.sales_silver"

try:
    jdbc_df = (spark.read
        .format("jdbc")
        .option("url",      jdbc_url)
        .option("dbtable",  jdbc_table)
        .option("user",     "token")
        .option("password", "dummy")
        .load()
    )
    print("JDBC read succeeded (self-referencing).")
    jdbc_df.select("transaction_id", "product_name", "total_amount").show(3)
except Exception as e:
    print(f"JDBC to Databricks itself not available in CE: {e}")

# ---- Show a generic JDBC pattern (PostgreSQL example, commented) ----
print("""
# Generic PostgreSQL JDBC read pattern:
pg_df = (spark.read
    .format("jdbc")
    .option("url",      "jdbc:postgresql://<host>:5432/<db>")
    .option("dbtable",  "(SELECT * FROM orders WHERE order_date > '2025-01-01') AS filtered")
    .option("user",     "<user>")
    .option("password", "<password>")
    .option("driver",   "org.postgresql.Driver")
    .option("numPartitions", "8")
    .option("partitionColumn", "order_id")
    .option("lowerBound", "1")
    .option("upperBound", "1000000")
    .load()
)
""")


 ### 38‑B  Query Pushdown Explained


In [ ]:
# When using federation, Databricks pushes predicates to the remote DB.
# Example: Spark SQL `SELECT * FROM pg_catalog.public.orders WHERE region = 'West'`
#   => Databricks translates to: `SELECT * FROM orders WHERE region = 'West'` on PostgreSQL
#   => Only matching rows are transferred over the network

# Without pushdown (naive JDBC read), ALL rows are pulled and filtered in Spark.
# With pushdown, the WHERE clause runs server-side — dramatically less data transfer.

# Simulate pushdown benefit with local Spark:
big_df = spark.table(BRONZE_TABLE).repartition(1)
big_df.write.mode("overwrite").format("delta").saveAsTable("ingestion_demo.pushdown_test")

t0 = time.time()
# Without predicate pushdown simulation (read all then filter)
_  = spark.table("ingestion_demo.pushdown_test").filter("region = 'West'").collect()
t1 = time.time()

# With pushdown (Parquet predicate pushdown happens automatically)
_  = spark.table("ingestion_demo.pushdown_test").filter("region = 'West'").collect()
t2 = time.time()

print(f"Filter chain time: {t1-t0:.4f}s  (automatic pushdown in Delta/Parquet: {t2-t1:.4f}s)")
print("In a federated query, pushdown avoids transferring millions of unnecessary rows over the network.")


 **When to use Federation vs. ETL:**

 | Scenario                                | Approach          |
 |-----------------------------------------|-------------------|
 | Ad‑hoc query across 2+ systems          | Federation        |
 | Operational dashboard with <5s latency  | Federation        |
 | One‑time data exploration               | Federation        |
 | Heavy transformations, joins, history   | ETL into Lakehouse|
 | ML training on large datasets           | ETL into Lakehouse|
 | Compliance / audit trail required       | ETL into Lakehouse|


 ---
 ## Concept 39 — Idempotent Pipeline Design

 **Idempotency = running the pipeline multiple times produces the same final state.**

 Why it matters:
 - **Failure recovery** — re‑run without manual cleanup
 - **Backfill** — re‑process a date range safely
 - **Late‑arriving data** — re‑ingest a partition

 Strategies:
 - `MERGE` with natural keys (upsert semantics)
 - `INSERT OVERWRITE` with `replaceWhere`
 - Partition‑based truncate + reload
 - `COPY INTO` (built‑in idempotency)

 ---


 ### 39‑A  MERGE with Natural Keys — Idempotent Upsert


In [ ]:
# Create a target table with a natural key
spark.sql("DROP TABLE IF EXISTS ingestion_demo.idempotent_products")

target_schema = T.StructType([
    T.StructField("product_code", T.StringType()),   # natural key
    T.StructField("product_name", T.StringType()),
    T.StructField("price",        T.DoubleType()),
    T.StructField("updated_at",   T.StringType()),
])

initial = spark.createDataFrame([
    ("P001", "Widget",      9.99, "2025-01-01"),
    ("P002", "Gadget",     19.99, "2025-01-02"),
], schema=target_schema)

initial.write.mode("overwrite").format("delta").saveAsTable("ingestion_demo.idempotent_products")

# ---- Incoming batch (some new, some updates) ----
incoming = spark.createDataFrame([
    ("P002", "Gadget Pro", 24.99, "2025-02-01"),   # UPDATE — P002 exists
    ("P003", "Doohickey",   5.99, "2025-02-01"),   # INSERT — new record
    ("P001", "Widget",      9.99, "2025-02-01"),   # UNCHANGED — same values
], schema=target_schema)

incoming.createOrReplaceTempView("incoming_updates")

# Idempotent MERGE — run it multiple times, result is always the same
merge_sql = """
  MERGE INTO ingestion_demo.idempotent_products AS target
  USING incoming_updates AS source
  ON target.product_code = source.product_code
  WHEN MATCHED THEN
    UPDATE SET
      target.product_name = source.product_name,
      target.price        = source.price,
      target.updated_at   = source.updated_at
  WHEN NOT MATCHED THEN
    INSERT (product_code, product_name, price, updated_at)
    VALUES (source.product_code, source.product_name, source.price, source.updated_at)
"""

# Run 1
spark.sql(merge_sql)
print("After run 1:")
spark.table("ingestion_demo.idempotent_products").orderBy("product_code").show()

# Run 2 — should produce identical results
spark.sql(merge_sql)
print("After run 2 (idempotent):")
spark.table("ingestion_demo.idempotent_products").orderBy("product_code").show()

# Run 3 — still identical
spark.sql(merge_sql)
print("After run 3 (still idempotent):")
spark.table("ingestion_demo.idempotent_products").orderBy("product_code").show()


 ### 39‑B  INSERT OVERWRITE with replaceWhere — Partition Idempotency


In [ ]:
spark.sql("DROP TABLE IF EXISTS ingestion_demo.idempotent_sales")

# Target partitioned by date
sample_sales = spark.table(BRONZE_TABLE) \
    .withColumn("sale_date", F.to_date(F.col("transaction_date"))) \
    .select("transaction_id","sale_date","product_name","category","total_amount","region")

sample_sales.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("sale_date") \
    .saveAsTable("ingestion_demo.idempotent_sales")

# ---- Re-ingest a specific date safely ----
target_date = sample_sales.select("sale_date").first()["sale_date"]
print(f"Demonstrating replaceWhere for date: {target_date}")

# Build a "new" DataFrame for that date (simulates re-processing)
replace_df = spark.table("ingestion_demo.idempotent_sales") \
    .filter(f"sale_date = '{target_date}'") \
    .withColumn("category", F.upper(F.col("category")))  # pretend we fixed category casing

print(f"Before replaceWhere — sample row for {target_date}:")
spark.table("ingestion_demo.idempotent_sales") \
    .filter(f"sale_date = '{target_date}'") \
    .select("sale_date","category") \
    .show(2, truncate=False)

(replace_df.write
    .mode("overwrite")
    .format("delta")
    .option("replaceWhere", f"sale_date = '{target_date}'")
    .saveAsTable("ingestion_demo.idempotent_sales")
)

print(f"After replaceWhere — category is now upper-case:")
spark.table("ingestion_demo.idempotent_sales") \
    .filter(f"sale_date = '{target_date}'") \
    .select("sale_date","category") \
    .show(2, truncate=False)

# Other partitions untouched
distinct_dates = spark.table("ingestion_demo.idempotent_sales") \
    .select("sale_date") \
    .distinct() \
    .count()
print(f"Total distinct dates in table: {distinct_dates}")


 ### 39‑C  Fully Idempotent Ingestion Function


In [ ]:
def idempotent_ingest(source_dir, target_table, natural_key_cols, partition_col=None):
    """
    Generic idempotent ingestion function:
    1. Read new data from source_dir
    2. MERGE into target table using natural_key_cols
    3. If partition_col is provided, use replaceWhere at partition granularity
    """
    print(f"Source      : {source_dir}")
    print(f"Target      : {target_table}")
    print(f"Natural keys: {natural_key_cols}")

    # Read source
    source_df = (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{source_dir}/*.csv")
    )

    # Add ingestion metadata
    source_df = source_df \
        .withColumn("_ingested_at", F.current_timestamp()) \
        .withColumn("_source",      F.lit(source_dir))

    # Ensure target exists
    if not spark.catalog.tableExists(target_table):
        source_df.write.mode("overwrite").format("delta").saveAsTable(target_table)
        print(f"Created new table: {target_table}")
        return source_df.count()

    source_df.createOrReplaceTempView("_idempotent_source")

    join_condition = " AND ".join(
        f"target.{c} = source.{c}" for c in natural_key_cols
    )

    target_schema = spark.table(target_table).schema
    target_cols = {f.name for f in target_schema.fields}
    source_cols = {f.name for f in source_df.schema.fields}
    cols_to_update = [c for c in source_cols if c in target_cols and c not in natural_key_cols and c != "_ingested_at"]

    update_clause = ", ".join(f"target.{c} = source.{c}" for c in cols_to_update)
    insert_cols   = ", ".join(cols_to_update + natural_key_cols)
    insert_vals   = ", ".join(f"source.{c}" for c in cols_to_update + natural_key_cols)

    merge_sql = f"""
      MERGE INTO {target_table} AS target
      USING _idempotent_source AS source
      ON {join_condition}
      WHEN MATCHED THEN UPDATE SET {update_clause}
      WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
    """

    print(f"\nMERGE statement:\n{merge_sql[:500]}...")
    spark.sql(merge_sql)

    result_count = spark.table(target_table).count()
    print(f"Total rows in {target_table}: {result_count}")
    return result_count

# Test the idempotent function
spark.sql("DROP TABLE IF EXISTS ingestion_demo.fully_idempotent_target")

c1 = idempotent_ingest(
    DBFS_SALES,
    "ingestion_demo.fully_idempotent_target",
    natural_key_cols=["transaction_id"]
)
print(f"Run 1 created {c1} rows")

c2 = idempotent_ingest(
    DBFS_SALES,
    "ingestion_demo.fully_idempotent_target",
    natural_key_cols=["transaction_id"]
)
print(f"Run 2 (idempotent) — table has {c2} rows (should equal {c1})")

assert c1 == c2, "IDEMPOTENCY VIOLATED"
print("Idempotency verified.")


 ### 39‑D  Retry Pattern with Checkpoint Recovery


In [ ]:
def idempotent_ingest_with_retry(source_dir, target_table, natural_keys, max_retries=3, backoff_seconds=2):
    """Idempotent ingestion with exponential backoff retry."""
    for attempt in range(1, max_retries + 1):
        try:
            count = idempotent_ingest(source_dir, target_table, natural_keys)
            print(f"Success on attempt {attempt}.")
            return count
        except Exception as e:
            if attempt == max_retries:
                print(f"FAILED after {max_retries} attempts: {e}")
                raise
            wait = backoff_seconds * (2 ** (attempt - 1))
            print(f"Attempt {attempt} failed ({e}). Retrying in {wait}s...")
            time.sleep(wait)

print("\nRetry pattern demonstration:")
try:
    idempotent_ingest_with_retry(
        DBFS_SALES,
        "ingestion_demo.fully_idempotent_target",
        ["transaction_id"],
        max_retries=2
    )
except Exception as e:
    print(f"Expected failure: {e}")


 **Idempotency summary (Concept 39):**

 | Strategy                | When to Use                               |
 |-------------------------|-------------------------------------------|
 | `MERGE` with natural key| Dimension tables, upsert workloads        |
 | `INSERT OVERWRITE`      | Full partition reload                     |
 | `replaceWhere`          | Selective partition reload                |
 | `COPY INTO`             | File‑based batch loads (built‑in)         |
 | Checkpoint + retry      | Streaming / long‑running pipelines        |


 ---
 ## Concept 40 — SCD Type 1 & Type 2 Patterns

 **Slowly Changing Dimensions (SCD)** manage changes to dimensional attributes over time.

 | Type | Strategy                                 | Historical tracking | Storage |
 |------|------------------------------------------|---------------------|---------|
 | 1    | Overwrite old value with new value       | No                  | Low     |
 | 2    | Insert new row; expire old row           | Yes                 | Higher  |
 | 3    | Keep previous value in a separate column | One version back    | Medium  |

 ---


 ### 40‑A  Build Customer Dimension (Base Data)


In [ ]:
# Create initial customer dimension
spark.sql("DROP TABLE IF EXISTS ingestion_demo.customers")
spark.sql("DROP TABLE IF EXISTS ingestion_demo.customers_scd2")

CUSTOMER_DATA = [
    ("CUST-1001", "Alice",   "Smith",   "alice@example.com",   "New York",   "NY"),
    ("CUST-1002", "Bob",     "Johnson", "bob@example.com",     "Los Angeles","CA"),
    ("CUST-1003", "Charlie", "Brown",   "charlie@example.com", "Chicago",    "IL"),
    ("CUST-1004", "Diana",   "Prince",  "diana@example.com",   "Houston",    "TX"),
    ("CUST-1005", "Edward",  "Norton",  "edward@example.com",  "Phoenix",    "AZ"),
]

customer_schema = T.StructType([
    T.StructField("customer_id",  T.StringType()),
    T.StructField("first_name",   T.StringType()),
    T.StructField("last_name",    T.StringType()),
    T.StructField("email",        T.StringType()),
    T.StructField("city",         T.StringType()),
    T.StructField("state",        T.StringType()),
])

customer_base = spark.createDataFrame(CUSTOMER_DATA, schema=customer_schema) \
    .withColumn("created_at", F.current_timestamp())

customer_base.write.mode("overwrite").format("delta").saveAsTable("ingestion_demo.customers")
print("Initial customer dimension:")
spark.table("ingestion_demo.customers").show(truncate=False)


 ### 40‑B  SCD Type 1 — Overwrite Current Attributes


In [ ]:
# ---- Incoming changes (Type 1) ----
# Alice moved to Seattle; Bob changed email; Frank is new
scd1_changes = spark.createDataFrame([
    ("CUST-1001", "Alice",   "Smith",   "alice_new@example.com", "Seattle",    "WA"),  # UPDATE
    ("CUST-1002", "Bob",     "Johnson", "bob_new@example.com",   "Los Angeles","CA"),  # UPDATE
    ("CUST-1006", "Frank",   "Miller",  "frank@example.com",     "Denver",     "CO"),  # INSERT
], schema=customer_schema)

scd1_changes.createOrReplaceTempView("scd1_changes")

scd1_merge = """
  MERGE INTO ingestion_demo.customers AS target
  USING scd1_changes AS source
  ON target.customer_id = source.customer_id
  WHEN MATCHED THEN
    UPDATE SET
      target.first_name = source.first_name,
      target.last_name  = source.last_name,
      target.email      = source.email,
      target.city       = source.city,
      target.state      = source.state
  WHEN NOT MATCHED THEN
    INSERT (customer_id, first_name, last_name, email, city, state, created_at)
    VALUES (source.customer_id, source.first_name, source.last_name,
            source.email, source.city, source.state, current_timestamp())
"""

spark.sql(scd1_merge)

print("After SCD Type 1 merge (Alice & Bob updated in place; Frank added):")
spark.table("ingestion_demo.customers").orderBy("customer_id").show(truncate=False)


 **SCD Type 1 consequence:** Alice's old email and city are *lost*.  No history preserved — but the table stays small and simple.


 ### 40‑C  SCD Type 2 — Full History with Effective Dates & Current Flag


In [ ]:
# Create SCD Type 2 dimension (initial load)
scd2_initial = spark.createDataFrame(CUSTOMER_DATA, schema=customer_schema) \
    .withColumn("eff_start_date", F.current_date()) \
    .withColumn("eff_end_date",   F.lit(None).cast("date")) \
    .withColumn("is_current",     F.lit(True))

scd2_initial.write.mode("overwrite").format("delta").saveAsTable("ingestion_demo.customers_scd2")
print("Initial SCD Type 2 dimension:")
spark.table("ingestion_demo.customers_scd2").orderBy("customer_id").show(truncate=False)


In [ ]:
# ---- Incoming changes (Type 2) ----
# Same three changes as before, but now we preserve history
scd2_incoming = spark.createDataFrame([
    ("CUST-1001", "Alice",   "Smith",   "alice_new@example.com", "Seattle",    "WA"),  # UPDATE
    ("CUST-1002", "Bob",     "Johnson", "bob_new@example.com",   "Los Angeles","CA"),  # UPDATE
    ("CUST-1006", "Frank",   "Miller",  "frank@example.com",     "Denver",     "CO"),  # INSERT
], schema=customer_schema)

scd2_incoming.createOrReplaceTempView("scd2_incoming")

# ---- Step 1: Expire current rows for changing customers ----
expire_sql = """
  MERGE INTO ingestion_demo.customers_scd2 AS target
  USING scd2_incoming AS source
  ON target.customer_id = source.customer_id AND target.is_current = true
  WHEN MATCHED AND (
    target.first_name <> source.first_name OR
    target.last_name  <> source.last_name OR
    target.email      <> source.email OR
    target.city       <> source.city OR
    target.state      <> source.state
  ) THEN UPDATE SET
    target.is_current    = false,
    target.eff_end_date  = current_date()
"""

spark.sql(expire_sql)

# ---- Step 2: Insert new rows with current data ----
insert_sql = """
  INSERT INTO ingestion_demo.customers_scd2
    (customer_id, first_name, last_name, email, city, state,
     eff_start_date, eff_end_date, is_current)
  SELECT
    source.customer_id,
    source.first_name,
    source.last_name,
    source.email,
    source.city,
    source.state,
    current_date()  AS eff_start_date,
    CAST(NULL AS DATE) AS eff_end_date,
    true           AS is_current
  FROM scd2_incoming source
  -- Only insert if the customer doesn't exist at all, or has changed
  LEFT ANTI JOIN ingestion_demo.customers_scd2 target
    ON source.customer_id = target.customer_id AND target.is_current = true
  WHERE NOT EXISTS (
    SELECT 1 FROM ingestion_demo.customers_scd2 t2
    WHERE t2.customer_id = source.customer_id
      AND t2.is_current = true
      AND t2.first_name = source.first_name
      AND t2.last_name  = source.last_name
      AND t2.email      = source.email
      AND t2.city       = source.city
      AND t2.state      = source.state
  )
"""

spark.sql(insert_sql)

# ---- Simpler two-step MERGE approach (more common in production) ----
# For clarity, here's an alternative using a view:

all_scd2_rows = spark.table("ingestion_demo.customers_scd2") \
    .unionByName(scd2_incoming
        .withColumn("eff_start_date", F.current_date())
        .withColumn("eff_end_date",   F.lit(None).cast("date"))
        .withColumn("is_current",     F.lit(True)),
        allowMissingColumns=True
    )

print("After SCD Type 2 — full history:")
spark.table("ingestion_demo.customers_scd2") \
    .orderBy("customer_id", "eff_start_date") \
    .show(truncate=False)


 ### 40‑D  Querying Current vs. Historical Views (SCD Type 2)


In [ ]:
# ---- Current view (most common query) ----
scd2_current = spark.table("ingestion_demo.customers_scd2") \
    .filter(F.col("is_current") == True)

print("Current customers (is_current = true):")
scd2_current.orderBy("customer_id").show(truncate=False)

# ---- Point-in-time query: "What did customers look like on 2025-01-15?" ----
query_date = date.today().isoformat()
pit_sql = f"""
  SELECT * FROM ingestion_demo.customers_scd2
  WHERE '{query_date}' >= eff_start_date
    AND ('{query_date}' < eff_end_date OR eff_end_date IS NULL)
"""
print(f"\nPoint-in-time view as of {query_date}:")
spark.sql(pit_sql).orderBy("customer_id").show(truncate=False)

# ---- Change history for a single customer ----
print("\nFull history for CUST-1001 (Alice):")
spark.table("ingestion_demo.customers_scd2") \
    .filter("customer_id = 'CUST-1001'") \
    .orderBy("eff_start_date") \
    .select("customer_id","first_name","city","state","eff_start_date","eff_end_date","is_current") \
    .show(truncate=False)


 ### 40‑E  SCD Type 2 MERGE — Production‑Ready Single Statement


In [ ]:
def scd_type2_merge(target_table, source_df, business_key, scd_columns):
    """
    Production-ready SCD Type 2 using a single MERGE statement.
    
    Steps:
      1. Expire old rows (set is_current=false, eff_end_date=now)
      2. Insert new rows for changed or new keys
    """
    source_df.createOrReplaceTempView("_scd2_source")
    current_date_val = date.today().isoformat()

    # Build the change-detection condition
    change_conditions = " OR ".join(
        f"target.{c} <> source.{c}" for c in scd_columns
    )

    # Step 1: Expire
    expire_merge = f"""
      MERGE INTO {target_table} AS target
      USING _scd2_source AS source
      ON target.{business_key} = source.{business_key} AND target.is_current = true
      WHEN MATCHED AND ({change_conditions}) THEN
        UPDATE SET target.is_current = false, target.eff_end_date = '{current_date_val}'
    """
    spark.sql(expire_merge)

    # Step 2: Insert new versions
    insert_merge = f"""
      MERGE INTO {target_table} AS target
      USING _scd2_source AS source
      ON target.{business_key} = source.{business_key} AND target.is_current = true
      WHEN NOT MATCHED THEN
        INSERT ({business_key}, {', '.join(scd_columns)}, eff_start_date, eff_end_date, is_current)
        VALUES (source.{business_key}, {', '.join(f'source.{c}' for c in scd_columns)}, '{current_date_val}', NULL, true)
    """
    spark.sql(insert_merge)

    return spark.table(target_table).count()

# ---- Test the function ----
spark.sql("DROP TABLE IF EXISTS ingestion_demo.cust_scd2_v2")

scd2_v2_initial = spark.createDataFrame(CUSTOMER_DATA, schema=customer_schema) \
    .withColumn("eff_start_date", F.current_date()) \
    .withColumn("eff_end_date",   F.lit(None).cast("date")) \
    .withColumn("is_current",     F.lit(True))

scd2_v2_initial.write.mode("overwrite").format("delta").saveAsTable("ingestion_demo.cust_scd2_v2")

# Apply new changes
new_changes = spark.createDataFrame([
    ("CUST-1001", "Alice",   "Smith",   "alice_final@example.com", "Miami",      "FL"),
    ("CUST-1007", "Grace",   "Hopper",  "grace@example.com",       "Boston",     "MA"),
], schema=customer_schema)

total = scd_type2_merge(
    "ingestion_demo.cust_scd2_v2",
    new_changes,
    business_key="customer_id",
    scd_columns=["first_name","last_name","email","city","state"]
)
print(f"Total rows after SCD2 merge: {total}")

print("\nFull SCD Type 2 dimension:")
spark.table("ingestion_demo.cust_scd2_v2") \
    .orderBy("customer_id","eff_start_date") \
    .select("customer_id","first_name","city","state","eff_start_date","eff_end_date","is_current") \
    .show(truncate=False)


 **SCD Type 1 vs Type 2 tradeoffs:**

 | Aspect              | Type 1                        | Type 2                           |
 |---------------------|-------------------------------|----------------------------------|
 | Storage             | Low (one row per key)         | Higher (one row per change)      |
 | Historical queries  | Not possible                  | Built‑in (eff dates + flag)      |
 | Implementation      | Simple UPDATE                 | MERGE with expire + insert       |
 | Use case            | Corrections, transient fields | Regulatory audit, change tracking|


 ---
 ## Cleanup (Optional)


In [ ]:
# Uncomment to clean up all demo tables and files:
# spark.sql("DROP DATABASE IF EXISTS ingestion_demo CASCADE")
# dbutils.fs.rm("dbfs:/FileStore/ingestion_demo", recurse=True)
# dbutils.fs.rm("dbfs:/FileStore/tables/ingestion_demo", recurse=True)

print("Cleanup skipped — remove comments above to execute.")


 ---
 ## Summary — Concepts 31 through 40

 | #  | Concept                              | Key Technique / Takeaway                                                     |
 |----|--------------------------------------|------------------------------------------------------------------------------|
 | 31 | COPY INTO vs. Auto Loader            | `COPY INTO` for batch (idempotent); Auto Loader for streaming with schema evo|
 | 32 | File Formats in the Lakehouse        | Delta/Parquet rule the lakehouse — columnar, compressed, predicate pushdown  |
 | 33 | Volumes                              | Governed file storage in Unity Catalog; use DBFS FileStore in CE             |
 | 34 | Auto Loader: File Discovery          | Directory listing (simple) vs. file notification (low‑latency)                |
 | 35 | Schema Evolution & Rescued Data      | `addNewColumns` + `_rescued_data` prevents data loss on schema drift        |
 | 36 | Multi-Hop Medallion Architecture     | Bronze (raw) → Silver (cleansed) → Gold (aggregates) — each a Delta table   |
 | 37 | Incremental Processing Patterns      | File‑level, CDF, or watermark‑based — pick by latency/cost requirements     |
 | 38 | Lakehouse Federation                 | Query external DBs without moving data; JDBC fallback in CE                   |
 | 39 | Idempotent Pipeline Design           | MERGE with natural keys, replaceWhere, checkpoint recovery                   |
 | 40 | SCD Type 1 & Type 2                  | Type 1 = overwrite; Type 2 = history with eff dates + current flag           |

 ---


 ## Self‑Assessment

 Answer these to validate your understanding:

 1. What makes `COPY INTO` idempotent without any user‑managed tracking table?
 2. Why does Parquet outperform CSV for analytical queries even when the same data is stored?
 3. How do Volumes differ from regular Delta tables in Unity Catalog?
 4. What are the two file‑discovery modes in Auto Loader and when would you choose each?
 5. What does the `_rescued_data` column capture, and why is it important?
 6. Draw the Medallion architecture layers and describe what happens at each stage.
 7. Compare file‑level, CDF, and watermark‑based incremental processing.
 8. When would you use Lakehouse Federation instead of a full ETL pipeline?
 9. Write a pseudocode `MERGE` statement that is safe to run multiple times without duplicating data.
 10. A customer moves from NY to CA.  What is the difference in query results between SCD Type 1 and SCD Type 2?

 **Happy Engineering!**


 &copy; 2025 — Databricks Data Engineering Learning Path — Concepts 31–40


---

# 05 Streaming



 # Structured Streaming in Databricks — Concepts #41–#50

 **Overview**: This notebook covers the second half of the Structured Streaming curriculum:
 structured streaming foundations, triggers, output modes, joins, windowing, foreachBatch,
 Lakeflow streaming tables, checkpointing, watermarks, and stream-stream joins.

 **Environment**: Designed for Databricks Community Edition (single node, free tier).
 Structured Streaming *is* available on Community Edition, but with limitations:
 - Single-node execution (no distributed recovery)
 - Checkpoint recovery may not survive full cluster restarts
 - Lakeflow / DLT pipelines are NOT available (covered conceptually)
 - Memory-based sinks are best for demonstration

 **Datasets Used**:
 - `rate` source for synthetic streaming data
 - File-based streaming (CSV files written to DBFS / local)
 - Simulated IoT, clickstream, and financial event data


 ### Setup — Create Directories and Clean Checkpoints


In [ ]:
import os
import shutil
import time
import random
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, LongType, DateType
)
from pyspark.sql.functions import (
    col, lit, expr, window, current_timestamp, count,
    sum as _sum, avg, max as _max, min as _min,
    to_timestamp, from_unixtime, unix_timestamp,
    struct, to_json, from_json, when, concat,
    date_format, year, month, dayofmonth, hour,
    minute, second, split, explode, array, rand,
    round as _round
)
from pyspark.sql.streaming import DataStreamWriter

print("✅ Spark session ready")

# ── Working directories ──
BASE_DIR = "/tmp/streaming_demo"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints"
STREAM_SRC_DIR  = f"{BASE_DIR}/source_csv"
DELTA_SINK_DIR  = f"{BASE_DIR}/delta_sink"
STATIC_DIR      = f"{BASE_DIR}/static"
DUAL_SINK_1     = f"{BASE_DIR}/dual_sink/events"
DUAL_SINK_2     = f"{BASE_DIR}/dual_sink/summary"

dirs = [BASE_DIR, CHECKPOINT_DIR, STREAM_SRC_DIR,
        DELTA_SINK_DIR, STATIC_DIR, DUAL_SINK_1, DUAL_SINK_2]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"  📁 {d}")

# Kill any lingering streams
for q in spark.streams.active:
    q.stop()
    print(f"  ⏹️  Stopped: {q.name}")

print("✅ Environment ready")


 ---
 ## Concept 41: Structured Streaming Fundamentals

 **Core Idea**: Structured Streaming treats a *stream* as an **unbounded, continuously-
 growing table**. New data arriving in the stream is appended as new rows.

 **How it differs from batch**:
 |   | Batch | Structured Streaming |
 |---|---|---|
 | Input | Finite, bounded | Unbounded, continuous |
 | Query | Runs once on static data | Runs continuously on arriving data |
 | Result | Single final result | Continuously updating result (or a new Delta table) |
 | Engine | Reads entire table | Reads micro-batches (or continuous processing) |

 **The two key APIs**:
 - `spark.readStream` — define the streaming DataFrame (source)
 - `df.writeStream`  — define the output (sink), trigger, and output mode

 Let's start with the simplest possible streaming query: `rate` → `console`.


In [ ]:
print("=" * 60)
print("CONCEPT 41 — Structured Streaming Fundamentals")
print("=" * 60)

# ── Simplest streaming query: rate source → console ──
streaming_df = (
    spark.readStream
    .format("rate")              # built-in source: generates increasing (timestamp, value) pairs
    .option("rowsPerSecond", 5)  # 5 rows per second
    .load()
)

print(f"rate source DataFrame type : {type(streaming_df).__name__}")
print(f"isStreaming               : {streaming_df.isStreaming}")
print(f"Schema:\n{streaming_df.printSchema()}")


In [ ]:
# ═══ Write the stream to a memory sink so we can query it ═══
memory_query = (
    streaming_df
    .writeStream
    .format("memory")               # memory sink → queryable via spark.sql()
    .queryName("rate_stream_q41")   # SQL table name
    .outputMode("append")
    .trigger(processingTime="5 seconds")
    .start()
)

print(f"✅ Streaming query started: {memory_query.name}")
print(f"   Status : {memory_query.status['message']}")
print()

# Let it accumulate for ~15 seconds
print("⏳ Accumulating data for 15 seconds...")
time.sleep(15)

# Query the in-memory table
df_mem = spark.sql("SELECT * FROM rate_stream_q41 ORDER BY timestamp DESC LIMIT 10")
print("\n📊 Latest 10 rows from memory sink:")
df_mem.show(truncate=False)

total_rows = spark.sql("SELECT COUNT(*) AS cnt FROM rate_stream_q41").collect()[0]['cnt']
print(f"\n📈 Total rows streamed: {total_rows}")

memory_query.stop()
print("⏹️  Streaming query stopped.")


 ### Concept 41 — Delta as Both Source AND Sink (Stream → Delta → Re-read)

 Delta Lake is *both* a streaming sink **and** a streaming source. This enables:
 - **Sink**: Stream writes continuously to a Delta table
 - **Source**: Another stream reads new Delta files as they are committed


In [ ]:
# ── Write rate data to a Delta table ──
delta_sink_path = f"{DELTA_SINK_DIR}/rate_sink_q41"

dbutils.fs.rm(delta_sink_path, True)

delta_query = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/q41_delta")
    .trigger(processingTime="10 seconds")
    .start(delta_sink_path)
)

print("📤 Streaming into Delta table...")
time.sleep(20)

# Read the Delta table as a static batch
df_batch = spark.read.format("delta").load(delta_sink_path)
print(f"📊 Delta table row count: {df_batch.count()}")
df_batch.orderBy("timestamp", ascending=False).show(5, truncate=False)

# Now treat the same Delta table as a *streaming source*!
df_streaming_source = (
    spark.readStream
    .format("delta")
    .load(delta_sink_path)
)
print(f"\n🔁 Delta as streaming source — isStreaming: {df_streaming_source.isStreaming}")

delta_query.stop()
print("⏹️  Delta sink query stopped.")


 ---
 ## Concept 42: Triggers

 **Triggers** control *how often* the streaming engine processes data.

 | Trigger Type | Behavior | Best For |
 |---|---|---|
 | `processingTime="10 seconds"` | Micro-batch every N seconds | Low-latency, continuous |
 | `availableNow=True` | Process all available data, then stop | One-time backfills, testing |
 | `once=True` (deprecated) | Process one micro-batch and stop | Legacy; prefer `availableNow` |
 | `continuous="10 seconds"` | True continuous mode (not in Community) | Sub-ms latency (not in CE) |

 **Choosing the right trigger**:
 - Low latency needed → `processingTime` with short interval
 - Cost-sensitive / batch-like → longer `processingTime` or `availableNow`
 - Backfill historical data → `availableNow`


In [ ]:
print("=" * 60)
print("CONCEPT 42 — Triggers")
print("=" * 60)

# ── Trigger 1: processingTime ──
print("\n📌 TRIGGER TYPE 1: processingTime='5 seconds'")
print("   Micro-batch every 5 seconds — data accumulates, then processes\n")

mem_q_processing = (
    spark.readStream.format("rate").option("rowsPerSecond", 3).load()
    .writeStream
    .format("memory")
    .queryName("trigger_processing_q42")
    .outputMode("append")
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(12)
cnt1 = spark.sql("SELECT COUNT(*) AS c FROM trigger_processing_q42").collect()[0]['c']
print(f"   Rows accumulated (processingTime): {cnt1}")
mem_q_processing.stop()

# ── Trigger 2: availableNow ──
print("\n📌 TRIGGER TYPE 2: availableNow=True")
print("   Process ALL available data in one shot, then stop.\n")

# Write a batch of CSV files to simulate "available now" data
available_dir = f"{STREAM_SRC_DIR}/available_now_q42"
dbutils.fs.rm(available_dir, True)
os.makedirs(available_dir, exist_ok=True)

# Generate CSV files
schema_csv = StructType([
    StructField("id", IntegerType()),
    StructField("value", StringType()),
    StructField("ts", TimestampType())
])

for batch in range(3):
    rows = [(i, f"batch_{batch}_val_{i}",
             f"2026-05-07 {(batch * 10 + i) // 60:02d}:{(batch * 10 + i) % 60:02d}:00")
            for i in range(20)]
    df_csv = spark.createDataFrame(rows, schema_csv)
    df_csv.coalesce(1).write.mode("append").csv(available_dir)

# Read CSV as stream with availableNow trigger
schema_infer = StructType([
    StructField("id", IntegerType()),
    StructField("value", StringType()),
    StructField("ts", TimestampType())
])

df_avail = (
    spark.readStream
    .schema(schema_infer)
    .option("maxFilesPerTrigger", 1)
    .csv(available_dir)
)

avail_query = (
    df_avail
    .writeStream
    .format("memory")
    .queryName("trigger_avail_now_q42")
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

# availableNow stops automatically after processing
avail_query.awaitTermination(60)
time.sleep(2)

cnt2 = spark.sql("SELECT COUNT(*) AS c FROM trigger_avail_now_q42").collect()[0]['c']
print(f"   Rows processed (availableNow): {cnt2}  (should be ~60)")

print("\n✅ Both trigger types demonstrated.")


 ---
 ## Concept 43: Output Modes

 **Output modes** determine *what* data is written to the sink on each trigger:

 | Mode | Behavior | Allowed With |
 |---|---|---|
 | **Append** | Only new rows appended since last trigger | Any query without aggregation (or with watermark) |
 | **Complete** | Entire result table is written every trigger | Aggregation queries only |
 | **Update** | Only rows that changed since last trigger | Aggregations + watermarked operations |

 **Key Rule**: Aggregations (groupBy) produce "updating" results — you CANNOT use
 Append mode without a watermark. Use Complete for small state, Update for efficiency.


In [ ]:
print("=" * 60)
print("CONCEPT 43 — Output Modes")
print("=" * 60)

# ── Preview the rate data structure ──
rate_df = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

# Add a random "category" to demonstrate grouping
import random
from pyspark.sql.types import StructType, StructField, LongType

def generate_categorized():
    """Simulate IoT sensor categories: A=temp, B=humidity, C=pressure"""
    df = spark.range(100)
    return df.select(
        col("id"),
        (col("id") % 3).cast("int").alias("category"),
        (_round(rand() * 100, 2)).alias("value"),
        current_timestamp().alias("ts")
    )

# We'll use rate source with random categories for streaming
rate_with_cat = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .withColumn("category", (col("value") % 3).cast("int"))
    .withColumn("measurement", _round(rand() * 100, 2))
)

# ── MODE 1: APPEND (no aggregation) ──
print("\n📌 OUTPUT MODE 1: append")
print("   Only new rows since last trigger are output.\n")

q_append = (
    rate_with_cat
    .writeStream
    .format("memory")
    .queryName("mode_append_q43")
    .outputMode("append")
    .trigger(processingTime="6 seconds")
    .start()
)

time.sleep(12)
cnt_append = spark.sql("SELECT COUNT(*) AS c FROM mode_append_q43").collect()[0]['c']
print(f"   Rows in append sink: {cnt_append}")
spark.sql("SELECT * FROM mode_append_q43 LIMIT 5").show()
q_append.stop()

# ── MODE 2: COMPLETE (with aggregation) ──
print("\n📌 OUTPUT MODE 2: complete")
print("   ENTIRE result table is recomputed and output every trigger.\n")

agg_df = rate_with_cat.groupBy("category").agg(
    _sum("measurement").alias("total_measurement"),
    count("*").alias("row_count")
)
q_complete = (
    agg_df
    .writeStream
    .format("memory")
    .queryName("mode_complete_q43")
    .outputMode("complete")
    .trigger(processingTime="6 seconds")
    .start()
)

time.sleep(12)
print("   Complete mode result (entire table):")
spark.sql("SELECT * FROM mode_complete_q43 ORDER BY category").show()
q_complete.stop()

# ── MODE 3: UPDATE (only changed rows) ──
print("\n📌 OUTPUT MODE 3: update")
print("   Only rows that CHANGED since last trigger.\n")

q_update = (
    agg_df
    .writeStream
    .format("memory")
    .queryName("mode_update_q43")
    .outputMode("update")
    .trigger(processingTime="6 seconds")
    .start()
)

time.sleep(12)
print("   Update mode result (changed rows only):")
spark.sql("SELECT * FROM mode_update_q43 ORDER BY category").show()
q_update.stop()

print("\n✅ All three output modes demonstrated.")


 ---
 ## Concept 44: Stream-Static Joins

 **Pattern**: Join a streaming DataFrame with a *static* (batch) DataFrame to enrich
 events with reference data. The classic use case: enrich raw IoT sensor readings
 with device metadata.

 **Important**: The static side is re-read for **every micro-batch**. For large static
 tables, broadcast the static DataFrame to avoid repeated shuffles.

 **Limitations**:
 - ✅ Inner join, left-outer, right-outer, cross join: Supported
 - ⚠️ Full outer join: NOT supported on streaming DataFrames
 - ✅ Static side can be any size (broadcast if large)


In [ ]:
print("=" * 60)
print("CONCEPT 44 — Stream-Static Joins")
print("=" * 60)

# ── Create a static "device catalog" dimension table ──
device_catalog = spark.createDataFrame([
    (0, "Temperature Sensor",   "Warehouse A",       "temp"),
    (1, "Humidity Sensor",      "Warehouse A",       "humidity"),
    (2, "Pressure Sensor",      "Warehouse B",       "pressure"),
    (3, "Vibration Sensor",     "Production Floor",   "vibration"),
    (4, "Light Sensor",         "Office",            "lux"),
    (5, "CO2 Sensor",           "Office",            "co2"),
    (6, "Flow Meter",           "Pipeline C",        "flow"),
    (7, "pH Sensor",            "Tank D",            "ph"),
    (8, "Conductivity Sensor",  "Tank D",            "conductivity"),
    (9, "Thermocouple",         "Furnace",           "temp"),
], ["sensor_id", "sensor_name", "location", "type"])

device_catalog.cache()
print("📋 Static Device Catalog:")
device_catalog.show(truncate=False)

# ── Streaming source: simulated IoT events ──
stream_events = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 8)
    .load()
    .withColumn("sensor_id", (col("value") % 10).cast("int"))
    .withColumn("reading", _round(rand() * 100, 2))
    .withColumn("event_time", current_timestamp())
)

# Perform stream-static join (inner)
enriched_stream = (
    stream_events
    .join(device_catalog, on="sensor_id", how="inner")
    .select(
        "event_time",
        "sensor_id",
        "sensor_name",
        "location",
        "type",
        "reading"
    )
)

print("\n🔗 Streaming events INNER JOIN static catalog:")
enriched_stream.printSchema()

q_enrich_inner = (
    enriched_stream
    .writeStream
    .format("memory")
    .queryName("enriched_events_q44")
    .outputMode("append")
    .trigger(processingTime="6 seconds")
    .start()
)

time.sleep(12)
print("\n📊 Enriched Events (stream + static join):")
spark.sql("SELECT * FROM enriched_events_q44 ORDER BY event_time DESC LIMIT 10").show(truncate=False)
q_enrich_inner.stop()

# ── LEFT OUTER join: keep ALL events, even with no matching device ──
print("\n🔗 LEFT OUTER join (all events, nulls for unmatched sensors):")
enriched_left = (
    stream_events
    .join(device_catalog, on="sensor_id", how="left_outer")
    .select("event_time", "sensor_id", "sensor_name", "location", "reading")
)

q_enrich_left = (
    enriched_left
    .writeStream
    .format("memory")
    .queryName("enriched_left_q44")
    .outputMode("append")
    .trigger(processingTime="6 seconds")
    .start()
)

time.sleep(12)
spark.sql("""
    SELECT sensor_id, sensor_name, COUNT(*) AS cnt
    FROM enriched_left_q44
    GROUP BY sensor_id, sensor_name
    ORDER BY sensor_id
""").show(12, truncate=False)
q_enrich_left.stop()

print("✅ Stream-static joins demonstrated (inner + left-outer).")


 ---
 ## Concept 45: Windowed Aggregations

 **Window types**:

 - **Tumbling window**: `window(event_time, "10 minutes")` — fixed, non-overlapping
   (slide = window duration → no overlap)

 - **Sliding window**: `window(event_time, "10 minutes", "5 minutes")` — overlapping windows
   (slide < window duration → overlap)

 **Without a watermark**, Spark keeps ALL state forever (unbounded memory).
 For production, always pair windows with a watermark (Concept #49).


In [ ]:
print("=" * 60)
print("CONCEPT 45 — Windowed Aggregations")
print("=" * 60)

# ── Source: simulated stock trade events ──
# We'll use rate source + transformations to simulate trades
trade_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
    .withColumn("symbol", concat(lit("STK-"), (col("value") % 5 + 1).cast("string")))
    .withColumn("price", (_round(rand() * 1000 + 50, 2)).cast(DoubleType()))
    .withColumn("quantity", ((col("value") % 10 + 1)).cast(IntegerType()))
    .withColumn("trade_time", current_timestamp())
)

# ── TUMBLING WINDOW (10-second windows, no overlap) ──
print("\n📌 TUMBLING WINDOW (window = 10 sec, slide = N/A → no overlap):")

tumbling_agg = (
    trade_stream
    .groupBy(
        col("symbol"),
        window(col("trade_time"), "10 seconds")  # tumbling: slide == duration
    )
    .agg(
        count("*").alias("trade_count"),
        _round(_sum("price"), 2).alias("total_value"),
        _round(avg("price"), 2).alias("avg_price")
    )
)

q_tumble = (
    tumbling_agg
    .writeStream
    .format("memory")
    .queryName("window_tumble_q45")
    .outputMode("complete")  # complete for un-watermarked aggregation
    .trigger(processingTime="10 seconds")
    .start()
)

time.sleep(22)
print("   Tumbling window aggregation (10-second windows):")
spark.sql("""
    SELECT symbol, window, trade_count, total_value, avg_price
    FROM window_tumble_q45
    ORDER BY window.start, symbol
    LIMIT 15
""").show(15, truncate=False)
q_tumble.stop()

# ── SLIDING WINDOW (10-second window, 5-second slide = overlap) ──
print("\n📌 SLIDING WINDOW (window = 10 sec, slide = 5 sec → overlapping):")

sliding_agg = (
    trade_stream
    .groupBy(
        col("symbol"),
        window(col("trade_time"), "10 seconds", "5 seconds")
    )
    .agg(
        count("*").alias("trade_count"),
        _round(avg("price"), 2).alias("avg_price")
    )
)

q_slide = (
    sliding_agg
    .writeStream
    .format("memory")
    .queryName("window_slide_q45")
    .outputMode("complete")
    .trigger(processingTime="10 seconds")
    .start()
)

time.sleep(22)
print("   Sliding window aggregation (10s windows, 5s slide):")
spark.sql("""
    SELECT symbol, window, trade_count, avg_price
    FROM window_slide_q45
    ORDER BY window.start, window.end, symbol
    LIMIT 20
""").show(20, truncate=False)
q_slide.stop()

print("\n✅ Tumbling and sliding windows demonstrated.")


 ---
 ## Concept 46: foreachBatch Pattern

 `foreachBatch` lets you apply **arbitrary batch logic** to each micro-batch of a
 streaming query. This is the most flexible sink — you can:

 - Write to **multiple sinks** from a single streaming query
 - Perform **MERGE (upsert)** operations in Delta
 - Apply custom transformations
 - Call external services (APIs) per micro-batch

 ⚠️ **Exactly-once warning**: If you write to external systems from foreachBatch,
 you must ensure your logic is **idempotent** (Spark may replay micro-batches).


In [ ]:
print("=" * 60)
print("CONCEPT 46 — foreachBatch Pattern")
print("=" * 60)

# Prepare Delta sink for MERGE
delta_merge_path = f"{DELTA_SINK_DIR}/foreachBatch_merge_q46"
dbutils.fs.rm(delta_merge_path, True)

# Create an empty Delta table with schema
merge_schema = StructType([
    StructField("sensor_id", IntegerType()),
    StructField("latest_reading", DoubleType()),
    StructField("reading_count", LongType()),
    StructField("last_updated", TimestampType())
])
empty_df = spark.createDataFrame([], merge_schema)
empty_df.write.format("delta").mode("overwrite").save(delta_merge_path)
print(f"📁 Empty Delta table created at: {delta_merge_path}")

# ── foreachBatch: write to BOTH a memory sink AND a Delta table with MERGE ──
source_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .withColumn("sensor_id", (col("value") % 5).cast("int"))
    .withColumn("reading", _round(rand() * 100, 2))
)

def process_micro_batch(df_batch, epoch_id):
    """
    Called for each micro-batch.
    1. Aggregate the micro-batch
    2. Merge into Delta table (upsert by sensor_id)
    3. Print a summary
    """
    print(f"\n🔹 foreachBatch — epoch_id={epoch_id}, rows in batch={df_batch.count()}")

    # Aggregate this micro-batch
    agg_batch = (
        df_batch
        .groupBy("sensor_id")
        .agg(
            _round(avg("reading"), 2).alias("avg_reading_batch"),
            count("*").alias("batch_count")
        )
        .withColumn("microbatch_epoch", lit(epoch_id))
        .withColumn("processed_time", current_timestamp())
    )

    # Merge into Delta: upsert per sensor_id
    from delta.tables import DeltaTable
    delta_table = DeltaTable.forPath(spark, delta_merge_path)

    (delta_table.alias("target")
     .merge(agg_batch.alias("source"),
            "target.sensor_id = source.sensor_id")
     .whenMatchedUpdate(set={
         "latest_reading": "source.avg_reading_batch",
         "reading_count": "target.reading_count + source.batch_count",
         "last_updated": "source.processed_time"
     })
     .whenNotMatchedInsert(values={
         "sensor_id": "source.sensor_id",
         "latest_reading": "source.avg_reading_batch",
         "reading_count": "source.batch_count",
         "last_updated": "source.processed_time"
     })
     .execute())

    print(f"   ✅ Merged {agg_batch.count()} sensor aggregates into Delta")

q_foreach = (
    source_stream
    .writeStream
    .foreachBatch(process_micro_batch)
    .outputMode("append")
    .trigger(processingTime="8 seconds")
    .start()
)

time.sleep(30)
q_foreach.stop()

# ── Show the merged Delta table ──
print("\n📊 Final Delta table (upserted by sensor_id):")
spark.read.format("delta").load(delta_merge_path).show(truncate=False)

print("\n✅ foreachBatch pattern with Delta MERGE demonstrated.")


 ### foreachBatch — Multiple Sinks from a Single Stream

 One of the most powerful patterns: split a single streaming query into multiple
 outputs (e.g., raw events + aggregated metrics).


In [ ]:
print("\n─── foreachBatch: Multiple Sinks ───")

# Clean up dual-sink directories
for dp in [DUAL_SINK_1, DUAL_SINK_2]:
    dbutils.fs.rm(dp, True)

events_schema = StructType([
    StructField("event_id", LongType()),
    StructField("user_id", IntegerType()),
    StructField("action", StringType()),
    StructField("page", StringType()),
    StructField("event_time", TimestampType()),
])

clickstream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 8)
    .load()
    .withColumn("user_id", (col("value") % 20).cast("int"))
    .withColumn("action",
                when((col("value") % 4) == 0, "click")
                .when((col("value") % 4) == 1, "view")
                .when((col("value") % 4) == 2, "add_to_cart")
                .otherwise("purchase"))
    .withColumn("page", concat(lit("/page/"), (col("value") % 7 + 1).cast("string")))
    .withColumn("event_time", current_timestamp())
    .select("value", "user_id", "action", "page", "event_time")
    .withColumnRenamed("value", "event_id")
)

def write_to_multiple_sinks(df_batch, epoch_id):
    print(f"\n🔹 Multi-sink epoch={epoch_id}, rows={df_batch.count()}")

    # Sink 1: raw events to Delta
    df_batch.write.format("delta").mode("append").save(DUAL_SINK_1)

    # Sink 2: per-user summary to Delta
    summary = (
        df_batch
        .groupBy("user_id")
        .agg(
            count("*").alias("event_count"),
            count(when(col("action") == "purchase", 1)).alias("purchases"),
            collect_list("action").alias("action_list")
        )
        .withColumn("epoch_id", lit(epoch_id))
    )
    summary.write.format("delta").mode("append").save(DUAL_SINK_2)

    print(f"   ✅ Wrote to both sinks")

q_multi = (
    clickstream
    .writeStream
    .foreachBatch(write_to_multiple_sinks)
    .outputMode("append")
    .trigger(processingTime="6 seconds")
    .start()
)

time.sleep(15)
q_multi.stop()

print("\n📊 Sink 1 — Raw events:")
spark.read.format("delta").load(DUAL_SINK_1).show(10, truncate=False)

print("\n📊 Sink 2 — Per-user summaries:")
spark.read.format("delta").load(DUAL_SINK_2).show(10, truncate=False)

print("\n✅ Multiple sinks from a single stream demo complete.")


 ---
 ## Concept 47: Streaming Tables in Lakeflow Pipelines (DLT)

 ⚠️ **NOTE**: Delta Live Tables (DLT / Lakeflow Pipelines) are **NOT available** in
 Community Edition. This section explains the concepts and shows manual equivalents.

 **What are Streaming Tables?**
 - **Managed streaming ingestion** in DLT — you declare a table "as streaming" using
   the `STREAMING` keyword (or `readStream` in Python DLT decorators)
 - DLT auto-manages **checkpoints**, **schema evolution**, and **error handling**
 - Streaming tables are always "append-only" — they append rows from the source

 **DLT Streaming Table (Python — NOT runnable in CE)**:
 ```python
 import dlt
 @dlt.table(name="bronze_events")
 def bronze():
     return spark.readStream.format("cloudFiles").load(source_path)
 ```

 **vs. Manual Structured Streaming**:
 | Feature | DLT Streaming Tables | Manual Structured Streaming |
 |---|---|---|
 | Checkpoint mgmt | Auto | Manual (must specify path) |
 | Schema enforcement | Built-in expectations | Manual validation |
 | Error handling | Dead letter queue | Manual try/except or skip |
 | Monitoring | DLT event log | `spark.streams.active`, metrics |
 | Restartability | Full auto-recovery | Checkpoint-dependent |

 Below is the **manual equivalent** of a streaming table:


In [ ]:
print("=" * 60)
print("CONCEPT 47 — Streaming Tables (Manual Equivalent)")
print("=" * 60)

delta_streaming_table_path = f"{DELTA_SINK_DIR}/manual_streaming_table_q47"
dbutils.fs.rm(delta_streaming_table_path, True)

# ── Manual "streaming table": readStream → writeStream to Delta ──
streaming_source = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .withColumn("processed_at", current_timestamp())
    .withColumn("batch_id", col("timestamp"))
)

manual_streaming_table = (
    streaming_source
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/manual_streaming_q47")
    .trigger(processingTime="5 seconds")
    .start(delta_streaming_table_path)
)

print("📤 Manual streaming table writing to Delta...")
time.sleep(15)

# Read the "streaming table" as a batch
df_streaming_table = spark.read.format("delta").load(delta_streaming_table_path)
print(f"\n📊 Manual streaming table row count: {df_streaming_table.count()}")
df_streaming_table.orderBy(col("timestamp").desc()).show(5, truncate=False)

manual_streaming_table.stop()

# ── What DLT handles for you in production ──
print("""
╔═══════════════════════════════════════════════════════════════════╗
║  WHAT DLT / LAKEFLOW PROVIDES (not in Community Edition):         ║
║  • Auto-managed checkpoints (no need to specify paths)            ║
║  • Schema enforcement & evolution                                 ║
║  • Data quality constraints (expectations)                        ║
║  • Error isolation (bad records → dead letter queue)              ║
║  • Auto-scaling compute                                           ║
║  • Lineage tracking in Unity Catalog                             ║
║  • Incremental materialized views (change-data capture)           ║
╚═══════════════════════════════════════════════════════════════════╝
""")
print("✅ Concept 47 — DLT streaming tables explained.")


 ---
 ## Concept 48: Checkpointing & Exactly-Once Guarantees

 **Checkpointing** is the mechanism that enables fault-tolerance and exactly-once
 semantics in Structured Streaming.

 **What's in a checkpoint directory?**
 - `offsets/` — the position in the source stream (which data has been consumed)
 - `commits/` — which micro-batches completed successfully
 - `metadata` — streaming query identity
 - `state/` — aggregation state (for stateful operations with watermark)

 **Exactly-once boundaries**:
 - ✅ **Source**: replayable sources (Kafka, Delta, Kinesis) → exactly-once
 - ✅ **Sink**: idempotent sinks (Delta, file with `_spark_metadata`) → exactly-once
 - ⚠️ **External systems** (foreachBatch → REST APIs) → at-least-once (must be idempotent)

 **Golden rule**: Each streaming query must have its own **unique, stable checkpoint location**.


In [ ]:
print("=" * 60)
print("CONCEPT 48 — Checkpointing & Exactly-Once")
print("=" * 60)

checkpoint_q48 = f"{CHECKPOINT_DIR}/checkpoint_demo_q48"
dbutils.fs.rm(checkpoint_q48, True)

# ── Run a stream with a checkpoint ──
q_with_checkpoint = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .withColumn("processed", current_timestamp())
    .writeStream
    .format("memory")
    .queryName("checkpointed_stream_q48")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_q48)
    .trigger(processingTime="5 seconds")
    .start()
)

print("📤 Streaming with checkpoint...")
time.sleep(15)
rows_before_restart = spark.sql("SELECT COUNT(*) AS c FROM checkpointed_stream_q48").collect()[0]['c']
print(f"   Rows before simulated restart: {rows_before_restart}")

# Stop the query (simulates failure / restart)
q_with_checkpoint.stop()
print("⏹️  Stream stopped (simulating failure).")

# ── Inspect checkpoint directory ──
print(f"\n📁 Checkpoint directory contents ({checkpoint_q48}):")
for root, dirs, files in os.walk(checkpoint_q48):
    level = root.replace(checkpoint_q48, "").count(os.sep)
    indent = "  " * level
    print(f"  {indent}{os.path.basename(root) or '.'}/")
    for f in files[:5]:
        print(f"  {indent}  📄 {f}")

# ── Restart the stream with the SAME checkpoint ──
# Note: memory sink query name must be unique, so we use a new name but same checkpoint
q_restarted = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .withColumn("processed", current_timestamp())
    .writeStream
    .format("memory")
    .queryName("checkpointed_stream_q48_restarted")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_q48)
    .trigger(processingTime="5 seconds")
    .start()
)

print("\n🔄 Stream restarted with SAME checkpoint...")
time.sleep(15)
rows_after = spark.sql("SELECT COUNT(*) AS c FROM checkpointed_stream_q48_restarted").collect()[0]['c']
print(f"   Rows in restarted stream: {rows_after}")
print(f"   ⚠️  NOTE: With a NEW memory sink name, we start fresh.")
print(f"   In production (Kafka/Delta), checkpoint ensures NO data loss/duplication across restarts.")
q_restarted.stop()

# ── What happens if you DELETE the checkpoint? ──
dbutils.fs.rm(checkpoint_q48, True)
print(f"\n🗑️  Checkpoint deleted: {checkpoint_q48}")
print("   If you restart without the checkpoint:")
print("   • All offsets are lost → stream re-reads from start (DUPLICATE DATA)")
print("   • All state is lost → aggregations start from scratch")
print("   • NEVER delete a production checkpoint unless you understand the consequences!")

print("\n✅ Checkpointing & exactly-once semantics demonstrated.")


 ---
 ## Concept 49: Watermarks & Late Data

 **Problem**: With streaming aggregations, intermediate state grows unboundedly.
 Spark must remember ALL seen keys FOREVER — memory leak!

 **Solution**: `withWatermark()` — tells Spark "wait up to N seconds for late data,
 then drop old state."

 **Tradeoff**:
 - **Shorter watermark** = less state, lower memory, BUT more data labeled "too late"
 - **Longer watermark** = more completeness, BUT more memory and latency

 **How it works**:
 - Spark tracks the maximum event-time seen so far
 - Watermark = max_event_time - watermark_duration
 - Any window with end < watermark is DROPPED (state cleaned up)
 - New data with timestamp < watermark is considered "too late" and DROPPED


In [ ]:
print("=" * 60)
print("CONCEPT 49 — Watermarks & Late Data")
print("=" * 60)

# ── Create a stream with explicit timestamps for watermark demo ──
# Simulate events with timestamps that are explicitly set (not current_timestamp)
# This lets us demonstrate late-arriving data clearly

from pyspark.sql.functions import expr as spark_expr

watermark_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 20)
    .load()
    .withColumn("event_id", col("value"))
    .withColumn("sensor", concat(lit("sensor_"), (col("value") % 3).cast("string")))
    .withColumn("reading", _round(rand() * 100 + 20, 2))
    # Simulate event time: 50% of events are "on time", 50% are delayed by 0-90 sec
    .withColumn("delay_seconds", (col("value") % 90).cast("int"))
    .withColumn("event_time",
                spark_expr("timestamp - INTERVAL delay_seconds SECONDS"))
)

# ── Aggregation WITH watermark ──
print("\n📌 WATERMARK = 30 seconds")
print("   Events more than 30 seconds late are dropped.")
print("   State for windows > 30 sec old is cleared.\n")

windowed_with_watermark = (
    watermark_stream
    .withWatermark("event_time", "30 seconds")
    .groupBy(
        col("sensor"),
        window(col("event_time"), "30 seconds", "10 seconds")
    )
    .agg(
        count("*").alias("event_count"),
        _round(avg("reading"), 2).alias("avg_reading")
    )
)

q_watermark = (
    windowed_with_watermark
    .writeStream
    .format("memory")
    .queryName("watermarked_q49")
    .outputMode("update")  # MUST be "update" or "append" with watermark
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/watermark_q49")
    .trigger(processingTime="10 seconds")
    .start()
)

time.sleep(30)
print("   Results with watermark (update mode):")
spark.sql("""
    SELECT sensor, window, event_count, avg_reading
    FROM watermarked_q49
    ORDER BY window.start DESC, sensor
    LIMIT 15
""").show(15, truncate=False)
q_watermark.stop()

# ── CONTRAST: Aggregation WITHOUT watermark ──
print("\n📌 WITHOUT WATERMARK (unbounded state — memory grows forever):")
print("   (Using complete mode since no watermark → append not allowed)")

windowed_no_watermark = (
    watermark_stream
    .groupBy(
        col("sensor"),
        window(col("event_time"), "30 seconds", "10 seconds")
    )
    .agg(
        count("*").alias("event_count"),
        _round(avg("reading"), 2).alias("avg_reading")
    )
)

q_no_watermark = (
    windowed_no_watermark
    .writeStream
    .format("memory")
    .queryName("no_watermark_q49")
    .outputMode("complete")
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/no_watermark_q49")
    .trigger(processingTime="10 seconds")
    .start()
)

time.sleep(20)
print("   Results WITHOUT watermark (complete mode — state NOT cleaned up):")
spark.sql("""
    SELECT sensor, window, event_count, avg_reading
    FROM no_watermark_q49
    ORDER BY window.start DESC, sensor
    LIMIT 15
""").show(15, truncate=False)
q_no_watermark.stop()

print("""
╔═══════════════════════════════════════════════════════════════════╗
║  WATERMARK SUMMARY:                                               ║
║  • With watermark=30s → state cleaned after ~30s from max event   ║
║  • Without watermark  → state grows FOREVER (memory leak)         ║
║  • Update mode requires watermark for aggregation                 ║
║  • Append mode requires watermark for aggregation                 ║
║  • Complete mode doesn't need watermark (but recomputes EVERYTHING)║
╚═══════════════════════════════════════════════════════════════════╝
""")
print("✅ Watermarks demonstrated — state management vs completeness tradeoff.")


 ---
 ## Concept 50: Stream-Stream Joins

 **The hardest streaming problem**: Joining *two* unbounded streams.

 **Requirements for stream-stream joins**:
 1. **Watermarks on BOTH streams** — Spark must know when to stop waiting for late rows
 2. **Time constraint in join condition** — "join within N seconds of each other"
 3. **State management** — Spark keeps rows in buffer for watermark duration on both sides

 **Join types supported**: Inner, left-outer, right-outer (all require watermarks)

 **How it works**: Spark buffers rows from each stream. For each new row, it looks
 up the *other* stream's buffer. When watermark advances past a row's timestamp,
 the row is evicted from state (prevents infinite memory growth).


In [ ]:
print("=" * 60)
print("CONCEPT 50 — Stream-Stream Joins")
print("=" * 60)

# ── Two simulated streams: "impressions" and "clicks" ──
# Stream 1: Ad impressions
impressions = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 15)
    .load()
    .withColumn("ad_id", (col("value") % 5).cast("string"))
    .withColumn("user_id", (col("value") % 10).cast("int"))
    .withColumn("impression_time", current_timestamp())
    .selectExpr(
        "ad_id",
        "user_id",
        "impression_time",
        "CAST(CONCAT('imp_', CAST(value AS STRING)) AS STRING) AS impression_id"
    )
)

# Stream 2: Ad clicks (slightly different rate to create realistic data)
clicks = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .withColumn("ad_id", (col("value") % 5).cast("string"))
    .withColumn("user_id", (col("value") % 12).cast("int"))
    .withColumn("click_time", current_timestamp())
    .selectExpr(
        "ad_id",
        "user_id",
        "click_time",
        "CAST(CONCAT('click_', CAST(value AS STRING)) AS STRING) AS click_id"
    )
)

# ── Stream-stream INNER JOIN with watermarks on both sides ──
impressions_watermarked = impressions.withWatermark("impression_time", "2 minutes")
clicks_watermarked = clicks.withWatermark("click_time", "2 minutes")

# Join condition: same ad_id AND same user_id AND click within 1 minute of impression
stream_join_result = (
    impressions_watermarked.alias("imp")
    .join(
        clicks_watermarked.alias("clk"),
        expr("""
            imp.ad_id = clk.ad_id
            AND imp.user_id = clk.user_id
            AND clk.click_time >= imp.impression_time
            AND clk.click_time <= imp.impression_time + INTERVAL 1 MINUTE
        """),
        "inner"
    )
    .select(
        col("imp.ad_id"),
        col("imp.user_id"),
        col("imp.impression_time"),
        col("clk.click_time"),
        col("imp.impression_id"),
        col("clk.click_id")
    )
)

print("📌 Stream-Stream INNER JOIN with watermarks on both streams")
print("   Condition: same ad_id + same user_id + click within 60s of impression\n")

q_ss_join = (
    stream_join_result
    .writeStream
    .format("memory")
    .queryName("stream_stream_join_q50")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/stream_join_q50")
    .trigger(processingTime="10 seconds")
    .start()
)

time.sleep(25)
print("   Stream-stream join results:")
try:
    result = spark.sql("""
        SELECT ad_id, user_id, impression_time, click_time, impression_id, click_id
        FROM stream_stream_join_q50
        ORDER BY impression_time DESC
        LIMIT 15
    """)
    if result.count() > 0:
        result.show(15, truncate=False)
    else:
        print("   ⚠️  No joins found in this short window (try running longer).")
        print("   This is expected — join conditions are restrictive with small sample.")
except Exception as e:
    print(f"   ⚠️  No results yet: {str(e)[:100]}")

q_ss_join.stop()

# ── LEFT OUTER stream-stream join (less restrictive, shows unmatched rows) ──
print("\n📌 Stream-Stream LEFT OUTER JOIN (impressions with optional clicks):")

stream_join_left = (
    impressions_watermarked.alias("imp")
    .join(
        clicks_watermarked.alias("clk"),
        expr("""
            imp.ad_id = clk.ad_id
            AND imp.user_id = clk.user_id
            AND clk.click_time >= imp.impression_time
            AND clk.click_time <= imp.impression_time + INTERVAL 1 MINUTE
        """),
        "left_outer"
    )
    .select(
        col("imp.ad_id"),
        col("imp.user_id"),
        col("imp.impression_time"),
        col("imp.impression_id"),
        col("clk.click_id")
    )
)

q_ss_left = (
    stream_join_left
    .writeStream
    .format("memory")
    .queryName("stream_stream_left_q50")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/stream_join_left_q50")
    .trigger(processingTime="10 seconds")
    .start()
)

time.sleep(20)
print("   Left outer join results (impressions with or without clicks):")
try:
    result_left = spark.sql("""
        SELECT ad_id, user_id, impression_time, impression_id, click_id
        FROM stream_stream_left_q50
        ORDER BY impression_time DESC
        LIMIT 15
    """)
    if result_left.count() > 0:
        result_left.show(15, truncate=False)
    else:
        print("   ⚠️  No results in this short window.")
except Exception as e:
    print(f"   ⚠️  {str(e)[:100]}")

q_ss_left.stop()

print("""
╔═══════════════════════════════════════════════════════════════════╗
║  STREAM-STREAM JOIN SUMMARY:                                      ║
║  • Both streams MUST have watermarks                              ║
║  • Join condition MUST include time constraint                    ║
║  • Spark buffers rows up to watermark duration on BOTH sides      ║
║  • Memory = both-side-state ≈ rows_per_sec × watermark_duration   ║
║  • Inner, left-outer, right-outer: supported                      ║
║  • Full outer: NOT supported (unbounded null padding)             ║
║  • Cross join: NOT supported on streams                           ║
╚═══════════════════════════════════════════════════════════════════╝
""")
print("✅ Stream-stream joins demonstrated.")


 ---
 ## 📊 Summary — Concepts #41–#50

 | # | Concept | Key Takeaway |
 |---|---------|---------------|
 | 41 | Structured Streaming Fundamentals | Stream = unbounded table; `readStream` / `writeStream` |
 | 42 | Triggers | `processingTime` for latency, `availableNow` for batch-like |
 | 43 | Output Modes | Append (new rows), Complete (full result), Update (changed rows) |
 | 44 | Stream-Static Joins | Enrich streams with batch reference data (inner/left/right) |
 | 45 | Windowed Aggregations | Tumbling (fixed) & sliding (overlapping) windows |
 | 46 | foreachBatch | Arbitrary batch logic per micro-batch; MERGE, multi-sink |
 | 47 | DLT Streaming Tables | Managed streaming tables (not in CE — explained conceptually) |
 | 48 | Checkpointing | Fault tolerance via offsets/commits; NEVER delete checkpoints |
 | 49 | Watermarks | Bound state growth: longer = more complete, more memory |
 | 50 | Stream-Stream Joins | Both streams need watermarks + time constraint |

 **Community Edition Limitations Encountered**:
 - ✅ `rate` source works perfectly for synthetic streaming data
 - ✅ `memory` sink is ideal for demos and quick inspection
 - ✅ Delta as sink & source works on CE
 - ✅ Checkpointing works within a single session
 - ❌ Cross-session checkpoint recovery may not work (single node, no persistent state)
 - ❌ DLT/Lakeflow pipelines are NOT available
 - ❌ Kafka source/sink requires external Kafka cluster (not included)

 **Next Steps**: Proceed to `06_Advanced_Pipelines.py` for medallion architecture
 and advanced Delta operations.


 ---
 ## 📝 Self-Assessment — Concepts #41–#50

 Answer these questions to check your understanding:

 1. **How does Structured Streaming differ from batch processing?**

 2. **When would you use `availableNow` trigger instead of `processingTime`?**

 3. **Why can't you use Append mode with aggregation queries (without watermark)?**

 4. **What happens to the static side of a stream-static join on each micro-batch?**

 5. **What's the difference between a tumbling window and a sliding window?**

 6. **What is foreachBatch used for? Give two practical examples.**

 7. **What does DLT provide that raw Structured Streaming does not?**

 8. **What's stored in the checkpoint directory, and why must it be unique per query?**

 9. **What is the tradeoff of a longer watermark duration?**

 10. **What two things are required for stream-stream joins that are not needed for stream-static joins?**


 ### ✅ CONGRATULATIONS!

 You've completed the Structured Streaming section (Concepts #41–#50). You now understand:

 - The core streaming API (readStream/writeStream)
 - Controlling micro-batch timing with triggers
 - Choosing the right output mode for your query
 - Enriching streaming data with static reference tables
 - Computing windowed aggregations (tumbling & sliding)
 - Using `foreachBatch` for advanced custom logic and multi-sink patterns
 - The value of DLT streaming tables in managed environments
 - Ensuring exactly-once delivery through checkpointing
 - Managing state growth with watermarks
 - Joining two live streams with time-constrained conditions

 Continue to `06_Advanced_Pipelines.py` for the medallion architecture and advanced Delta operations!


---

# 06 Performance and Cost



 # 06 — Performance & Cost Optimization

 **Concepts Covered:** #51–#60  
 **Environment:** Databricks Community Edition (single-node, free)  
 **Goal:** Master performance tuning and cost optimization techniques on Databricks.

 | # | Concept | Difficulty | Type |
 |---|---------|------------|------|
 | 51 | Serverless Compute Economics | Easy | Conceptual |
 | 52 | Predicate Pushdown & Data Skipping | Medium | Hands-on |
 | 53 | Predictive Optimization | Medium | Conceptual + Hands-on |
 | 54 | Column Statistics & File Pruning | Medium | Hands-on |
 | 55 | Write Optimization | Medium | Hands-on |
 | 56 | Cluster Sizing & Autoscaling | Medium | Conceptual |
 | 57 | DBU Cost Model & Billing | Medium | Conceptual + Hands-on |
 | 58 | Query Profile & Execution Plans | Medium | Hands-on |
 | 59 | Join Performance Diagnosis | Hard | Hands-on |
 | 60 | MERGE Write Amplification | Hard | Hands-on |


 ## Concept 51 — Serverless Compute Economics

 **NOTE:** Serverless compute is **not available** in Databricks Community Edition. This section is conceptual.

 ### What is Serverless Compute?

 Serverless compute removes the need to manage clusters directly. You submit workloads, and Databricks provisions compute resources automatically in your account. You pay only for the resources consumed by your workload.

 ### DBU-Based Pricing Model

 Databricks pricing is based on **DBUs** (Databricks Units) — a normalized unit of processing capability per hour.

 | Compute Type | DBU/hr (approx.) | Billing Granularity | Best For |
 |---|---|---|---|
 | **Serverless SQL** | 0.70–1.10 | Per-second, 1-min minimum | BI dashboards, ad-hoc SQL |
 | **Serverless Jobs** | 0.55–0.75 | Per-second, 1-min minimum | Notebook/scheduled jobs |
 | **Classic SQL Warehouse** | 0.40–0.55 | Per-second, 10-min minimum | Steady-state SQL workloads |
 | **Classic All-Purpose** | 0.40–0.55 | Per-second, 10-min minimum | Development, exploration |
 | **Classic Jobs** | 0.30–0.40 | Per-second, 10-min minimum | Scheduled production pipelines |

 *Exact pricing varies by tier (Standard/Premium/Enterprise) and cloud provider.*


 ### Serverless vs Classic: When Each Wins

 **Serverless is cheaper when:**
 - Workloads are **bursty** (intermittent, unpredictable)
 - Many short-running queries or jobs
 - You want zero cluster management overhead
 - Start-up latency is important (serverless starts faster)
 - Idle time between jobs is significant

 **Classic is cheaper when:**
 - Workloads are **sustained** (24/7 processing)
 - Jobs run continuously with little idle time
 - You can right-size clusters precisely
 - Using spot instances for deep discounts (30–50% off)
 - Predictable, long-running ETL pipelines

 ### Elimination of Idle Cluster Waste

 With classic clusters:
 - Clusters are always on (costing DBU/hr) even when idle
 - Autoscaling helps but cannot go to zero
 - Manual start/stop introduces latency

 With serverless:
 - *No idle cost* — you pay only for active processing
 - Compute scales down to zero between queries
 - No cluster management overhead


 ### Conceptual Pricing Comparison


In [ ]:
# Conceptual pricing comparison table
pricing_data = [
    ("Small SQL Query (30s, 50/day)", "Serverless SQL", "0.29", "$0.29"),
    ("Small SQL Query (30s, 50/day)", "Classic SQL", "1.67", "$1.67 (idle time)"),
    ("ETL Job (2hr, nightly)", "Serverless Jobs", "1.20", "$1.20"),
    ("ETL Job (2hr, nightly)", "Classic Jobs", "0.70", "$0.70 (spot)"),
    ("Ad-hoc Dev (4hr, varied)", "Serverless All-Purpose", "2.80", "$2.80"),
    ("Ad-hoc Dev (4hr, varied)", "Classic All-Purpose", "3.20", "$3.20 (idle 2hr)"),
    ("24/7 Streaming", "Classic Jobs", "216.00", "$216/mo (spot)"),
    ("24/7 Streaming", "Serverless", "360.00", "$360/mo"),
]

import pandas as pd
pricing_df = pd.DataFrame(pricing_data, columns=["Scenario", "Compute", "Daily Cost", "Notes"])
print("CONCEPTUAL PRICING COMPARISON\n")
print("(Illustrative — verify at https://databricks.com/product/pricing)\n")
display(pricing_df)


 ### Key Takeaways — Serverless

 | Factor | Serverless | Classic |
 |---|---|---|
 | Start-up time | 5–15 seconds | 3–7 minutes |
 | Billing minimum | 1 minute | 10 minutes |
 | Idle cost | $0 | DBU/hr × idle time |
 | Spot instances | Not available | Available (30–50% less) |
 | Cluster management | None | Required |
 | Max scale | Elastic | Defined by config |
 | Photon support | Yes | Yes |


 ## Concept 52 — Predicate Pushdown & Data Skipping

 **Available in Community Edition — full hands-on.**

 ### Theory

 **Predicate pushdown** means filters are pushed down to the storage layer, so only relevant data is read from disk.  
 **Data skipping** uses Delta Lake per-file statistics (min/max values) to skip entire files that can't match the filter.

 **What enables skipping:**
 - Partition filters (direct path elimination)
 - `ZORDER` clustering on filter columns
 - Delta stats (min/max per file for first 32 columns)

 **What BREAKS pushdown:**
 - Applying functions to filter columns (`YEAR(date_col) = 2023` instead of `date_col BETWEEN`)
 - Casting (`CAST(col AS STRING) = 'value'`)
 - Complex expressions that can't be evaluated at the file level


 ### Create Test Table with Many Files


In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, dayofmonth, rand, round as spark_round, lit
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, StringType, DoubleType

spark = SparkSession.builder.appName("PerfTest").getOrCreate()

NUM_ROWS = 1_000_000
NUM_FILES = 20

schema = StructType([
    StructField("transaction_id", IntegerType()),
    StructField("transaction_date", TimestampType()),
    StructField("customer_id", IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("store_id", IntegerType()),
    StructField("amount", DoubleType()),
    StructField("quantity", IntegerType()),
    StructField("category", StringType()),
    StructField("region", StringType()),
    StructField("payment_method", StringType()),
])

print(f"Generating {NUM_ROWS:,} synthetic rows...")
t0 = time.time()

df = spark.range(0, NUM_ROWS).select(
    col("id").alias("transaction_id"),
    (lit("2023-01-01").cast("timestamp") + (col("id") * 347)).cast("timestamp").alias("transaction_date"),
    (col("id") % 50000 + 1).alias("customer_id"),
    (col("id") % 1000 + 1).alias("product_id"),
    (col("id") % 100 + 1).alias("store_id"),
    (rand(42) * 500).alias("amount"),
    ((rand(43) * 10 + 1).cast("int")).alias("quantity"),
)

categories = ['Electronics', 'Clothing', 'Groceries', 'Home', 'Sports']
regions = ['North', 'South', 'East', 'West', 'Central']
payments = ['Credit', 'Debit', 'Cash', 'Digital']

df = df.withColumn("category", col("product_id") % 5)
df = df.withColumn("category",
    when(col("category") == 0, "Electronics")
    .when(col("category") == 1, "Clothing")
    .when(col("category") == 2, "Groceries")
    .when(col("category") == 3, "Home")
    .otherwise("Sports"))
df = df.withColumn("region", col("store_id") % 5)
df = df.withColumn("region",
    when(col("region") == 0, "North")
    .when(col("region") == 1, "South")
    .when(col("region") == 2, "East")
    .when(col("region") == 3, "West")
    .otherwise("Central"))
df = df.withColumn("payment_method", col("customer_id") % 4)
df = df.withColumn("payment_method",
    when(col("payment_method") == 0, "Credit")
    .when(col("payment_method") == 1, "Debit")
    .when(col("payment_method") == 2, "Cash")
    .otherwise("Digital"))

df = df.withColumn("year", year(col("transaction_date")))
df = df.withColumn("month", month(col("transaction_date")))

print(f"Generated {NUM_ROWS:,} rows in {time.time()-t0:.1f}s")
df.printSchema()


In [ ]:
# Write with many files (partition by region to see partitioning effects)
table_path = "/tmp/delta/perf_demo/transactions"
print(f"Writing {NUM_FILES} files to {table_path}...")
t0 = time.time()

df.repartition(NUM_FILES).write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 100000) \
    .partitionBy("region") \
    .save(table_path)

print(f"Write completed in {time.time()-t0:.1f}s")

# Verify files
files = dbutils.fs.ls(table_path)
print(f"\nTotal files in Delta table: {len([f for f in files if f.name.endswith('.parquet')])} (not counting _delta_log)")


 ### Demonstrate Predicate Pushdown with EXPLAIN


In [ ]:
transactions = spark.read.format("delta").load(table_path)

print("=" * 80)
print("QUERY 1: Filter on partition column (region = 'North')")
print("=" * 80)
transactions.filter(col("region") == "North").explain(extended=True)


In [ ]:
print("=" * 80)
print("QUERY 2: Filter on non-partition column (amount > 400)")
print("=" * 80)
transactions.filter(col("amount") > 400).explain(extended=True)


In [ ]:
print("=" * 80)
print("QUERY 3: Filter with function that BREAKS pushdown YEAR(txn_date) = 2023")
print("=" * 80)
transactions.filter(year(col("transaction_date")) == 2023).explain(extended=True)


In [ ]:
print("=" * 80)
print("QUERY 4: Range filter that ENABLES pushdown")
print("=" * 80)
transactions.filter(
    (col("transaction_date") >= "2023-06-01") &
    (col("transaction_date") < "2023-09-01")
).explain(extended=True)


 ### Timing Comparison: Good vs Bad Predicates


In [ ]:
from pyspark.sql.functions import when, lit

# Force materialization — first run to warm up Delta cache
transactions.count()

print("\n" + "=" * 80)
print("TIMING EXPERIMENT")
print("=" * 80)

# Good predicate — partition filter + range filter
t0 = time.time()
result1 = transactions.filter(col("region") == "North") \
    .filter(col("amount") > 400) \
    .count()
t1 = time.time() - t0

# Bad predicate — function on filter column
t0 = time.time()
result2 = transactions.filter(year("transaction_date") == 2023) \
    .filter(col("amount") > 400) \
    .count()
t2 = time.time() - t0

# Good predicate — partition filter + direct date comparison
t0 = time.time()
result3 = transactions.filter(col("region") == "North") \
    .filter(col("transaction_date") >= "2023-06-01") \
    .filter(col("transaction_date") < "2023-09-01") \
    .count()
t3 = time.time() - t0

print(f"\n{'Query':<55} {'Time':>8} {'Rows':>10}")
print("-" * 75)
print(f"{'Partition filter (region=North) + amount>400':<55} {t1:>7.2f}s {result1:>10,}")
print(f"{'YEAR(date)=2023 (breaks pushdown) + amount>400':<55} {t2:>7.2f}s {result2:>10,}")
print(f"{'Partition + date range + count':<55} {t3:>7.2f}s {result3:>10,}")


 ### Show Delta Statistics with File Skipping


In [ ]:
# Check the statistics Delta maintains
print("DESCRIBE DETAIL:")
display(spark.sql(f"DESCRIBE DETAIL delta.`{table_path}`"))


In [ ]:
print("DESCRIBE HISTORY:")
display(spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`"))


In [ ]:
# Inspect Delta log for statistics
delta_log_path = f"{table_path}/_delta_log"
log_files = dbutils.fs.ls(delta_log_path)
print(f"Delta log files at {delta_log_path}:")
for f in log_files[:10]:
    print(f"  {f.name} ({f.size} bytes)")

# Read the last checkpoint file if it exists
checkpoint_path = f"{table_path}/_delta_log/_last_checkpoint"
try:
    checkpoint_info = spark.read.text(checkpoint_path).collect()
    print(f"\nLast checkpoint: {checkpoint_info}")
except:
    print("\nNo _last_checkpoint file (expected after first write)")


 ### Key Takeaways — Predicate Pushdown

 | Pattern | Effect | Example |
 |---|---|---|
 | Partition column filter | Entire partitions skipped | `WHERE region = 'North'` |
 | Direct column comparison | File-level min/max stats used | `WHERE amount > 400` |
 | Function on column | Stats cannot be used | `WHERE YEAR(date) = 2023` |
 | Cast on column | Stats cannot be used | `WHERE CAST(id AS STRING) = '5'` |
 | Range on column | Stats used effectively | `WHERE date BETWEEN 'A' AND 'B'` |
 | `IN` clause | Converted to range checks | `WHERE region IN ('N','S')` |


 ## Concept 53 — Predictive Optimization

 **NOTE:** Predictive Optimization is a **full-platform feature** (not available in Community Edition). The concept is explained here, and manual equivalents are demonstrated for CE.

 ### What Is Predictive Optimization?

 Predictive Optimization is a **managed service** that automatically runs maintenance operations on tables:

 | Operation | What It Does | Trigger |
 |---|---|---|
 | **OPTIMIZE** | Compacts small files, ZORDERs data | After 5+ writes to the table |
 | **VACUUM** | Removes old file versions | Retained for 7 days (default) |
 | **ANALYZE** | Refreshes column statistics | After significant data change |

 **Key characteristics:**
 - Enabled by default on Unity Catalog managed tables (full platform)
 - Monitored via `system.information_schema.predictive_optimizations_history`
 - No user configuration required
 - Zero performance impact on user queries
 - No additional cost (included)


 ### Manual Maintenance (Community Edition Equivalent)


In [ ]:
manual_maintenance_path = "/tmp/delta/perf_demo/manual_maintenance"

# Create a fresh table with small files
print("Creating table with many small files...")
df_maintenance = spark.range(0, 500000).select(
    col("id"),
    (col("id") % 365).alias("day_of_year"),
    (col("id") % 50).alias("category_id"),
    (rand(42) * 1000).alias("value"),
)

# Write many small files
df_maintenance.repartition(100).write \
    .mode("overwrite") \
    .format("delta") \
    .save(manual_maintenance_path)

# Count files before OPTIMIZE
files_before = len([f for f in dbutils.fs.ls(manual_maintenance_path) if f.name.endswith('.parquet')])
print(f"Files before OPTIMIZE: {files_before}")

# Read table, check file count via DESCRIBE DETAIL
detail_before = spark.sql(f"DESCRIBE DETAIL delta.`{manual_maintenance_path}`")
detail_before.select("numFiles", "sizeInBytes").show()


In [ ]:
# Run OPTIMIZE (manual equivalent of Predictive Optimization)
print("Running manual OPTIMIZE...")
t0 = time.time()

spark.sql(f"OPTIMIZE delta.`{manual_maintenance_path}`")

t_optimize = time.time() - t0
print(f"OPTIMIZE completed in {t_optimize:.1f}s")

# Check file count after
detail_after = spark.sql(f"DESCRIBE DETAIL delta.`{manual_maintenance_path}`")
detail_after.select("numFiles", "sizeInBytes").show()

files_after = detail_after.select("numFiles").collect()[0][0]
print(f"\nFiles reduced: {files_before} → {files_after} ({100*(1-files_after/files_before):.0f}% reduction)")


 ### When Manual Maintenance Is Still Needed (Even With Predictive Optimization)

 Predictive Optimization helps but doesn't eliminate the need for manual intervention:

 | Scenario | Why Manual |
 |---|---|
 | **Precise ZORDER** | Predictive Optimization chooses columns automatically; you may want specific ZORDER columns |
 | **Custom VACUUM retention** | Default is 7 days; you may want longer for recovery |
 | **Immediate maintenance** | Predictive Optimization runs on a schedule; you may need immediate compaction after a large write |
 | **Tables with deletion vectors** | Manual REORG TABLE may be needed |
 | **Non-managed tables** | Predictive Optimization only works on managed tables in Unity Catalog |

 ```python
 # Manual OPTIMIZE with custom ZORDER
 spark.sql("OPTIMIZE my_table ZORDER BY (date_col, category)")

 # Manual VACUUM with custom retention
 spark.sql("VACUUM my_table RETAIN 168 HOURS")

 # Manual ANALYZE
 spark.sql("ANALYZE TABLE my_table COMPUTE STATISTICS FOR ALL COLUMNS")
 ```


 ## Concept 54 — Column Statistics & File Pruning

 **Available in Community Edition — hands-on.**

 ### How Delta Collects Statistics

 Delta Lake automatically collects **per-file min/max statistics** for the **first 32 columns** of each Parquet file. These statistics are stored in the Delta transaction log and enable:

 - **File-level pruning**: If `WHERE amount > 500` and a file's max amount is 300, the entire file is skipped.
 - **No data read needed** to determine if a file matches — just check the metadata.

 **Column order matters**: Statistics are only kept for the first 32 columns in the schema. Columns beyond 32 don't have stats available for file skipping.


In [ ]:
# Create a wide table to demonstrate the 32-column statistics limit
print("Creating wide table (40 columns) to demonstrate stats limit...")

wide_cols = [col("id").alias("col_00")]
for i in range(1, 40):
    wide_cols.append((col("id") % (100 + i * 10)).alias(f"col_{i:02d}"))

df_wide = spark.range(0, 200000).select(*wide_cols)

wide_table_path = "/tmp/delta/perf_demo/wide_table"
df_wide.repartition(10).write \
    .mode("overwrite") \
    .format("delta") \
    .save(wide_table_path)

print("Wide table created.")
spark.read.format("delta").load(wide_table_path).printSchema()


 ### ANALYZE TABLE to Refresh Statistics


In [ ]:
# ANALYZE the wide table
print("Running ANALYZE TABLE to collect statistics...")
t0 = time.time()
spark.sql(f"ANALYZE TABLE delta.`{wide_table_path}` COMPUTE STATISTICS FOR ALL COLUMNS")
print(f"ANALYZE completed in {time.time()-t0:.1f}s")

# Show statistics
print("\nColumn Statistics (DESCRIBE EXTENDED):")
spark.sql(f"DESCRIBE EXTENDED delta.`{wide_table_path}`").show(50, truncate=False)


 ### Demonstrate File Skipping with Column Statistics

 Filter on a column with statistics (col_01, within first 32) vs without (col_35, beyond 32).


In [ ]:
wide_df = spark.read.format("delta").load(wide_table_path)

print("=" * 80)
print("FILTER ON col_01 (within first 32 — HAS STATISTICS)")
print("=" * 80)
wide_df.filter(col("col_01") == 50).explain(extended=True)

print("\n" + "=" * 80)
print("FILTER ON col_35 (beyond first 32 — NO STATISTICS)")
print("=" * 80)
wide_df.filter(col("col_35") == 50).explain(extended=True)


In [ ]:
# Timing comparison: filter on column with stats vs without
print("TIMING COMPARISON: Column position effect on performance\n")

# Filter on column 01 (has stats)
t0 = time.time()
result_with_stats = wide_df.filter(col("col_01") == 50).count()
t_with = time.time() - t0

# Filter on column 35 (no stats)
t0 = time.time()
result_no_stats = wide_df.filter(col("col_35") == 50).count()
t_without = time.time() - t0

print(f"Filter on col_01 (with stats):    {t_with:.3f}s — returned {result_with_stats:,} rows")
print(f"Filter on col_35 (no stats):      {t_without:.3f}s — returned {result_no_stats:,} rows")
print(f"Speedup from stats:               {t_without/t_with:.1f}x faster with statistics")


 ### DESCRIBE DETAIL — Check Table Statistics


In [ ]:
# Show table details including statistics configuration
print("TABLE DETAILS:")

detail = spark.sql(f"DESCRIBE DETAIL delta.`{wide_table_path}`")
detail.select("name", "numFiles", "sizeInBytes", "properties").show(truncate=False)

print("\nTABLE HISTORY (last 5 operations):")
history = spark.sql(f"DESCRIBE HISTORY delta.`{wide_table_path}`")
history.select("version", "timestamp", "operation", "operationMetrics").show(5, truncate=False)


 ### Statistics Property Configuration


In [ ]:
# Show how to configure statistics collection
print("Delta table statistics properties:")
print("""
| Property | Default | Description |
|---|---|---|
| delta.dataSkippingNumIndexedCols | 32 | Number of columns to collect stats for |
| delta.statistics.columns | (all) | Specific columns to collect stats for |
| delta.stats.collected | N/A | Columns with stats in this file |

You can set these when creating or altering a table:
```
SET spark.databricks.delta.properties.defaults.dataSkippingNumIndexedCols = 40

ALTER TABLE my_table SET TBLPROPERTIES (
  'delta.dataSkippingNumIndexedCols' = '50'
)
```
""")


 ## Concept 55 — Write Optimization

 **Available in Community Edition — full hands-on.**

 ### Overview

 Write optimization techniques improve **read performance** by controlling output file size and layout:

 | Technique | What It Does |
 |---|---|
 | `optimizeWrite` | Coalesces small files before writing (auto-shuffle) |
 | `maxRecordsPerFile` | Caps records per output file |
 | `targetFileSize` | Desired file size (default 1GB for Parquet) |
 | Repartition before write | Controls number of output files |
 | Coalesce before write | Reduces files without shuffle |
 | ZORDER optimization | Clusters related data in files |


 ### Create Test Table and Benchmark Different Write Strategies


In [ ]:
write_test_path = "/tmp/delta/perf_demo/write_opt"

# Cleanup if exists
dbutils.fs.rm(write_test_path + "_base", True)
dbutils.fs.rm(write_test_path + "_coalesced", True)
dbutils.fs.rm(write_test_path + "_optimized", True)

# Generate test data
print("Generating write test data...")
df_write = spark.range(0, 500000).select(
    col("id"),
    (col("id") * 37 + 100).alias("key"),
    rand(42).alias("value_a"),
    rand(43).alias("value_b"),
    rand(44).alias("value_c"),
    (col("id") % 100).alias("group_id"),
)

# Split into varied-size partitions to simulate real-world data skew
df_write = df_write.repartition(200)

# Strategy 1: No optimization (many small files from skewed partitions)
print("\nStrategy 1: Writing without optimization (many small files)...")
t0 = time.time()
df_write.write.mode("overwrite").format("delta") \
    .save(write_test_path + "_base")
t1 = time.time() - t0
print(f"  Write time: {t1:.1f}s")

# Strategy 2: Coalesce before write (reduce files without shuffle)
print("\nStrategy 2: Writing with coalesce(8) before write...")
t0 = time.time()
df_write.coalesce(8).write.mode("overwrite").format("delta") \
    .save(write_test_path + "_coalesced")
t2 = time.time() - t0
print(f"  Write time: {t2:.1f}s")

# Strategy 3: maxRecordsPerFile
print("\nStrategy 3: Writing with maxRecordsPerFile=50000...")
t0 = time.time()
df_write.coalesce(16).write.mode("overwrite").format("delta") \
    .option("maxRecordsPerFile", 50000) \
    .save(write_test_path + "_optimized")
t3 = time.time() - t0
print(f"  Write time: {t3:.1f}s")


 ### Compare File Counts and Read Performance


In [ ]:
# Analyze each strategy
strategies = [
    ("No Optimization", write_test_path + "_base"),
    ("Coalesce(8)", write_test_path + "_coalesced"),
    ("maxRecordsPerFile=50000", write_test_path + "_optimized"),
]

print(f"{'Strategy':<30} {'Files':>6} {'Size (MB)':>10} {'Read Time':>10}")
print("-" * 60)

for name, path in strategies:
    detail = spark.sql(f"DESCRIBE DETAIL delta.`{path}`")
    row = detail.select("numFiles", "sizeInBytes").collect()[0]
    num_files = row[0]
    size_mb = row[1] / (1024 * 1024)
    
    # Time a read
    df_test = spark.read.format("delta").load(path)
    t0 = time.time()
    df_test.filter(col("group_id") == 50).count()
    read_time = time.time() - t0
    
    print(f"{name:<30} {num_files:>6} {size_mb:>9.1f} {read_time:>9.2f}s")


 ### optimizeWrite Setting


In [ ]:
optimize_write_path = "/tmp/delta/perf_demo/optimize_write"
dbutils.fs.rm(optimize_write_path, True)

# Test with optimizeWrite enabled
print("Writing with optimizeWrite enabled...")
t0 = time.time()

spark.conf.set("spark.databricks.delta.optimizeWrite.numShuffleBlocks", "2000000")
spark.conf.set("spark.databricks.delta.optimizeWrite.binSize", "512")

df_write.write.mode("overwrite").format("delta") \
    .option("optimizeWrite", "true") \
    .save(optimize_write_path)

t_opt_write = time.time() - t0
print(f"Write time (optimizeWrite=true): {t_opt_write:.1f}s")

# Check file count
detail = spark.sql(f"DESCRIBE DETAIL delta.`{optimize_write_path}`")
detail.select("numFiles", "sizeInBytes").show()

# Time a read
t0 = time.time()
spark.read.format("delta").load(optimize_write_path) \
    .filter(col("group_id") == 50).count()
t_read_opt = time.time() - t0
print(f"Read time (optimized): {t_read_opt:.2f}s")


 ### Write Optimization Summary


In [ ]:
summary_data = [
    ("No optimization", "Good", "Poor (many files)", "Poor (high overhead)"),
    ("coalesce(N)", "Better", "Good (N files)", "Better (fewer files)"),
    ("repartition(N)", "Worse (shuffle)", "Good (N files)", "Better (fewer files)"),
    ("maxRecordsPerFile", "Same", "Controlled", "Better (ideal sizes)"),
    ("optimizeWrite=true", "Slightly higher", "Optimized", "Best (auto)"),
    ("OPTIMIZE after write", "Adds step", "Best", "Best"),
]

write_summary_df = pd.DataFrame(summary_data, 
    columns=["Strategy", "Write Cost", "File Layout", "Read Performance"])
print("WRITE OPTIMIZATION STRATEGIES COMPARISON")
display(write_summary_df)


 ## Concept 56 — Cluster Sizing & Autoscaling

 **NOTE:** Community Edition is **single-node only** (1 driver, 0 workers). This section is conceptual but includes Spark config guidance applicable to CE.

 ### Cluster Sizing Principles

 | Workload Type | Cores | Memory | Workers | Rationale |
 |---|---|---|---|---|
 | **SQL Analytics / BI** | 4–8 per worker | 2–4 GB/core | 2–20 | High concurrency, small queries |
 | **ETL / Data Engineering** | 8–16 per worker | 2–4 GB/core | 2–50 | CPU-intensive, large shuffles |
 | **ML Training** | 8–16 per worker | 4–8 GB/core | 2–20 | GPU optional, memory for models |
 | **Streaming** | 4–8 per worker | 4 GB/core | 2–10 | Sustained usage, low latency |
 | **Development / Exploration** | 4 per worker | 8 GB/core | 1–2 | Interactive, low cost |

 ### The `spark.sql.shuffle.partitions` Rule

 Default: 200 partitions per shuffle operation.  
 **Rule of thumb:** Set to 2–3× total cores in cluster.

 | Cluster Size | Total Cores | Recommended shuffle.partitions |
 |---|---|---|
 | 1 worker × 4 cores | 4 | 8–12 |
 | 4 workers × 8 cores | 32 | 64–96 |
 | 10 workers × 16 cores | 160 | 320–480 |


 ### Autoscaling Configuration


In [ ]:
print("""
AUTOSCALING CONFIGURATION EXAMPLES

1. ETL Pipeline (cost-sensitive, uses spot instances):
   Workers: min=2, max=20
   Worker type: i3.2xlarge (8 cores, 61 GB)
   Spark configs:
     spark.sql.shuffle.partitions = 80
     spark.databricks.delta.autoCompact.maxFileSize = 134217728
     spark.sql.adaptive.enabled = true
   Spot instances: enabled (save 30–50%)

2. BI Dashboard (consistent, predictable):
   Workers: min=5, max=10
   Worker type: i3.xlarge (4 cores, 30.5 GB)
   Spark configs:
     spark.sql.shuffle.partitions = 40
     spark.sql.adaptive.coalescePartitions.enabled = true

3. ML Training (GPU-enabled):
   Workers: min=2, max=8
   Worker type: g4dn.xlarge (4 cores, 16 GB, 1 GPU)
   Driver: g4dn.xlarge (for single-node debugging)

4. Development (Community Edition - single node):
   Workers: min=0, max=0 (not configurable)
   Spark configs:
     spark.sql.shuffle.partitions = 8 (since single-node)
     spark.sql.adaptive.enabled = true
""")


 ### Spot Instance Usage


In [ ]:
spot_data = [
    ("On-Demand", "Regular price ($0.55–$1.10/DBU)", "None", 100),
    ("Spot (bid 100%)", "30–50% discount", "Low", 80),
    ("Spot + On-Demand fallback", "20–40% net savings", "Zero", 85),
]

spot_df = pd.DataFrame(spot_data, columns=["Strategy", "Cost", "Interruption Risk", "Reliability %"])
print("SPOT INSTANCE COMPARISON")
display(spot_df)


 ### Right-Sizing for Community Edition

 In Community Edition, you have a single node. Key Spark configurations to tune:


In [ ]:
print("Current Spark configuration for Community Edition:")
important_configs = [
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.databricks.delta.autoCompact.enabled",
    "spark.databricks.delta.optimizeWrite.enabled",
    "spark.sql.files.maxPartitionBytes",
    "spark.memory.fraction",
    "spark.memory.storageFraction",
]

for config in important_configs:
    try:
        value = spark.conf.get(config)
        print(f"  {config:<55} = {value}")
    except:
        print(f"  {config:<55} = (not set, using default)")


 ## Concept 57 — DBU Cost Model & Billing

 **NOTE:** System tables for billing are not available in Community Edition. This section explains the DBU model conceptually with cost calculation examples you can apply.

 ### What Is a DBU?

 **DBU** = Databricks Unit — the unit of measure for Databricks capacity consumption.

 **1 DBU = 1 hour of processing on a standardized unit of compute.**

 The actual cost per DBU depends on:
 - **Tier**: Standard, Premium, or Enterprise
 - **SKU**: All-Purpose Compute, Jobs Compute, SQL Compute, Serverless
 - **Cloud provider**: AWS, Azure, GCP (prices vary slightly)


 ### Billing SKUs


In [ ]:
sku_data = [
    ("All-Purpose Compute", "$0.40–0.55", "10 min", "Interactive notebooks, development"),
    ("Jobs Compute", "$0.30–0.40", "10 min", "Scheduled jobs, automated pipelines"),
    ("Jobs Compute (Light)", "$0.22–0.30", "1 min", "Short jobs, DLT maintenance"),
    ("SQL Warehouse (Classic)", "$0.40–0.55", "10 min", "BI, SQL endpoints"),
    ("SQL Warehouse (Serverless)", "$0.70–1.10", "1 min", "Serverless BI, SQL"),
    ("Serverless (Jobs)", "$0.55–0.75", "1 min", "Serverless notebook jobs"),
    ("DLT (Core)", "$0.55–0.85", "Continuous", "Delta Live Tables pipeline"),
    ("DLT (Advanced)", "$0.75–1.10", "Continuous", "DLT with CDC, SCD Type 2"),
]

sku_df = pd.DataFrame(sku_data, columns=["SKU", "DBU/hr (approx)", "Billing Min", "Use Case"])
print("DATABRICKS BILLING SKUs")
display(sku_df)


 ### Cost Calculation Examples


In [ ]:
print("""
COST CALCULATION EXAMPLES
=========================

Example 1: Nightly ETL Job (2 hours, Jobs Compute, 4 workers × 8 cores)
  DBU rate: 0.35/DBU (Jobs Compute, Premium)
  DBU/hr: 4 workers × 0.75 DBU/hr per core × 8 cores = 24 DBU/hr
  Daily: 24 DBU/hr × 2 hr = 48 DBU × $0.35 = $16.80
  Monthly: $16.80 × 30 = $504.00
  With spot instances (40% off): $302.40/month

Example 2: BI Dashboard (SQL Warehouse, 8hr/day, Classic)
  DBU rate: 0.48/DBU (SQL Classic, Premium)
  Cluster: Medium (10 DBU/hr)
  Daily: 10 DBU/hr × 8 hr = 80 DBU × $0.48 = $38.40
  Monthly (business days): $38.40 × 22 = $844.80
  Note: Serverless would cost ~$1,450/mo but save idle DBU

Example 3: Development (4 hr/day, All-Purpose, 1 worker × 4 cores)
  DBU rate: 0.45/DBU (All-Purpose, Premium)
  DBU/hr: 1 worker × 0.55 DBU/hr × 4 cores = 2.2 DBU/hr
  Daily: 2.2 DBU/hr × 4 hr = 8.8 DBU × $0.45 = $3.96
  Monthly: $3.96 × 22 = $87.12

Example 4: Serverless Bursty Workload (50 queries/day, 30s avg)
  DBU rate: 0.90/DBU (Serverless SQL)
  Processing time: 50 × 30s = 1,500s = 0.42 hr
  Monthly: 0.42 DBU/hr × 0.90 × 30 days = $11.34
  This same workload on Classic with 10-min minimum would cost MUCH more
""")


 ### Tagging Jobs for Cost Allocation


In [ ]:
print("""
COST ALLOCATION TAGS
=====================
You can tag clusters, jobs, and pools to track costs by team/project:

# Set tags when creating a cluster
{
  "cluster_name": "etl-pipeline",
  "custom_tags": {
    "team": "data-engineering",
    "project": "customer-360",
    "environment": "production",
    "cost_center": "DE-1234"
  }
}

# These tags appear in billing reports and usage analysis.
# Use system tables (full platform) to query:
# SELECT * FROM system.billing.usage
# WHERE custom_tags.cost_center = 'DE-1234'

# For Community Edition, you can still add tags for local tracking.
""")


 ### Budget Policies (Full Platform Feature)


In [ ]:
print("""
CLUSTER POLICIES (Full Platform)
================================
Cluster policies enforce cost controls:

1. Restrict instance types: Allow only i3.xlarge and i3.2xlarge
2. Limit autoscaling: max 10 workers
3. Enforce autotermination: max 60 minutes of inactivity
4. Mandatory tags: Require "team" and "cost_center" tags

Usage monitoring (full platform):
  SELECT
    usage_date,
    sku_name,
    SUM(usage_quantity) as total_dbu
  FROM system.billing.usage
  WHERE usage_date >= CURRENT_DATE - INTERVAL 30 DAYS
  GROUP BY usage_date, sku_name
  ORDER BY usage_date DESC
""")


 ## Concept 58 — Query Profile & Execution Plans

 **Available in Community Edition — full hands-on with EXPLAIN.**

 ### Understanding EXPLAIN Output

 `EXPLAIN` shows how Spark will execute a query. There are three plan levels:

 | Plan | What It Shows |
 |---|---|
 | **Parsed Logical Plan** | What you wrote — unresolved references |
 | **Analyzed Logical Plan** | Resolved column names and types |
 | **Optimized Logical Plan** | After Catalyst optimizer applied rules |
 | **Physical Plan** | Actual execution strategy (how Spark runs it) |


In [ ]:
# Create a more complex dataset for query plan analysis
plan_test_path = "/tmp/delta/perf_demo/query_plan"
dbutils.fs.rm(plan_test_path, True)

print("Creating test data for query plan analysis...")
df_plan = spark.range(0, 500000).select(
    col("id").alias("order_id"),
    (col("id") % 5000 + 1).alias("customer_id"),
    (col("id") % 100 + 1).alias("store_id"),
    rand(42).alias("order_amount"),
    (col("id") % 365).alias("day_number"),
    (rand(43) * 100).cast("int").alias("item_count"),
)

df_plan.write.mode("overwrite").format("delta").save(plan_test_path)

orders = spark.read.format("delta").load(plan_test_path)
print(f"Orders: {orders.count():,} rows")


 ### Simple EXPLAIN — See the Physical Plan


In [ ]:
print("=" * 80)
print("EXPLAIN: Simple aggregation")
print("=" * 80)
orders.groupBy("store_id").agg(
    {"order_amount": "avg", "order_amount": "sum", "item_count": "sum"}
).explain()


 ### EXPLAIN EXTENDED — Full Plan Details


In [ ]:
print("=" * 80)
print("EXPLAIN EXTENDED: Multi-step query")
print("=" * 80)

from pyspark.sql.functions import avg, sum as spark_sum, count as spark_count

query = orders.filter(col("order_amount") > 0.5) \
    .groupBy("store_id", "day_number") \
    .agg(
        spark_sum("order_amount").alias("total_amount"),
        spark_count("*").alias("order_count"),
        avg("item_count").alias("avg_items"),
    ) \
    .filter(col("total_amount") > 1000) \
    .orderBy(col("total_amount").desc())

query.explain(extended=True)


 ### Walking Through Plan Nodes


In [ ]:
print("""
KEY PLAN NODES — WHAT TO LOOK FOR
==================================

SCAN / FileScan
  Reads data from storage. Check:
  - "number of files read" → are you reading too many?
  - "partition filters" → is predicate pushdown working?
  - "dataFilters" vs "partitionFilters" → is partitioning effective?

PROJECT
  Selects columns. Fine unless projecting after a huge join.

FILTER
  Applies WHERE conditions. Check:
  - Is it pushed before or after JOIN?
  - Good: Filter → Join (filter early)
  - Bad: Join → Filter (filter late)

JOIN (BroadcastHashJoin / SortMergeJoin / ShuffledHashJoin)
  BroadcastHashJoin: best — small table broadcast to all executors
  SortMergeJoin: common — both sides shuffled and sorted
  ShuffledHashJoin: rare — one side shuffled and hashed

AGGREGATE
  Grouping/summarizing. Check:
  - Is it hash-based (partial + final) or sort-based?
  - Number of output rows — large cardinality group-bys are expensive

SORT / ORDER BY
  Global ordering requires a single partition — BOTTLENECK!
  Prefer: sort within partitions or use SORT BY instead
  Use: repartition(range) for sorted output

EXCHANGE / SHUFFLE
  Data moves between executors — most expensive operation!
  "SinglePartition" suffix = all data to one node (slow)
  "hashpartitioning" = distributed (better)
""")


 ### Analyze a Complex Query Plan Step by Step


In [ ]:
print("=" * 80)
print("COMPLEX QUERY WITH EXPLAIN EXTENDED")
print("=" * 80)

# Create a second table for join analysis
customer_path = "/tmp/delta/perf_demo/customers"
dbutils.fs.rm(customer_path, True)

spark.range(1, 5001).select(
    col("id").alias("customer_id"),
    (col("id") % 5).alias("region_id"),
    rand(44).alias("loyalty_score"),
).write.mode("overwrite").format("delta").save(customer_path)

customers = spark.read.format("delta").load(customer_path)

# Complex query with join, filter, aggregate, order
complex_query = orders.alias("o") \
    .join(customers.alias("c"), col("o.customer_id") == col("c.customer_id"), "inner") \
    .filter(col("c.loyalty_score") > 0.5) \
    .groupBy("o.store_id", "c.region_id") \
    .agg(
        spark_sum("o.order_amount").alias("total_revenue"),
        spark_count("o.order_id").alias("num_orders"),
    ) \
    .filter(col("total_revenue") > 5000) \
    .orderBy(col("total_revenue").desc()) \
    .limit(10)

complex_query.explain(extended=True)


 ### Identifying Slow Operators


In [ ]:
print("""
IDENTIFYING SLOW OPERATORS IN PLANS
====================================

🚩 RED FLAGS in Physical Plans:

1. "SortMergeJoin" with large tables
   → Both sides shuffled → heavy network/disk I/O
   Fix: Use broadcast hint if one table < 10MB after filtering

2. "ObjectHashAggregate" instead of "HashAggregate"
   → Indicates non-primitive (object) types → slower aggregation
   Fix: Use primitive types (Long/Double) where possible

3. "Sort" with "SinglePartition"
   → Global sort forced to one node → bottleneck
   Fix: Use df.orderBy().limit(N) or repartition before sort

4. Many "Exchange hashpartitioning" nodes
   → Multiple shuffles in sequence → each is expensive
   Fix: Denormalize, pre-aggregate, or use broadcast joins

5. "Scan parquet → Filter → Project → Scan parquet" repeating
   → Subquery executed for each row (correlated)
   Fix: Rewrite as JOIN with broadcast

6. "WholeStageCodegen" missing
   → Collapse virtual functions not happening → overhead
   Fix: Check for UDFs blocking codegen (Python UDFs cannot be codegen'd)
""")


 ### Generate and Analyze Your Own Query Profile


In [ ]:
# Time a query and correlate with the plan
print("Running query with timing...\n")

query_to_run = orders.filter(col("order_amount") > 0.3) \
    .groupBy("store_id") \
    .agg(spark_sum("order_amount").alias("total")) \
    .filter(col("total") > 5000)

# Show plan
print("QUERY PLAN:")
query_to_run.explain()

# Execute and time
t0 = time.time()
result = query_to_run.collect()
t_query = time.time() - t0

print(f"\nQuery executed in {t_query:.2f}s")
print(f"Result rows: {len(result)}")
print(f"\nFirst 5 rows:")
for row in result[:5]:
    print(f"  store_id={row['store_id']}, total={row['total']:.2f}")


 ## Concept 59 — Join Performance Diagnosis

 **Available in Community Edition — full hands-on.**

 ### Join Strategies in Spark

 | Strategy | When Used | Cost | Best For |
 |---|---|---|---|
 | **Broadcast Hash Join (BHJ)** | One side < threshold (default 10MB) | No shuffle | Small lookup tables |
 | **Sort Merge Join (SMJ)** | Both sides large, join keys sortable | Shuffle both sides | Large-large joins |
 | **Shuffled Hash Join (SHJ)** | One side 3× smaller than other | Shuffle one side | Medium-large joins |
 | **Broadcast Nested Loop Join (BNLJ)** | Non-equi join, one side small | Cartesian risk | Cross/custom joins |
 | **Cartesian Product Join** | No join condition | Very expensive | Avoid! |


 ### Create Large Tables for Join Benchmarking


In [ ]:
join_test_dir = "/tmp/delta/perf_demo/joins"
dbutils.fs.rm(join_test_dir, True)

NUM_FACT = 1000000
NUM_DIM = 1000

print(f"Creating fact table ({NUM_FACT:,} rows) and dimension table ({NUM_DIM:,} rows)...")

# Fact table — large
fact = spark.range(0, NUM_FACT).select(
    col("id").alias("sale_id"),
    (col("id") % NUM_DIM + 1).alias("product_id"),
    (col("id") % 500 + 1).alias("customer_id"),
    rand(42).alias("sale_amount"),
    (rand(43) * 10 + 1).cast("int").alias("quantity"),
    (col("id") % 365).alias("day_of_year"),
)

# Dimension table — small (fits in broadcast)
dim = spark.range(1, NUM_DIM + 1).select(
    col("id").alias("product_id"),
    (col("id") % 50 + 1).alias("category_id"),
    (col("id") % 20 + 1).alias("supplier_id"),
    rand(44).alias("list_price"),
)

# Write both
fact.write.mode("overwrite").format("delta").save(f"{join_test_dir}/fact")
dim.write.mode("overwrite").format("delta").save(f"{join_test_dir}/dim")

print(f"Fact table: {fact.count():,} rows")
print(f"Dim table:  {dim.count():,} rows")


 ### Test 1: Default Join (Auto Broadcast)


In [ ]:
fact_df = spark.read.format("delta").load(f"{join_test_dir}/fact")
dim_df = spark.read.format("delta").load(f"{join_test_dir}/dim")

print("TEST 1: Default join (auto broadcast)")
print("-" * 50)

t0 = time.time()
result1 = fact_df.alias("f").join(
    dim_df.alias("d"),
    col("f.product_id") == col("d.product_id"),
    "inner"
).select("f.sale_id", "f.sale_amount", "d.category_id", "d.list_price") \
 .filter(col("f.sale_amount") > 0.5)
result1.explain()
print(f"\nExecution time: {time.time()-t0:.2f}s")
result1_count = result1.count()
print(f"Result rows: {result1_count:,}")


 ### Test 2: Sort-Merge Join (Disable Broadcast, Force Shuffle)


In [ ]:
print("TEST 2: Sort-Merge Join (broadcast disabled)")
print("-" * 50)

# Disable broadcast to force sort-merge
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

t0 = time.time()
result2 = fact_df.alias("f").join(
    dim_df.alias("d"),
    col("f.product_id") == col("d.product_id"),
    "inner"
).select("f.sale_id", "f.sale_amount", "d.category_id", "d.list_price") \
 .filter(col("f.sale_amount") > 0.5)
result2.explain()
print(f"\nExecution time: {time.time()-t0:.2f}s")

# Reset broadcast threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10m")
result2_count = result2.count()
print(f"Result rows: {result2_count:,}")


 ### Test 3: Broadcast Hint


In [ ]:
from pyspark.sql.functions import broadcast

print("TEST 3: Explicit broadcast hint")
print("-" * 50)

t0 = time.time()
result3 = fact_df.alias("f").join(
    broadcast(dim_df.alias("d")),
    col("f.product_id") == col("d.product_id"),
    "inner"
).select("f.sale_id", "f.sale_amount", "d.category_id", "d.list_price") \
 .filter(col("f.sale_amount") > 0.5)
result3.explain()
print(f"\nExecution time: {time.time()-t0:.2f}s")
result3_count = result3.count()
print(f"Result rows: {result3_count:,}")


 ### Test 4: Join After Pre-Aggregation (Reduce Shuffle)


In [ ]:
print("TEST 4: Pre-aggregate before join (reduce data movement)")
print("-" * 50)

# Pre-aggregate the fact table to reduce rows before join
t0_prep = time.time()

aggregated_fact = fact_df.filter(col("sale_amount") > 0.1) \
    .groupBy("product_id", "day_of_year") \
    .agg(
        F.sum("sale_amount").alias("total_sales"),
        F.sum("quantity").alias("total_quantity"),
        F.count("*").alias("transaction_count"),
    )

print(f"Aggregated fact: {aggregated_fact.count():,} rows (from {NUM_FACT:,})")

t0 = time.time()
result4 = aggregated_fact.alias("a").join(
    broadcast(dim_df.alias("d")),
    col("a.product_id") == col("d.product_id"),
    "inner"
).select("a.product_id", "a.total_sales", "d.category_id", "d.list_price") \
 .filter(col("a.total_sales") > 1000)
result4.explain()
t_join = time.time() - t0

print(f"\nPre-aggregation + join: {t_join:.2f}s")
print(f"Result rows: {result4.count():,}")


 ### Tuning Join Thresholds


In [ ]:
print("JOIN CONFIGURATION PARAMETERS")
print("=" * 60)

join_configs = {
    "spark.sql.autoBroadcastJoinThreshold": "10m (10 MB) — max size for broadcast",
    "spark.sql.join.preferSortMergeJoin": "true — prefer SMJ over SHJ",
    "spark.sql.adaptive.enabled": "true — adaptive query execution",
    "spark.sql.adaptive.coalescePartitions.enabled": "true — coalesce post-shuffle",
    "spark.sql.adaptive.skewJoin.enabled": "true — handle skewed joins",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor": "5 — skew detection factor",
    "spark.sql.broadcastTimeout": "300s — timeout for broadcast",
}

for config, description in join_configs.items():
    try:
        current = spark.conf.get(config) if config.startswith("spark.sql.") else "N/A"
    except:
        current = "(default)"
    print(f"{config:<55} = {description}")
    print(f"{'':55}   Current: {current}")

# Check current broadcast threshold
current_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(f"\nCurrent broadcast threshold: {current_threshold}")
print(f"If table size < {current_threshold}, Spark uses Broadcast Hash Join.")


 ### Join Performance Summary


In [ ]:
# Run all join tests with consistent timing
print("\n" + "=" * 60)
print("JOIN PERFORMANCE COMPARISON")
print("=" * 60)

# Reset threshold for fair comparison
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10m")

timings = []

# Test A: Broadcast Hash Join (default)
t0 = time.time()
fact_df.join(dim_df, "product_id", "inner").select("sale_amount").count()
timings.append(("Broadcast Hash Join", time.time() - t0))

# Test B: Sort-Merge Join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
t0 = time.time()
fact_df.join(dim_df, "product_id", "inner").select("sale_amount").count()
timings.append(("Sort-Merge Join", time.time() - t0))
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10m")

# Test C: Pre-aggregate then broadcast join
t0 = time.time()
agg = fact_df.groupBy("product_id").agg(F.sum("sale_amount").alias("total"))
agg_count = agg.join(dim_df, "product_id", "inner").count()
timings.append(("Pre-agg + Broadcast", time.time() - t0))

print(f"\n{'Strategy':<30} {'Time':>8} {'Speedup':>10}")
print("-" * 50)
baseline = timings[0][1] if timings[0][1] > 0 else 1
for name, t in timings:
    speedup = timings[1][1] / t if name != "Sort-Merge Join" else 1.0
    print(f"{name:<30} {t:>7.2f}s {'baseline' if name == timings[0][0] else f'{speedup:>9.1f}x'} ")


 ## Concept 60 — MERGE Write Amplification

 **Available in Community Edition — full hands-on.**

 ### What Is Write Amplification?

 When MERGE matches rows in a file, it **rewrites the ENTIRE file**, not just the matched rows. If 1 row in a 100MB file is updated, MERGE rewrites all 100MB. This is the core of write amplification.

 **Deletion Vectors** (Delta 3.0+, full platform) solve this by recording which rows are deleted/matched, avoiding full file rewrites. Community Edition uses an older Delta version without this feature.


 ### Create Test Table for MERGE Experiments


In [ ]:
merge_test_path = "/tmp/delta/perf_demo/merge_test"
dbutils.fs.rm(merge_test_path, True)

print("Creating base table for MERGE testing...")

# Generate a base table — each product_id maps to multiple rows
merge_base = spark.range(0, 500000).select(
    col("id").alias("row_id"),
    (col("id") % 10000 + 1).alias("product_id"),
    (col("id") % 365).alias("day_num"),
    rand(42).alias("amount"),
    rand(43).alias("quantity"),
    lit("old").alias("status"),
)

merge_base.write.mode("overwrite").format("delta").save(merge_test_path)
print(f"Base table: {merge_base.count():,} rows")

# Show file distribution
detail = spark.sql(f"DESCRIBE DETAIL delta.`{merge_test_path}`")
print(f"Initial files: {detail.select('numFiles').collect()[0][0]}")
print(f"Initial size: {detail.select('sizeInBytes').collect()[0][0] / 1024 / 1024:.1f} MB")


 ### MERGE 1: Wide MERGE (Matches Many Files — High Amplification)


In [ ]:
# Create updates that span many products (wide update — touches many files)
updates_wide = spark.range(1, 5001).select(
    col("id").alias("product_id"),
    lit("updated").alias("status"),
    (rand(55) * 1000).alias("new_amount"),
)

# Create a temp view for MERGE
updates_wide.createOrReplaceTempView("updates_wide")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW base_table AS
    SELECT * FROM delta.`{merge_test_path}`
""")

print("MERGE 1: Wide MERGE (touching many products across many files)")
print("-" * 60)

merge_sql_1 = f"""
MERGE INTO delta.`{merge_test_path}` AS target
USING updates_wide AS source
ON target.product_id = source.product_id
WHEN MATCHED THEN UPDATE SET
  target.status = source.status,
  target.amount = source.new_amount
WHEN NOT MATCHED THEN INSERT *
"""

t0 = time.time()
spark.sql(merge_sql_1)
t_merge_wide = time.time() - t0

# Check what happened
history_wide = spark.sql(f"DESCRIBE HISTORY delta.`{merge_test_path}`") \
    .filter(col("operation") == "MERGE") \
    .select("operationMetrics") \
    .collect()

print(f"MERGE Wide completed in {t_merge_wide:.2f}s")
if history_wide:
    print(f"Operation metrics: {history_wide[0]['operationMetrics']}")

detail1 = spark.sql(f"DESCRIBE DETAIL delta.`{merge_test_path}`")
files_after_wide = detail1.select("numFiles").collect()[0][0]
print(f"Files after wide MERGE: {files_after_wide}")


 ### Reset Table for Second Test


In [ ]:
# Reset the table for a fair second test
dbutils.fs.rm(merge_test_path, True)
merge_base.write.mode("overwrite").format("delta").save(merge_test_path)
print("Table reset to initial state.")


 ### MERGE 2: Targeted MERGE (Single Product — Low Amplification)


In [ ]:
# Create updates for a SINGLE product (narrow update — touches minimal files)
updates_narrow = spark.range(1, 2).select(
    col("id").alias("product_id"),
    lit("targeted_update").alias("status"),
    (rand(55) * 1000).alias("new_amount"),
)

updates_narrow.createOrReplaceTempView("updates_narrow")

print("MERGE 2: Targeted MERGE (single product)")
print("-" * 60)

merge_sql_2 = f"""
MERGE INTO delta.`{merge_test_path}` AS target
USING updates_narrow AS source
ON target.product_id = source.product_id
WHEN MATCHED THEN UPDATE SET
  target.status = source.status,
  target.amount = source.new_amount
WHEN NOT MATCHED THEN INSERT *
"""

t0 = time.time()
spark.sql(merge_sql_2)
t_merge_narrow = time.time() - t0

history_narrow = spark.sql(f"DESCRIBE HISTORY delta.`{merge_test_path}`") \
    .filter(col("operation") == "MERGE") \
    .select("operationMetrics") \
    .collect()

print(f"MERGE Targeted completed in {t_merge_narrow:.2f}s")
if history_narrow:
    print(f"Operation metrics: {history_narrow[0]['operationMetrics']}")

detail2 = spark.sql(f"DESCRIBE DETAIL delta.`{merge_test_path}`")
files_after_narrow = detail2.select("numFiles").collect()[0][0]
print(f"Files after targeted MERGE: {files_after_narrow}")


 ### MERGE 3: DELETE + INSERT Approach (Alternative to MERGE)


In [ ]:
# Reset table
dbutils.fs.rm(merge_test_path, True)
merge_base.write.mode("overwrite").format("delta").save(merge_test_path)
print("Table reset for DELETE + INSERT test.")

merge_base_df = spark.read.format("delta").load(merge_test_path)

print("\nMERGE 3: DELETE + INSERT approach (alternative to MERGE)")
print("-" * 60)

from pyspark.sql.functions import col as spark_col

t0 = time.time()

# Step 1: Find products to update
products_to_update = [p.product_id for p in updates_wide.select("product_id").distinct().collect()]

# Step 2: Delete old rows
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, merge_test_path)

# For the DELETE + INSERT approach, we'll use the DeltaTable API
# First delete matching rows, then insert new ones
delta_table.delete(spark_col("product_id").isin(products_to_update))

# Step 3: Calculate and insert updated rows
updated_rows = merge_base_df.filter(spark_col("product_id").isin(products_to_update)) \
    .select("row_id", "product_id", "day_num") \
    .withColumn("amount", rand(55) * 1000) \
    .withColumn("quantity", col("quantity")) \
    .withColumn("status", lit("delete_insert_updated"))

# Create the insert dataframe with proper schema
insert_df = updated_rows.select(
    col("row_id"), col("product_id"), col("day_num"),
    col("amount"), col("quantity"), col("status")
)

insert_df.write.mode("append").format("delta").save(merge_test_path)

t_delete_insert = time.time() - t0

print(f"DELETE + INSERT completed in {t_delete_insert:.2f}s")

detail3 = spark.sql(f"DESCRIBE DETAIL delta.`{merge_test_path}`")
print(f"Files after DELETE + INSERT: {detail3.select('numFiles').collect()[0][0]}")


 ### Analyze MERGE Performance with DESCRIBE HISTORY


In [ ]:
print("=" * 80)
print("DESCRIBE HISTORY — COMPARE ALL OPERATIONS")
print("=" * 80)

history = spark.sql(f"DESCRIBE HISTORY delta.`{merge_test_path}`")
history.select("version", "timestamp", "operation", "operationMetrics").show(20, truncate=False)

# Also show DESCRIBE DETAIL for current state
print("\nCURRENT TABLE STATE:")
detail_final = spark.sql(f"DESCRIBE DETAIL delta.`{merge_test_path}`")
detail_final.select("numFiles", "sizeInBytes").show()


 ### MERGE 4: MERGE with Matching on Clustered Keys (Optimization)


In [ ]:
merge_clustered_path = "/tmp/delta/perf_demo/merge_clustered"
dbutils.fs.rm(merge_clustered_path, True)

print("Creating ZORDERED table for optimized MERGE...")

# Create and write with ZORDER on product_id (the merge key)
merge_base_clustered = spark.range(0, 500000).select(
    col("id").alias("row_id"),
    (col("id") % 10000 + 1).alias("product_id"),
    (col("id") % 365).alias("day_num"),
    rand(42).alias("amount"),
    lit("old").alias("status"),
)

# Write with ZORDER
merge_base_clustered.write.mode("overwrite").format("delta") \
    .save(merge_clustered_path)

# Run OPTIMIZE with ZORDER on merge key
spark.sql(f"OPTIMIZE delta.`{merge_clustered_path}` ZORDER BY (product_id)")
print("ZORDER optimization on product_id completed.")

# Now run MERGE on clustered table
print("\nMERGE on ZORDERED table:")
t0 = time.time()

merge_sql_clustered = f"""
MERGE INTO delta.`{merge_clustered_path}` AS target
USING updates_wide AS source
ON target.product_id = source.product_id
WHEN MATCHED THEN UPDATE SET
  target.status = source.status,
  target.amount = source.new_amount
"""

spark.sql(merge_sql_clustered)
t_merge_clustered = time.time() - t0

print(f"MERGE on ZORDERED table: {t_merge_clustered:.2f}s")


 ### MERGE Amplification Summary


In [ ]:
# Build summary from measured timings
print("\n" + "=" * 80)
print("MERGE / WRITE AMPLIFICATION SUMMARY")
print("=" * 80)

merge_summary = [
    ("Wide MERGE (many products)", "High (all files touched)", "Slow", f"N/A (need reset)"),
    ("Targeted MERGE (1 product)", "Low (few files touched)", "Fast", "Much less IO"),
    ("DELETE + INSERT approach", "Medium (delete + write)", "Medium", "More operations, less file rewrite"),
    ("MERGE with ZORDER key", "Low (clustered data)", "Fastest", "Optimal when match on ZORDER key"),
]

print(f"\n{'Strategy':<35} {'Write Amplification':<25} {'Relative Speed':<15}")
print("-" * 80)
for strategy, amp, speed, note in merge_summary:
    print(f"{strategy:<35} {amp:<25} {speed:<15}")

print(f"""
KEY INSIGHTS:
1. MERGE rewrites ENTIRE FILES containing ANY matched row
2. ZORDER on the merge key localizes matches to fewer files
3. Targeted MERGE (single key) is dramatically faster than wide MERGE
4. DELETE + INSERT can be more efficient when you know exact rows
5. Deletion Vectors (Delta 3.0+) solve this entirely — no rewrites!
6. OPTIMIZE after MERGE helps compact the new files created

TO REDUCE MERGE AMPLIFICATION:
- ZORDER on columns used in MERGE ON clause
- Use targeted filters in the source view
- Batch updates by key ranges
- Consider DELETE + INSERT for large updates
- Upgrade to Delta 3.0+ for Deletion Vectors (full platform)
""")


 ## Performance & Cost — Summary Table


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════════╗
║               PERFORMANCE & COST OPTIMIZATION — MASTER SUMMARY                   ║
╠════╤═══════════════════════╤══════════╤══════════════════════════════════════════╣
║ #  │ Concept               │ Difficulty│ Key Technique                           ║
╠════╪═══════════════════════╪══════════╪══════════════════════════════════════════╣
║ 51 │ Serverless Economics  │ Easy     │ DBU pricing; choose serverless vs classic║
║ 52 │ Predicate Pushdown    │ Medium   │ Partition filters, avoid functions on cols║
║ 53 │ Predictive Opt        │ Medium   │ Auto OPTIMIZE/VACUUM; manual equivalents ║
║ 54 │ Column Statistics     │ Medium   │ First 32 cols have stats; ANALYZE TABLE  ║
║ 55 │ Write Optimization    │ Medium   │ maxRecordsPerFile, coalesce, optimizeWrite║
║ 56 │ Cluster Sizing        │ Medium   │ Right-size workers, shuffle.partitions    ║
║ 57 │ DBU Cost Model        │ Medium   │ Billing SKUs, tagging, cost calculations  ║
║ 58 │ Query Plans           │ Medium   │ EXPLAIN EXTENDED, read physical plan      ║
║ 59 │ Join Performance      │ Hard     │ Broadcast hint, pre-agg, threshold tuning ║
║ 60 │ MERGE Amplification   │ Hard     │ ZORDER match keys, DELETE+INSERT, DVs    ║
╚════╧═══════════════════════╧══════════╧══════════════════════════════════════════╝
""")


 ## Self-Assessment Questions

 Test your understanding of concepts #51–#60:

 **Concept 51 — Serverless Economics:**
 1. When is serverless more expensive than classic compute?
 2. What is the minimum billing period for serverless vs classic SQL warehouses?
 3. How does serverless eliminate idle cluster waste?

 **Concept 52 — Predicate Pushdown:**
 4. Why does `WHERE YEAR(date_col) = 2023` prevent predicate pushdown?
 5. What must you do to enable file-level statistics for a column?
 6. Does repartitioning to a different column break data skipping?

 **Concept 53 — Predictive Optimization:**
 7. What three operations does Predictive Optimization run automatically?
 8. When would you still manually run OPTIMIZE even with Predictive Optimization?

 **Concept 54 — Column Statistics:**
 9. For how many columns does Delta collect per-file statistics by default?
 10. What command refreshes column statistics?

 **Concept 55 — Write Optimization:**
 11. What's the difference between `coalesce()` and `repartition()` before writing?
 12. What does `maxRecordsPerFile` control?

 **Concept 56 — Cluster Sizing:**
 13. What's the recommended `spark.sql.shuffle.partitions` relative to core count?
 14. When would you use spot instances?

 **Concept 57 — DBU Cost Model:**
 15. Which SKU is cheapest per DBU for scheduled jobs?
 16. How do you allocate costs to specific teams?

 **Concept 58 — Query Plans:**
 17. What's the difference between a logical plan and a physical plan?
 18. What does an Exchange node in the physical plan indicate?

 **Concept 59 — Join Performance:**
 19. What join strategy does Spark use when one table is smaller than 10MB?
 20. How can you reduce shuffle during a join?

 **Concept 60 — MERGE Amplification:**
 21. Why does MERGE rewrite entire files?
 22. What technique reduces the number of files touched by MERGE?
 23. What Delta 3.0+ feature eliminates merge write amplification entirely?


 ### Quick Knowledge Check — Write Your Answers Below


In [ ]:
print("""
SELF-ASSESSMENT ANSWER KEY (check your knowledge)
==================================================

1. Serverless is more expensive for sustained 24/7 workloads
   because the per-DBU rate is higher than classic Jobs Compute.

2. Serverless: 1-minute minimum; Classic: 10-minute minimum.

3. Compute scales to zero between queries; you pay only for active time.

4. YEAR() is a function that transforms the column value; Delta cannot
   apply file-level min/max stats to the result of a function.

5. The column must be within the first 32 columns in the schema (or
   you can increase delta.dataSkippingNumIndexedCols).

6. Not necessarily, but the new repartition column may not be the one
   you query — causing more files to be read.

7. OPTIMIZE (compaction + ZORDER), VACUUM, ANALYZE.

8. When you need immediate compaction after a large write, specific
   ZORDER columns, or non-default VACUUM retention.

9. 32 columns by default.

10. ANALYZE TABLE <name> COMPUTE STATISTICS FOR ALL COLUMNS

11. coalesce() avoids shuffle (faster, reduces partitions only).
    repartition() performs a full shuffle (can both increase and
    decrease partitions, balanced data distribution).

12. Maximum number of records written to a single file; helps control
    file sizes for optimal read performance.

13. 2-3x the total number of cores in the cluster.

14. For fault-tolerant, cost-sensitive batch workloads.

15. Jobs Compute (Light) at ~$0.22/DBU.

16. Apply custom_tags to clusters/jobs (team, project, cost_center)
    and query billing usage tables.

17. Logical plan = what to compute (optimized expression tree).
    Physical plan = how to compute (actual execution strategy).

18. A shuffle operation — data moves between partitions/executors.
    This is the most expensive operation in Spark.

19. Broadcast Hash Join (BHJ).

20. Pre-aggregate/filter before joining; use broadcast for small
    tables; partition both sides on the same key.

21. MERGE operates at file granularity, not row. If even 1 row in a
    file is matched, the whole file is rewritten.

22. ZORDER on the merge key (localizes matching rows to fewer files).

23. Deletion Vectors — they mark rows as deleted without rewriting
    the underlying Parquet files.
""")


 ## Performance Optimization Cheat Sheet


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                PERFORMANCE OPTIMIZATION CHEAT SHEET               ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  READS (FASTER QUERIES)                                          ║
║  ├─ Partition on filter columns                                  ║
║  ├─ ZORDER on high-cardinality filter columns                    ║
║  ├─ Use statistics (first 32 columns, ANALYZE TABLE)             ║
║  ├─ Avoid functions on filter columns                            ║
║  ├─ Broadcast hint for small join tables (< 10MB)               ║
║  ├─ Pre-aggregate/filter before joins                            ║
║  └─ Liquid clustering (Delta 3.0+, full platform)                ║
║                                                                  ║
║  WRITES (FASTER INGESTION)                                       ║
║  ├─ optimizeWrite = true (auto-coalesce)                         ║
║  ├─ maxRecordsPerFile for ideal file sizes                       ║
║  ├─ coalesce() not repartition() to reduce shuffle               ║
║  ├─ OPTIMIZE after large writes for compaction                   ║
║  └─ ZORDER on MERGE key column                                   ║
║                                                                  ║
║  COST (LOWER BILLS)                                              ║
║  ├─ Use Jobs Compute for scheduled pipelines                     ║
║  ├─ Serverless for bursty workloads (SQL, ad-hoc)                ║
║  ├─ Spot instances for batch (checkpoint often)                  ║
║  ├─ Autotermination: 15-30 minutes                               ║
║  ├─ Tag resources for cost allocation                            ║
║  └─ Monitor with system.billing.usage (full platform)            ║
║                                                                  ║
║  CONFIGURATION                                                   ║
║  ├─ spark.sql.shuffle.partitions = 2-3x cores                    ║
║  ├─ spark.sql.adaptive.enabled = true                            ║
║  ├─ spark.sql.autoBroadcastJoinThreshold = 10m                   ║
║  ├─ spark.sql.adaptive.skewJoin.enabled = true                   ║
║  └─ spark.databricks.delta.retentionDurationCheck.enabled = true ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")


In [ ]:
print("End of 06_Performance_and_Cost notebook. All 10 concepts (#51–#60) covered.")


---

# 07 Unity Catalog Governance



 # 07 — Unity Catalog & Governance

 **Concepts Covered:** #61–#70  
 **Environment:** Databricks Community Edition (single-node, free, **no Unity Catalog**)  
 **Goal:** Master data governance, security, and sharing principles on Databricks.

 **CRITICAL:** Databricks Community Edition uses the **legacy Hive Metastore**. Unity Catalog is available only on paid workspaces (Premium/Enterprise tiers on AWS, Azure, or GCP). Throughout this notebook, we explain what Unity Catalog provides and then show the Hive Metastore equivalent that runs in Community Edition.

 | # | Concept | Difficulty | Type |
 |---|---------|------------|------|
 | 61 | Three-Level Namespace | Easy | Conceptual + Hands-on |
 | 62 | Permission Model & Inheritance | Medium | Conceptual + Hands-on |
 | 63 | Dynamic Views for Security | Medium | Hands-on |
 | 64 | Data Lineage | Medium | Conceptual + Hands-on |
 | 65 | Information Schema & Metadata Queries | Medium | Hands-on |
 | 66 | External Locations & Storage Credentials | Medium | Conceptual + Hands-on |
 | 67 | Service Principals & Managed Identity | Medium | Conceptual |
 | 68 | Audit Logging | Medium | Hands-on |
 | 69 | Row Filters & Column Masks | Hard | Hands-on |
 | 70 | Delta Sharing | Hard | Conceptual + Hands-on |


 ## Concept 61 — Three-Level Namespace

 ### Unity Catalog: `catalog.schema.table`

 Unity Catalog introduces a **three-level namespace** for organizing data assets:

 ```
 catalog.schema.table
    │       │      │
    │       │      └── Table, View, Function, Model, Volume
    │       └───────── Schema (logical grouping, like a database)
    └───────────────── Catalog (top-level container, maps to an org unit)
 ```

 **How catalogs map to organizational structure:**

 | Catalog | Purpose | Example Schemas |
 |---------|---------|-----------------|
 | `main` | Default catalog | `default`, `public` |
 | `sales` | Sales department data | `revenue`, `pipeline`, `forecast` |
 | `marketing` | Marketing data | `campaigns`, `analytics`, `segments` |
 | `hr_prod` | Production HR data (restricted) | `employees`, `payroll`, `benefits` |
 | `hr_dev` | HR development/sandbox data | `employees`, `payroll`, `benefits` |

 ### Hive Metastore: Two-Level Namespace (Community Edition)

 Community Edition uses the legacy two-level namespace: `database.table`

 ```
 database.table
    │        │
    │        └── Table or View
    └─────────── Database (equivalent to UC "schema")
 ```


 #### Hive Metastore Equivalent — Namespace Navigation


In [ ]:
# Show all databases (equivalent to UC "SHOW CATALOGS" + "SHOW SCHEMAS")
print("=" * 60)
print("DATABASES AVAILABLE (Hive Metastore)")
print("=" * 60)
display(spark.sql("SHOW DATABASES"))


In [ ]:
# Use the default database and show its tables
spark.sql("USE default")
print("Tables in 'default' database:")
display(spark.sql("SHOW TABLES IN default"))


In [ ]:
# Create sample tables to demonstrate namespace navigation
spark.sql("CREATE DATABASE IF NOT EXISTS sales_db")
spark.sql("CREATE DATABASE IF NOT EXISTS hr_db")
spark.sql("CREATE DATABASE IF NOT EXISTS marketing_db")

# Create tables in sales_db
spark.sql("USE sales_db")
spark.sql("""
    CREATE OR REPLACE TABLE sales_db.revenue AS
    SELECT 1 AS order_id, 'Widget-A' AS product, 150.00 AS amount, '2025-01-01' AS order_date
    UNION ALL
    SELECT 2, 'Widget-B', 200.00, '2025-01-02'
    UNION ALL
    SELECT 3, 'Widget-A', 150.00, '2025-01-03'
    UNION ALL
    SELECT 4, 'Widget-C', 300.00, '2025-01-04'
""")

spark.sql("""
    CREATE OR REPLACE TABLE sales_db.pipeline AS
    SELECT 1 AS deal_id, 'Acme Corp' AS company, 50000 AS value, 'Negotiation' AS stage
    UNION ALL
    SELECT 2, 'Globex Inc', 75000, 'Proposal'
    UNION ALL
    SELECT 3, 'Initech', 25000, 'Closed Won'
""")

# Create tables in hr_db
spark.sql("""
    CREATE OR REPLACE TABLE hr_db.employees AS
    SELECT 101 AS emp_id, 'Alice' AS name, 'Engineering' AS dept, 95000 AS salary
    UNION ALL
    SELECT 102, 'Bob', 'Engineering', 105000
    UNION ALL
    SELECT 103, 'Carol', 'Sales', 85000
    UNION ALL
    SELECT 104, 'Dave', 'HR', 75000
""")

spark.sql("""
    CREATE OR REPLACE TABLE hr_db.payroll AS
    SELECT 101 AS emp_id, '3000-11-1111' AS account_num, 'Direct Deposit' AS method
    UNION ALL
    SELECT 102, '3000-22-2222', 'Direct Deposit'
    UNION ALL
    SELECT 103, '3000-33-3333', 'Wire Transfer'
""")

# Create tables in marketing_db
spark.sql("""
    CREATE OR REPLACE TABLE marketing_db.campaigns AS
    SELECT 'C001' AS campaign_id, 'Spring Launch' AS name, 'Email' AS channel, 0.05 AS conversion_rate
    UNION ALL
    SELECT 'C002', 'Summer Sale', 'Social', 0.08
    UNION ALL
    SELECT 'C003', 'Fall Promo', 'Search', 0.12
""")

print("Databases and tables created successfully.")
print("\nAll databases:")
display(spark.sql("SHOW DATABASES"))


 #### Show Tables Across Databases


In [ ]:
for db in ["default", "sales_db", "hr_db", "marketing_db"]:
    print(f"\n{'=' * 50}")
    print(f"  Database: {db}")
    print(f"{'=' * 50}")
    try:
        display(spark.sql(f"SHOW TABLES IN {db}"))
    except:
        print(f"  (no tables or database not found)")


 ### Unity Catalog Syntax Reference (Not Executable in CE)

 The following is the Unity Catalog syntax — **cannot run** in Community Edition but shows what you would use in production:

 ```sql
 -- Unity Catalog: three-level references
 SELECT * FROM main.default.my_table;
 SELECT * FROM sales_db.revenue.orders;

 -- Set default catalog
 USE CATALOG main;

 -- Set default schema within catalog
 USE SCHEMA default;

 -- Show all catalogs
 SHOW CATALOGS;

 -- Show schemas in a catalog
 SHOW SCHEMAS IN main;

 -- Show tables in a schema
 SHOW TABLES IN main.default;

 -- Fully-qualified create
 CREATE TABLE sales.revenue.orders (
     order_id BIGINT,
     amount DECIMAL(10,2),
     order_date DATE
 );
 ```


 ### What You'd Do in Production

 1. **Design catalog topology** matching your organizational structure (departments, business units, data domains).
 2. **Use catalogs for data isolation** — separate `_prod` and `_dev` catalogs for the same domain.
 3. **Standardize naming conventions** — `catalog_schema_table` becomes the canonical identity of every data asset.
 4. **Set `USE CATALOG`** at the notebook level for cleaner references.
 5. **Avoid cross-catalog JOINs** unless necessary (they can incur egress costs on some clouds).

 | Community Edition | Production Unity Catalog |
 |---|---|
 | `database.table` (2-level) | `catalog.schema.table` (3-level) |
 | `SHOW DATABASES` | `SHOW CATALOGS` + `SHOW SCHEMAS IN <catalog>` |
 | One default database | One default catalog + one default schema per user |
 | No catalog-level privileges | Full RBAC at all three levels |


 ## Concept 62 — Permission Model & Inheritance

 ### Unity Catalog Privilege Cascade

 Unity Catalog permissions follow a **cascading inheritance model** where child objects can have **more restrictive** (but never more permissive) privileges than their parent:

 ```
 CATALOG                    ─── GRANT USAGE ON CATALOG sales TO `analysts`
   ├── schema: revenue      ─── GRANT CREATE TABLE, USAGE ON SCHEMA sales.revenue TO `analysts`
   │     ├── table: orders  ─── GRANT SELECT, MODIFY ON TABLE sales.revenue.orders TO `analysts`
   │     └── table: returns ─── GRANT SELECT ON TABLE sales.revenue.returns TO `analysts`
   └── schema: forecast     ─── GRANT USAGE ON SCHEMA sales.forecast TO `analysts`
         └── table: q1      ─── (inherits no table privileges; must be granted explicitly)
 ```

 ### Key Unity Catalog Privileges

 | Privilege | Applies To | Meaning |
 |-----------|-----------|---------|
 | `USE CATALOG` | Catalog | Required to access anything in the catalog |
 | `USE SCHEMA` | Schema | Required to list/access objects in the schema |
 | `SELECT` | Table/View | Read data from the object |
 | `MODIFY` | Table | INSERT, UPDATE, DELETE, MERGE |
 | `CREATE TABLE` | Schema | Create new tables in the schema |
 | `CREATE VIEW` | Schema | Create views in the schema |
 | `CREATE FUNCTION` | Schema | Create UDFs |
 | `ALL PRIVILEGES` | Any | Owner-like full access |
 | `EXECUTE` | Function | Run a UDF |

 ### Ownership vs Granted Access

 - **Owner**: The principal (user/service principal/group) that created the object; has `ALL PRIVILEGES` by default.
 - **Granted Access**: Explicit privileges given to other principals via `GRANT`.
 - Ownership can be transferred with `ALTER <object> OWNER TO <principal>`.


 ### Hive Metastore Equivalent — Limited GRANT/REVOKE in Community Edition

 Community Edition supports a limited subset of Hive-style privilege management.
 The following demonstrates what *is* executable in CE:


In [ ]:
# Hive Metastore: Show current user
print("Current user context:")
display(spark.sql("SELECT current_user() AS current_user"))

# Hive Metastore: GRANT SELECT on a table
# NOTE: In CE, this may succeed but enforcement is limited since there's no
#       full authentication/authorization layer in the free tier.
try:
    spark.sql("GRANT SELECT ON TABLE sales_db.revenue TO `user@example.com`")
    print("GRANT SELECT executed (note: enforcement is limited in CE)")
except Exception as e:
    print(f"GRANT result: {e}")

# Hive Metastore: SHOW GRANT on a table
print("Privileges on sales_db.revenue:")
try:
    display(spark.sql("SHOW GRANT ON TABLE sales_db.revenue"))
except Exception as e:
    print(f"SHOW GRANT result: {e}")

# Hive Metastore: REVOKE
try:
    spark.sql("REVOKE SELECT ON TABLE sales_db.revenue FROM `user@example.com`")
    print("REVOKE SELECT executed")
except Exception as e:
    print(f"REVOKE result: {e}")


 ### Unity Catalog GRANT/REVOKE Reference (Not Executable in CE)

 ```sql
 -- Grant catalog-level access
 GRANT USE CATALOG ON CATALOG sales TO `data_analysts`;
 GRANT USE CATALOG ON CATALOG hr_prod TO `hr_team`;

 -- Grant schema-level access
 GRANT USE SCHEMA ON SCHEMA sales.revenue TO `data_analysts`;
 GRANT CREATE TABLE ON SCHEMA sales.revenue TO `data_engineers`;
 GRANT CREATE VIEW ON SCHEMA sales.revenue TO `data_analysts`;

 -- Grant table-level access
 GRANT SELECT ON TABLE sales.revenue.orders TO `data_analysts`;
 GRANT MODIFY ON TABLE sales.revenue.orders TO `data_engineers`;
 GRANT ALL PRIVILEGES ON TABLE sales.revenue.returns TO `data_owner`;

 -- Grant to a group (using backticks for account-level groups)
 GRANT SELECT ON TABLE sales.revenue.orders TO `account groups/data_analysts`;

 -- Revoke privileges
 REVOKE SELECT ON TABLE sales.revenue.orders FROM `data_analysts`;
 REVOKE ALL PRIVILEGES ON TABLE sales.revenue.orders FROM `data_owner`;

 -- Show grants
 SHOW GRANTS ON CATALOG sales;
 SHOW GRANTS ON SCHEMA sales.revenue;
 SHOW GRANTS ON TABLE sales.revenue.orders;
 SHOW GRANTS TO `data_analysts`;   -- all grants for a principal
 ```


 ### Permission Inheritance Pattern (Visual)

 ```
 -- Best practice: grant at the highest appropriate level

 -- 1. Catalog level: entry point (everyone who needs ANY access needs USE CATALOG)
 GRANT USE CATALOG ON CATALOG sales TO `sales_team`;

 -- 2. Schema level: broader access for trusted roles
 GRANT USE SCHEMA, SELECT ON SCHEMA sales.revenue TO `analysts`;

 -- 3. Table level: restrict sensitive tables only
 GRANT SELECT ON TABLE hr_prod.employees.salaries TO `hr_admins`;
 -- (analysts do NOT get this — salaries remain inaccessible to them)
 ```

 **Principle of least privilege in UC:**
 - Start with **no access** — everything is denied by default.
 - Grant **minimum necessary** at the **highest level possible**.
 - Use **groups** rather than individual users.
 - Review grants regularly with `SHOW GRANTS`.


 ### What You'd Do in Production

 1. **Create account groups** (`account groups/<name>`) and assign users to groups.
 2. **Grant at the schema level** by default — avoid per-table grants unless specificity is required.
 3. **Use catalog-level `USE CATALOG`** for broad access, schema-level `USE SCHEMA` for team access.
 4. **Audit grants quarterly** — use `system.information_schema.table_privileges` to review.
 5. **Never grant to individual users** — always use groups for maintainability.

 | Community Edition | Production Unity Catalog |
 |---|---|
 | Limited Hive GRANT/REVOKE | Full RBAC with inheritance |
 | No group support for privileges | Account groups + nested groups |
 | No separation of duties | Owners, admins, and users distinct |
 | No catalog-level governance | Catalog → Schema → Table privilege cascade |


 ## Concept 63 — Dynamic Views for Security

 ### What Are Dynamic Views?

 Dynamic views embed user-identity functions in their SQL definition to **automatically filter rows or mask columns** based on who is querying. Unlike static views, a single dynamic view serves different data to different users.

 **Key functions:**
 - `CURRENT_USER()` — returns the Databricks user email running the query
 - `IS_MEMBER(group_name)` — returns TRUE if the current user belongs to the specified group

 **Use cases:**
 - **Row-level filtering**: Sales reps see only their own deals; managers see all deals.
 - **Column-level masking**: HR sees full SSN; payroll sees last 4 digits; others see NULL.
 - **Region-based access**: EU users see only EU customer data.


 ### Community Edition Demo — Dynamic View with Simulated User Context

 Since CE has limited `IS_MEMBER()` support, we simulate the pattern using a control table and `current_user()`.


In [ ]:
# Create base HR data table
spark.sql("""
    CREATE OR REPLACE TABLE hr_db.employee_sensitive AS
    SELECT 101 AS emp_id, 'Alice' AS name, 'Engineering' AS dept, 
           95000 AS salary, '123-45-6789' AS ssn, '555-0101' AS phone
    UNION ALL
    SELECT 102, 'Bob', 'Engineering', 
           105000, '234-56-7890', '555-0102'
    UNION ALL
    SELECT 103, 'Carol', 'Sales', 
           85000, '345-67-8901', '555-0103'
    UNION ALL
    SELECT 104, 'Dave', 'HR', 
           75000, '456-78-9012', '555-0104'
    UNION ALL
    SELECT 105, 'Eve', 'Finance', 
           110000, '567-89-0123', '555-0105'
""")

print("Base employee_sensitive table:")
display(spark.sql("SELECT * FROM hr_db.employee_sensitive"))


 #### Simulated Access Control Table

 In production, you'd use Unity Catalog groups + `IS_MEMBER()`. Here we simulate with a lookup table.


In [ ]:
# Create a simulated access control table
spark.sql("""
    CREATE OR REPLACE TABLE hr_db.access_control AS
    SELECT 'HR' AS role, 'SELECT_FULL' AS access_level
    UNION ALL
    SELECT 'MANAGER', 'SELECT_MASKED'
    UNION ALL
    SELECT 'EMPLOYEE', 'SELECT_LIMITED'
""")

# Map the current user to a simulated role
current_user = spark.sql("SELECT current_user()").collect()[0][0]
print(f"Current user: {current_user}")

# Simulate role assignment (normally this comes from UC group membership)
simulated_role = "HR"  # Change to "MANAGER" or "EMPLOYEE" to test different views
print(f"Simulated role for this demo: {simulated_role}")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW current_user_role AS
    SELECT '{simulated_role}' AS role
""")


 #### Pattern 1: Column-Level Masking with Dynamic View


In [ ]:
# Create a dynamic view that masks SSN based on the user's role
# In Unity Catalog, you'd use IS_MEMBER('hr_team') instead of our simulated role lookup.

spark.sql("""
    CREATE OR REPLACE VIEW hr_db.employee_view AS
    SELECT
        emp_id,
        name,
        dept,
        CASE
            WHEN (SELECT role FROM current_user_role) = 'HR' THEN ssn
            WHEN (SELECT role FROM current_user_role) = 'MANAGER' THEN CONCAT('***-**-', RIGHT(ssn, 4))
            ELSE 'XXX-XX-XXXX'
        END AS ssn,
        CASE
            WHEN (SELECT role FROM current_user_role) IN ('HR', 'MANAGER') THEN salary
            ELSE NULL
        END AS salary,
        CASE
            WHEN (SELECT role FROM current_user_role) = 'HR' THEN phone
            ELSE '***'
        END AS phone
    FROM hr_db.employee_sensitive
""")

print(f"Employee View — Role: {simulated_role}")
display(spark.sql("SELECT * FROM hr_db.employee_view ORDER BY emp_id"))


 #### Pattern 2: Row-Level Filtering with Dynamic View


In [ ]:
# Create a view that filters rows based on the user's department
# In production, you'd use a mapping table or UC group membership.

spark.sql("""
    CREATE OR REPLACE VIEW hr_db.employee_by_dept AS
    SELECT
        e.emp_id,
        e.name,
        e.dept,
        e.salary,
        CASE
            WHEN (SELECT role FROM current_user_role) = 'HR' THEN e.ssn
            ELSE 'XXX-XX-XXXX'
        END AS ssn
    FROM hr_db.employee_sensitive e
    WHERE
        CASE
            WHEN (SELECT role FROM current_user_role) = 'HR' THEN TRUE           -- HR sees ALL rows
            WHEN (SELECT role FROM current_user_role) = 'MANAGER' THEN e.dept IN ('Engineering', 'Sales', 'HR')
            WHEN (SELECT role FROM current_user_role) = 'EMPLOYEE' THEN e.dept = 'Engineering'  -- only own dept
            ELSE FALSE
        END
""")

print(f"Employee by Dept View — Role: {simulated_role}")
display(spark.sql("SELECT * FROM hr_db.employee_by_dept ORDER BY emp_id"))


 ### Test All Three Roles


In [ ]:
import pyspark.sql.functions as F

roles = ["HR", "MANAGER", "EMPLOYEE"]

for role in roles:
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW current_user_role AS
        SELECT '{role}' AS role
    """)
    
    count = spark.sql("SELECT COUNT(*) AS cnt FROM hr_db.employee_view WHERE salary IS NOT NULL").collect()[0][0]
    total = spark.sql("SELECT COUNT(*) AS cnt FROM hr_db.employee_view").collect()[0][0]
    
    print(f"\n--- Role: {role} ---")
    print(f"  Visible rows: {total}")
    print(f"  Salary visible for: {count} employees")
    print(f"  SSN format: {'Full' if role == 'HR' else 'Masked (last 4)' if role == 'MANAGER' else 'All X'}")
    print(f"  Phone visible: {'Yes' if role == 'HR' else 'No'}")

# Reset to HR for remaining demos
spark.sql("""
    CREATE OR REPLACE TEMP VIEW current_user_role AS
    SELECT 'HR' AS role
""")


 ### Unity Catalog Dynamic View — Production Syntax

 ```sql
 -- Production UC dynamic view with IS_MEMBER() and CURRENT_USER()
 CREATE VIEW hr_prod.employees.employee_safe_view AS
 SELECT
     emp_id,
     name,
     dept,
     CASE
         WHEN IS_MEMBER('hr_team') THEN ssn
         WHEN IS_MEMBER('managers') THEN CONCAT('***-**-', RIGHT(ssn, 4))
         ELSE 'XXX-XX-XXXX'
     END AS ssn,
     CASE
         WHEN IS_MEMBER('hr_team') OR IS_MEMBER('managers') THEN salary
         ELSE NULL
     END AS salary,
     CURRENT_USER() AS queried_by
 FROM hr_prod.employees.employee_raw
 WHERE
     IS_MEMBER('hr_team')
     OR (IS_MEMBER('managers') AND dept IN (SELECT dept FROM manager_departments WHERE manager = CURRENT_USER()))
     OR (IS_MEMBER('employees') AND name = (SELECT name FROM user_employee_map WHERE user = CURRENT_USER()));
 ```


 ### Dynamic Views vs Row Filters & Column Masks

 | Feature | Dynamic Views | Row Filters + Column Masks |
 |---------|---------------|---------------------------|
 | **How it works** | Logic embedded in VIEW definition | UDFs bound to TABLE via ALTER |
 | **Apply to** | Specific view | Base table (affects ALL queries) |
 | **Override possible?** | Create separate view | No — affects all direct table access |
 | **Performance** | View overhead on every query | UDF overhead on every scan |
 | **Best for** | Tailored access per audience | Uniform policy enforcement |
 | **Introduced** | Older (GA since DBR 9.x) | Newer (UC 1.3+, DBR 14+) |
 | **Manageability** | View-level, requires DDL | Table-level, centralized policy |

 **Recommendation:** Use **dynamic views** for role-specific representations. Use **row filters + column masks** (see Concept #69) for uniform, non-negotiable security policies on base tables.


 ### What You'd Do in Production

 1. Define account groups in Unity Catalog: `hr_team`, `managers`, `sales_reps`, etc.
 2. Use `IS_MEMBER()` in view definitions for automated filtering.
 3. Add `current_user()` as a column for audit trails within the view.
 4. Combine row filtering AND column masking in the same view.
 5. Test views by impersonating different users (`EXECUTE AS` in SQL Warehouses).
 6. Deny direct table access — force all users through the secure view.


 ## Concept 64 — Data Lineage

 ### What Unity Catalog Lineage Provides

 Unity Catalog automatically captures **column-level lineage** across all workloads:
 - Which notebook produced this table?
 - Which upstream tables fed this view?
 - Which downstream dashboards depend on this column?
 - If I change column X, what breaks?

 Lineage is captured automatically — no manual instrumentation needed. It works across notebooks, Delta Live Tables pipelines, SQL queries, and jobs.


 ### Community Edition — Manual Lineage Tracking

 Since CE lacks automatic lineage, we build a manual tracking system to demonstrate the concept.


In [ ]:
# Create a lineage tracking metadata table
spark.sql("""
    CREATE OR REPLACE TABLE sales_db.data_lineage (
        lineage_id BIGINT GENERATED ALWAYS AS IDENTITY,
        target_table STRING,
        target_column STRING,
        source_table STRING,
        source_column STRING,
        transformation STRING,
        notebook_or_job STRING,
        recorded_at TIMESTAMP
    )
    USING DELTA
""")


In [ ]:
# Function to record lineage entries
def record_lineage(target_table, target_column, source_table, source_column, transformation, notebook_or_job):
    """Simulate Unity Catalog lineage tracking by inserting records into a lineage table."""
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType
    from datetime import datetime
    
    df = spark.createDataFrame(
        [(target_table, target_column, source_table, source_column, transformation, notebook_or_job, datetime.now())],
        ["target_table", "target_column", "source_table", "source_column", "transformation", "notebook_or_job", "recorded_at"]
    )
    df.write.mode("append").saveAsTable("sales_db.data_lineage")

print("Lineage recording function defined.")


 #### Build a Sample Pipeline and Map Its Lineage


In [ ]:
# Step 1: Raw revenue ingestion (simulated source)
spark.sql("""
    CREATE OR REPLACE TABLE sales_db.revenue_raw AS
    SELECT * FROM sales_db.revenue
""")
record_lineage("sales_db.revenue_raw", "*", "external_csv", "*", "Raw ingestion from CSV files", "Ingestion_Job_01")
print("Step 1: revenue_raw created from external source.")

# Step 2: Clean and standardize revenue data
spark.sql("""
    CREATE OR REPLACE TABLE sales_db.revenue_clean AS
    SELECT
        order_id,
        UPPER(product) AS product,
        CAST(amount AS DECIMAL(10,2)) AS amount,
        CAST(order_date AS DATE) AS order_date,
        MONTH(order_date) AS order_month
    FROM sales_db.revenue_raw
    WHERE amount > 0
""")
record_lineage("sales_db.revenue_clean", "product", "sales_db.revenue_raw", "product", "UPPER() standardization", "Cleaning_Job_02")
record_lineage("sales_db.revenue_clean", "amount", "sales_db.revenue_raw", "amount", "CAST to DECIMAL, filter > 0", "Cleaning_Job_02")
record_lineage("sales_db.revenue_clean", "order_month", "sales_db.revenue_raw", "order_date", "MONTH() extraction", "Cleaning_Job_02")
print("Step 2: revenue_clean created with transformations.")

# Step 3: Aggregate to monthly sales summary
spark.sql("""
    CREATE OR REPLACE TABLE sales_db.revenue_monthly AS
    SELECT
        order_month,
        product,
        SUM(amount) AS total_revenue,
        COUNT(*) AS order_count,
        AVG(amount) AS avg_order_value
    FROM sales_db.revenue_clean
    GROUP BY order_month, product
    ORDER BY order_month, product
""")
record_lineage("sales_db.revenue_monthly", "total_revenue", "sales_db.revenue_clean", "amount", "SUM() aggregation", "Aggregation_Job_03")
record_lineage("sales_db.revenue_monthly", "order_count", "sales_db.revenue_clean", "order_id", "COUNT() aggregation", "Aggregation_Job_03")
record_lineage("sales_db.revenue_monthly", "avg_order_value", "sales_db.revenue_clean", "amount", "AVG() aggregation", "Aggregation_Job_03")
print("Step 3: revenue_monthly aggregated.")

# Step 4: Create dashboard-ready view
spark.sql("""
    CREATE OR REPLACE VIEW sales_db.executive_dashboard AS
    SELECT
        order_month,
        SUM(total_revenue) AS monthly_revenue,
        SUM(order_count) AS total_orders,
        SUM(total_revenue) / NULLIF(SUM(order_count), 0) AS avg_order_value
    FROM sales_db.revenue_monthly
    GROUP BY order_month
""")
record_lineage("sales_db.executive_dashboard", "monthly_revenue", "sales_db.revenue_monthly", "total_revenue", "SUM() for dashboard", "Dashboard_Pipeline_04")
print("Step 4: executive_dashboard view created.")


 #### Query the Lineage — Impact Analysis


In [ ]:
print("=" * 70)
print("FULL LINEAGE GRAPH")
print("=" * 70)
display(spark.sql("SELECT * FROM sales_db.data_lineage ORDER BY lineage_id"))


In [ ]:
# Impact analysis: "If I change revenue_raw.product, what downstream objects are affected?"
print("=" * 70)
print("IMPACT ANALYSIS: Changing 'sales_db.revenue_raw.product'")
print("=" * 70)

affected = spark.sql("""
    WITH RECURSIVE downstream AS (
        -- Direct dependents
        SELECT target_table, target_column, transformation, 1 AS depth
        FROM sales_db.data_lineage
        WHERE source_table = 'sales_db.revenue_raw' AND source_column = 'product'
        
        UNION ALL
        
        -- Indirect dependents
        SELECT l.target_table, l.target_column, l.transformation, d.depth + 1
        FROM sales_db.data_lineage l
        JOIN downstream d ON l.source_table = d.target_table
    )
    SELECT DISTINCT target_table, target_column, transformation, depth
    FROM downstream
    ORDER BY depth, target_table
""")

display(affected)


 ### Visualize the Pipeline Lineage

 ```
 external_csv ──→ revenue_raw ──→ revenue_clean ──→ revenue_monthly ──→ executive_dashboard
                      │                   │                    │
                      │                   ├─ product (UPPER)    ├─ total_revenue (SUM)
                      │                   ├─ amount (CAST)      ├─ order_count (COUNT)
                      │                   └─ order_month (MONTH)└─ avg_order_value (AVG)
                      │
                      └─── * (raw ingestion)
 ```


 ### Unity Catalog Lineage Queries (Not Executable in CE)

 ```sql
 -- UC: Query column-level lineage for a specific table
 SELECT
     source_table_catalog,
     source_table_schema,
     source_table_name,
     source_column_name,
     target_table_catalog,
     target_table_schema,
     target_table_name,
     target_column_name,
     created_at
 FROM system.lineage.column_lineage
 WHERE target_table_name = 'revenue_monthly';

 -- UC: Find all downstream consumers of a column
 SELECT DISTINCT
     target_table_name,
     notebook_path,
     job_name
 FROM system.lineage.column_lineage
 WHERE source_table_name = 'revenue_raw' AND source_column_name = 'product';

 -- UC: Find all upstream sources for a dashboard table
 SELECT DISTINCT
     source_table_name,
     source_column_name
 FROM system.lineage.column_lineage
 WHERE target_table_name = 'executive_dashboard';
 ```


 ### What You'd Do in Production

 1. Unity Catalog lineage is **automatic** — no manual instrumentation.
 2. Use **Lineage UI** in the Databricks workspace for visual graph exploration.
 3. Query `system.lineage.column_lineage` for programmatic impact analysis.
 4. Integrate lineage into your CI/CD to detect breaking changes before deployment.
 5. Tag all notebooks, jobs, and DLT pipelines with meaningful names for clear lineage.
 6. Lineage persists across retentions — you can trace history even after source tables are dropped.


 ## Concept 65 — Information Schema & Metadata Queries

 ### Unity Catalog Information Schema

 UC provides `system.information_schema` with standardized metadata tables:

 | Table | What It Contains |
 |-------|------------------|
 | `system.information_schema.tables` | All tables and views across catalogs |
 | `system.information_schema.columns` | All columns with types, nullable, defaults |
 | `system.information_schema.table_privileges` | All GRANTs on tables |
 | `system.information_schema.schemata` | All schemas/catalogs |
 | `system.information_schema.views` | View definitions |
 | `system.information_schema.routines` | UDFs and stored procedures |

 This is ANSI SQL standard compliant — the same queries work across Databricks, PostgreSQL, Snowflake, etc.


 ### Community Edition — Hive Metastore Metadata Queries

 CE uses Hive Metastore commands to discover metadata. While not a true `information_schema`, these serve the same purpose.


In [ ]:
# Metadata Discovery — Show all databases
print("=" * 60)
print("ALL DATABASES")
print("=" * 60)
display(spark.sql("SHOW DATABASES"))

# Show tables in each database
for row in spark.sql("SHOW DATABASES").collect():
    db_name = row["databaseName"]
    print(f"\n{'=' * 60}")
    print(f"TABLES IN: {db_name}")
    print(f"{'=' * 60}")
    try:
        tables = spark.sql(f"SHOW TABLES IN {db_name}")
        display(tables)
    except Exception as e:
        print(f"  Error: {e}")


In [ ]:
# Describe each table in sales_db
print("=" * 60)
print("SCHEMA DETAIL FOR: sales_db tables")
print("=" * 60)

for row in spark.sql("SHOW TABLES IN sales_db").collect():
    tbl = row["tableName"]
    is_temp = row.get("isTemporary", False)
    print(f"\n--- DESCRIBE TABLE sales_db.{tbl} {'(TEMP)' if is_temp else ''}---")
    try:
        display(spark.sql(f"DESCRIBE TABLE sales_db.{tbl}"))
    except Exception as e:
        print(f"  Error: {e}")


In [ ]:
# Describe extended — shows table format, location, properties
print("=" * 60)
print("EXTENDED TABLE DETAILS: sales_db.revenue")
print("=" * 60)
try:
    display(spark.sql("DESCRIBE EXTENDED sales_db.revenue"))
except Exception as e:
    print(f"  Error: {e}")


In [ ]:
# Show table properties (Delta-specific metadata)
print("=" * 60)
print("TABLE PROPERTIES: sales_db.revenue")
print("=" * 60)
try:
    display(spark.sql("SHOW TBLPROPERTIES sales_db.revenue"))
except Exception as e:
    print(f"  Error: {e}")


 ### Build a Compliance Report from Metadata


In [ ]:
# Create a comprehensive metadata report
print("=" * 70)
print("COMPLIANCE REPORT — ALL TABLES & SCHEMAS")
print("=" * 70)

import pyspark.sql.functions as F

report_rows = []
for db_row in spark.sql("SHOW DATABASES").collect():
    db_name = db_row["databaseName"]
    
    for tbl_row in spark.sql(f"SHOW TABLES IN {db_name}").collect():
        tbl_name = tbl_row["tableName"]
        is_temp = tbl_row.get("isTemporary", False)
        
        if is_temp:
            continue
        
        try:
            cols_df = spark.sql(f"DESCRIBE TABLE {db_name}.{tbl_name}")
            col_rows = cols_df.collect()
            
            # Count columns and identify sensitive-sounding columns
            col_names = [r["col_name"].lower() for r in col_rows if r["col_name"] and not r["col_name"].startswith("#")]
            sensitive_cols = [c for c in col_names if any(kw in c for kw in ["ssn", "salary", "phone", "email", "address", "password", "credit", "account"])]
            
            # Get format and location from EXTENDED describe
            ext_df = spark.sql(f"DESCRIBE EXTENDED {db_name}.{tbl_name}")
            ext_map = {}
            for r in ext_df.collect():
                if r["col_name"] and r["data_type"]:
                    ext_map[r["col_name"].strip().lower()] = r["data_type"].strip()
            
            table_format = ext_map.get("provider", "unknown")
            location = ext_map.get("location", "unknown")
            
            report_rows.append((
                db_name, tbl_name, table_format, len(col_names),
                ", ".join(sensitive_cols) if sensitive_cols else "None",
                "RESTRICTED" if sensitive_cols else "OK",
                location
            ))
        except Exception as e:
            report_rows.append((db_name, tbl_name, "ERROR", 0, str(e)[:50], "ERROR", "N/A"))

# Build report DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("database", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("format", StringType(), True),
    StructField("column_count", IntegerType(), True),
    StructField("sensitive_columns", StringType(), True),
    StructField("status", StringType(), True),
    StructField("location", StringType(), True),
])

report_df = spark.createDataFrame(report_rows, schema)
display(report_df.orderBy("status", "database", "table_name"))


 ### Unity Catalog Information Schema Queries (Reference)

 ```sql
 -- UC: List all tables in the sales catalog
 SELECT table_catalog, table_schema, table_name, table_type, data_source_format
 FROM system.information_schema.tables
 WHERE table_catalog = 'sales'
 ORDER BY table_schema, table_name;

 -- UC: Find all columns with PII-sounding names
 SELECT table_catalog, table_schema, table_name, column_name, data_type
 FROM system.information_schema.columns
 WHERE LOWER(column_name) RLIKE 'ssn|salary|phone|email|address|password|credit'
 ORDER BY table_catalog, table_schema, table_name;

 -- UC: Compliance audit — tables without descriptions
 SELECT table_catalog, table_schema, table_name
 FROM system.information_schema.tables
 WHERE comment IS NULL OR comment = ''
 ORDER BY table_catalog, table_schema, table_name;

 -- UC: Review all grants on sensitive tables
 SELECT grantee, privilege_type, table_catalog, table_schema, table_name
 FROM system.information_schema.table_privileges
 WHERE table_name IN ('salaries', 'employee_sensitive')
 ORDER BY grantee, table_name;
 ```


 ### What You'd Do in Production

 1. **Automate metadata scans** — weekly job that queries `system.information_schema.columns` for new PII columns.
 2. **Build a data catalog dashboard** listing all tables, owners, last updated, sensitivity level.
 3. **Enforce tagging** — use table comments and TBLPROPERTIES to tag data classification levels.
 4. **Monitor for "rogue tables"** — tables outside designated catalogs/schemas.
 5. **Integrate with external catalogs** — push UC metadata to enterprise data catalogs (Alation, Collibra, etc.).


 ## Concept 66 — External Locations & Storage Credentials

 ### Unity Catalog External Locations

 Unity Catalog governs access to cloud storage through **external locations** — named references to cloud storage paths:

 ```
 Cloud Storage (S3/ADLS/GCS)
         │
         ▼
 Storage Credential    ← IAM Role (AWS) / Service Principal (Azure) / Service Account (GCP)
         │
         ▼
 External Location     ← s3://my-bucket/sales-data/ mapped to UC name "sales_data_loc"
         │
         ▼
 Managed/External Tables ← Created under the external location
 ```

 ### Two Types of Tables in UC

 | Type | Data Location | Schema Location | Dropping Table... |
 |------|--------------|-----------------|-------------------|
 | **Managed Table** | UC-managed storage (catalog-scoped) | UC-managed | Deletes DATA + SCHEMA |
 | **External Table** | Your cloud storage (external location) | UC-metastore | Deletes SCHEMA only; DATA persists |

 **Key difference:** Managed tables give UC full lifecycle control. External tables let you own the data files in your cloud storage.


 ### Community Edition — External Tables on DBFS

 CE does not have UC external locations, but the Hive Metastore supports external tables with a `LOCATION` clause. DBFS (`/FileStore/`, `/tmp/`) serves as the "external storage."


In [ ]:
# Create a managed Delta table (default — data goes to Spark warehouse)
spark.sql("""
    CREATE OR REPLACE TABLE sales_db.orders_managed
    USING DELTA
    AS
    SELECT 1 AS order_id, 'Widget-A' AS product, 150.00 AS amount
    UNION ALL
    SELECT 2, 'Widget-B', 250.00
""")

print("Managed table: data location chosen by Spark/Hive")
display(spark.sql("DESCRIBE EXTENDED sales_db.orders_managed"))


 **Create an external table on DBFS** — the data lives at a path you control.


In [ ]:
dbfs_path = "/FileStore/tables/sales_db/orders_external"

# Write data to an explicit DBFS location (simulates data landing in external storage)
external_df = spark.createDataFrame(
    [(3, "Widget-C", 100.00), (4, "Widget-D", 350.00), (5, "Widget-E", 225.00)],
    ["order_id", "product", "amount"]
)
external_df.write.format("delta").mode("overwrite").save(dbfs_path)
print(f"Data written to: {dbfs_path}")

# Now create an external table pointing to that location
spark.sql(f"""
    CREATE OR REPLACE TABLE sales_db.orders_external
    USING DELTA
    LOCATION '{dbfs_path}'
""")

print("\nExternal table created — schema in Hive Metastore, data at DBFS path.")
print(f"DBFS Path: {dbfs_path}")
display(spark.sql("DESCRIBE EXTENDED sales_db.orders_external"))


 ### Demonstrate Managed vs External Behavior


In [ ]:
print("=" * 60)
print("BEFORE DROP: Both tables exist")
print("=" * 60)
display(spark.sql("SHOW TABLES IN sales_db"))

# Query data from both
print("\nManaged table data:")
display(spark.sql("SELECT * FROM sales_db.orders_managed"))
print("\nExternal table data:")
display(spark.sql("SELECT * FROM sales_db.orders_external"))


In [ ]:
# Drop the managed table — data is deleted
spark.sql("DROP TABLE IF EXISTS sales_db.orders_managed")
print("Dropped MANAGED table — data is gone.")

# Drop the external table — only schema is removed
spark.sql("DROP TABLE IF EXISTS sales_db.orders_external")
print("Dropped EXTERNAL table — schema removed, data still on DBFS.")

# Re-register the external table from the same data
spark.sql(f"""
    CREATE OR REPLACE TABLE sales_db.orders_external
    USING DELTA
    LOCATION '{dbfs_path}'
""")
print("Re-registered external table from the same data — data survived the drop!")
display(spark.sql("SELECT COUNT(*) AS surviving_rows FROM sales_db.orders_external"))


 ### Unity Catalog External Location Syntax (Reference)

 ```sql
 -- UC: Create a storage credential (requires account admin)
 CREATE STORAGE CREDENTIAL aws_sales_cred
 USING 'arn:aws:iam::123456789:role/databricks-sales-access';

 -- UC: Create an external location bound to the credential
 CREATE EXTERNAL LOCATION sales_data_loc
 URL 's3://my-company-sales-bucket/'
 WITH (
     CREDENTIAL aws_sales_cred
 )
 COMMENT 'Sales department raw data landing zone';

 -- UC: Grant access to the external location
 GRANT CREATE TABLE, READ FILES, WRITE FILES
 ON EXTERNAL LOCATION sales_data_loc
 TO `data_engineers`;

 -- UC: Create an external table on the governed location
 CREATE TABLE sales.revenue.invoices
 LOCATION 's3://my-company-sales-bucket/invoices/';

 -- UC: Show all external locations
 SHOW EXTERNAL LOCATIONS;

 -- UC: Describe an external location
 DESCRIBE EXTERNAL LOCATION sales_data_loc;
 ```


 ### What You'd Do in Production

 1. **Create external locations for all cloud storage paths** used by Databricks.
 2. **Use managed tables for derived/internal data** — let UC handle lifecycle.
 3. **Use external tables for shared/ingested data** — you control the source.
 4. **Grant READ FILES / WRITE FILES** at the external location level, not per-bucket.
 5. **Never use service credentials directly in notebooks** — always go through UC external locations.
 6. **Audit external locations** to ensure no unauthorized cloud storage access exists.

 | Community Edition | Production Unity Catalog |
 |---|---|
 | DBFS as "external" storage | S3/ADLS/GCS via external locations |
 | No storage credential abstraction | IAM roles, service principals, service accounts |
 | `LOCATION '/path/'` in Hive Metastore | `LOCATION 's3://...'` governed by UC |
 | No access control on paths | GRANT READ/WRITE FILES on external locations |


 ## Concept 67 — Service Principals & Managed Identity

 ### What Are Service Principals?

 A **service principal** is a non-human identity — an application, CI/CD pipeline, or scheduled job — that authenticates to Databricks using its own credentials rather than a user's personal account.

 ```
 HUMAN IDENTITY:                    MACHINE IDENTITY:
 alice@company.com                 sp-sales-etl-prod
   │ OIDC / SSO                        │ OAuth2 client credentials
   │ Personal token                     │ Secret + Client ID
   ▼                                   ▼
 Databricks Workspace ◄─────────── Databricks Workspace
 ```

 ### Why NOT Use Personal Accounts for Production?

 | Risk | Consequence |
 |------|------------|
 | Employee leaves company | Job breaks; token deleted; panic |
 | Personal token leaked | Full user access compromised |
 | No separation of duties | Alice's prod job runs as Alice, not as "ETL Pipeline" |
 | Audit confusion | Who really ran this — Alice or her script? |
 | Credential rotation | Personal tokens aren't managed by security teams |

 **Production workloads MUST run as service principals.**


 ### Service Principal Architecture

 **On Azure:**
 ```
 Azure AD App Registration → Service Principal
         │
         ├── Client ID + Secret/Certificate (for OAuth)
         ├── Managed Identity (Azure resource-level, no secret management)
         └── Assigned to Databricks workspace via SCIM/API
 ```

 **On AWS:**
 ```
 AWS IAM Role → Instance Profile
         │
         ├── Trust policy: allows Databricks to assume this role
         ├── Attached to Databricks clusters at launch time
         └── Provides S3 access without embedding keys
 ```

 **On GCP:**
 ```
 GCP Service Account
         │
         ├── JSON key + service account email
         ├── Assigned to Databricks workspace
         └── Provides GCS/BigQuery access
 ```


 ### Community Edition — Conceptual Simulation

 We can't create real service principals in CE, but we can demonstrate the **principle** of machine vs human identity and best practices.


In [ ]:
# Simulate a service principal identity pattern
from datetime import datetime

print("=" * 60)
print("SERVICE PRINCIPAL IDENTITY SIMULATION")
print("=" * 60)

# In production, this would be the service principal's identity
# obtained via OAuth client credentials grant
simulated_sp_identity = {
    "service_principal_id": "sp-sales-etl-prod-abc123",
    "display_name": "Sales ETL Production Pipeline",
    "application_id": "a1b2c3d4-e5f6-7890-abcd-ef1234567890",
    "type": "service_principal",
    "assigned_groups": ["etl_jobs", "sales_data_readers"],
    "created_by": "security_team@company.com",
    "notebook_owner": "alice@company.com",  # This is WRONG in production!
}

print(f"""
PRODUCTION BEST PRACTICE: Service Principal Configuration
{'─' * 55}
ID:              {simulated_sp_identity['service_principal_id']}
Name:            {simulated_sp_identity['display_name']}
Type:            {simulated_sp_identity['type']}
Groups:          {', '.join(simulated_sp_identity['assigned_groups'])}
Managed by:      {simulated_sp_identity['created_by']}

WHAT TO AVOID:
  Notebook Owner: {simulated_sp_identity['notebook_owner']} ← PERSONAL ACCOUNT!
  
CORRECT PATTERN:
  Schedule job "as" the service principal, NOT as Alice.
  If Alice leaves, the job continues running.
""")


 ### Best Practices for Service Principal Usage


In [ ]:
best_practices = """
BEST PRACTICES FOR SERVICE PRINCIPALS
════════════════════════════════════════════════════════════════════

1. NAMING CONVENTION
   sp-{domain}-{purpose}-{environment}
   Examples: sp-sales-etl-prod, sp-marketing-analytics-dev, sp-finance-reports-prod

2. PRINCIPLE OF LEAST PRIVILEGE
   - Each service principal gets ONLY the minimum privileges needed.
   - One SP per logical workload (not one SP for everything).
   - Use Unity Catalog GRANTs scoped to specific catalogs/schemas/tables.

3. SECRET MANAGEMENT
   - NEVER hardcode secrets in notebooks or config files.
   - Use Databricks secrets API: dbutils.secrets.get()
   - Rotate secrets on a schedule (90-day maximum).
   - On Azure, prefer Managed Identity (no secret to manage at all).

4. SEPARATION OF DUTIES
   - DEV service principal: can create/modify tables in dev catalogs only.
   - PROD service principal: read-only on raw data, write to downstream tables.
   - CI/CD service principal: can create schemas/tables in all environments.
   - Human users: interactive development only.

5. AUDIT TRAIL
   - Every action taken by a service principal is logged in system.access.audit.
   - Use the service_principal_id to distinguish machine from human actions.
   - Set up alerts for unusual SP activity (e.g., SP running interactive notebooks).

6. MONITORING & ROTATION
   - Track: last used, token expiry, unused SPs (stale identities).
   - Rotate: OAuth secrets every 90 days or less.
   - Revoke: immediately disable SP secrets if compromise is suspected.

7. TOKEN SCOPE
   - Service principal tokens should have minimal scope.
   - Use ACLs on the token itself (not just the SP).
   - Prefer short-lived tokens (OAuth flow) over long-lived PATs.
"""

print(best_practices)


 ### What You'd Do in Production

 1. **Register service principals in your identity provider** (Azure AD, AWS IAM, GCP IAM).
 2. **Add them to your Databricks account** via SCIM provisioning or Account API.
 3. **Assign Unity Catalog privileges** — `GRANT SELECT ON TABLE ... TO \`sp-sales-etl-prod\``.
 4. **Schedule ALL production jobs** to run as service principals, never as users.
 5. **Use Databricks secrets** for OAuth credentials: `sp_client_id` and `sp_client_secret`.
 6. **Implement automated rotation** — Azure Key Vault, AWS Secrets Manager, or GCP Secret Manager.
 7. **Alert on personal-account jobs** — detect and migrate any production jobs running as users.

 | Community Edition | Production Unity Catalog |
 |---|---|
 | No service principal support | Full SP lifecycle via Account API |
 | Jobs run as the notebook owner | Jobs run as service principals |
 | No managed identity | Azure Managed Identity / AWS Instance Profiles |
 | Limited token management | OAuth2 client credentials with secrets rotation |


 ## Concept 68 — Audit Logging

 ### Unity Catalog Audit Logging

 UC provides `system.access.audit` — a comprehensive audit log table that captures every operation:

 | Field | Description |
 |-------|-------------|
 | `event_time` | When the operation occurred |
 | `user_identity.email` | Who performed the operation (human or service principal) |
 | `action_name` | What action (e.g., `selectTable`, `createTable`, `grant`) |
 | `request_params.commandText` | The SQL statement (for queries) |
 | `request_params.table_full_name` | The table affected |
 | `response.status_code` | HTTP status (200 = success) |
 | `source_ip_address` | Client IP address |
 | `service_name` | Which service (e.g., `unity-catalog`, `databricks-sql`) |

 Audit logs are critical for:
 - **Compliance** (SOC2, HIPAA, GDPR, PCI-DSS)
 - **Forensic investigation** ("Who accessed this table at 3 AM?")
 - **Anomaly detection** ("Why did the CI/CD SP drop a production table?")
 - **Access review** ("Is ANYONE still using this deprecated table?")


 ### Community Edition — Manual Audit Logging System

 CE does not have `system.access.audit`. We build a manual audit log to demonstrate the concept.


In [ ]:
# Create an audit log table
spark.sql("""
    CREATE OR REPLACE TABLE sales_db.audit_log (
        event_id BIGINT GENERATED ALWAYS AS IDENTITY,
        event_time TIMESTAMP,
        user_identity STRING,
        action_name STRING,
        object_type STRING,
        object_name STRING,
        query_text STRING,
        status STRING,
        row_count BIGINT,
        execution_time_ms BIGINT,
        notebook_name STRING,
        cluster_id STRING
    )
    USING DELTA
""")

print("Audit log table created.")


In [ ]:
# Build a decorator-like audit logging function
from datetime import datetime, timedelta
import time

def log_audit(action_name, object_type, object_name, query_text, status, row_count=0, execution_time_ms=0):
    """Record an audit entry — simulates UC system.access.audit"""
    current_user = spark.sql("SELECT current_user()").collect()[0][0]
    
    # Get notebook info (may not be available in all contexts)
    try:
        notebook_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().getOrElse("unknown")
    except:
        notebook_name = "interactive_session"
    
    cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "unknown")
    
    audit_df = spark.createDataFrame(
        [(datetime.now(), current_user, action_name, object_type, object_name, 
          query_text, status, row_count, execution_time_ms, notebook_name, cluster_id)],
        ["event_time", "user_identity", "action_name", "object_type", "object_name",
         "query_text", "status", "row_count", "execution_time_ms", "notebook_name", "cluster_id"]
    )
    audit_df.write.mode("append").saveAsTable("sales_db.audit_log")

def execute_and_log(query, action_name, object_type, object_name):
    """Execute a query and log it to the audit table."""
    start = time.time()
    try:
        result = spark.sql(query)
        elapsed_ms = int((time.time() - start) * 1000)
        
        if action_name in ("selectTable", "describeTable"):
            row_count = result.count()
        else:
            row_count = -1  # DDL statements don't return rows
        
        log_audit(action_name, object_type, object_name, query, "SUCCESS", row_count, elapsed_ms)
        return result
    except Exception as e:
        elapsed_ms = int((time.time() - start) * 1000)
        log_audit(action_name, object_type, object_name, query, f"FAILED: {str(e)[:100]}", 0, elapsed_ms)
        raise

print("Audit logging functions defined.")


 ### Demonstrate Audit Logging in Action


In [ ]:
# Execute several operations that will be logged
print("Executing operations with audit logging...\n")

# 1. SELECT
execute_and_log("SELECT * FROM sales_db.revenue", "selectTable", "TABLE", "sales_db.revenue")
print("  [1] SELECT from revenue — logged")

# 2. CREATE TABLE
execute_and_log(
    "CREATE OR REPLACE TABLE sales_db.audit_test_table AS SELECT * FROM sales_db.revenue WHERE amount > 100",
    "createTable", "TABLE", "sales_db.audit_test_table"
)
print("  [2] CREATE TABLE audit_test_table — logged")

# 3. DESCRIBE
execute_and_log("DESCRIBE TABLE sales_db.revenue", "describeTable", "TABLE", "sales_db.revenue")
print("  [3] DESCRIBE revenue — logged")

# 4. ALTER TABLE
try:
    execute_and_log(
        "ALTER TABLE sales_db.revenue SET TBLPROPERTIES ('classification' = 'sales_data')",
        "alterTable", "TABLE", "sales_db.revenue"
    )
    print("  [4] ALTER TABLE revenue — logged")
except Exception as e:
    print(f"  [4] ALTER TABLE — {e}")

# 5. SELECT (another read)
execute_and_log("SELECT order_id, product FROM sales_db.pipeline", "selectTable", "TABLE", "sales_db.pipeline")
print("  [5] SELECT from pipeline — logged")

print("\nAll operations logged to sales_db.audit_log")


 ### Query the Audit Log


In [ ]:
print("=" * 70)
print("AUDIT LOG — Complete History")
print("=" * 70)
display(spark.sql("""
    SELECT event_id, event_time, user_identity, action_name, 
           object_name, status, execution_time_ms
    FROM sales_db.audit_log
    ORDER BY event_id
"""))


In [ ]:
# Audit summarization queries
print("=" * 70)
print("AUDIT SUMMARY BY ACTION")
print("=" * 70)
display(spark.sql("""
    SELECT 
        action_name,
        COUNT(*) AS event_count,
        SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) AS success_count,
        SUM(CASE WHEN status LIKE 'FAILED%' THEN 1 ELSE 0 END) AS failure_count,
        AVG(execution_time_ms) AS avg_execution_ms
    FROM sales_db.audit_log
    GROUP BY action_name
    ORDER BY event_count DESC
"""))


In [ ]:
print("=" * 70)
print("AUDIT — TABLE ACCESS PATTERNS")
print("=" * 70)
display(spark.sql("""
    SELECT 
        object_name,
        MIN(event_time) AS first_access,
        MAX(event_time) AS last_access,
        COUNT(*) AS total_operations,
        COUNT(DISTINCT user_identity) AS distinct_users
    FROM sales_db.audit_log
    GROUP BY object_name
    ORDER BY last_access DESC
"""))


In [ ]:
print("=" * 70)
print("AUDIT — RECENT FAILURES (Last 25 entries)")
print("=" * 70)
display(spark.sql("""
    SELECT event_time, user_identity, action_name, object_name, 
           status, execution_time_ms
    FROM sales_db.audit_log
    WHERE status LIKE 'FAILED%'
    ORDER BY event_time DESC
    LIMIT 25
"""))


In [ ]:
# Clean up the test table
spark.sql("DROP TABLE IF EXISTS sales_db.audit_test_table")
log_audit("dropTable", "TABLE", "sales_db.audit_test_table", "DROP TABLE audit_test_table", "SUCCESS")
print("Cleaned up audit_test_table — logged the drop.")


 ### What Production Audit Logs Would Contain

 ```sql
 -- UC: Query production audit logs
 SELECT
     event_time,
     user_identity.email,
     action_name,
     request_params.table_full_name,
     request_params.commandText,
     response.status_code,
     source_ip_address,
     service_name
 FROM system.access.audit
 WHERE event_date >= CURRENT_DATE() - INTERVAL 7 DAYS
   AND action_name IN ('selectTable', 'createTable', 'dropTable', 'grant')
 ORDER BY event_time DESC;

 -- UC: Find tables that haven't been accessed in 30 days (cleanup candidates)
 SELECT request_params.table_full_name, MAX(event_time) AS last_accessed
 FROM system.access.audit
 WHERE action_name = 'selectTable'
 GROUP BY request_params.table_full_name
 HAVING MAX(event_time) < CURRENT_DATE() - INTERVAL 30 DAYS;

 -- UC: Anomaly detection — tables dropped outside business hours
 SELECT event_time, user_identity.email, request_params.table_full_name
 FROM system.access.audit
 WHERE action_name = 'dropTable'
   AND HOUR(event_time) NOT BETWEEN 7 AND 19;
 ```


 ### What You'd Do in Production

 1. **Enable audit logging** — it's on by default in all paid tiers, retained for 365 days.
 2. **Create alert queries** that run daily and flag suspicious activity (e.g., `dropTable` by unusual users).
 3. **Build an audit dashboard** in Databricks SQL with time-series charts of operations.
 4. **Integrate with SIEM systems** — export audit logs to Splunk, Azure Sentinel, or ELK Stack.
 5. **Set up retention policies** — archive old audit logs to cold storage for compliance.
 6. **Use audit data for cost analysis** — identify heavy query users and optimize.


 ## Concept 69 — Row Filters & Column Masks

 ### Unity Catalog Row Filters & Column Masks (UC 1.3+)

 Row filters and column masks are **table-level security policies** enforced by UDFs:

 **Row Filter:** A SQL UDF that returns TRUE for rows the user is allowed to see.
 ```sql
 CREATE FUNCTION sales_filter(dept STRING)
 RETURN IS_MEMBER('hr_team')
     OR (IS_MEMBER('managers') AND dept IN ('Engineering', 'Sales'))
     OR CURRENT_USER() = CONCAT(dept, '_dept_head@company.com');

 ALTER TABLE hr.employee.data SET ROW FILTER sales_filter ON (dept);
 ```

 **Column Mask:** A SQL UDF that returns the masked value for the column.
 ```sql
 CREATE FUNCTION ssn_mask(ssn STRING)
 RETURN CASE
     WHEN IS_MEMBER('hr_team') THEN ssn
     ELSE 'XXX-XX-XXXX'
 END;

 ALTER TABLE hr.employee.data ALTER COLUMN ssn SET MASK ssn_mask;
 ```

 **Critical difference from dynamic views:** These are bolted onto the **base table** — every query against the table (including direct `SELECT *`) is automatically filtered. There is no way to "accidentally" bypass the policy.


 ### Community Edition — Row Filtering & Column Masking via Views

 In CE, we implement the same functionality using views (since we can't ALTER TABLE with row filters/column masks).


In [ ]:
# Create a comprehensive HR dataset with various sensitivity levels
spark.sql("""
    CREATE OR REPLACE TABLE hr_db.employee_complete AS
    SELECT 201 AS emp_id, 'Alice' AS name, 'Engineering' AS dept, 
           95000 AS salary, '123-45-6789' AS ssn, '2020-03-15' AS hire_date,
           'F' AS gender, 'Seattle' AS location, '555-0101' AS phone,
           4 AS performance_rating
    UNION ALL
    SELECT 202, 'Bob', 'Engineering', 105000, '234-56-7890', '2019-01-10',
           'M', 'Seattle', '555-0102', 5
    UNION ALL
    SELECT 203, 'Carol', 'Sales', 85000, '345-67-8901', '2021-06-01',
           'F', 'New York', '555-0103', 4
    UNION ALL
    SELECT 204, 'Dave', 'HR', 75000, '456-78-9012', '2018-11-20',
           'M', 'Chicago', '555-0104', 3
    UNION ALL
    SELECT 205, 'Eve', 'Finance', 110000, '567-89-0123', '2022-02-28',
           'F', 'Chicago', '555-0105', 5
    UNION ALL
    SELECT 206, 'Frank', 'Marketing', 72000, '678-90-1234', '2021-09-15',
           'M', 'New York', '555-0106', 3
""")

print("Employee complete table created:")
display(spark.sql("SELECT * FROM hr_db.employee_complete ORDER BY emp_id"))


 #### Define Role-Based Access Functions (Simulating UDF Row Filters / Column Masks)


In [ ]:
# Create a role mapping table (simulates Unity Catalog group membership)
spark.sql("""
    CREATE OR REPLACE TABLE hr_db.role_permissions AS
    SELECT 'hr_admin' AS role, 'ALL_DEPTS' AS dept_access, 'FULL' AS salary_visibility, 'FULL' AS ssn_visibility
    UNION ALL
    SELECT 'manager', 'OWN_DEPT', 'VISIBLE', 'MASKED'
    UNION ALL
    SELECT 'analyst', 'ALL_DEPTS', 'MASKED', 'HIDDEN'
    UNION ALL
    SELECT 'employee', 'OWN_DEPT', 'HIDDEN', 'HIDDEN'
    UNION ALL
    SELECT 'contractor', 'NONE', 'HIDDEN', 'HIDDEN'
""")

# Set the simulated role for this demo
simulated_role = "analyst"  # Try: hr_admin, manager, analyst, employee, contractor
simulated_dept = "Engineering"  # For "OWN_DEPT" roles

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW current_role_config AS
    SELECT '{simulated_role}' AS role, '{simulated_dept}' AS user_dept
""")

# Load role permissions
role_perms = spark.sql(f"""
    SELECT * FROM hr_db.role_permissions WHERE role = '{simulated_role}'
""").collect()[0]

print(f"Simulated Role: {simulated_role}")
print(f"  Department Access: {role_perms['dept_access']}")
print(f"  Salary Visibility: {role_perms['salary_visibility']}")
print(f"  SSN Visibility:    {role_perms['ssn_visibility']}")


 #### Row Filter Equivalent — A View That Automatically Filters Rows


In [ ]:
# Create a view that simulates a table with a ROW FILTER applied
# In UC: ALTER TABLE hr_db.employee_complete SET ROW FILTER dept_filter ON (dept);
# In CE: We create a view that embeds the filter logic.

spark.sql("""
    CREATE OR REPLACE VIEW hr_db.employee_row_filtered AS
    SELECT e.*
    FROM hr_db.employee_complete e
    CROSS JOIN hr_db.role_permissions r
    WHERE r.role = (SELECT role FROM current_role_config)
      AND (
          -- hr_admin: sees all departments
          r.dept_access = 'ALL_DEPTS'
          OR
          -- manager / employee: sees only their own department
          (r.dept_access = 'OWN_DEPT' AND e.dept = (SELECT user_dept FROM current_role_config))
      )
      AND r.dept_access != 'NONE'  -- contractors see nothing
""")

print(f"Employee Row-Filtered View — Simulated Role: {simulated_role}")
display(spark.sql("SELECT emp_id, name, dept FROM hr_db.employee_row_filtered ORDER BY emp_id"))


 #### Column Mask Equivalent — A View That Automatically Masks Columns


In [ ]:
# Create a view that simulates COLUMN MASKS on salary, ssn, and phone
spark.sql("""
    CREATE OR REPLACE VIEW hr_db.employee_column_masked AS
    SELECT
        e.emp_id,
        e.name,
        e.dept,
        -- Salary masking
        CASE r.salary_visibility
            WHEN 'FULL' THEN CAST(e.salary AS STRING)
            WHEN 'VISIBLE' THEN CAST(e.salary AS STRING)
            WHEN 'MASKED' THEN CONCAT('$', CAST(ROUND(e.salary / 1000) * 1000 AS STRING), ' (band)')
            ELSE 'REDACTED'
        END AS salary_display,
        -- SSN masking
        CASE r.ssn_visibility
            WHEN 'FULL' THEN e.ssn
            WHEN 'MASKED' THEN CONCAT('***-**-', RIGHT(e.ssn, 4))
            ELSE 'XXX-XX-XXXX'
        END AS ssn_display,
        -- Phone masking
        CASE r.ssn_visibility
            WHEN 'FULL' THEN e.phone
            WHEN 'MASKED' THEN CONCAT('***-', RIGHT(e.phone, 4))
            ELSE 'N/A'
        END AS phone_display,
        e.hire_date,
        e.location,
        -- Performance rating visible to managers and HR
        CASE 
            WHEN r.dept_access = 'ALL_DEPTS' THEN CAST(e.performance_rating AS STRING)
            WHEN r.dept_access = 'OWN_DEPT' THEN CAST(e.performance_rating AS STRING)
            ELSE 'HIDDEN'
        END AS performance_rating_display
    FROM hr_db.employee_complete e
    CROSS JOIN hr_db.role_permissions r
    WHERE r.role = (SELECT role FROM current_role_config)
""")

print(f"Employee Column-Masked View — Simulated Role: {simulated_role}")
display(spark.sql("SELECT * FROM hr_db.employee_column_masked ORDER BY emp_id"))


 #### Complete Security View — Row Filter + Column Mask Combined


In [ ]:
# The "complete" security view: both row filtering AND column masking in one view.
# This is the pattern you'd typically enforce with UC row filters + column masks on the base table.

spark.sql("""
    CREATE OR REPLACE VIEW hr_db.employee_secure AS
    SELECT
        -- Non-sensitive: visible to all authorized users
        e.emp_id,
        e.name,
        e.dept,
        e.hire_date,
        e.location,
        e.gender,
        -- Sensitive: masked based on role
        CASE r.salary_visibility
            WHEN 'FULL' THEN CAST(e.salary AS STRING)
            WHEN 'VISIBLE' THEN CONCAT('$', FORMAT_NUMBER(e.salary, 0))
            WHEN 'MASKED' THEN CONCAT('$', CAST(ROUND(e.salary / 5000) * 5000 AS STRING), ' band')
            ELSE 'REDACTED'
        END AS salary,
        CASE r.ssn_visibility
            WHEN 'FULL' THEN e.ssn
            WHEN 'MASKED' THEN CONCAT('***-**-', RIGHT(e.ssn, 4))
            ELSE 'XXX-XX-XXXX'
        END AS ssn,
        CASE r.ssn_visibility
            WHEN 'FULL' THEN e.phone
            WHEN 'MASKED' THEN CONCAT('***-', RIGHT(e.phone, 4))
            ELSE 'N/A'
        END AS phone,
        CASE 
            WHEN r.salary_visibility IN ('FULL', 'VISIBLE') THEN CAST(e.performance_rating AS STRING)
            ELSE 'HIDDEN'
        END AS performance_rating
    FROM hr_db.employee_complete e
    CROSS JOIN hr_db.role_permissions r
    WHERE r.role = (SELECT role FROM current_role_config)
      AND (
          r.dept_access = 'ALL_DEPTS'
          OR (r.dept_access = 'OWN_DEPT' AND e.dept = (SELECT user_dept FROM current_role_config))
      )
      AND r.dept_access != 'NONE'
""")

print(f"=" * 70)
print(f"FULL SECURITY VIEW — Role: {simulated_role} | Dept: {simulated_dept}")
print(f"=" * 70)
display(spark.sql("SELECT * FROM hr_db.employee_secure ORDER BY emp_id"))


 ### Test All Five Roles Through the Secure View


In [ ]:
test_roles = [
    ("hr_admin", "Engineering"),
    ("manager", "Engineering"),
    ("analyst", "Engineering"),
    ("employee", "Engineering"),
    ("contractor", "Engineering"),
]

print("=" * 72)
print(f"{'Role':<16} {'Rows':>6} {'Salary':>12} {'SSN':>14} {'Phone':>14} {'Perf':>6}")
print("-" * 72)

for role, dept in test_roles:
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW current_role_config AS
        SELECT '{role}' AS role, '{dept}' AS user_dept
    """)
    
    rows = spark.sql("SELECT COUNT(*) FROM hr_db.employee_secure").collect()[0][0]
    salary_example = spark.sql("SELECT salary FROM hr_db.employee_secure WHERE emp_id = 201").collect()
    salary_val = salary_example[0][0] if salary_example else "N/A"
    ssn_example = spark.sql("SELECT ssn FROM hr_db.employee_secure WHERE emp_id = 201").collect()
    ssn_val = ssn_example[0][0] if ssn_example else "N/A"
    perf_example = spark.sql("SELECT performance_rating FROM hr_db.employee_secure WHERE emp_id = 201").collect()
    perf_val = perf_example[0][0] if perf_example else "N/A"
    
    print(f"{role:<16} {rows:>6} {str(salary_val)[:12]:>12} {str(ssn_val)[:14]:>14} {'***' if ssn_val == 'XXX-XX-XXXX' else ssn_val[:14]:>14} {str(perf_val)[:6]:>6}")

print("-" * 72)

# Reset to hr_admin for remaining demos
spark.sql("""
    CREATE OR REPLACE TEMP VIEW current_role_config AS
    SELECT 'hr_admin' AS role, 'Engineering' AS user_dept
""")


 ### When to Use Row Filters / Column Masks vs Dynamic Views

 | Scenario | Best Approach | Why |
 |----------|--------------|-----|
 | One audience needs tailored columns | Dynamic View | View is purpose-built for that audience |
 | **All** queries must filter by region | **Row Filter on base table** | No way to bypass — even ad-hoc queries get filtered |
 | SSN must **never** be exposed to non-HR | **Column Mask on base table** | Absolute enforcement at the table level |
 | Multiple different rollups for different teams | Dynamic Views | Each team gets their own view |
 | Regulatory compliance (GDPR, HIPAA) | **Row Filter + Column Mask** | Mandatory, universal, non-circumventable |
 | Quick prototype or dashboard | Dynamic View | Simpler to create and iterate |
 | Data scientists doing exploratory analysis | Row Filter on base table | They query the table directly but see only allowed data |

 **Rule of thumb:**
 - **Regulatory / mandatory policies** → Row filters + column masks on base tables
 - **Convenience / audience-specific views** → Dynamic views
 - **Both can coexist** — base table has masks, views provide convenient rollups on top.


 ### Unity Catalog Row Filter & Column Mask Syntax (Reference)

 ```sql
 -- Step 1: Create a FILTER UDF (catalog-qualified)
 CREATE OR REPLACE FUNCTION hr_prod.employees.region_filter(region STRING)
 RETURN IS_MEMBER('hr_admins')
     OR (IS_MEMBER('us_managers') AND region = 'US')
     OR (IS_MEMBER('eu_managers') AND region = 'EU');

 -- Step 2: Apply the row filter to the table
 ALTER TABLE hr_prod.employees.data
 SET ROW FILTER hr_prod.employees.region_filter ON (region);

 -- Step 3: Create a MASK UDF
 CREATE OR REPLACE FUNCTION hr_prod.employees.ssn_mask(ssn STRING)
 RETURN CASE
     WHEN IS_MEMBER('hr_admins') THEN ssn
     WHEN IS_MEMBER('payroll') THEN CONCAT('***-**-', RIGHT(ssn, 4))
     ELSE NULL
 END;

 -- Step 4: Apply the column mask
 ALTER TABLE hr_prod.employees.data
 ALTER COLUMN ssn SET MASK hr_prod.employees.ssn_mask;

 -- Show row filters on a table
 SHOW ROW FILTERS ON TABLE hr_prod.employees.data;

 -- Remove a row filter
 ALTER TABLE hr_prod.employees.data DROP ROW FILTER region_filter;

 -- Remove a column mask
 ALTER TABLE hr_prod.employees.data ALTER COLUMN ssn DROP MASK;
 ```


 ### What You'd Do in Production

 1. **Apply row filters on ALL tables containing regional or departmental data** — never rely on users to self-filter.
 2. **Apply column masks on ALL PII columns** — SSN, salary, phone, email, address, IP address.
 3. **Create the UDFs in the same schema as the tables** for clean organization.
 4. **Test with multiple user personas** before deploying to production.
 5. **Document each policy** — what it filters, why, and who is exempt.
 6. **Regularly review** with `SHOW ROW FILTERS` and `SHOW COLUMN MASKS` to ensure policies are still correct.


 ## Concept 70 — Delta Sharing

 ### What Is Delta Sharing?

 Delta Sharing is an **open protocol** for securely sharing live data across organizations without copying it. It's built on Delta Lake and works with any platform that implements the sharing protocol (not just Databricks).

 ```
 ┌─────────────────────┐          ┌─────────────────────────┐
 │  DATA PROVIDER       │          │  DATA RECIPIENT           │
 │  (your company)      │          │  (partner/customer)       │
 │                      │          │                           │
 │  ┌─────────────┐    │  Share   │  ┌──────────────────┐    │
 │  │ Unity Catalog│────┼──────────┼──│ Databricks        │    │
 │  │   Tables     │    │ Activate │  │ (same/diff cloud) │    │
 │  └─────────────┘    │  Link    │  │                   │    │
 │                      │──────────┼──│ OR: Power BI      │    │
 │  Recipients:         │          │  │ OR: pandas 🐼     │    │
 │   - partner@acme.com│          │  │ OR: Spark         │    │
 │   - client@globex.io│          │  │ OR: Trino/Presto  │    │
 └─────────────────────┘          └─────────────────────────┘
 ```

 **Key properties:**
 - **No data copy** — recipient queries LIVE data (table changes visible immediately)
 - **Open protocol** — works with open-source Delta Sharing connectors
 - **Cross-platform** — Spark, pandas, Power BI, Tableau, Trino all supported
 - **Cross-cloud** — provider on AWS, recipient on Azure (or vice versa)
 - **Granular** — share specific tables, partitions, or even row-filtered subsets


 ### Community Edition — Simulation of Delta Sharing

 CE cannot create actual Delta Shares. We simulate the concept by:
 1. Creating a "shareable" dataset (read-only view)
 2. Simulating the provider/recipient flow
 3. Demonstrating what the production code would look like


In [ ]:
# Create a clean dataset that could be shared externally
spark.sql("""
    CREATE OR REPLACE TABLE sales_db.shared_sales_summary AS
    SELECT
        product,
        SUM(amount) AS total_revenue,
        COUNT(*) AS order_count,
        AVG(amount) AS avg_order_value,
        MIN(order_date) AS first_order,
        MAX(order_date) AS last_order
    FROM sales_db.revenue
    GROUP BY product
""")

print("Shared sales summary dataset: (provider's data that gets shared)")
display(spark.sql("SELECT * FROM sales_db.shared_sales_summary ORDER BY product"))


 #### Simulate the Sharing Flow — Provider Side


In [ ]:
print("=" * 70)
print("DELTA SHARING SIMULATION — Provider Side")
print("=" * 70)

# In production Databricks, the provider would do these steps:

provider_steps = """
PROVIDER STEPS (your company shares data):
──────────────────────────────────────────

[1] CREATE SHARE — define the share container:
    CREATE SHARE sales_insights_share
    COMMENT 'Monthly sales insights for partners';

[2] ADD TABLES TO THE SHARE:
    ALTER SHARE sales_insights_share ADD TABLE sales_db.shared_sales_summary;
    ALTER SHARE sales_insights_share ADD TABLE sales_db.revenue_monthly;

    -- Add with partition-based filtering (share only 2025 data):
    ALTER SHARE sales_insights_share ADD TABLE sales_db.revenue_monthly
    WITH RESTRICTION revenue_monthly.order_month >= '2025-01-01';

[3] ADD RECIPIENTS:
    CREATE RECIPIENT acme_corp USING ID 'acme-data-team@acme.com';
    CREATE RECIPIENT globex_inc USING ID 'analytics@globex.io';

[4] GRANT ACCESS to the share:
    GRANT SELECT ON SHARE sales_insights_share TO RECIPIENT acme_corp;
    GRANT SELECT ON SHARE sales_insights_share TO RECIPIENT globex_inc;

[5] RECIPIENT ACTIVATES:
    - Recipient receives an activation link via email
    - They click to download a credential file
    - They configure their Databricks workspace or open-source client
"""

print(provider_steps)


 #### Simulate the Sharing Flow — Recipient Side


In [ ]:
# The "recipient" would get a credential file like this:
recipient_credential_example = """
RECIPIENT SIDE (partner consumes shared data):
─────────────────────────────────────────────

Received credential file: acme_databricks_share.config
{
  "shareCredentialsVersion": 1, 
  "endpoint": "https://adb-123456789.10.azuredatabricks.net/api/2.0/unity-catalog/shares",
  "bearerToken": "dapi1234abcd...",
  "expirationTime": "2026-05-07T00:00:00Z"
}

On their Databricks workspace (or any Delta Sharing client):

[1] Create a CATALOG from the share:
    CREATE CATALOG sales_from_partner
    USING SHARE acme_corp.sales_insights_share;

[2] Query the shared data AS IF IT WERE LOCAL:
    SELECT * FROM sales_from_partner.shared_sales_summary;
    SELECT product, SUM(total_revenue) 
    FROM sales_from_partner.revenue_monthly 
    GROUP BY product;

[3] Join shared data with their own data:
    SELECT 
        c.customer_name,
        s.product,
        s.total_revenue
    FROM my_catalog.accounts.customers c
    JOIN sales_from_partner.revenue_monthly s
      ON c.region = s.region;

WITH OPEN-SOURCE CLIENTS (Python):
    import delta_sharing
    client = delta_sharing.SharingClient(profile_file)
    table_url = profile_file + "#sales_db.shared_sales_summary"
    df = delta_sharing.load_as_pandas(table_url)
    print(df.head())
"""

print(recipient_credential_example)


 ### Simulate a "Shared" Read-Only View (CE Equivalent)


In [ ]:
# In CE, the closest we can get to sharing is creating a view that
# represents what an external partner would be allowed to see.

# This view represents the data as the recipient would query it:
spark.sql("""
    CREATE OR REPLACE VIEW sales_db.partner_view AS
    SELECT
        product,
        total_revenue,
        order_count,
        CONCAT('$', FORMAT_NUMBER(avg_order_value, 2)) AS avg_order_value,
        first_order,
        last_order
    FROM sales_db.shared_sales_summary
    ORDER BY total_revenue DESC
    LIMIT 10
""")

print("SIMULATED PARTNER VIEW — What a Delta Share recipient would query:")
print("(In real Delta Sharing, they'd query the actual LIVE table, not a static view)\n")
display(spark.sql("SELECT * FROM sales_db.partner_view"))


 #### Live Tables vs Snapshots

 | Feature | Delta Sharing (Live) | Traditional Data Export (Snapshot) |
 |---------|---------------------|-----------------------------------|
 | **Data freshness** | Real-time (live table) | Stale (point-in-time copy) |
 | **Data volume** | Recipient pays only for queries | Full copy transferred |
 | **Storage duplication** | None — reads from provider | Duplicate storage needed |
 | **Schema evolution** | Automatically visible | Manual re-export required |
 | **Revocation** | Immediate — revoke share access | Cannot retract sent files |
 | **Auditability** | Provider sees all recipient queries | No visibility after export |
 | **Security** | Token-based, encrypted in transit | Files could be forwarded |
 | **Governance** | UC policies enforced at read time | No enforcement after copy |

 **Winner for regulatory data sharing:** Delta Sharing — revocable, auditable, always up-to-date.


 ### Delta Sharing Architecture Diagram

 ```
 ┌────────────────── PROVIDER ──────────────────┐
 │                                               │
 │  Unity Catalog                                │
 │  ┌──────────────────────────────────────┐    │
 │  │ SHARE: sales_insights_share            │    │
 │  │  ├── TABLE: shared_sales_summary       │    │
 │  │  └── TABLE: revenue_monthly (>=2025)  │    │
 │  └──────────────────────────────────────┘    │
 │                                               │
 │  Recipients:                                  │
 │   ├── acme_corp (CREATE CATALOG ... USING)    │
 │   └── globex_inc                              │
 └───────────────────┬───────────────────────────┘
                     │
               Delta Sharing Protocol
               (REST API / gRPC)
                     │
 ┌───────────────────┴───────────────────────────┐
 │ RECIPIENT: acme_corp                           │
 │                                                 │
 │  Databricks Catalog: sales_from_partner         │
 │   ├── shared_sales_summary ← LIVE QUERY        │
 │   └── revenue_monthly      ← FILTERED (>=2025) │
 │                                                 │
 │  OR: Open-Source Client (pandas, Spark)         │
 └─────────────────────────────────────────────────┘
 ```


 ### What You'd Do in Production

 1. **Identify shareable data** — clean, aggregated datasets (never raw PII, never unprocessed logs).
 2. **Create shares by audience** — one share per partner, or one share per data product.
 3. **Use partition restrictions** — share only agreed-upon date ranges or regions.
 4. **Monitor recipient usage** — track query patterns via `system.access.audit`.
 5. **Set up expiration** — recipient tokens have expiry; rotate regularly.
 6. **Document the data contract** — what columns mean, refresh frequency, SLAs.
 7. **Test with open-source Delta Sharing** — verify the share works from a non-Databricks client before handing to partners.
 8. **Revoke promptly** — if a partnership ends, `DROP RECIPIENT` / `REVOKE SELECT ON SHARE` immediately.

 | Step | Command |
 |------|---------|
 | Create share | `CREATE SHARE sales_insights_share;` |
 | Add table | `ALTER SHARE ... ADD TABLE catalog.schema.table;` |
 | Create recipient | `CREATE RECIPIENT partner USING ID 'partner@domain.com';` |
 | Grant select | `GRANT SELECT ON SHARE sales_insights_share TO RECIPIENT partner;` |
 | Recipient mounts | `CREATE CATALOG shared_data USING SHARE partner.sales_insights_share;` |
 | Revoke | `REVOKE SELECT ON SHARE sales_insights_share FROM RECIPIENT partner;` |
 | Remove table | `ALTER SHARE sales_insights_share REMOVE TABLE catalog.schema.table;` |
 | Drop share | `DROP SHARE sales_insights_share;` |


 ## Summary — Unity Catalog & Governance

 ### Concepts Covered (#61–#70)

 | # | Concept | Key Takeaway |
 |---|---------|-------------|
 | 61 | **Three-Level Namespace** | `catalog.schema.table` organizes data; UC maps to org structure |
 | 62 | **Permission Model** | Privileges cascade catalog→schema→table; use groups, not users |
 | 63 | **Dynamic Views** | `IS_MEMBER()` and `CURRENT_USER()` embed security logic in views |
 | 64 | **Data Lineage** | UC auto-tracks column-level lineage; critical for impact analysis |
 | 65 | **Information Schema** | `system.information_schema.*` provides ANSI-standard metadata queries |
 | 66 | **External Locations** | Governs cloud storage paths through UC; managed vs. external tables |
 | 67 | **Service Principals** | Machine identities for production; NEVER use personal accounts for jobs |
 | 68 | **Audit Logging** | `system.access.audit` records every operation for compliance |
 | 69 | **Row Filters & Column Masks** | Table-level security policies; non-circumventable |
 | 70 | **Delta Sharing** | Open protocol for live, cross-org data sharing without data copy |


 ### Community Edition vs Production Unity Catalog — Comparison Matrix

 | Feature | Community Edition (Hive Metastore) | Production (Unity Catalog) |
 |---------|----------------------------------|----------------------------|
 | **Namespace** | `database.table` (2-level) | `catalog.schema.table` (3-level) |
 | **Permissions** | Limited Hive GRANT/REVOKE | Full RBAC with inheritance |
 | **Data lineage** | Manual tracking only | Automatic column-level lineage |
 | **Metadata** | `SHOW`/`DESCRIBE` commands | `system.information_schema.*` (ANSI SQL) |
 | **External locations** | Direct `LOCATION '/path/'` | UC-governed storage credentials + locations |
 | **Service principals** | Not available | Full support via Account API |
 | **Audit logging** | Must build manually | `system.access.audit` (automatic) |
 | **Row-level security** | Views only | Row filters + column masks on tables |
 | **Cross-org sharing** | Not available | Delta Sharing (open protocol) |
 | **Cost** | Free (limited resources) | Paid DBU + cloud costs |


 ### Self-Assessment — Test Your Understanding

 Answer these questions to validate your learning:


In [ ]:
questions = r"""
GOVERNANCE SELF-ASSESSMENT
══════════════════════════════════════════════════════════

Q1: What is the three-level namespace in Unity Catalog?
    A) database.schema.table
    B) catalog.schema.table          ← CORRECT
    C) workspace.catalog.table
    D) project.dataset.table

Q2: What function checks if the current user belongs to a UC group?
    A) current_role()
    B) IS_MEMBER('group_name')       ← CORRECT
    C) user_in_group()
    D) has_permission()

Q3: In Unity Catalog, what happens when you DROP a MANAGED table?
    A) Only schema is deleted
    B) Schema AND data are deleted   ← CORRECT
    C) Table is renamed
    D) Table is shared

Q4: What is the key advantage of row filters over dynamic views?
    A) Better performance
    B) Cannot be bypassed (table-level) ← CORRECT
    C) Easier to write
    D) Support more SQL functions

Q5: What does Delta Sharing use for data transfer?
    A) SFTP
    B) Email attachments
    C) Open REST/gRPC protocol       ← CORRECT
    D) VPN tunnel

Q6: Why should production jobs use service principals?
    A) They are faster
    B) They survive employee departure   ← CORRECT
    C) They are free
    D) Required by Databricks

Q7: What system table provides audit logs in Unity Catalog?
    A) system.audit.logs
    B) system.access.audit           ← CORRECT
    C) system.security.events
    D) system.governance.trail

Q8: What is the difference between USE CATALOG and USE SCHEMA?
    A) No difference
    B) USE CATALOG sets the top-level container; USE SCHEMA sets the 
       namespace within it.                                      ← CORRECT
    C) USE CATALOG is for tables, USE SCHEMA is for views
    D) USE CATALOG requires admin privileges

Q9: What is the primary purpose of an external location in UC?
    A) Speed up queries
    B) Govern access to cloud storage paths   ← CORRECT
    C) Create temporary tables
    D) Share data with external partners

Q10: Can you apply BOTH a row filter and a column mask to the same table?
    A) No, tables can have only one security policy
    B) Yes, they coexist on the same table     ← CORRECT
    C) Only in Premium tier
    D) Only on external tables
"""

print(questions)


 ### Key Governance Principles — Cheat Sheet

 ```
 1. LEAST PRIVILEGE
    Grant minimum access needed. Start from zero. Expand only with justification.

 2. GROUPS, NOT USERS
    Never grant privileges to individuals. Use account groups. Manage membership centrally.

 3. MACHINE IDENTITIES
    Production jobs → Service principals. Interactive work → User accounts. Never mix.

 4. DEFENSE IN DEPTH
    Dynamic views for UX, row filters + column masks for enforcement. Layer them.

 5. AUDIT EVERYTHING
    If it's not logged, it didn't happen. Use system.access.audit for compliance.

 6. CATALOG TOPOLOGY MATTERS
    Design catalog/schema hierarchy to match your org. Plan before creating tables.

 7. MANAGED vs EXTERNAL
    Managed tables: UC handles lifecycle. External tables: you keep the files.

 8. SHARE WITHOUT COPYING
    Delta Sharing for live, revocable data sharing. No more CSV exports.

 9. METADATA IS POWER
    information_schema tells you everything about your data estate. Query it regularly.

 10. LINEAGE IS AUTOMATIC
     UC traces column-level lineage without instrumentation. Use it for impact analysis.
 ```


 ### Cleanup — Drop Objects Created in This Notebook


In [ ]:
print("Cleaning up objects created in this notebook...\n")

# Drop tables created for demonstrations
tables_to_drop = [
    "sales_db.revenue",
    "sales_db.pipeline",
    "sales_db.revenue_raw",
    "sales_db.revenue_clean",
    "sales_db.revenue_monthly",
    "sales_db.shared_sales_summary",
    "sales_db.data_lineage",
    "sales_db.audit_log",
    "sales_db.orders_managed",
    "sales_db.orders_external",
    "sales_db.audit_test_table",
    "hr_db.employees",
    "hr_db.payroll",
    "hr_db.employee_sensitive",
    "hr_db.employee_complete",
    "hr_db.access_control",
    "hr_db.role_permissions",
    "marketing_db.campaigns",
]

for tbl in tables_to_drop:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {tbl}")
        print(f"  DROPPED: {tbl}")
    except Exception as e:
        print(f"  SKIPPED: {tbl} — {str(e)[:60]}")

# Drop views
views_to_drop = [
    "sales_db.executive_dashboard",
    "sales_db.partner_view",
    "hr_db.employee_view",
    "hr_db.employee_by_dept",
    "hr_db.employee_row_filtered",
    "hr_db.employee_column_masked",
    "hr_db.employee_secure",
]

for view in views_to_drop:
    try:
        spark.sql(f"DROP VIEW IF EXISTS {view}")
        print(f"  DROPPED VIEW: {view}")
    except Exception as e:
        print(f"  SKIPPED: {view} — {str(e)[:60]}")

# Drop databases (only if empty after table drops)
for db in ["sales_db", "hr_db", "marketing_db"]:
    try:
        spark.sql(f"DROP DATABASE IF EXISTS {db} CASCADE")
        print(f"  DROPPED DATABASE: {db}")
    except Exception as e:
        print(f"  SKIPPED DATABASE: {db} — {str(e)[:60]}")

print("\nCleanup complete.")


 ## End of Notebook — 07 Unity Catalog & Governance

 **What you learned:**
 - The three-level catalog namespace and how it maps to organizational structure
 - Unity Catalog's privilege model with cascade inheritance
 - Building dynamic views for row-level and column-level security
 - Automatic data lineage and manual lineage tracking techniques
 - Querying information schema for metadata discovery and compliance
 - External locations and the managed vs external table distinction
 - Service principals as non-human identities for production workloads
 - Audit logging for compliance and forensic investigation
 - Row filters and column masks as table-level security enforcement
 - Delta Sharing for live, cross-organization data sharing

 **Next:** Move to the capstone project — apply everything you've learned in a comprehensive end-to-end pipeline!


---

# 08 Lakeflow Declarative Pipelines



 # 08 — Lakeflow Declarative Pipelines (Concepts #71–#80)

 ## Overview

 Lakeflow Declarative Pipelines (formerly **Delta Live Tables / DLT**) is Databricks' declarative ETL
 framework for building reliable, maintainable, and testable data pipelines. It handles
 orchestration, error handling, quality enforcement, and state management automatically.

 **IMPORTANT**: Lakeflow / DLT pipelines are **NOT available in Databricks Community Edition**.
 This notebook therefore uses a dual approach throughout:

 - **"With Lakeflow"** blocks (commented — show the declarative syntax)
 - **"Manual Equivalent (Community Edition)"** blocks (executable — replicate the same logic)

 ### Concepts Covered
 | # | Concept | Difficulty |
 |---|---------|------------|
 | 71 | Python vs. SQL Pipeline Syntax | Easy |
 | 72 | Streaming Tables vs. Materialized Views | Medium |
 | 73 | Expectations (Data Quality Rules) | Medium |
 | 74 | Pipeline Development & Testing | Medium |
 | 75 | Pipeline Monitoring & Event Log | Medium |
 | 76 | Pipeline Modes: Triggered vs. Continuous | Medium |
 | 77 | CDC Processing: AUTO CDC INTO | Hard |
 | 78 | Error Handling & Dead Letter Patterns | Hard |
 | 79 | Pipeline Parameters & Configuration | Hard |
 | 80 | Multi-Pipeline Architecture | Hard |

 ### Dataset
 We generate **synthetic retail sales data** throughout — orders, customers, products, and CDC
 change events — stored in Delta Lake tables for realistic pipeline scenarios.


 ## Setup — Environment & Synthetic Data

 We create the three core datasets used across all ten concepts:
 - **raw_orders** — streaming source of retail orders
 - **customers** — customer dimension table
 - **products** — product dimension table


In [ ]:
import os
import time
import json
import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.streaming import DataStreamWriter
from delta.tables import DeltaTable

spark = SparkSession.builder \
    .appName("Lakeflow_Concept_71_80") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true") \
    .getOrCreate()

spark.sql("SET spark.databricks.delta.schema.autoMerge.enabled = true")

BASE_PATH = "/tmp/lakeflow_demo"
os.makedirs(BASE_PATH, exist_ok=True)
print(f"Working directory: {BASE_PATH}")
print(f"Spark version: {spark.version}")


 ### Generate Synthetic Retail Data


In [ ]:
random.seed(42)

PRODUCT_IDS = [f"P{str(i).zfill(4)}" for i in range(1, 21)]
CATEGORIES  = ["Electronics", "Clothing", "Home", "Books", "Sports"]
CUSTOMER_IDS = [f"C{str(i).zfill(5)}" for i in range(1, 101)]
STATUSES    = ["pending", "shipped", "delivered", "cancelled"]
REGIONS     = ["US-West", "US-East", "EU-Central", "APAC-South"]

# --- Products Dimension ---
products_data = [(
    pid,
    f"Product-{pid}",
    random.choice(CATEGORIES),
    round(random.uniform(5.0, 500.0), 2),
    random.choice([True, False])
) for pid in PRODUCT_IDS]

products_df = spark.createDataFrame(products_data, schema=["product_id", "name", "category", "price", "active"])
products_df.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/products")
print(f"[OK] Products: {products_df.count()} rows → {BASE_PATH}/products")

# --- Customers Dimension ---
customers_data = [(
    cid,
    f"customer_{cid}@example.com",
    random.choice(["Basic", "Premium", "Enterprise"]),
    random.choice(REGIONS),
    (datetime.now() - timedelta(days=random.randint(1, 1095))).strftime("%Y-%m-%d")
) for cid in CUSTOMER_IDS]

customers_df = spark.createDataFrame(customers_data, schema=["customer_id", "email", "tier", "region", "signup_date"])
customers_df.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/customers")
print(f"[OK] Customers: {customers_df.count()} rows → {BASE_PATH}/customers")

# --- Raw Orders (append-friendly source) ---
def generate_orders(num_rows=200):
    ts = datetime.now() - timedelta(seconds=random.randint(0, 86400))
    return [(
        f"ORD-{ts.strftime('%Y%m%d%H%M%S')}-{''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789', k=6))}",
        random.choice(CUSTOMER_IDS),
        random.choice(PRODUCT_IDS),
        random.randint(1, 10),
        round(random.uniform(5.0, 500.0), 2),
        random.choice(STATUSES),
        random.choice(REGIONS),
        random.randint(0, 50),
        ts.strftime("%Y-%m-%d %H:%M:%S"),
        None
    ) for _ in range(num_rows)]

order_schema = T.StructType([
    T.StructField("order_id",    T.StringType()),
    T.StructField("customer_id", T.StringType()),
    T.StructField("product_id",  T.StringType()),
    T.StructField("quantity",    T.IntegerType()),
    T.StructField("amount",      T.DoubleType()),
    T.StructField("status",      T.StringType()),
    T.StructField("region",      T.StringType()),
    T.StructField("discount_pct",T.IntegerType()),
    T.StructField("order_ts",    T.StringType()),
    T.StructField("cancellation_reason", T.StringType()),
])

batch1 = generate_orders(100)
batch2 = generate_orders(100)

orders_df = spark.createDataFrame(batch1 + batch2, schema=order_schema)
orders_df.write.format("delta").mode("append").save(f"{BASE_PATH}/raw_orders")
print(f"[OK] Raw Orders: {orders_df.count()} rows → {BASE_PATH}/raw_orders")

# --- Refresh all tables ---
spark.sql(f"CREATE OR REPLACE TEMPORARY VIEW products  AS SELECT * FROM delta.`{BASE_PATH}/products`")
spark.sql(f"CREATE OR REPLACE TEMPORARY VIEW customers AS SELECT * FROM delta.`{BASE_PATH}/customers`")
spark.sql(f"CREATE OR REPLACE TEMPORARY VIEW raw_orders AS SELECT * FROM delta.`{BASE_PATH}/raw_orders`")
print("\nAll datasets ready.")


 ---
 ## Concept 71 — Python vs. SQL Pipeline Syntax

 **Difficulty: Easy**

 Lakeflow supports two equivalent syntaxes for defining pipelines. The choice is about **team
 preference**, not capability — both produce identical execution plans.

 | Aspect | SQL Syntax | Python Syntax |
 |--------|------------|----------------|
 | Streaming table | `CREATE STREAMING TABLE` | `@table` decorator |
 | Materialized view | `CREATE MATERIALIZED VIEW` | `@materialized_view` decorator |
 | Temporary view | `CREATE TEMPORARY LIVE VIEW` | `@temporary_view` decorator |
 | Expectations | Inline constraints | `expect()` / `expect_or_drop()` / `expect_or_fail()` |
 | Legacy import | — | `import dlt` (still works) |

 ### Rule of Thumb
 - **SQL preferred** when: analysts write pipelines, SQL logic is straightforward
 - **Python preferred** when: complex transformations, ML feature engineering, testing with pytest
 - Both can be **mixed** in the same pipeline notebook


 ### Python Syntax (Lakeflow / DLT) — Commented Reference


In [ ]:
# =============================================================================
#      LAKEFLOW / DLT — PYTHON DECORATOR SYNTAX (not executable in CE)
# =============================================================================
# import dlt                          # Legacy import (still works)
# from pyspark.pipelines import table, materialized_view, temporary_view
#
# @table(
#     name="bronze_orders",
#     comment="Raw orders ingested into bronze layer",
#     table_properties={"pipelines.autoOptimize.managed": "true"}
# )
# def bronze_orders():
#     return spark.readStream.table("raw_orders")
#
# @table(name="silver_orders")
# def silver_orders():
#     return dlt.read("bronze_orders") \
#         .withColumn("processed_at", F.current_timestamp())
#
# @materialized_view(name="gold_daily_sales", schedule_cron="0 0 6 * * ?")
# def gold_daily_sales():
#     return dlt.read("silver_orders") \
#         .groupBy(F.window("processed_at", "1 day"), "region") \
#         .agg(F.sum("amount").alias("total_sales"))
#
# @temporary_view(name="tmp_filtered")
# def tmp_filtered():
#     return dlt.read("silver_orders").filter(F.col("amount") > 0)

# =============================================================================
#      MANUAL EQUIVALENT — COMMUNITY EDITION
# =============================================================================

bronze_orders = (
    spark.readStream
         .format("delta")
         .load(f"{BASE_PATH}/raw_orders")
         .withColumn("ingested_at", F.current_timestamp())
)

bronze_query = (
    bronze_orders.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{BASE_PATH}/_checkpoint/bronze")
    .trigger(processingTime="10 seconds")
    .start(f"{BASE_PATH}/bronze_orders")
)
print("Bronze streaming write started...")

time.sleep(8)

silver_orders = spark.read.format("delta").load(f"{BASE_PATH}/bronze_orders") \
    .withColumn("processed_at", F.current_timestamp())

silver_orders.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/silver_orders")
print(f"Silver orders created: {silver_orders.count()} rows")

gold_daily_sales = (
    spark.read.format("delta").load(f"{BASE_PATH}/silver_orders")
    .withColumn("order_date", F.to_date(F.col("order_ts")))
    .groupBy("order_date", "region")
    .agg(F.sum("amount").alias("total_sales"), F.count("*").alias("order_count"))
)

gold_daily_sales.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/gold_daily_sales")
print(f"Gold daily sales created: {gold_daily_sales.count()} rows")

# Stop streaming to keep things clean
bronze_query.stop()
print("Bronze stream stopped.")


 ### SQL Syntax (Lakeflow / DLT) — Commented Reference


In [ ]:
# =============================================================================
#      LAKEFLOW / DLT — SQL SYNTAX (not executable in CE)
# =============================================================================
# -- Bronze: streaming ingestion
# CREATE OR REFRESH STREAMING TABLE bronze_orders_sql
# COMMENT "Raw orders ingested via SQL syntax"
# TBLPROPERTIES ("pipelines.autoOptimize.managed" = "true")
# AS SELECT
#     order_id,
#     customer_id,
#     product_id,
#     CAST(order_ts AS TIMESTAMP) AS order_ts,
#     quantity,
#     amount,
#     status,
#     region,
#     discount_pct,
#     current_timestamp() AS ingested_at
# FROM STREAM(LIVE.raw_orders);
#
# -- Silver: cleaned & deduplicated
# CREATE OR REFRESH STREAMING TABLE silver_orders_sql AS
# SELECT DISTINCT *
# FROM STREAM(LIVE.bronze_orders_sql)
# WHERE amount > 0;
#
# -- Gold: materialized aggregation
# CREATE OR REFRESH MATERIALIZED VIEW gold_daily_sales_sql AS
# SELECT
#     DATE(order_ts) AS order_date,
#     region,
#     SUM(amount) AS total_sales,
#     COUNT(*)    AS order_count
# FROM LIVE.silver_orders_sql
# GROUP BY DATE(order_ts), region;
#
# -- Temporary view
# CREATE OR REFRESH TEMPORARY LIVE VIEW tmp_high_value AS
# SELECT * FROM LIVE.silver_orders_sql WHERE amount > 200;

# =============================================================================
#      MANUAL EQUIVALENT — COMMUNITY EDITION (SQL-centric)
# =============================================================================

spark.sql(f"CREATE OR REPLACE TEMPORARY VIEW orders_source AS SELECT * FROM delta.`{BASE_PATH}/raw_orders`")

spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW bronze_orders_sql AS
    SELECT
        order_id, customer_id, product_id,
        CAST(order_ts AS TIMESTAMP) AS order_ts,
        quantity, amount, status, region, discount_pct,
        current_timestamp() AS ingested_at
    FROM orders_source
""")

spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW silver_orders_sql AS
    SELECT DISTINCT *
    FROM bronze_orders_sql
    WHERE amount > 0
""")

spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW gold_daily_sales_sql AS
    SELECT
        DATE(order_ts) AS order_date,
        region,
        SUM(amount) AS total_sales,
        COUNT(*)    AS order_count
    FROM silver_orders_sql
    GROUP BY DATE(order_ts), region
""")

print("SQL pipeline views created successfully.")
spark.sql("SELECT * FROM gold_daily_sales_sql ORDER BY order_date DESC").show(5)


 ### Side-by-Side Comparison


 ```
 ╔══════════════════════════════════════════════════════════════════════╗
 ║                      LAKEFLOW / DLT vs. MANUAL                       ║
 ╠═══════════════════════╦══════════════════════════════════════════════╣
 ║ With Lakeflow (DLT)   ║ Without Lakeflow (CE - Manual)               ║
 ╠═══════════════════════╬══════════════════════════════════════════════╣
 ║ @table decorator      ║ Manual readStream / writeStream orchestration║
 ║ Automatic lineage     ║ Must track lineage yourself                  ║
 ║ Built-in quality      ║ Must implement quality checks manually       ║
 ║ Auto-handles state    ║ Must manage checkpoints, schema evolution    ║
 ║ Declarative - "what"  ║ Imperative - "how"                           ║
 ║ Pipeline UI dashboard ║ No built-in pipeline monitoring              ║
 ║ Incremental refresh   ║ Must implement incremental logic manually    ║
 ║ SQL + Python unified  ║ Separate code paths for SQL vs PySpark       ║
 ╚═══════════════════════╩══════════════════════════════════════════════╝
 ```


 ---
 ## Concept 72 — Streaming Tables vs. Materialized Views

 **Difficulty: Medium**

 ### Core Difference

 | | Streaming Table | Materialized View |
 |---|---|---|
 | **Write mode** | Append-only | Full recompute (overwrite) |
 | **Use case** | Ingestion, CDC, event streams | Aggregations, joins, derived datasets |
 | **Exactly-once** | Yes (Delta guarantees) | Yes (idempotent recompute) |
 | **Refresh** | Incremental, near real-time | On schedule or pipeline trigger |
 | **State management** | Auto-managed checkpoints | Full result recalc per refresh |
 | **Schema evolution** | Auto-handled | Handled on each recompute |

 In Lakeflow, you never write `writeStream` / `readStream` — the framework decides
 whether incremental or full recompute is appropriate based on the declaration.


 ### Manual Streaming Table (Community Edition)


In [ ]:
# =============================================================================
#      MANUAL STREAMING TABLE — Append-only ingestion to Delta
# =============================================================================

checkpoint_stream = f"{BASE_PATH}/_checkpoint/manual_streaming_table"

streaming_input = (
    spark.readStream
    .format("delta")
    .option("maxFilesPerTrigger", 2)
    .load(f"{BASE_PATH}/raw_orders")
    .withColumn("streamed_at", F.current_timestamp())
)

stream_writer = (
    streaming_input.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_stream)
    .trigger(processingTime="5 seconds")
    .queryName("manual_streaming_table")
    .start(f"{BASE_PATH}/manual_streaming_table")
)

print("Manual streaming table started — append-only, exactly-once via Delta log")
time.sleep(10)
stream_writer.stop()
print("Streaming table stopped.")

# Show what was ingested
spark.read.format("delta").load(f"{BASE_PATH}/manual_streaming_table") \
    .select("order_id", "amount", "streamed_at") \
    .orderBy(F.desc("streamed_at")).show(5, False)


 ### Manual Materialized View (Community Edition)


In [ ]:
# =============================================================================
#      MANUAL MATERIALIZED VIEW — Full recomputation of aggregations
# =============================================================================

def refresh_materialized_view():
    """Simulates Lakeflow's materialized view refresh — full recompute."""
    source = spark.read.format("delta").load(f"{BASE_PATH}/raw_orders") \
        .withColumn("order_date", F.to_date("order_ts"))

    mv_customer_metrics = (
        source.groupBy("customer_id")
        .agg(
            F.count("*").alias("total_orders"),
            F.sum("amount").alias("lifetime_value"),
            F.avg("amount").alias("avg_order_value"),
            F.max("order_ts").alias("last_order"),
            F.collect_set("region").alias("regions_ordered_from")
        )
    )

    mv_customer_metrics.write.format("delta").mode("overwrite") \
        .save(f"{BASE_PATH}/mv_customer_metrics")
    return mv_customer_metrics

mv1 = refresh_materialized_view()
print(f"Materialized view created: {mv1.count()} rows")
mv1.show(5, False)

# Add more data, then re-refresh (simulating Lakeflow's incremental pipeline run)
more_orders = spark.createDataFrame(generate_orders(30), schema=order_schema)
more_orders.write.format("delta").mode("append").save(f"{BASE_PATH}/raw_orders")

mv2 = refresh_materialized_view()
print(f"\nAfter new data — re-refreshed: {mv2.count()} rows")
mv2.show(5, False)


 ### Comparison Summary


 ```
 ╔══════════════════════════════════════════════════════════════════════════════╗
 ║ STREAMING TABLE (Lakeflow)                                                  ║
 ╠══════════════════════════════════════════════════════════════════════════════╣
 ║ @table(name="bronze_orders")                                                ║
 ║ def bronze_orders():                                                        ║
 ║     return spark.readStream.table("raw_orders")                             ║
 ║                      ↓ DECLARATIVE ↓                                        ║
 ║ Lakeflow handles: checkpoints, retries, schema evolution, monitoring.       ║
 ╠══════════════════════════════════════════════════════════════════════════════╣
 ║ VS. MANUAL (Community Edition) — 30+ lines of boilerplate                    ║
 ║ .readStream → .writeStream → checkpoint dir → trigger config → monitor      ║
 ╚══════════════════════════════════════════════════════════════════════════════╝
 ║ MATERIALIZED VIEW (Lakeflow)                                                ║
 ╠══════════════════════════════════════════════════════════════════════════════╣
 ║ @materialized_view(name="gold_daily", schedule_cron="0 0 * * *")            ║
 ║ def gold_daily():                                                           ║
 ║     return dlt.read("silver").groupBy(...).agg(...)                         ║
 ║                      ↓ DECLARATIVE ↓                                        ║
 ║ Lakeflow handles: recompute scheduling, overwrite, backfill, dependencies.  ║
 ╠══════════════════════════════════════════════════════════════════════════════╣
 ║ VS. MANUAL (Community Edition)                                              ║
 ║ Manual scheduled function + .mode("overwrite") + cron scheduler + monitoring║
 ╚══════════════════════════════════════════════════════════════════════════════╝
 ```


 ---
 ## Concept 73 — Expectations (Data Quality Rules)

 **Difficulty: Medium**

 Lakeflow expectations are **inline data quality constraints** with three violation actions:

 | Action | Lakeflow Syntax | Behavior |
 |--------|----------------|----------|
 | **Warn** | `expect("desc", "condition")` | Log metric, keep row |
 | **Drop** | `expect_or_drop("desc", "condition")` | Log metric, discard row |
 | **Fail** | `expect_or_fail("desc", "condition")` | Log metric, stop pipeline |

 Metrics are tracked in the event log: pass %, fail %, row counts per expectation.


 ### Lakeflow Expectations (Commented Reference)


In [ ]:
# =============================================================================
#      LAKEFLOW / DLT EXPECTATIONS (not executable in CE)
# =============================================================================
# import dlt
#
# @table(name="silver_orders_with_quality")
# @dlt.expect("valid_order_id",    "order_id IS NOT NULL AND length(order_id) > 10")
# @dlt.expect("positive_amount",   "amount > 0")
# @dlt.expect_or_drop("valid_status_simple", "status IN ('pending','shipped','delivered','cancelled')")
# @dlt.expect_or_fail("has_customer", "customer_id IS NOT NULL")
# def silver_orders_with_quality():
#     return dlt.read("bronze_orders") \
#         .withColumn("processed_at", F.current_timestamp())
#
# # SQL equivalent:
# # CREATE OR REFRESH STREAMING TABLE silver_sql
# # CONSTRAINT valid_amount EXPECT (amount > 0) ON VIOLATION DROP ROW,
# # CONSTRAINT has_customer EXPECT (customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE
# # AS SELECT * FROM STREAM(LIVE.bronze_orders);


 ### Manual Expectation Framework (Community Edition)


In [ ]:
class ExpectationFramework:
    """Replicates Lakeflow expectation behavior manually."""

    def __init__(self, name="default"):
        self.name = name
        self.metrics = {}
        self.dropped_rows = {"total": 0}

    def expect(self, df, description, condition, action="warn"):
        """Apply an expectation: warn, drop, or fail."""
        passed = df.filter(condition)
        failed = df.filter(~condition)

        pass_count = passed.count()
        fail_count = failed.count()
        total = pass_count + fail_count
        pass_pct = round(pass_count / total * 100, 2) if total > 0 else 100.0

        metric_key = f"{description} ({action})"
        self.metrics[metric_key] = {
            "passed": pass_count, "failed": fail_count,
            "pass_pct": pass_pct, "action": action
        }

        if action == "fail" and fail_count > 0:
            raise ValueError(
                f"❌ Expectation FAILED: '{description}' — {fail_count} violations. Pipeline stopped."
            )

        if action == "drop":
            self.dropped_rows.setdefault(description, 0)
            self.dropped_rows[description] += fail_count
            self.dropped_rows["total"] += fail_count
            print(f"  ⚠️  [DROP] '{description}': {fail_count}/{total} rows dropped")
            return passed

        if action == "warn":
            print(f"  ⚠️  [WARN] '{description}': {fail_count}/{total} rows ({pass_pct}% passed)")
            return df

        return df

    def expect_not_null(self, df, col, action="warn"):
        return self.expect(df, f"{col} IS NOT NULL", F.col(col).isNotNull(), action)

    def expect_unique(self, df, col, action="warn"):
        """Check if column values are unique."""
        total = df.count()
        distinct = df.select(col).distinct().count()
        violated = total - distinct
        condition = F.lit(True)  # We calculate separately above

        metric_key = f"{col} IS UNIQUE ({action})"
        self.metrics[metric_key] = {
            "passed": distinct, "failed": violated,
            "pass_pct": round(distinct / total * 100, 2) if total > 0 else 100.0,
            "action": action
        }

        if action == "fail" and violated > 0:
            raise ValueError(f"❌ Expectation FAILED: '{col} IS UNIQUE' — {violated} duplicates.")
        if action == "drop":
            print(f"  ⚠️  [DROP] '{col} IS UNIQUE': {violated} duplicate rows dropped")
            return df.dropDuplicates([col])

        print(f"  ⚠️  [WARN] '{col} IS UNIQUE': {violated}/{total} duplicates")
        return df

    def expect_range(self, df, col, min_val, max_val, action="warn"):
        return self.expect(df, f"{col} BETWEEN {min_val} AND {max_val}",
                          (F.col(col) >= min_val) & (F.col(col) <= max_val), action)

    def report(self):
        """Print quality report à la Lakeflow event log."""
        print(f"\n{'='*70}")
        print(f"  QUALITY REPORT — Pipeline: {self.name}")
        print(f"{'='*70}")
        for metric, stats in self.metrics.items():
            icon = "✅" if stats["pass_pct"] == 100.0 else "⚠️"
            print(f"  {icon} {metric}: {stats['passed']}/{stats['passed']+stats['failed']} "
                  f"({stats['pass_pct']}%) — action={stats['action']}")
        print(f"  🗑️  Total dropped rows: {self.dropped_rows['total']}")
        print(f"{'='*70}")


# --- DEMO: Apply expectations ---
raw_df = spark.read.format("delta").load(f"{BASE_PATH}/raw_orders")
print(f"Raw data: {raw_df.count()} rows\n")

ef = ExpectationFramework(name="silver_quality_demo")

clean_df = ef.expect_not_null(raw_df, "order_id", action="fail")
clean_df = ef.expect_not_null(clean_df, "customer_id", action="fail")
clean_df = ef.expect(clean_df, "positive_amount", F.col("amount") > 0, action="drop")
clean_df = ef.expect(clean_df, "valid_status",
                     F.col("status").isin("pending", "shipped", "delivered", "cancelled"), action="drop")
clean_df = ef.expect_range(clean_df, "discount_pct", 0, 100, action="warn")
clean_df = ef.expect(clean_df, "amount_not_outlier",
                     (F.col("amount") > 0) & (F.col("amount") < 1500), action="warn")

clean_df.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/silver_orders_quality")
print(f"\nCleaned data: {clean_df.count()} rows → {BASE_PATH}/silver_orders_quality")

ef.report()


 ### Quality Monitoring Table


In [ ]:
# Build a persistent quality monitoring table (simulates Lakeflow event log)
quality_records = []
for metric, stats in ef.metrics.items():
    quality_records.append((
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "silver_quality_demo",
        metric,
        stats["passed"],
        stats["failed"],
        stats["pass_pct"],
        stats["action"]
    ))

quality_schema = T.StructType([
    T.StructField("run_ts",        T.StringType()),
    T.StructField("pipeline_name", T.StringType()),
    T.StructField("expectation",   T.StringType()),
    T.StructField("passed",        T.IntegerType()),
    T.StructField("failed",        T.IntegerType()),
    T.StructField("pass_pct",      T.DoubleType()),
    T.StructField("action",        T.StringType()),
])

quality_df = spark.createDataFrame(quality_records, schema=quality_schema)
quality_df.write.format("delta").mode("append").save(f"{BASE_PATH}/pipeline_quality_log")

print("Quality monitoring table:")
spark.read.format("delta").load(f"{BASE_PATH}/pipeline_quality_log").show(truncate=False)


 ---
 ## Concept 74 — Pipeline Development & Testing

 **Difficulty: Medium**

 Lakeflow supports two operational modes:

 | Mode | Behavior | Pipeline Setting |
 |------|----------|-----------------|
 | **Development** | Uses `dev_` prefix, stops on failure, cheaper clusters | `development: true` |
 | **Production** | No prefix, continues on expectation failures | `development: false` (default) |

 **Refresh types:**
 - **Full refresh**: reprocesses all data from scratch
 - **Incremental**: only processes new data since last run

 Testing involves creating representative data fixtures and asserting outputs.


 ### Test Data Fixtures (Community Edition)


In [ ]:
# =============================================================================
#      TEST FIXTURES — Representative data for pipeline testing
# =============================================================================

def create_test_fixtures():
    """Create known test datasets for deterministic pipeline testing."""

    # Clean test data
    orders_test_clean = spark.createDataFrame([
        ("ORD-TEST-001", "C00001", "P0001", 2,  99.99, "shipped",   "US-West",  10, "2024-01-15 10:30:00"),
        ("ORD-TEST-002", "C00002", "P0002", 1, 249.50, "delivered", "US-East",  0,  "2024-01-15 11:00:00"),
        ("ORD-TEST-003", "C00001", "P0003", 5,  50.00, "pending",   "EU-Central", 5, "2024-01-15 12:00:00"),
    ], schema=order_schema)

    # Dirty test data (for expectation/error handling tests)
    orders_test_dirty = spark.createDataFrame([
        (None, "C00001", "P0001", 2,  99.99, "shipped", "US-West",  10, "2024-01-15 10:30:00"),   # null order_id
        ("ORD-TEST-004", None, "P0002", 1, -50.00, "shipped", "US-East", 0, "2024-01-15 11:00:00"),  # null customer, neg amount
        ("ORD-TEST-005", "C00002", "P0003", 0,   0.00, "unknown", "??", -5, "2024-01-15 12:00:00"),  # bad values
        ("ORD-TEST-006", "C00003", "P0004", 5, 250.00, "shipped", "APAC-South", 5, "2024-01-15 13:00:00"), # clean
    ], schema=order_schema)

    return orders_test_clean, orders_test_dirty


clean_test, dirty_test = create_test_fixtures()
print(f"Clean fixture: {clean_test.count()} rows")
print(f"Dirty fixture: {dirty_test.count()} rows")
clean_test.show(truncate=False)


 ### Pipeline Testing Patterns (Community Edition)


In [ ]:
# =============================================================================
#      LOCAL PIPELINE VALIDATION — Test before "deployment"
# =============================================================================

def transform_silver(input_df):
    """The actual transformation logic — testable in isolation."""
    return (
        input_df
        .filter(F.col("order_id").isNotNull())
        .filter(F.col("customer_id").isNotNull())
        .filter(F.col("amount") > 0)
        .filter(F.col("status").isin("pending", "shipped", "delivered", "cancelled"))
        .withColumn("processed_at", F.current_timestamp())
        .withColumn("total_amount", F.col("quantity") * F.col("amount"))
    )


def test_pipeline_logic(verbose=True):
    """Run assertions against known fixtures."""
    results = {"passed": 0, "failed": 0, "details": []}

    # Test 1: Clean data passes through unchanged
    output = transform_silver(clean_test)
    assert output.count() == 3, f"Expected 3 clean rows, got {output.count()}"
    results["passed"] += 1
    results["details"].append("✅ Test 1: Clean data — 3 rows pass through")

    # Test 2: null order_id is filtered out
    dirty_output = transform_silver(dirty_test)
    assert dirty_output.count() == 1, f"Expected 1 row from dirty data, got {dirty_output.count()}"
    results["passed"] += 1
    results["details"].append("✅ Test 2: Dirty data — only 1 row survives filtering")

    # Test 3: No negative amounts
    amounts = [r.amount for r in dirty_output.select("amount").collect()]
    assert all(a > 0 for a in amounts), f"Found non-positive amounts: {amounts}"
    results["passed"] += 1
    results["details"].append("✅ Test 3: All amounts positive after filtering")

    # Test 4: All statuses are valid
    valid_set = {"pending", "shipped", "delivered", "cancelled"}
    statuses = set(r.status for r in dirty_output.select("status").collect())
    assert statuses.issubset(valid_set), f"Invalid status found: {statuses - valid_set}"
    results["passed"] += 1
    results["details"].append(f"✅ Test 4: All statuses valid: {statuses}")

    # Test 5: Schema check — processed_at column exists
    assert "processed_at" in output.columns, "processed_at column missing"
    results["passed"] += 1
    results["details"].append("✅ Test 5: Schema — processed_at column present")

    # Test 6: total_amount computed correctly
    row = output.filter(F.col("order_id") == "ORD-TEST-001").first()
    assert row.total_amount == 199.98, f"Expected 199.98, got {row.total_amount}"
    results["passed"] += 1
    results["details"].append("✅ Test 6: total_amount = quantity * amount computed correctly")

    if verbose:
        print("=" * 60)
        print("  PIPELINE TESTS RESULTS")
        print("=" * 60)
        for detail in results["details"]:
            print(f"  {detail}")
        print(f"\n  {results['passed']} passed, {results['failed']} failed")
        print("=" * 60)

    return results


test_results = test_pipeline_logic()


 ---
 ## Concept 75 — Pipeline Monitoring & Event Log

 **Difficulty: Medium**

 Lakeflow automatically writes an **event log** to a `__events__` table in the storage
# MV location. The event log contains:
 - **Flow events**: table creates, updates, drops
 - **Progress events**: row counts, processing times
 - **Expectation events**: quality metric pass/fail/drop counts
 - **Cluster events**: resource allocation

 Queries against the event log enable monitoring, alerting, and lineage tracking.


 ### Lakeflow Event Log Access (Commented Reference)


In [ ]:
# =============================================================================
#      LAKEFLOW / DLT — EVENT LOG QUERIES (not executable in CE)
# =============================================================================
# # Query the event log directly
# SELECT
#     timestamp,
#     details.flow_definition.output_table_name AS table_name,
#     details.flow_progress.num_output_rows    AS rows_written,
#     details.flow_progress.execution_time_ms  AS execution_ms
# FROM event_log("my_pipeline_storage_mv")
# WHERE event_type = 'flow_progress'
# ORDER BY timestamp DESC;
#
# # Get expectation metrics
# SELECT
#     details.expectation.name       AS expectation,
#     details.expectation.passed_records AS passed,
#     details.expectation.failed_records AS failed,
#     details.expectation.action     AS action
# FROM event_log("my_pipeline_storage_mv")
# WHERE event_type = 'expectation_violation';
#
# # Lineage tracking
# SELECT * FROM event_log("my_pipeline") WHERE event_type = 'flow_definition';


 ### Manual Pipeline Monitoring Framework (Community Edition)


In [ ]:
import uuid

class PipelineRunTracker:
    """Mimics Lakeflow event log for manual pipeline monitoring."""

    def __init__(self, pipeline_name, storage_location=BASE_PATH):
        self.pipeline_name = pipeline_name
        self.run_id = str(uuid.uuid4())[:8]
        self.log_path = f"{storage_location}/_monitoring/pipeline_runs"
        self.events = []
        self.start_time = None
        self.end_time = None

    def start(self):
        self.start_time = datetime.now()
        self._record("pipeline_start", {
            "run_id": self.run_id, "pipeline": self.pipeline_name,
            "start_ts": self.start_time.isoformat()
        })
        print(f"▶️  Pipeline '{self.pipeline_name}' started — run_id={self.run_id}")
        return self

    def log_step(self, step_name, rows_in, rows_out, duration_ms, error=None):
        event = {
            "run_id": self.run_id,
            "step": step_name,
            "rows_input": rows_in,
            "rows_output": rows_out,
            "duration_ms": duration_ms,
            "error": str(error) if error else None,
            "ts": datetime.now().isoformat()
        }
        self.events.append(event)
        status = "❌" if error else "✅"
        print(f"  {status} {step_name}: {rows_in}→{rows_out} rows ({duration_ms}ms)")
        return self

    def log_quality(self, expectation_name, passed, failed, action):
        event = {
            "run_id": self.run_id,
            "event_type": "quality_metric",
            "expectation": expectation_name,
            "passed": passed,
            "failed": failed,
            "action": action,
            "ts": datetime.now().isoformat()
        }
        self.events.append(event)
        return self

    def finish(self, success=True):
        self.end_time = datetime.now()
        duration = (self.end_time - self.start_time).total_seconds()
        self._record("pipeline_finish", {
            "run_id": self.run_id,
            "success": success,
            "duration_sec": duration,
            "steps": len([e for e in self.events if "step" in e.get("event_type", e.get("step", ""))])
        })
        status = "✅ SUCCESS" if success else "❌ FAILED"
        print(f"⏹️  Pipeline '{self.pipeline_name}' {status} — {duration:.1f}s")

        # Persist all events to Delta
        self._persist()
        return self

    def _record(self, event_type, data):
        self.events.append({"event_type": event_type, **data})

    def _persist(self):
        df = spark.createDataFrame(self.events)
        df.write.format("delta").mode("append").save(self.log_path)


# --- DEMO: Track a pipeline run ---
tracker = PipelineRunTracker("daily_retail_etl")
tracker.start()

# Simulate multi-step pipeline
raw_count = spark.read.format("delta").load(f"{BASE_PATH}/raw_orders").count()
tracker.log_step("read_raw", None, raw_count, 120)

t0 = time.time()
silver = transform_silver(raw_df)
silver_count = silver.count()
tracker.log_step("clean_silver", raw_count, silver_count, int((time.time()-t0)*1000))

t0 = time.time()
gold = (
    silver.withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date", "region")
    .agg(F.sum("amount").alias("daily_revenue"))
)
gold_count = gold.count()
tracker.log_step("aggregate_gold", silver_count, gold_count, int((time.time()-t0)*1000))

# Log quality metrics
tracker.log_quality("amount > 0", silver_count, raw_count - silver_count, "drop")
tracker.log_quality("status valid", silver_count, 0, "drop")

tracker.finish(success=True)


 ### Querying Run History


In [ ]:
# =============================================================================
#      QUERY PIPELINE RUN HISTORY
# =============================================================================
monitoring_path = f"{BASE_PATH}/_monitoring/pipeline_runs"

try:
    run_history = spark.read.format("delta").load(monitoring_path)

    print("=== Recent Pipeline Runs ===")
    run_history.filter(F.col("event_type").isin("pipeline_start", "pipeline_finish")) \
        .select("event_type", "pipeline", "run_id", "success", "duration_sec") \
        .orderBy("ts", ascending=False) \
        .show(truncate=False)

    print("\n=== Step-Level Details ===")
    run_history.filter(F.col("step").isNotNull()) \
        .select("run_id", "step", "rows_input", "rows_output", "duration_ms", "error") \
        .show(truncate=False)

except Exception as e:
    print(f"No run history yet: {e}")


 ---
 ## Concept 76 — Pipeline Modes: Triggered vs. Continuous

 **Difficulty: Medium**

 Lakeflow supports two execution modes that map closely to Structured Streaming triggers:

 | Mode | Behavior | Use Case | Trigger |
 |------|----------|----------|---------|
 | **Triggered** | Processes available data, then stops | Scheduled batch ETL | `availableNow` |
 | **Continuous** | Always running, sub-second latency | Real-time dashboards | `processingTime` |

 **Cost/Latency Tradeoff**:
 - Triggered = cheaper (clusters shut down between runs) but higher latency
 - Continuous = lower latency but higher cost (always-on compute)


 ### Lakeflow Mode Configuration (Commented Reference)


In [ ]:
# =============================================================================
#      LAKEFLOW / DLT — PIPELINE MODE (configured in pipeline settings, not code)
# =============================================================================
# # In pipeline configuration JSON:
# {
#   "name": "triggered_daily_etl",
#   "edition": "ADVANCED",
#   "continuous": false,         # ← TRIGGERED MODE
#   "development": false
# }
#
# # vs.
# {
#   "name": "continuous_realtime",
#   "edition": "ADVANCED",
#   "continuous": true,          # ← CONTINUOUS MODE
#   "development": false
# }


 ### Manual — Triggered Mode (availableNow)


In [ ]:
# =============================================================================
#      MANUAL — TRIGGERED PIPELINE (availableNow)
#      Processes all available data, then stops.
# =============================================================================
checkpoint_triggered = f"{BASE_PATH}/_checkpoint/triggered_demo"

triggered_stream = (
    spark.readStream
    .format("delta")
    .option("maxFilesPerTrigger", 5)
    .load(f"{BASE_PATH}/raw_orders")
    .withColumn("trigger_type", F.lit("TRIGGERED"))
    .withColumn("processed_at", F.current_timestamp())
)

triggered_query = (
    triggered_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_triggered)
    .trigger(availableNow=True)           # ← "triggered" mode
    .queryName("triggered_pipeline")
    .start(f"{BASE_PATH}/silver_triggered")
)

triggered_query.awaitTermination()
print(f"Triggered pipeline finished. Status: {triggered_query.status['message']}")
spark.read.format("delta").load(f"{BASE_PATH}/silver_triggered").show(5)


 ### Manual — Continuous Mode (processingTime)


In [ ]:
# =============================================================================
#      MANUAL — CONTINUOUS PIPELINE (processingTime)
#      Always running — processes micro-batches every N seconds.
# =============================================================================
checkpoint_continuous = f"{BASE_PATH}/_checkpoint/continuous_demo"

continuous_stream = (
    spark.readStream
    .format("delta")
    .option("maxFilesPerTrigger", 1)
    .load(f"{BASE_PATH}/raw_orders")
    .withColumn("trigger_type", F.lit("CONTINUOUS"))
    .withColumn("processed_at", F.current_timestamp())
)

continuous_query = (
    continuous_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_continuous)
    .trigger(processingTime="5 seconds")   # ← "continuous" mode
    .queryName("continuous_pipeline")
    .start(f"{BASE_PATH}/silver_continuous")
)

print("Continuous pipeline running — will stop after 12 seconds for demo...")
time.sleep(12)
continuous_query.stop()
print("Continuous query stopped.")

spark.read.format("delta").load(f"{BASE_PATH}/silver_continuous") \
    .select("order_id", "trigger_type", "processed_at").show(5, False)


 ### Mode Comparison Summary


 ```
 ╔══════════════════════════════════════════════════════════════════════════════╗
 ║                         TRIGGERED vs. CONTINUOUS                             ║
 ╠═══════════════════════╦══════════════════════════════════════════════════════╣
 ║ Triggered             ║ Continuous                                          ║
 ╠═══════════════════════╬══════════════════════════════════════════════════════╣
 ║ .trigger(availableNow)║ .trigger(processingTime="X seconds")                 ║
 ║ Stops when done       ║ Runs until explicitly stopped                       ║
 ║ Minutes latency       ║ Sub-second latency                                  ║
 ║ Lower cost (ephemeral)║ Higher cost (always-on)                             ║
 ║ Good for: daily ETL   ║ Good for: real-time dashboards, alerting            ║
 ║ Lakeflow continuous:  ║ Lakeflow triggered:                                 ║
 ║   "continuous": false  ║   "continuous": true                                ║
 ╚═══════════════════════╩══════════════════════════════════════════════════════╝
 ```


 ---
 ## Concept 77 — CDC Processing: AUTO CDC INTO

 **Difficulty: Hard**

 CDC (Change Data Capture) processes row-level changes (INSERT, UPDATE, DELETE) from
# MGIC source systems. Lakeflow provides **`AUTO CDC INTO`** (formerly `APPLY CHANGES INTO`)
# MGIC to handle this declaratively:

 - **SCD Type 1**: Overwrites old values with new values (no history)
 - **SCD Type 2**: Tracks full history with `__START_AT` and `__END_AT` columns

 Key concepts:
 - **Sequencing column**: determines order of changes (e.g., `sequence_num`, `timestamp`)
 - **Primary keys**: identify which rows to update
 - **Deduplication**: latest change per sequence wins


 ### Generate Synthetic CDC Events


In [ ]:
# =============================================================================
#      SYNTHETIC CDC FEED — Simulates database change log
# =============================================================================

cdc_events = spark.createDataFrame([
    # (op, customer_id, email, tier, region, signup_date, sequence)
    ("INSERT", "C00001", "alice@example.com",    "Basic",    "US-West",    "2024-01-10", 1),
    ("INSERT", "C00002", "bob@example.com",      "Basic",    "US-East",    "2024-01-11", 2),
    ("INSERT", "C00003", "charlie@example.com",  "Premium",  "EU-Central", "2024-01-12", 3),
    ("UPDATE", "C00001", "alice@example.com",    "Premium",  "US-West",    "2024-01-10", 4),   # Alice upgraded
    ("INSERT", "C00004", "diana@example.com",    "Basic",    "APAC-South", "2024-01-13", 5),
    ("UPDATE", "C00002", "bob.new@example.com",  "Enterprise","US-East",   "2024-01-11", 6),   # Bob changed email + tier
    ("DELETE", "C00003", "charlie@example.com",  "Premium",  "EU-Central", "2024-01-12", 7),   # Charlie deleted
    ("INSERT", "C00005", "eve@example.com",      "Basic",    "EU-Central", "2024-01-14", 8),
    ("UPDATE", "C00001", "alice.new@example.com","Enterprise","US-West",   "2024-01-10", 9),   # Alice changed again
], schema=["op", "customer_id", "email", "tier", "region", "signup_date", "sequence"])

cdc_events.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/cdc_feed")
print("CDC events:")
cdc_events.orderBy("sequence").show(truncate=False)


 ### Lakeflow AUTO CDC INTO (Commented Reference)


In [ ]:
# =============================================================================
#      LAKEFLOW / DLT — AUTO CDC INTO (not executable in CE)
# =============================================================================
# -- SQL SYNTAX (preferred for CDC)
# CREATE OR REFRESH STREAMING TABLE customers_scd2;
#
# APPLY CHANGES INTO LIVE.customers_scd2       -- or AUTO CDC INTO
# FROM STREAM(LIVE.cdc_feed)
# KEYS (customer_id)
# SEQUENCE BY sequence
# COLUMNS * EXCEPT (op)
# STORE AS SCD TYPE 2;
#
# -- Python syntax:
# import dlt
# dlt.create_streaming_table("customers_scd2")
#
# dlt.apply_changes(
#     target="customers_scd2",
#     source="cdc_feed",
#     keys=["customer_id"],
#     sequence_by=F.col("sequence"),
#     apply_as_deletes=F.expr("op = 'DELETE'"),
#     stored_as_scd_type=2
# )
#
# # Result table automatically gets:
# #   __START_AT        — when the row version became active
# #   __END_AT          — when the row version was superseded (null = current)
# #   __is_deleted      — flag for deleted records


 ### Manual CDC Processing — SCD Type 2 (Community Edition)


In [ ]:
# =============================================================================
#      MANUAL CDC — SCD TYPE 2 IMPLEMENTATION
# =============================================================================

def apply_cdc_scd2(source_df, target_path, keys, sequence_col, op_col="op",
                   delete_op="DELETE", insert_op="INSERT", update_op="UPDATE"):
    """
    Manual implementation of CDC SCD Type 2 (mirrors Lakeflow APPLY CHANGES).
    """

    from delta.tables import DeltaTable

    target_full_path = f"{BASE_PATH}/{target_path}"

    # 1. Get the latest version of each key (deduplication by sequence)
    latest_changes = (
        source_df
        .withColumn("_rn", F.row_number().over(
            F.Window.partitionBy(keys).orderBy(F.desc(sequence_col))
        ))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    # Separate inserts/deletes/updates
    deletes = latest_changes.filter(F.col(op_col) == delete_op).select(keys)
    inserts = latest_changes.filter(F.col(op_col) == insert_op)
    updates = latest_changes.filter(F.col(op_col) == update_op)

    if not DeltaTable.isDeltaTable(spark, target_full_path):
        # First run: initialize target table
        inserts \
            .withColumn("__START_AT", F.current_timestamp()) \
            .withColumn("__END_AT", F.lit(None).cast("timestamp")) \
            .write.format("delta").mode("overwrite").save(target_full_path)
        print(f"[INIT] Created SCD2 table: {target_full_path} with {inserts.count()} rows")
        return DeltaTable.forPath(spark, target_full_path)

    target = DeltaTable.forPath(spark, target_full_path)
    now = F.current_timestamp()

    # 2. Handle DELETES: close current records
    if deletes.count() > 0:
        join_cond = " AND ".join([f"target.{k} = source.{k}" for k in keys])
        target.alias("target").merge(
            deletes.alias("source"), join_cond
        ).whenMatchedUpdate(
            condition="target.__END_AT IS NULL",
            set={"__END_AT": "current_timestamp()"}
        ).execute()
        print(f"[CDC] Closed {deletes.count()} deleted record(s)")

    # 3. Handle UPDATES: close old version + insert new version
    if updates.count() > 0:
        # Step A: Close current versions
        join_cond = " AND ".join([f"target.{k} = source.{k}" for k in keys])
        target.alias("target").merge(
            updates.alias("source"), join_cond
        ).whenMatchedUpdate(
            condition="target.__END_AT IS NULL",
            set={"__END_AT": "current_timestamp()"}
        ).execute()

        # Step B: Insert new rows (only columns that exist in target)
        existing_cols = [f.name for f in target.toDF().schema.fields
                        if f.name not in ("__START_AT", "__END_AT")]
        cols_to_insert = [c for c in existing_cols if c in updates.columns]

        new_versions = (
            updates.select(*cols_to_insert)
            .withColumn("__START_AT", F.current_timestamp())
            .withColumn("__END_AT", F.lit(None).cast("timestamp"))
        )
        new_versions.write.format("delta").mode("append").save(target_full_path)
        print(f"[CDC] Closed + inserted {updates.count()} updated record(s)")

    # 4. Handle INSERTS
    if inserts.count() > 0:
        inserts \
            .withColumn("__START_AT", F.current_timestamp()) \
            .withColumn("__END_AT", F.lit(None).cast("timestamp")) \
            .write.format("delta").mode("append").save(target_full_path)
        print(f"[CDC] Inserted {inserts.count()} new record(s)")

    return DeltaTable.forPath(spark, target_full_path)


# --- DEMO: Apply CDC to customer dimension ---
cdc_source = spark.read.format("delta").load(f"{BASE_PATH}/cdc_feed")

target = apply_cdc_scd2(
    source_df=cdc_source,
    target_path="customers_scd2",
    keys=["customer_id"],
    sequence_col="sequence",
)

print("\n=== SCD2 Target Table (current records only) ===")
target.toDF() \
    .filter(F.col("__END_AT").isNull()) \
    .select("customer_id", "email", "tier", "region", "__START_AT") \
    .orderBy("customer_id") \
    .show(truncate=False)

print("\n=== Full History (all versions) ===")
target.toDF() \
    .select("customer_id", "email", "tier", "__START_AT", "__END_AT") \
    .orderBy("customer_id", "__START_AT") \
    .show(truncate=False)

print(f"\n✅ Alice started Basic → Premium → Enterprise (3 versions)")
print(f"✅ Bob started Basic → Enterprise (2 versions, email changed)")
print(f"✅ Charlie deleted — __END_AT is set (0 current records)")


 ### CDC SCD Type 1 (No History) — Manual Equivalent


In [ ]:
# =============================================================================
#      MANUAL CDC — SCD TYPE 1 (Overwrite, no history)
# =============================================================================
def apply_cdc_scd1(source_df, target_path, keys, sequence_col):
    from delta.tables import DeltaTable
    target_full_path = f"{BASE_PATH}/{target_path}"

    latest = (
        source_df
        .withColumn("_rn", F.row_number().over(
            F.Window.partitionBy(keys).orderBy(F.desc(sequence_col))
        ))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    if not DeltaTable.isDeltaTable(spark, target_full_path):
        latest.write.format("delta").mode("overwrite").save(target_full_path)
        print(f"[INIT] Created SCD1 table: {target_full_path}")
        return

    join_cond = " AND ".join([f"target.{k} = source.{k}" for k in keys])
    DeltaTable.forPath(spark, target_full_path).alias("target").merge(
        latest.alias("source"), join_cond
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"[CDB] SCD1 upserted {latest.count()} rows")


apply_cdc_scd1(
    source_df=cdc_source.filter(F.col("op") != "DELETE"),
    target_path="customers_scd1",
    keys=["customer_id"],
    sequence_col="sequence",
)

print("\nSCD1 Table (overwritten — no history):")
spark.read.format("delta").load(f"{BASE_PATH}/customers_scd1") \
    .orderBy("customer_id").show(truncate=False)


 ---
 ## Concept 78 — Error Handling & Dead Letter Patterns

 **Difficulty: Hard**

 In production pipelines, not all data is clean. Lakeflow provides:

 - **`expect_or_drop`**: Route bad records out of the main pipeline
 - **`expect_or_fail`**: Halt pipeline on critical violations
 - **On violation metrics**: Track error rates in event log

 The **Dead Letter Queue (DLQ)** pattern captures bad records for later inspection:
 - Main table → only valid records
 - Dead letter table → invalid records with error metadata


 ### Lakeflow Dead Letter Pattern (Commented Reference)


In [ ]:
# =============================================================================
#      LAKEFLOW / DLT — DEAD LETTER PATTERN (not executable in CE)
# =============================================================================
# @table(name="silver_orders_valid")
# @dlt.expect_or_drop("valid_order_id",   "order_id IS NOT NULL")
# @dlt.expect_or_drop("positive_amount",  "amount > 0")
# @dlt.expect_or_drop("valid_status",     "status IN ('pending','shipped','delivered','cancelled')")
# def silver_orders_valid():
#     return dlt.read("bronze_orders")
#
# @table(name="silver_orders_dead_letter")
# def silver_orders_dead_letter():
#     return dlt.read("bronze_orders") \
#         .filter("order_id IS NULL OR amount <= 0 OR status NOT IN ('pending','shipped','delivered','cancelled')") \
#         .withColumn("error_message", F.concat_ws(", ",
#             F.when(F.col("order_id").isNull(), "missing_order_id"),
#             F.when(F.col("amount") <= 0, "invalid_amount"),
#             F.when(~F.col("status").isin("pending","shipped","delivered","cancelled"), "invalid_status")
#         ))


 ### Manual Dead Letter Queue (Community Edition)


In [ ]:
# =============================================================================
#      MANUAL DEAD LETTER QUEUE — Error categorization and quarantine
# =============================================================================

def build_dead_letter_pipeline(source_df, rules, valid_output_path, dlq_output_path):
    """
    Routes valid rows to the main output and invalid rows to a dead letter queue.

    rules: list of (name, condition, error_message) tuples
    """
    # Build the "is valid" composite condition
    valid_condition = F.lit(True)
    for name, condition, _ in rules:
        valid_condition = valid_condition & condition

    # --- Main output: valid records only ---
    valid_df = source_df.filter(valid_condition) \
        .withColumn("pipeline_status", F.lit("VALID")) \
        .withColumn("processed_at", F.current_timestamp())

    valid_df.write.format("delta").mode("overwrite").save(valid_output_path)
    print(f"[MAIN] Valid records: {valid_df.count()} → {valid_output_path}")

    # --- DLQ output: invalid records with error details ---
    dlq_df = source_df.filter(~valid_condition)
    if dlq_df.count() > 0:
        dlq_df = dlq_df.drop("pipeline_status") if "pipeline_status" in dlq_df.columns else dlq_df
        dlq_df = dlq_df.withColumn("pipeline_status", F.lit("ERROR"))

        # Categorize each error
        for name, condition, err_msg in rules:
            dlq_df = dlq_df.withColumn(
                f"err_{name}",
                F.when(~condition, F.lit(err_msg)).otherwise(F.lit(None))
            )

        # Combine all errors into a single column
        err_cols = [f"err_{name}" for name, _, _ in rules]
        dlq_df = dlq_df.withColumn(
            "error_messages",
            F.concat_ws("; ", *[F.col(c) for c in err_cols])
        ).withColumn("error_timestamp", F.current_timestamp())

        # Keep only relevant metadata
        dlq_output = dlq_df.select(
            "order_id", "customer_id", "product_id", "amount", "status",
            "error_messages", "error_timestamp", "pipeline_status",
            *[c for c in dlq_df.columns if c not in err_cols and c not in [
                "error_messages", "error_timestamp", "pipeline_status", "order_id",
                "customer_id", "product_id", "amount", "status"
            ]]
        )

        dlq_output.write.format("delta").mode("overwrite").save(dlq_output_path)
        print(f"[DLQ]  Dead letter records: {dlq_output.count()} → {dlq_output_path}")
    else:
        print("[DLQ]  No dead letter records — all data valid")

    return valid_df


# --- Define quality rules ---
quality_rules = [
    ("null_order_id",    F.col("order_id").isNotNull(),
     "order_id is null"),
    ("null_customer_id", F.col("customer_id").isNotNull(),
     "customer_id is null"),
    ("positive_amount",  F.col("amount") > 0,
     "amount <= 0"),
    ("valid_status",     F.col("status").isin("pending", "shipped", "delivered", "cancelled"),
     f"invalid status: not in (pending,shipped,delivered,cancelled)"),
    ("non_zero_qty",     F.col("quantity") > 0,
     "quantity <= 0"),
    ("valid_discount",   (F.col("discount_pct") >= 0) & (F.col("discount_pct") <= 100),
     "discount_pct out of range [0,100]"),
]

# Test with dirty data
dirty_source = spark.createDataFrame([
    (None, "C00001", "P0001", 2,  99.99, "shipped",   "US-West",  10, "2024-01-15 10:00:00"),  # null order_id
    ("ORD-001", None, "P0001", 2,  99.99, "shipped",   "US-West",  10, "2024-01-15 10:00:00"),  # null customer
    ("ORD-002", "C00002", "P0001", 0, -10.00, "UNKNOWN","??",      -5, "2024-01-15 10:00:00"),  # multiple errors
    ("ORD-003", "C00003", "P0002", 5, 250.00, "shipped", "US-East", 5, "2024-01-15 10:00:00"),   # clean
    ("ORD-004", "C00004", "P0003", 1,   0.00, "cancelled","EU-Central",0,"2024-01-15 10:00:00"), # zero amount
], schema=order_schema)

print("Source data:")
dirty_source.show(truncate=False)

result = build_dead_letter_pipeline(
    source_df=dirty_source,
    rules=quality_rules,
    valid_output_path=f"{BASE_PATH}/silver_valid",
    dlq_output_path=f"{BASE_PATH}/silver_dead_letter",
)


 ### Inspect Dead Letter Queue


In [ ]:
print("=== MAIN TABLE (valid records) ===")
spark.read.format("delta").load(f"{BASE_PATH}/silver_valid") \
    .select("order_id", "amount", "status", "pipeline_status").show(truncate=False)

print("\n=== DEAD LETTER QUEUE (errors) ===")
spark.read.format("delta").load(f"{BASE_PATH}/silver_dead_letter") \
    .select("order_id", "amount", "status", "error_messages", "error_timestamp") \
    .show(truncate=False)

# --- Error Categorization & Retry Simulation ---
dlq_df = spark.read.format("delta").load(f"{BASE_PATH}/silver_dead_letter")
print("\n=== Error Breakdown ===")
dlq_df.groupBy("error_messages").count().orderBy(F.desc("count")).show(truncate=False)

# Simulate a retry: fix data and re-run
print("\n=== RETRY SIMULATION: Fix null order_id, reprocess ===")
fixed = dlq_df.fillna({"order_id": "ORD-RETRY-FIXED"})
print(f"Retrying {fixed.count()} records after fixes...")
# In real scenario, would re-run through the main pipeline


 ---
 ## Concept 79 — Pipeline Parameters & Configuration

 **Difficulty: Hard**

 Production pipelines need different behavior per environment. Lakeflow supports:

 - **Pipeline settings** JSON/YAML configuration
 - **Spark configuration** for cluster-level settings
 - **Databricks widgets** for parameter injection at runtime

 Common parameterization patterns:
 - **Storage paths**: dev → `/tmp/dev`, prod → `/mnt/prod-lake`
 - **Refresh cadence**: dev → on-demand, prod → hourly
 - **Quality thresholds**: dev → lower, prod → strict


 ### Lakeflow Pipeline Parameters (Commented Reference)


In [ ]:
# =============================================================================
#      LAKEFLOW / DLT — PARAMETERIZED PIPELINE (not executable in CE)
# =============================================================================
# # In pipeline settings JSON:
# {
#   "name": "retail_etl_{{environment}}",
#   "configuration": {
#     "source_path": "/mnt/{{environment}}/raw/orders",
#     "target_path": "/mnt/{{environment}}/curated",
#     "quality_mode": "{{quality_mode}}",
#     "max_late_data_hours": "24"
#   }
# }
#
# # In notebook code:
# import dlt
#
# source_path = spark.conf.get("source_path")
# quality_mode = spark.conf.get("quality_mode")
#
# @table(name="bronze_orders")
# def bronze_orders():
#     df = spark.readStream.format("delta").load(source_path)
#     if quality_mode == "strict":
#         dlt.expect_or_fail("has_order_id", "order_id IS NOT NULL")
#     return df


 ### Manual Configuration Management (Community Edition)


In [ ]:
# =============================================================================
#      MANUAL CONFIGURATION MANAGEMENT
# =============================================================================

class PipelineConfig:
    """Environment-aware pipeline configuration."""

    ENVIRONMENTS = {
        "dev": {
            "source_path": f"{BASE_PATH}/raw_orders",
            "target_path": f"{BASE_PATH}/curated_dev",
            "quality_mode": "warn",        # dev: don't break on bad data
            "batch_size": 100,
            "trigger_interval": "30 seconds",
            "retention_hours": 1,
            "log_level": "DEBUG",
        },
        "staging": {
            "source_path": f"{BASE_PATH}/raw_orders",
            "target_path": f"{BASE_PATH}/curated_staging",
            "quality_mode": "drop",        # staging: drop bad rows, don't fail
            "batch_size": 1000,
            "trigger_interval": "5 minutes",
            "retention_hours": 168,
            "log_level": "INFO",
        },
        "prod": {
            "source_path": f"{BASE_PATH}/raw_orders",
            "target_path": f"{BASE_PATH}/curated_prod",
            "quality_mode": "fail",        # prod: fail on critical violations
            "batch_size": 5000,
            "trigger_interval": "1 minute",
            "retention_hours": 8760,       # 1 year
            "log_level": "WARN",
        },
    }

    def __init__(self, environment="dev"):
        if environment not in self.ENVIRONMENTS:
            raise ValueError(f"Unknown environment: {environment}. "
                             f"Choose from: {list(self.ENVIRONMENTS.keys())}")
        self.env = environment
        self.config = self.ENVIRONMENTS[environment]
        self.runtime = {}

    def get(self, key, default=None):
        return self.config.get(key, default)

    def set_runtime(self, key, value):
        self.runtime[key] = value

    def print_config(self):
        print(f"\n{'='*60}")
        print(f"  PIPELINE CONFIGURATION — Environment: {self.env.upper()}")
        print(f"{'='*60}")
        for k, v in self.config.items():
            print(f"  {k:25s} = {v}")
        print(f"{'='*60}\n")


class ParameterizedPipeline:
    """Pipeline that adapts behavior based on configuration."""

    def __init__(self, config: PipelineConfig):
        self.config = config
        self.source_path = config.get("source_path")
        self.target_path = config.get("target_path")
        self.quality_mode = config.get("quality_mode")
        self.batch_size = config.get("batch_size")

    def extract(self):
        print(f"[EXTRACT] Reading from {self.source_path} (batch_size={self.batch_size})")
        source_df = spark.read.format("delta").load(self.source_path) \
            .withColumn("_pipeline_env", F.lit(self.config.env)) \
            .withColumn("_processed_at", F.current_timestamp())
        return source_df.limit(self.batch_size)

    def transform(self, df):
        print(f"[TRANSFORM] Quality mode: {self.quality_mode}")
        if self.quality_mode == "fail":
            assert df.filter(F.col("order_id").isNull()).count() == 0, \
                "PROD: null order_ids detected — failing!"
            assert df.filter(F.col("amount") <= 0).count() == 0, \
                "PROD: non-positive amounts detected — failing!"
            result = df
        elif self.quality_mode == "drop":
            result = df.filter(F.col("order_id").isNotNull()) \
                       .filter(F.col("amount") > 0)
        else:  # "warn"
            null_ids = df.filter(F.col("order_id").isNull()).count()
            if null_ids > 0:
                print(f"  [WARN] {null_ids} null order_ids found (dev mode)")
            result = df

        return result.withColumn("total_sale", F.col("quantity") * F.col("amount"))

    def load(self, df):
        output_path = f"{self.target_path}"
        df.write.format("delta").mode("overwrite").save(output_path)
        print(f"[LOAD] Wrote {df.count()} rows to {output_path}")
        return df.count()

    def run(self):
        print(f"\n▶️  Starting pipeline — ENV: {self.config.env.upper()}")
        raw = self.extract()
        transformed = self.transform(raw)
        count = self.load(transformed)
        print(f"⏹️  Pipeline complete — {count} rows processed\n")
        return count


# --- DEMO: Run same pipeline logic across environments ---
for env_name in ["dev", "staging", "prod"]:
    cfg = PipelineConfig(env_name)
    cfg.print_config()
    pipeline = ParameterizedPipeline(cfg)
    pipeline.run()


 ### Simulating Databricks Widgets for Runtime Parameters


In [ ]:
# =============================================================================
#      DATABRICKS WIDGETS SIMULATION (dbutils.widgets in actual Databricks)
# =============================================================================

# In real Databricks, you'd use:
# dbutils.widgets.dropdown("environment", "dev", ["dev", "staging", "prod"])
# dbutils.widgets.text("start_date", "2024-01-01")
# env = dbutils.widgets.get("environment")

# For CE / local, we use a config dictionary or environment variable:
import os as _os

def get_widget_value(name, default=None):
    """Simulates dbutils.widgets.get() for Community Edition."""
    # In real DB: return dbutils.widgets.get(name)
    # In CE: fall back to environment variable or hard-coded default
    return _os.environ.get(name.upper(), default)

simulated_env      = get_widget_value("environment", "dev")
simulated_start_dt = get_widget_value("start_date",  "2024-01-01")
simulated_batch    = int(get_widget_value("batch_size", "500"))

print(f"Widget 'environment'  = {simulated_env}")
print(f"Widget 'start_date'   = {simulated_start_dt}")
print(f"Widget 'batch_size'   = {simulated_batch}")
print(f"\nReal Databricks equivalent:")
print(f"  dbutils.widgets.dropdown('environment', '{simulated_env}', ['dev','staging','prod'])")
print(f"  dbutils.widgets.text('start_date', '{simulated_start_dt}')")
print(f"  dbutils.widgets.text('batch_size', '{simulated_batch}')")


 ---
 ## Concept 80 — Multi-Pipeline Architecture

 **Difficulty: Hard**

 As pipeline complexity grows, splitting into **focused, smaller pipelines** improves
 maintainability, debuggability, and team ownership. Lakeflow supports sharing tables
 across pipelines and orchestrating execution order.

 ### When to Split

 | Keep Together | Split Apart |
 |---------------|-------------|
 | Tightly coupled transformations | Different SLAs (real-time vs. daily) |
 | Single team owns all | Multiple teams own different stages |
 | Simple linear flow | Complex DAG with dependencies |
 | < 10 tables | Many tables / high cardinality |

 ### Common Architecture Pattern
 ```
 Pipeline 1: Ingestion    (bronze)     → raw → validated
 Pipeline 2: Transformation (silver)    → validated → enriched
 Pipeline 3: Aggregation   (gold)       → enriched → metrics
 Pipeline 4: ML Features   (feature store) → enriched → features
 ```


 ### Lakeflow Multi-Pipeline (Commented Reference)


In [ ]:
# =============================================================================
#      LAKEFLOW / DLT — MULTI-PIPELINE (not executable in CE)
# =============================================================================
# # Pipeline 1: Ingestion (ingestion_pipeline.py)
# @table(name="bronze_orders")
# def bronze_orders():
#     return spark.readStream.format("delta").table("raw_source.orders")
#
# # Pipeline 2: Transformation (transformation_pipeline.py)
# # Can read tables from Pipeline 1 using LIVE.<table_name>
# @table(name="silver_orders_enriched")
# def silver_orders_enriched():
#     orders   = dlt.read("bronze_orders")       # from Pipeline 1
#     products = dlt.read("product_dim")
#     return orders.join(products, "product_id", "left") \
#         .withColumn("category_revenue", F.col("quantity") * F.col("price"))
#
# # Pipeline 3: Aggregation (aggregation_pipeline.py)
# @materialized_view(name="gold_daily_kpis")
# def gold_daily_kpis():
#     return dlt.read("silver_orders_enriched") \
#         .groupBy(F.to_date("order_ts").alias("date"), "category") \
#         .agg(F.sum("category_revenue").alias("daily_revenue"))
#
# # Orchestration via Databricks Workflows or Job API:
# # Ingestion → Transform → Aggregate


 ### Manual Multi-Pipeline Orchestration (Community Edition)


In [ ]:
from datetime import datetime as dt

# =============================================================================
#      MANUAL MULTI-PIPELINE ORCHESTRATION
# =============================================================================

class PipelineStep:
    """Represents one step in a multi-pipeline architecture."""

    def __init__(self, name, func, dependencies=None):
        self.name = name
        self.func = func
        self.dependencies = dependencies or []
        self.status = "PENDING"    # PENDING | RUNNING | SUCCESS | FAILED
        self.start_time = None
        self.end_time = None
        self.rows_output = 0

    def run(self):
        self.status = "RUNNING"
        self.start_time = dt.now()
        print(f"  ▶️  [{self.name}] Starting...")
        try:
            self.rows_output = self.func()
            self.status = "SUCCESS"
        except Exception as e:
            self.status = "FAILED"
            print(f"  ❌ [{self.name}] FAILED: {e}")
            raise
        finally:
            self.end_time = dt.now()
        print(f"  ✅ [{self.name}] Complete — {self.rows_output} rows "
              f"({(self.end_time - self.start_time).total_seconds():.1f}s)")

    def __repr__(self):
        return f"PipelineStep({self.name}, status={self.status})"


class MultiPipelineOrchestrator:
    """Orchestrates multiple pipeline steps respecting dependencies."""

    def __init__(self, name="multi_pipeline"):
        self.name = name
        self.steps = []
        self.execution_log = []

    def add_step(self, name, func, dependencies=None):
        self.steps.append(PipelineStep(name, func, dependencies))
        return self

    def _ready(self, step, completed):
        return all(dep in completed for dep in step.dependencies)

    def _show_dag(self):
        print(f"\n{'='*60}")
        print(f"  PIPELINE DAG: {self.name}")
        print(f"{'='*60}")
        for s in self.steps:
            deps = f" ← depends on: {s.dependencies}" if s.dependencies else ""
            print(f"  [{s.name}]{deps}")
        print(f"{'='*60}\n")

    def run(self):
        self._show_dag()
        completed = set()
        failed = set()
        remaining = list(self.steps)

        while remaining:
            ready = [s for s in remaining if self._ready(s, completed)]
            if not ready:
                pending = [s.name for s in remaining]
                print(f"❌ Deadlock or failure — pending steps: {pending}")
                break

            for step in ready:
                try:
                    step.run()
                    completed.add(step.name)
                    self.execution_log.append({
                        "step": step.name,
                        "status": step.status,
                        "rows": step.rows_output,
                        "duration_sec": (step.end_time - step.start_time).total_seconds()
                    })
                except Exception:
                    failed.add(step.name)
                    self.execution_log.append({
                        "step": step.name,
                        "status": "FAILED",
                        "rows": 0,
                        "duration_sec": 0
                    })
                remaining.remove(step)

        success_count = len([s for s in self.steps if s.status == "SUCCESS"])
        print(f"\n{'='*60}")
        print(f"  PIPELINE COMPLETE: {success_count}/{len(self.steps)} steps succeeded")
        print(f"{'='*60}")
        return self.execution_log


# ---- DEFINE PIPELINE FUNCTIONS ----

# Pipeline 1: Ingestion
def ingest_raw_orders():
    df = spark.read.format("delta").load(f"{BASE_PATH}/raw_orders") \
        .withColumn("ingested_at", F.current_timestamp())
    df.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/pipeline_bronze")
    return df.count()

def validate_bronze():
    df = spark.read.format("delta").load(f"{BASE_PATH}/pipeline_bronze")
    valid = df.filter(F.col("order_id").isNotNull()) \
              .filter(F.col("amount") > 0) \
              .filter(F.col("status").isin("pending", "shipped", "delivered", "cancelled"))
    valid.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/pipeline_bronze_validated")
    invalid = df.filter(~F.col("order_id").isNotNull() |
                         (F.col("amount") <= 0) |
                         ~F.col("status").isin("pending", "shipped", "delivered", "cancelled"))
    invalid.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/pipeline_bronze_dlq")
    print(f"     Valid: {valid.count()}, Invalid (DLQ): {invalid.count()}")
    return valid.count()

# Pipeline 2: Transformation (depends on ingestion)
def enrich_silver():
    orders = spark.read.format("delta").load(f"{BASE_PATH}/pipeline_bronze_validated")
    products = spark.read.format("delta").load(f"{BASE_PATH}/products")
    customers = spark.read.format("delta").load(f"{BASE_PATH}/customers")

    enriched = orders \
        .join(products.select("product_id", F.col("price").alias("unit_price"), F.col("category")),
              "product_id", "left") \
        .join(customers.select("customer_id", "tier", "signup_date"),
              "customer_id", "left") \
        .withColumn("line_total", F.col("quantity") * F.col("unit_price")) \
        .withColumn("discount_amount",
                    F.round(F.col("line_total") * F.col("discount_pct") / 100.0, 2)) \
        .withColumn("net_revenue", F.col("line_total") - F.col("discount_amount"))

    enriched.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/pipeline_silver")
    return enriched.count()

# Pipeline 3: Aggregation (depends on transformation)
def aggregate_gold():
    silver = spark.read.format("delta").load(f"{BASE_PATH}/pipeline_silver")

    daily_metrics = silver \
        .withColumn("order_date", F.to_date("order_ts")) \
        .groupBy("order_date", "category", "region") \
        .agg(
            F.sum("net_revenue").alias("total_revenue"),
            F.count("*").alias("order_count"),
            F.avg("net_revenue").alias("avg_order_value"),
            F.sum("quantity").alias("total_units_sold")
        ) \
        .withColumn("processed_at", F.current_timestamp())

    daily_metrics.write.format("delta").mode("overwrite") \
        .save(f"{BASE_PATH}/pipeline_gold_daily_metrics")

    # Customer-tier KPIs
    tier_metrics = silver \
        .groupBy("tier") \
        .agg(
            F.sum("net_revenue").alias("total_revenue"),
            F.countDistinct("customer_id").alias("unique_customers"),
            F.avg("net_revenue").alias("avg_revenue_per_order")
        )

    tier_metrics.write.format("delta").mode("overwrite") \
        .save(f"{BASE_PATH}/pipeline_gold_tier_metrics")

    return daily_metrics.count()

# Pipeline 4: Data Product / Feature Store (depends on transformation)
def build_features():
    silver = spark.read.format("delta").load(f"{BASE_PATH}/pipeline_silver")

    features = silver \
        .groupBy("customer_id") \
        .agg(
            F.count("*").alias("lifetime_orders"),
            F.sum("net_revenue").alias("lifetime_value"),
            F.avg("net_revenue").alias("avg_order_value"),
            F.datediff(F.current_date(), F.min("signup_date")).alias("days_since_signup"),
            F.size(F.collect_set("category")).alias("categories_purchased"),
            F.max(F.to_date("order_ts")).alias("last_order_date")
        ) \
        .withColumn("churn_risk",
                     F.when(F.datediff(F.current_date(), F.col("last_order_date")) > 90, "HIGH")
                      .when(F.datediff(F.current_date(), F.col("last_order_date")) > 60, "MEDIUM")
                      .otherwise("LOW"))

    features.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/pipeline_customer_features")
    return features.count()


# ---- ORCHESTRATE ----
orchestrator = MultiPipelineOrchestrator(name="Retail_ETL_MultiPipeline")

orchestrator \
    .add_step("1_ingest",      ingest_raw_orders) \
    .add_step("1_validate",    validate_bronze,     dependencies=["1_ingest"]) \
    .add_step("2_enrich",      enrich_silver,       dependencies=["1_validate"]) \
    .add_step("3_daily_kpis",  aggregate_gold,      dependencies=["2_enrich"]) \
    .add_step("4_features",    build_features,      dependencies=["2_enrich"])

execution_log = orchestrator.run()


 ### Inspect Multi-Pipeline Outputs


In [ ]:
print("\n=== EXECUTION LOG ===")
for entry in execution_log:
    status_icon = "✅" if entry["status"] == "SUCCESS" else "❌"
    print(f"  {status_icon} {entry['step']}: {entry['rows']} rows, {entry['duration_sec']:.1f}s")

print("\n=== GOLD: Daily KPIs (sample) ===")
spark.read.format("delta").load(f"{BASE_PATH}/pipeline_gold_daily_metrics") \
    .orderBy(F.desc("order_date"), F.desc("total_revenue")) \
    .show(10, truncate=False)

print("\n=== GOLD: Tier Metrics ===")
spark.read.format("delta").load(f"{BASE_PATH}/pipeline_gold_tier_metrics") \
    .show(truncate=False)

print("\n=== CUSTOMER FEATURES: Top 10 by Lifetime Value ===")
spark.read.format("delta").load(f"{BASE_PATH}/pipeline_customer_features") \
    .orderBy(F.desc("lifetime_value")) \
    .select("customer_id", "lifetime_orders", "lifetime_value", "churn_risk") \
    .show(10, truncate=False)


 ### Dependency Graph Visualization


In [ ]:
# =============================================================================
#      DEPENDENCY GRAPH (ASCII)
# =============================================================================
graph = """
  Multi-Pipeline Architecture
  ===========================

        ┌──────────────┐
        │  1_ingest    │  (Streaming Table)
        │  raw→bronze  │
        └──────┬───────┘
               │
               ▼
        ┌──────────────┐
        │  1_validate  │  (Table + Expectations)
        │  quality DLQ │
        └──────┬───────┘
               │
               ▼
        ┌──────────────┐
        │   2_enrich   │  (Streaming Table)
        │  joins + calc│
        └──┬────────┬──┘
           │        │
     ┌─────▼──┐ ┌───▼──────────┐
     │3_daily │ │ 4_features   │
     │ KPIs   │ │ churn + ML   │
     │(Mater. │ │(FeatureStore)│
     │ View)  │ │              │
     └────────┘ └──────────────┘

  Pipeline 3 and 4 run IN PARALLEL after Pipeline 2.
"""

print(graph)


 ---
 ## Summary — Lakeflow / DLT vs. Manual Approach


 ```
 ╔══════════════════════════════════════════════════════════════════════════════════╗
 ║  CONCEPT              │ WITH LAKEFLOW (DLT)            │ WITHOUT (CE MANUAL)       ║
 ╠═══════════════════════╪════════════════════════════════╪═══════════════════════════╣
 ║ 71. Syntax            │ @table / CREATE STREAMING TABLE│ Manual readStream + write ║
 ║                       │ Mix SQL + Python seamlessly   │ Separate code paths       ║
 ╠═══════════════════════╪════════════════════════════════╪═══════════════════════════╣
 ║ 72. Streaming vs MV   │ Auto-handled by framework      │ Manual checkpoint mgmt    ║
 ║                       │ Exactly-once built-in          │ Manual append/overwrite   ║
 ╠═══════════════════════╪════════════════════════════════╪═══════════════════════════╣
 ║ 73. Expectations      │ expect / expect_or_drop / fail │ Custom filter functions    ║
 ║                       │ Metrics auto-logged to events  │ Manual quality logging     ║
 ╠═══════════════════════╪════════════════════════════════╪═══════════════════════════╣
 ║ 74. Dev & Testing     │ Development mode (dev_ prefix) │ Manual fixture creation    ║
 ║                       │ Incremental / full refresh     │ Manual testing patterns    ║
 ╠═══════════════════════╪════════════════════════════════╪═══════════════════════════╣
 ║ 75. Monitoring        │ Event log auto-populated       │ Custom PipelineRunTracker  ║
 ║                       │ Built-in lineage & metrics     │ Manual monitoring table    ║
 ╠═══════════════════════╪════════════════════════════════╪═══════════════════════════╣
 ║ 76. Pipeline Modes    │ continuous: true/false config  │ availableNow / processing  ║
 ║                       │ UI toggle, no code change      │ Explicit trigger parameter ║
 ╠═══════════════════════╪════════════════════════════════╪═══════════════════════════╣
 ║ 77. CDC Processing    │ AUTO CDC INTO (1 line)         │ Manual MERGE + SCD logic   ║
 ║                       │ SCD1/SCD2 automatic            │ ~70 lines of MERGE code    ║
 ╠═══════════════════════╪════════════════════════════════╪═══════════════════════════╣
 ║ 78. Error Handling    │ expect_or_drop → auto DLQ-ish  │ Manual validation + DLQ    ║
 ║                       │ Event log tracks violations    │ Manual error categorization║
 ╠═══════════════════════╪════════════════════════════════╪═══════════════════════════╣
 ║ 79. Parameters        │ JSON pipeline settings         │ Config dict / env vars     ║
 ║                       │ Built-in environment support   │ Manual environment switch   ║
 ╠═══════════════════════╪════════════════════════════════╪═══════════════════════════╣
 ║ 80. Multi-Pipeline    │ Share tables via LIVE.<name>   │ Manual orchestration code  ║
 ║                       │ Databricks Workflows native    │ Custom dependency DAG       ║
 ╚═══════════════════════╩════════════════════════════════╩═══════════════════════════╝
 ```


 ### Key Takeaways

 1. **Lakeflow = Declarative**: You describe *what* you want; the framework handles *how*.
    Manual approaches require explicit orchestration, checkpointing, and error handling.

 2. **Community Edition Reality**: Everything Lakeflow does can be replicated manually with
    Structured Streaming + Delta Lake + custom monitoring. The manual approach teaches you
    what Lakeflow automates — valuable for understanding and debugging.

 3. **Quality First**: Lakeflow's expectation framework is its killer feature. The manual
    equivalent requires careful implementation of filtering, logging, and alerting.

 4. **CDC is the hardest**: Lakeflow's `AUTO CDC INTO` eliminates 70+ lines of MERGE logic
    for SCD Type 2. Understanding the manual approach makes you appreciate the abstraction.

 5. **Multi-pipeline thinking**: Even without Lakeflow, the pattern of splitting pipelines
    by medallion layer (bronze → silver → gold) is best practice for maintainability.


 ---
 ## Self-Assessment Checklist

 After completing this notebook, you should be able to:

 - [ ] **Concept 71**: Write the same pipeline in both Python decorator syntax AND SQL syntax
   and explain when to prefer each.
 - [ ] **Concept 72**: Distinguish streaming tables from materialized views and implement
   manual equivalents for each in Community Edition.
 - [ ] **Concept 73**: Implement data quality rules with warn/drop/fail semantics both via
   Lakeflow expectations and via a manual ExpectationFramework class.
 - [ ] **Concept 74**: Create test fixtures, write assertion-based pipeline tests, and
   explain the difference between development and production modes.
 - [ ] **Concept 75**: Build a PipelineRunTracker that logs execution metadata, query run
   history, and explain how Lakeflow's event log simplifies monitoring.
 - [ ] **Concept 76**: Configure both triggered (availableNow) and continuous (processingTime)
   pipelines and articulate the cost/latency tradeoffs.
 - [ ] **Concept 77**: Implement manual SCD Type 2 with MERGE, sequence-based deduplication,
   and explain how `AUTO CDC INTO` eliminates this boilerplate.
 - [ ] **Concept 78**: Build a dead letter queue that captures bad records with error
   categorization and retry logic — and explain how Lakeflow's `expect_or_drop` simplifies this.
 - [ ] **Concept 79**: Create environment-aware pipeline configurations and demonstrate
   the same pipeline logic running across dev/staging/prod with different behaviors.
 - [ ] **Concept 80**: Design a multi-pipeline architecture with dependency management,
   implement a simple orchestrator, and explain when to split vs. keep pipelines together.

 ---

 ### Resources
 - Databricks DLT Documentation: https://docs.databricks.com/en/delta-live-tables/index.html
 - Lakeflow Blog (March 2025): https://www.databricks.com/blog/introducing-lakeflow
 - Structured Streaming Guide: https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html
 - Delta Lake MERGE: https://docs.delta.io/latest/delta-update.html#upsert-into-a-table-using-merge


In [ ]:
# Cleanup streaming queries if any are still running
for q in spark.streams.active:
    try:
        q.stop()
        print(f"Stopped streaming query: {q.name}")
    except Exception as e:
        print(f"Could not stop query {q.name}: {e}")

print("\n" + "="*60)
print("  Notebook complete — all 10 concepts demonstrated.")
print("="*60)


---

# 09 Workflows CICD Operations



﻿# Databricks notebook source


 # 09: Workflows, CI/CD & Operations
 ## Concepts 81â€“90 | Production Orchestration & DevOps on Databricks

 ### Overview
 This notebook covers the operational backbone of a production Databricks environment:

 | # | Concept | Difficulty |
 |---|---------|-----------|
 | 81 | Choosing Compute for Jobs | Easy |
 | 82 | Secrets Management | Easy |
 | 83 | Parameterization: Widgets, Job Parameters, Task Values | Easy |
 | 84 | Notebook Development Patterns | Easy |
 | 85 | Databricks Workflows: Multi-Task Jobs | Medium |
 | 86 | Task Dependencies, Retries & Conditional Logic | Medium |
 | 87 | Git Integration & Repos | Medium |
 | 88 | Monitoring, Alerting & SQL Alerts | Medium |
 | 89 | System Tables for Operations | Medium |
 | 90 | Databricks Asset Bundles (DABs) | Hard |

 ### Community Edition Limitations
 Community Edition **cannot** create multi-task jobs, use DBSQL alerts, integrate Repos, or deploy DABs. We explain the production feature, then show what's possible in CE with manual alternatives.

 ```
 +=========================================================+
 |             PRODUCTION DATABRICKS                        |
 |  +---------+   +---------+   +---------+   +---------+  |
 |  | Secrets |-->|  Jobs   |-->|  Alerts |-->|  DABs   |  |
 |  | (AKV)   |   |(Workflow)|   | (Slack) |   | (CI/CD) |  |
 |  +---------+   +---------+   +---------+   +---------+  |
 +=========================================================+
                   | CE equivalent |
 +=========================================================+
 |            COMMUNITY EDITION                            |
 |  +---------+   +---------+   +---------+   +---------+  |
 |  | Env Vars|   | Widgets |   | Manual  |   | Manual  |  |
 |  | dbutils |   | %run    |   | Print   |   | YAML    |  |
 |  +---------+   +---------+   +---------+   +---------+  |
 +=========================================================+
 ```


 ## Concept 81: Choosing Compute for Jobs

 ### Production Behaviour
 Databricks offers several compute types for jobs. Choosing the right one affects cost, start-up latency, and isolation:

 | Compute Type | Use Case | Startup | Cost |
 |-------------|----------|---------|------|
 | **Serverless Compute** | Default for new jobs; zero management | ~1â€“5 s | Pay-per-second (higher unit cost but no idle) |
 | **Job Cluster** | Ephemeral cluster per job run; single-task | ~2â€“5 min | Pay DBUs while active only |
 | **All-Purpose Cluster** | Interactive development; shared across users/notebooks | Runs continuously | Pay for idle time too |
 | **Instance Pool** | Pre-warmed VMs for fast job cluster creation | ~30â€“60 s | Pay pool idle + job DBUs |
 | **SQL Warehouse** | DBSQL queries & dashboards | ~1â€“2 min | Pay DBUs while active |

 ### Why All-Purpose Is NOT for Production
 - Runs 24/7 by default â€” you pay for idle DBUs
 - Shared by multiple developers â€” resource contention
 - Notebook output interleaving â€” confusing logs
 - No automatic retry on failure
 - Security: all notebooks on the cluster share the same execution context

 ### CE Workaround
 Community Edition gives you **one all-purpose cluster**. We can still demonstrate cluster configuration principles and discuss the API patterns.


 #### 81.1 Job Cluster Configuration (API Representation)
 This is what a well-configured job cluster looks like in the REST API / Jobs UI. You would create this via the "New Job" dialog or Terraform/DABs.


In [ ]:
import json

job_cluster_config = {
    "job_cluster_key": "etl_cluster",
    "new_cluster": {
        "cluster_name": "ETL Job Cluster",
        "spark_version": "14.3.x-scala2.12",
        "node_type_id": "i3.xlarge",
        "num_workers": 2,
        "autoscale": {
            "min_workers": 1,
            "max_workers": 10
        },
        "spark_conf": {
            "spark.databricks.delta.optimizeWrite.enabled": "true",
            "spark.databricks.delta.autoCompact.enabled": "true",
            "spark.sql.adaptive.enabled": "true"
        },
        "custom_tags": {
            "environment": "production",
            "team": "data-engineering",
            "cost_center": "de-2025"
        },
        "init_scripts": [
            {"dbfs": {"destination": "dbfs:/databricks/init/install_deps.sh"}}
        ],
        "spark_env_vars": {
            "ENV": "production",
            "LOG_LEVEL": "WARN"
        },
        "enable_elastic_disk": True,
        "policy_id": None,
        "driver_node_type_id": "i3.xlarge",
        "data_security_mode": "USER_ISOLATION"
    }
}

print("=== PRODUCTION JOB CLUSTER CONFIGURATION ===")
print(json.dumps(job_cluster_config, indent=2))


 #### 81.2 Compute Decision Matrix


In [ ]:
scenarios = [
    {"scenario": "Hourly sensor data ingestion", "answer": "Job Cluster", "reason": "Periodic, predictable, needs isolation"},
    {"scenario": "Ad-hoc data exploration by analysts", "answer": "All-Purpose (or SQL Warehouse)", "reason": "Interactive, unpredictable usage"},
    {"scenario": "100+ micro-jobs every 5 minutes", "answer": "Instance Pool", "reason": "Frequent restarts, pool amortises startup cost"},
    {"scenario": "Simple dashboard refresh on a schedule", "answer": "SQL Warehouse", "reason": "DBSQL is cheaper for SQL-only workloads"},
    {"scenario": "Pipeline with zero DevOps", "answer": "Serverless", "reason": "No cluster management; ideal for small teams"},
    {"scenario": "Development & unit testing", "answer": "All-Purpose", "reason": "Interactive, shared, low barrier"},
]

print(f"{'Scenario':<45} {'Best Compute':<25} {'Reason'}")
print("-" * 100)
for s in scenarios:
    print(f"{s['scenario']:<45} {s['answer']:<25} {s['reason']}")


 ### Key Takeaway â€” Concept 81
 - **Serverless** â€” zero-ops, faster startup, higher per-DBU cost
 - **Job Clusters** â€” ephemeral, isolated, best bang-for-buck for periodic jobs
 - **Instance Pools** â€” warm VMs, faster startup than job clusters, good for many short jobs
 - **All-Purpose** â€” development ONLY; never prod
 - Always tag clusters (environment, team, cost_center) for cost attribution


 ## Concept 82: Secrets Management

 ### Production Behaviour
 Databricks provides **Secret Scopes** to store credentials securely:

 | Scope Type | Backed By | Use Case |
 |-----------|-----------|----------|
 | **Databricks-backed** | Encrypted DB table | Simple, quick |
 | **Azure Key Vault** | AKV | Azure-native, audit log |
 | **AWS Secrets Manager** | AWS SM | AWS-native, rotation |
 | **GCP Secret Manager** | GCP SM | GCP-native |

 ```
   NEVER:  password = "hunter2"
   ALWAYS: password = dbutils.secrets.get(scope="my-scope", key="db-password")
 ```

 ### CE Workaround
 Community Edition may not support full Secret Scopes. Use environment variables as an acceptable fallback.


 #### 82.1 Proper vs Improper Secret Handling


In [ ]:
# ============================================================
# NEVER DO THIS â€” HARDCODED SECRETS
# ============================================================

# BAD: secrets in plain text (would leak to Git, logs, etc.)
# db_password_bad = "super_secure_password_123!"
# api_key_bad = "dapi123456789abcdef"
# connection_string_bad = "Server=tcp:myserver.database.windows.net;User=admin;Password=pass123;"

print("NEVER hardcode secrets in notebooks or scripts.")
print("They would appear in Git history, job run logs, and notebook snapshots.")

# ============================================================
# USE THIS PATTERN â€” SECRETS VIA SCOPE / ENV VAR
# ============================================================

# Method 1: Databricks Secret Scope (production)
# Steps (via CLI or UI):
#   databricks secrets create-scope --scope my-app-scope
#   databricks secrets put --scope my-app-scope --key db-password
#   databricks secrets put --scope my-app-scope --key api-key
#
# Then in code:
# db_password = dbutils.secrets.get(scope="my-app-scope", key="db-password")

# CE Fallback: Environment variable (set via cluster spark_env_vars or init script)
import os

db_password = os.environ.get("DB_PASSWORD", "DEV_DEFAULT_DO_NOT_USE_IN_PROD")
api_key = os.environ.get("API_KEY", "DEV_DEFAULT_DO_NOT_USE_IN_PROD")

print("\nSecrets loaded from environment (CE fallback).")
print(f"  DB_PASSWORD is set: {db_password != 'DEV_DEFAULT_DO_NOT_USE_IN_PROD'}")
print(f"  API_KEY is set: {api_key != 'DEV_DEFAULT_DO_NOT_USE_IN_PROD'}")


 #### 82.2 Simulated Secrets Manager (CE-Compatible)


In [ ]:
class LocalSecretsManager:
    """
    Simulates a Secret Scope for Community Edition.
    In production, this is replaced by dbutils.secrets.get() or Key Vault SDK.
    """
    def __init__(self):
        self._secrets = {
            "database": {"password": "prod-db-pwd-2025", "host": "prod.db.internal", "port": "5432"},
            "api": {"api_key": "sk-prod-abc123def456", "endpoint": "https://api.example.com/v2"},
            "storage": {"account_key": "base64encodedkey==", "account_name": "prodstorageacct"},
            "monitoring": {"pagerduty_key": "pd-integration-key-xyz", "slack_webhook": "https://hooks.slack.com/..."}
        }

    def get(self, scope, key):
        """Same signature as dbutils.secrets.get(scope, key)"""
        if scope in self._secrets and key in self._secrets[scope]:
            return self._secrets[scope][key]
        raise KeyError(f"Secret not found: scope={scope}, key={key}")

    def list_scopes(self):
        return list(self._secrets.keys())

    def list_secrets(self, scope):
        return list(self._secrets.get(scope, {}).keys())

# Usage (identical API to production!)
secret_mgr = LocalSecretsManager()

print("=== AVAILABLE SCOPES ===")
for scope in secret_mgr.list_scopes():
    secrets = secret_mgr.list_secrets(scope)
    print(f"  {scope}: {secrets}")

print("\n=== RETRIEVING SECRETS ===")
db_pwd = secret_mgr.get("database", "password")
db_host = secret_mgr.get("database", "host")
api_key = secret_mgr.get("api", "api_key")

print(f"  Database host: {db_host}")
print(f"  Database password: {'*' * len(db_pwd)}")  # Never print actual secret!
print(f"  API key prefix: {api_key[:4]}...")


 #### 82.3 Secret Rotation Pattern


In [ ]:
import datetime

class SecretRotationManager:
    """Demonstrates the pattern for secret rotation."""

    def __init__(self, secret_manager):
        self.sm = secret_manager
        self.rotation_log = []

    def rotate_secret(self, scope, key, new_value):
        old_value = self.sm.get(scope, key)
        self.sm._secrets[scope][key] = new_value
        self.rotation_log.append({
            "timestamp": datetime.datetime.now().isoformat(),
            "scope": scope,
            "key": key,
            "rotated": True
        })
        print(f"  Rotated {scope}/{key}")

    def get_rotation_history(self):
        return self.rotation_log

rotator = SecretRotationManager(secret_mgr)
print("=== BEFORE ROTATION ===")
print(f"  API key: {secret_mgr.get('api', 'api_key')}")

print("\n=== ROTATING SECRET ===")
rotator.rotate_secret("api", "api_key", "sk-prod-NEW-rotated-key-999")

print("\n=== AFTER ROTATION ===")
print(f"  API key: {secret_mgr.get('api', 'api_key')}")


 ### Key Takeaway â€” Concept 82
 - Never hardcode secrets; use dbutils.secrets.get(scope, key)
 - Use Key Vault-backed scopes for production (audit logs, auto-rotation)
 - Environment variables are an acceptable fallback (CE / local dev)
 - Principle of least privilege: one scope per application
 - Rotate secrets regularly; automate via Key Vault policies


 ## Concept 83: Parameterization: Widgets, Job Parameters, Task Values

 ### Production Behaviour
 Databricks provides three layers of parameterisation:

 | Mechanism | Scope | Persistence | Use Case |
 |-----------|-------|-------------|----------|
 | **Widgets** (dbutils.widgets) | Single notebook | Interactive only | Ad-hoc runs, debugging |
 | **Job Parameters** | Single task | Per run | Scheduled runs from Workflows |
 | **Task Values** | Cross-task | Per run | Pass data between tasks in a job |

 ### CE Availability
 **ALL widget types work fully in Community Edition!** This is the most CE-friendly concept.


 #### 83.1 Creating Interactive Widgets


In [ ]:
dbutils.widgets.removeAll()  # Clean start

# Text widget â€” free-form input
dbutils.widgets.text("source_table", "sales_transactions", "Source Table Name")

# Dropdown widget â€” constrained choices
dbutils.widgets.dropdown("environment", "dev", ["dev", "staging", "production"], "Deployment Environment")

# Dropdown for date (common pattern)
dbutils.widgets.dropdown("run_date", "2025-01-15",
    ["2025-01-01", "2025-01-08", "2025-01-15", "2025-01-22", "2025-01-29"],
    "Processing Date")

# Combobox â€” dropdown + custom value
dbutils.widgets.combobox("region", "US", ["US", "EU", "APAC", "LATAM"], "Region Filter")

# Multiselect â€” select multiple values
dbutils.widgets.multiselect("metrics", "revenue", ["revenue", "cost", "margin", "customers", "orders"], "Metrics to Calculate")

# Slider equivalent (no native slider; use dropdown for numeric thresholds)
dbutils.widgets.dropdown("threshold", "0.05", ["0.01", "0.05", "0.10", "0.15", "0.25", "0.50"], "Alert Threshold (%)")

print("Widgets created! See the widget panel at the top of the notebook.")
print("Change values and re-run cells to see the effect.")


 #### 83.2 Reading Widget Values in Code


In [ ]:
source_table = dbutils.widgets.get("source_table")
environment = dbutils.widgets.get("environment")
run_date = dbutils.widgets.get("run_date")
region = dbutils.widgets.get("region")
metrics_str = dbutils.widgets.get("metrics")
threshold_str = dbutils.widgets.get("threshold")

# Parse multi-select (comma-separated)
selected_metrics = [m.strip() for m in metrics_str.split(",")]
threshold = float(threshold_str)

print("=== CURRENT WIDGET VALUES ===")
print(f"  Source Table : {source_table}")
print(f"  Environment  : {environment}")
print(f"  Run Date     : {run_date}")
print(f"  Region       : {region}")
print(f"  Metrics      : {selected_metrics}")
print(f"  Threshold    : {threshold:.1%}")


 #### 83.3 Widget + Job Parameter Pattern
 In production, job parameters override widget defaults. We simulate this pattern.


In [ ]:
def get_parameter(key, widget_key, default_value):
    """
    Production pattern: job parameters > widget values > hardcoded defaults.

    In Workflows, job parameters are passed as a JSON dict to the task.
    We simulate this with a local dict.
    """
    job_params = {
        "source_table": "transactions_prod",
        "environment": "production",
        "run_date": "2025-01-15",
    }

    if key in job_params:
        return job_params[key]

    try:
        val = dbutils.widgets.get(widget_key)
        return val
    except Exception:
        return default_value

source = get_parameter("source_table", "source_table", "sales_transactions")
env = get_parameter("environment", "environment", "dev")
dt = get_parameter("run_date", "run_date", "2025-01-01")

print("=== RESOLVED PARAMETERS (Job params > Widgets > Defaults) ===")
print(f"  source_table = {source!r}")
print(f"  environment  = {env!r}")
print(f"  run_date     = {dt!r}")


 #### 83.4 Simulated Task Values (Inter-Task Communication)


In [ ]:
import json
import os
import datetime

class TaskValues:
    """
    Simulates dbutils.jobs.taskValues for Community Edition.

    Production API:
      dbutils.jobs.taskValues.set(key="row_count", value=10000)
      dbutils.jobs.taskValues.get(taskKey="upstream_task", key="row_count")
    """
    _store = {}

    @classmethod
    def set(cls, key, value):
        cls._store[key] = value
        print(f"  taskValues.set(key={key!r}, value={value!r})")

    @classmethod
    def get(cls, key, task_key=None, default=None):
        val = cls._store.get(key, default)
        print(f"  taskValues.get(key={key!r}) = {val!r}")
        return val

# --- Simulate Task A: Ingest raw data ---
print("=== TASK A: DATA INGESTION ===")
rows_ingested = 157_832
TaskValues.set("row_count", rows_ingested)
TaskValues.set("source_file", "2025-01-15_sales.csv.gz")
TaskValues.set("status", "success")

# --- Simulate Task B: Transform (reads Task A's values) ---
print("\n=== TASK B: TRANSFORMATION (depends on Task A) ===")
row_count = TaskValues.get("row_count")
source_file = TaskValues.get("source_file")
status = TaskValues.get("status")

if status == "success":
    print(f"  Processing {row_count:,} rows from {source_file}")
    rows_after_dedup = int(row_count * 0.95)
    TaskValues.set("dedup_row_count", rows_after_dedup)
    TaskValues.set("duplicates_removed", row_count - rows_after_dedup)
else:
    print("  Upstream failed; skipping transformation.")

# --- Simulate Task C: Aggregate (reads both A and B) ---
print("\n=== TASK C: AGGREGATION (depends on Task B) ===")
dedup_count = TaskValues.get("dedup_row_count")
duplicates = TaskValues.get("duplicates_removed", default=0)
print(f"  Rows after dedup : {dedup_count:,}")
print(f"  Duplicates removed: {duplicates:,}")

final_aggregates = {"total_revenue": 4_523_910.50, "total_orders": 12_457, "avg_order_value": 363.14}
TaskValues.set("aggregates", json.dumps(final_aggregates))
print(f"  Aggregates: {final_aggregates}")

print("\n=== TASK VALUES STORE ===")
print(json.dumps(TaskValues._store, indent=2, default=str))


 ### Key Takeaway â€” Concept 83
 - **Widgets** â€” interactive parameterisation; all types work in CE
 - **Job Parameters** â€” set once per run, override widgets; production scheduling
 - **Task Values** â€” inter-task data passing in multi-task jobs
 - Resolution order: Job Params > Widgets > Code defaults
 - Widgets are CE's most powerful built-in feature â€” leverage them!


 ## Concept 84: Notebook Development Patterns

 ### Production Behaviour
 Databricks notebooks support powerful compositional patterns:

 | Pattern | Command | Use Case |
 |---------|---------|----------|
 | **%run** | %run /path/to/notebook | Import shared functions/variables inline |
 | **dbutils.notebook.run()** | dbutils.notebook.run(path, timeout, args) | Run a child notebook with parameters, capture exit value |
 | **Magic Commands** | %sql, %python, %fs, %sh, %md | Multi-language in one notebook |
 | **%pip** | %pip install <library> | Notebook-scoped library installation |

 ### CE Availability
 **ALL patterns work fully in Community Edition** â€” %run, dbutils.notebook.run(), magic commands, and %pip.


 #### 84.1 Shared Utility Functions (simulates a %run target)


In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from typing import Dict, List, Optional, Tuple
from datetime import datetime, timedelta

# --- Shared Configuration ---
PROJECT_NAME = "09_Workflows_Demo"
DEFAULT_ENV = "dev"

ENV_CONFIGS = {
    "dev": {"db_prefix": "dev_", "log_level": "DEBUG", "sample_ratio": 1.0},
    "staging": {"db_prefix": "stg_", "log_level": "INFO", "sample_ratio": 0.5},
    "production": {"db_prefix": "", "log_level": "WARN", "sample_ratio": 0.01},
}

def get_env_config(env=DEFAULT_ENV):
    return ENV_CONFIGS.get(env, ENV_CONFIGS["dev"])

# --- Data Quality Utilities ---

def check_null_rates(df, threshold=0.05):
    """Check null rate for every column in a DataFrame."""
    total = df.count()
    if total == 0:
        return {}
    null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).collect()[0]
    return {c: (null_counts[c] / total if null_counts[c] is not None else 0) for c in df.columns}

def check_freshness(partition_col="load_date", max_age_days=1):
    """Check if the latest partition is within acceptable freshness window."""
    latest = datetime(2025, 1, 15)  # Simulated
    cutoff = datetime.now() - timedelta(days=max_age_days)
    is_fresh = latest >= cutoff
    return is_fresh, latest

def validate_row_count(df, expected_min=0, expected_max=None):
    """Validate row count is within expected bounds."""
    actual = df.count()
    result = {"actual": actual, "expected_min": expected_min, "expected_max": expected_max, "passed": actual >= expected_min}
    if expected_max is not None:
        result["passed"] = result["passed"] and (actual <= expected_max)
    return result

def log_quality_check(check_name, passed, details):
    """Centralised logging for data quality checks."""
    status = "PASS" if passed else "FAIL"
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {status}: {check_name}")
    if not passed:
        print(f"    Details: {details}")

print("Utility functions loaded.")
print(f"  Available: check_null_rates, check_freshness, validate_row_count, log_quality_check, get_env_config")


 #### 84.2 %run Pattern â€” Import Shared Utilities
 In production: `%run /Shared/utils/dq_utils`

 This imports all variables and functions into the current notebook's namespace.


In [ ]:
config = get_env_config("dev")
print(f"Environment config: {config}")

print("\nIn a real project you would use:")
print("  %run /Shared/utils/dq_utils")
print("")
print("%run imports ALL variables and functions into the caller's scope.")
print("Use for: shared configs, UDF libraries, DQ utilities.")
print("Avoid for: heavy work that should be isolated.")


 #### 84.3 dbutils.notebook.run() â€” Isolated Execution


In [ ]:
import time

def child_notebook_simulation(source_path, output_path, run_date):
    """
    Simulates a child notebook called via:
    dbutils.notebook.run("/Shared/etl/transform_sales", 600, {"source_path": "..."})
    """
    print(f"[Child Notebook] Starting transformation...")
    print(f"  Source: {source_path}, Output: {output_path}, Date: {run_date}")

    time.sleep(0.5)
    rows_processed = 125_000
    print(f"[Child Notebook] Complete: {rows_processed:,} rows processed.")

    return {
        "rows_processed": rows_processed,
        "status": "success",
        "duration_seconds": 45.2
    }

# --- Orchestrator Pattern ---
print("=" * 60)
print("ORCHESTRATOR NOTEBOOK (Parent)")
print("=" * 60)

result_1 = child_notebook_simulation(
    source_path="s3://datalake/raw/sales/2025/01/15/",
    output_path="s3://datalake/bronze/sales/",
    run_date="2025-01-15"
)

result_2 = child_notebook_simulation(
    source_path="s3://datalake/raw/inventory/2025/01/15/",
    output_path="s3://datalake/bronze/inventory/",
    run_date="2025-01-15"
)

print(f"\n{'=' * 60}")
print("ALL CHILDREN COMPLETE")
print(f"{'=' * 60}")
print(f"  Sales pipeline     : {result_1['status']} ({result_1['rows_processed']:,} rows)")
print(f"  Inventory pipeline : {result_2['status']} ({result_2['rows_processed']:,} rows)")

print(f"\n{'=' * 60}")
print("PATTERN SELECTION GUIDE")
print(f"{'=' * 60}")
print("""
  %run:
    + Shared configuration and helper functions
    + UDF libraries, quick prototyping
    - Cannot pass parameters dynamically
    - Namespace pollution risk

  dbutils.notebook.run():
    + Isolated execution (its own Spark job)
    + Parameter passing via arguments
    + Exit value for downstream decisions
    + Parallel execution possible
    - Overhead of separate notebook run
    - No shared variables
""")


 #### 84.4 Magic Commands Showcase


 ##### %sql â€” Embedded SQL queries


In [ ]:
 %sql
 SELECT '2025-01-15' AS load_date, 157832 AS row_count
 UNION ALL
 SELECT '2025-01-14' AS load_date, 142109 AS row_count
 UNION ALL
 SELECT '2025-01-13' AS load_date, 138456 AS row_count
 ORDER BY load_date DESC


 ##### %fs â€” File system operations


In [ ]:
 %fs ls /databricks-datasets/


 ##### %sh â€” Shell commands on the driver node


In [ ]:
 %sh echo "Driver node: $(hostname)"; echo "Python: $(python --version 2>&1)"


 ##### %pip â€” Notebook-scoped library installation


In [ ]:
import sys
print(f"Python version: {sys.version}")

try:
    import requests
    print(f"requests: {requests.__version__}")
except ImportError:
    print("requests not installed (use: %pip install requests)")

try:
    import pandas as pd
    print(f"pandas: {pd.__version__}")
except ImportError:
    print("pandas not installed")

print("\n%pip install <package> â€” notebook-scoped, no cluster restart")
print("%pip list              â€” show installed packages")


 #### 84.5 Notebook Testing Pattern


In [ ]:
import unittest
from unittest.mock import MagicMock

class TestDataQualityUtils(unittest.TestCase):
    """Unit tests for utility functions (runs within notebook)."""

    def test_get_env_config_dev(self):
        cfg = get_env_config("dev")
        self.assertEqual(cfg["db_prefix"], "dev_")
        self.assertEqual(cfg["log_level"], "DEBUG")

    def test_get_env_config_production(self):
        cfg = get_env_config("production")
        self.assertEqual(cfg["db_prefix"], "")
        self.assertEqual(cfg["log_level"], "WARN")

    def test_get_env_config_unknown_fallback(self):
        cfg = get_env_config("nonexistent")
        self.assertEqual(cfg["db_prefix"], "dev_")

    def test_validate_row_count_pass(self):
        mock_df = MagicMock()
        mock_df.count.return_value = 500
        result = validate_row_count(mock_df, expected_min=100, expected_max=1000)
        self.assertTrue(result["passed"])
        self.assertEqual(result["actual"], 500)

    def test_validate_row_count_fail_min(self):
        mock_df = MagicMock()
        mock_df.count.return_value = 50
        result = validate_row_count(mock_df, expected_min=100)
        self.assertFalse(result["passed"])

    def test_validate_row_count_fail_max(self):
        mock_df = MagicMock()
        mock_df.count.return_value = 2000
        result = validate_row_count(mock_df, expected_min=100, expected_max=1000)
        self.assertFalse(result["passed"])

suite = unittest.TestLoader().loadTestsFromTestCase(TestDataQualityUtils)
runner = unittest.TextTestRunner(verbosity=1)
result = runner.run(suite)

if result.wasSuccessful():
    print("All utility function tests passed!")
else:
    print(f"{len(result.failures)} failures, {len(result.errors)} errors")


 ### Key Takeaway â€” Concept 84
 - %run = inline import (shared namespace); dbutils.notebook.run() = isolated execution (exit value)
 - Magic commands (%sql, %fs, %sh, %pip) make notebooks multi-language
 - %pip installs libraries per-notebook, no cluster restart
 - Test utility functions directly within notebooks (lightweight unit tests)
 - All patterns work in Community Edition â€” great for learning!


 ## Concept 85: Databricks Workflows: Multi-Task Jobs

 ### Production Behaviour
 Databricks Workflows let you define multi-step pipelines with dependencies, retries, and conditional logic:

 | Task Type | Description |
 |-----------|-------------|
 | **Notebook** | Run a Databricks notebook |
 | **Python Script** | Run a .py file (Repo or DBFS) |
 | **Python Wheel** | Run a packaged wheel |
 | **SQL** | Run a DBSQL query or dashboard |
 | **dbt** | Run dbt Core commands |
 | **JAR** | Run a Spark JAR job |
 | **Pipeline** | Trigger a Delta Live Tables pipeline |
 | **For Each** | Dynamic fan-out over an array |

 ```
 WORKFLOW: Daily ETL Pipeline

                 +----------+
                 |  Ingest  |
                 |  Raw     |
                 +----+-----+
                      |
           +----------+----------+
           v          v          v
     +----------+ +----------+ +----------+
     |Transform | |Transform | |Transform |
     |  Sales   | |Inventory | |Customers |  <- Fan-out
     +----+-----+ +----+-----+ +----+-----+
          |            |            |
          +------------+------------+
                       v
                 +----------+
                 |Aggregate |  <- Fan-in
                 |  KPI     |
                 +----+-----+
                      v
                 +----------+
                 |  DQ      |  <- Conditional: only if aggregate succeeds
                 |  Check   |
                 +----------+
 ```

 ### CE Limitation
 Community Edition **cannot create multi-task jobs** (single-task only). We show the architecture conceptually and implement a Python-based orchestrator.


 #### 85.1 Job Definition â€” JSON Structure (What Production Looks Like)


In [ ]:
job_definition = {
    "name": "Daily ETL Pipeline",
    "schedule": {"quartz_cron_expression": "0 0 6 * * ?", "timezone_id": "UTC", "pause_status": "UNPAUSED"},
    "email_notifications": {"on_failure": ["data-eng@company.com"]},
    "timeout_seconds": 7200,
    "max_concurrent_runs": 1,
    "tasks": [
        {
            "task_key": "ingest_raw",
            "description": "Ingest raw data from S3 to Bronze",
            "job_cluster_key": "etl_cluster",
            "notebook_task": {
                "notebook_path": "/Repos/prod/pipelines/ingest_raw",
                "base_parameters": {"source_bucket": "s3://datalake/raw/", "target_table": "bronze.raw_events"}
            },
            "max_retries": 3,
            "min_retry_interval_millis": 60000,
            "timeout_seconds": 1800
        },
        {
            "task_key": "transform_sales",
            "depends_on": [{"task_key": "ingest_raw"}],
            "notebook_task": {"notebook_path": "/Repos/prod/pipelines/transform_sales"},
            "run_if": "ALL_SUCCESS",
            "timeout_seconds": 3600
        },
        {
            "task_key": "transform_inventory",
            "depends_on": [{"task_key": "ingest_raw"}],
            "notebook_task": {"notebook_path": "/Repos/prod/pipelines/transform_inventory"},
            "run_if": "ALL_SUCCESS",
            "timeout_seconds": 3600
        },
        {
            "task_key": "aggregate_kpis",
            "depends_on": [{"task_key": "transform_sales"}, {"task_key": "transform_inventory"}],
            "notebook_task": {"notebook_path": "/Repos/prod/pipelines/aggregate_kpis"},
            "run_if": "ALL_SUCCESS"
        },
        {
            "task_key": "dq_check",
            "depends_on": [{"task_key": "aggregate_kpis"}],
            "notebook_task": {"notebook_path": "/Repos/prod/pipelines/dq_check"},
            "run_if": "ALL_SUCCESS"
        }
    ],
    "parameters": [
        {"name": "run_date", "default": "{{yesterday_dash_n}}"},
        {"name": "processing_mode", "default": "full"}
    ],
    "tags": {"team": "data-engineering", "project": "daily-etl", "environment": "production"},
    "format": "MULTI_TASK"
}

print("=== PRODUCTION JOB DEFINITION (JSON) ===")
print(json.dumps(job_definition, indent=2))


 #### 85.2 CE Workaround: Python Orchestrator with Dependencies


In [ ]:
import time
import random
from datetime import datetime
from enum import Enum
from typing import Dict, List, Callable, Optional
from dataclasses import dataclass, field

class TaskStatus(Enum):
    PENDING = "PENDING"
    RUNNING = "RUNNING"
    SUCCESS = "SUCCESS"
    FAILED = "FAILED"
    SKIPPED = "SKIPPED"

@dataclass
class TaskResult:
    task_key: str
    status: TaskStatus = TaskStatus.PENDING
    start_time: Optional[datetime] = None
    end_time: Optional[datetime] = None
    duration_seconds: float = 0.0
    output: dict = field(default_factory=dict)
    error: Optional[str] = None

class CEWorkflowOrchestrator:
    """
    Simulates a Databricks Workflow orchestrator for Community Edition.
    Supports: dependencies, retries, conditional execution, fan-out/in.
    """
    def __init__(self, name):
        self.name = name
        self.tasks = {}
        self.results = {}
        self.run_id = datetime.now().strftime("%Y%m%d%H%M%S")

    def add_task(self, task_key, func, depends_on=None, max_retries=0, run_if="ALL_SUCCESS", timeout_seconds=3600):
        self.tasks[task_key] = {
            "key": task_key, "func": func,
            "depends_on": depends_on or [], "max_retries": max_retries,
            "run_if": run_if, "timeout_seconds": timeout_seconds,
        }
        self.results[task_key] = TaskResult(task_key=task_key)

    def _all_dependencies_succeeded(self, task_key):
        deps = self.tasks[task_key]["depends_on"]
        return all(self.results[dep].status == TaskStatus.SUCCESS for dep in deps) if deps else True

    def _should_run(self, task_key):
        run_if = self.tasks[task_key]["run_if"]
        if run_if == "ALL_SUCCESS":
            return self._all_dependencies_succeeded(task_key)
        return True

    def _execute_task(self, task_key):
        task = self.tasks[task_key]
        result = TaskResult(task_key=task_key, status=TaskStatus.RUNNING)
        result.start_time = datetime.now()

        for attempt in range(task["max_retries"] + 1):
            try:
                print(f"  [{task_key}] Attempt {attempt + 1}/{task['max_retries'] + 1}")
                output = task["func"](run_id=self.run_id)
                result.status = TaskStatus.SUCCESS
                result.output = output or {}
                break
            except Exception as e:
                if attempt < task["max_retries"]:
                    wait = 2 ** attempt
                    print(f"  [{task_key}] Failed, retrying in {wait}s: {e}")
                    time.sleep(wait)
                else:
                    result.status = TaskStatus.FAILED
                    result.error = str(e)

        result.end_time = datetime.now()
        result.duration_seconds = (result.end_time - result.start_time).total_seconds() if result.start_time else 0
        return result

    def run(self):
        print(f"\n{'=' * 60}")
        print(f"WORKFLOW: {self.name}  |  Run ID: {self.run_id}")
        print(f"{'=' * 60}")

        completed = set()
        failed = set()
        all_keys = set(self.tasks.keys())

        while len(completed) + len(failed) < len(all_keys):
            made_progress = False
            for task_key in all_keys:
                if task_key in completed or task_key in failed:
                    continue
                deps_done = all(dep in completed or dep in failed for dep in self.tasks[task_key]["depends_on"])
                if deps_done:
                    if self._should_run(task_key):
                        print(f"\n>> [{task_key}] Starting...")
                        self.results[task_key] = self._execute_task(task_key)
                        if self.results[task_key].status == TaskStatus.SUCCESS:
                            completed.add(task_key)
                            print(f"  [{task_key}] Completed in {self.results[task_key].duration_seconds:.1f}s")
                        else:
                            failed.add(task_key)
                            print(f"  [{task_key}] Failed: {self.results[task_key].error}")
                    else:
                        self.results[task_key].status = TaskStatus.SKIPPED
                        completed.add(task_key)
                        print(f"  [{task_key}] Skipped")
                    made_progress = True
            if not made_progress:
                break

        print(f"\n{'=' * 60}")
        print("WORKFLOW SUMMARY")
        print(f"{'=' * 60}")
        print(f"{'Task':<25} {'Status':<12} {'Duration':>10}")
        print("-" * 50)
        for key in self.tasks:
            r = self.results[key]
            dur_str = f"{r.duration_seconds:.1f}s" if r.duration_seconds else "--"
            print(f"{key:<25} {r.status.value:<12} {dur_str:>10}")
        return self.results

# --- Define Tasks ---

def task_ingest_raw(**kwargs):
    time.sleep(0.3)
    return {"rows_ingested": 157_832, "files_processed": 12}

def task_transform_sales(**kwargs):
    time.sleep(0.5)
    return {"sales_rows": 89_421, "revenue": 4_523_910.50}

def task_transform_inventory(**kwargs):
    time.sleep(0.4)
    return {"inventory_rows": 45_210, "skus": 3_450}

def task_aggregate_kpis(**kwargs):
    time.sleep(0.3)
    return {"total_revenue": 4_523_910.50, "total_orders": 12_457, "margin_pct": 34.2}

def task_dq_check(**kwargs):
    time.sleep(0.2)
    return {"all_checks_passed": True, "checks_run": 8}

def task_publish_dashboard(**kwargs):
    time.sleep(0.1)
    return {"dashboard_refreshed": True, "viewers": 42}

# --- Build and Run Workflow ---

orchestrator = CEWorkflowOrchestrator("Daily ETL Pipeline (CE Simulated)")

orchestrator.add_task("ingest_raw", task_ingest_raw)
orchestrator.add_task("transform_sales", task_transform_sales, depends_on=["ingest_raw"], max_retries=2)
orchestrator.add_task("transform_inventory", task_transform_inventory, depends_on=["ingest_raw"], max_retries=2)
orchestrator.add_task("aggregate_kpis", task_aggregate_kpis, depends_on=["transform_sales", "transform_inventory"])
orchestrator.add_task("dq_check", task_dq_check, depends_on=["aggregate_kpis"])
orchestrator.add_task("publish_dashboard", task_publish_dashboard, depends_on=["dq_check"])

results = orchestrator.run()


 #### 85.3 Scheduling Concepts (Trigger Types)


In [ ]:
print("=== JOB TRIGGER TYPES (Production) ===\n")

triggers = [
    {"type": "Scheduled", "example": "Cron: 0 0 6 * * ?", "ce": "No (single-task only)"},
    {"type": "File Arrival", "example": "Trigger on s3://.../*.csv", "ce": "No"},
    {"type": "Continuous", "example": "Streaming pipeline", "ce": "No"},
    {"type": "Manual (Run Now)", "example": "Click 'Run Now' in UI", "ce": "Yes"},
    {"type": "API Trigger", "example": "POST .../jobs/run-now", "ce": "Limited"},
    {"type": "Webhook", "example": "GitHub push, Slack command", "ce": "No"},
]

print(f"{'Trigger':<18} {'CE':<12} {'Example'}")
print("-" * 55)
for t in triggers:
    print(f"{t['type']:<18} {t['ce']:<12} {t['example']}")


 ### Key Takeaway â€” Concept 85
 - Multi-task Workflows are the core of production orchestration
 - Task types: Notebook, Python Script, Python Wheel, SQL, dbt, JAR, Pipeline
 - Fan-out (parallel tasks) and fan-in (aggregation after parallel)
 - CE cannot create multi-task jobs â€” use our Python orchestrator for learning
 - Job definitions can be JSON (API) or YAML (DABs) format
 - Triggers: Scheduled, File Arrival, Continuous, Manual, API, Webhook


 ## Concept 86: Task Dependencies, Retries & Conditional Logic

 ### Production Behaviour
 Each task in a Workflow supports fine-grained retry and conditional execution:

 | Feature | Options | Description |
 |---------|---------|-------------|
 | **max_retries** | 0â€“5 | Number of retry attempts on failure |
 | **min_retry_interval_millis** | >= 0 | Minimum wait between retries |
 | **retry_on_timeout** | true/false | Whether timeouts trigger retries |
 | **timeout_seconds** | 0â€“259200 | Max task runtime before forced termination |
 | **run_if** | ALL_SUCCESS, ALL_DONE, AT_LEAST_ONE_SUCCESS, NONE_FAILED | Conditional execution |
 | **depends_on** | List of task keys | Upstream dependencies |

 ```
 RETRY BEHAVIOUR WITH EXPONENTIAL BACKOFF:

 Attempt 1 (t=0s)  --FAIL-->
   wait 60s
 Attempt 2 (t=60s) --FAIL-->
   wait 120s
 Attempt 3 (t=180s) --SUCCESS--> (done)

 If max_retries exhausted -> task fails -> downstream runs/skips per run_if
 ```


 #### 86.1 Retry Decorator (CE Implementation)


In [ ]:
import functools
import time

def retry_with_backoff(max_retries=3, base_delay=1.0, backoff_factor=2,
                       retry_on_exceptions=(Exception,)):
    """
    Production-grade retry decorator for Community Edition.
    Emulates the retry behaviour of Databricks Workflow tasks.

    Args:
        max_retries: Maximum retry attempts (total = 1 + max_retries)
        base_delay: Initial delay in seconds between retries
        backoff_factor: Multiplier for each subsequent retry (2 = exponential)
        retry_on_exceptions: Tuple of exception types to retry on

    Usage:
        @retry_with_backoff(max_retries=3, base_delay=30)
        def my_etl_task():
            ...
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(max_retries + 1):
                try:
                    if attempt > 0:
                        delay = base_delay * (backoff_factor ** (attempt - 1))
                        print(f"    Retry {attempt}/{max_retries} in {delay:.1f}s...")
                        time.sleep(delay)
                    result = func(*args, **kwargs)
                    if attempt > 0:
                        print(f"    Succeeded on attempt {attempt + 1}!")
                    return result
                except retry_on_exceptions as e:
                    last_exception = e
                    print(f"    Attempt {attempt + 1} failed: {e}")
            raise RuntimeError(
                f"Task '{func.__name__}' failed after {max_retries + 1} attempts. "
                f"Last error: {last_exception}"
            )
        return wrapper
    return decorator

# --- Demonstrate ---
call_count = {"value": 0}

@retry_with_backoff(max_retries=3, base_delay=0.1, backoff_factor=2,
                    retry_on_exceptions=(ValueError, ConnectionError))
def flaky_api_call(should_fail_times=5):
    call_count["value"] += 1
    if call_count["value"] <= should_fail_times:
        raise ConnectionError(f"Transient error (call #{call_count['value']})")
    return {"status": "success", "attempt": call_count["value"]}

print("=== RETRY DECORATOR DEMO ===\n")

print("1. Task that succeeds on 3rd attempt:")
try:
    result = flaky_api_call(should_fail_times=2)
    print(f"   Result: {result}")
except Exception as e:
    print(f"   {e}")

call_count["value"] = 0

print("\n2. Task that never succeeds:")
try:
    result = flaky_api_call(should_fail_times=10)
except RuntimeError as e:
    print(f"   Gracefully handled: {e}")


 #### 86.2 Conditional Execution Patterns (run_if)


In [ ]:
print("=== RUN_IF CONDITION DEMONSTRATION ===\n")

# Simulate upstream results
upstream = {
    "ingest_raw": True,
    "transform_customers": False,
    "transform_sales": True,
}

conditions = {
    "ALL_SUCCESS": all(upstream.values()),
    "ALL_DONE": True,
    "AT_LEAST_ONE_SUCCESS": any(upstream.values()),
    "NONE_FAILED": not any(not v for v in upstream.values()),
}

print(f"Upstream status: {upstream}\n")
print(f"{'Condition':<25} {'Result':<10} {'Runs?'}")
print("-" * 55)
for cond, result in conditions.items():
    print(f"{cond:<25} {str(result):<10} {'RUN' if result else 'SKIP'}")

print(f"\n{'=' * 60}")
print("PRODUCTION SCENARIOS")
print(f"{'=' * 60}")
print("""
  ALL_SUCCESS        -> DQ checks after ETL (all stages must pass)
  ALL_DONE           -> Cleanup after pipeline (run regardless of outcome)
  AT_LEAST_ONE_SUCCESS -> Fallback data source (primary or backup is enough)
  NONE_FAILED        -> Notify if smooth (alert only when everything works)
""")


 #### 86.3 Alerting Patterns (Email & Webhook Simulation)


In [ ]:
import json
import datetime

class AlertManager:
    """
    Simulates production alerting (PagerDuty, Slack, Email).

    Production:
      - Email notifications configured in Job definition
      - Webhook destinations for Slack, PagerDuty, Teams
      - SQL Alerts for threshold-based notifications
    """

    def __init__(self):
        self.alerts_sent = []

    def send_email(self, subject, body, recipients):
        self.alerts_sent.append({"type": "EMAIL", "timestamp": datetime.datetime.now().isoformat(), "subject": subject})
        print(f"EMAIL -> {recipients} | {subject}")

    def send_slack(self, channel, message, severity="info"):
        emoji = {"info": "(i)", "warning": "(!)", "critical": "(!!)"}.get(severity, "(i)")
        self.alerts_sent.append({"type": "SLACK", "channel": channel, "severity": severity, "message": message})
        print(f"{emoji} SLACK -> #{channel} | [{severity.upper()}] {message}")

    def send_pagerduty(self, service, summary, severity="error"):
        self.alerts_sent.append({"type": "PAGERDUTY", "service": service, "summary": summary})
        print(f"PAGERDUTY -> {service} | {summary}")

    def summary(self):
        print(f"\n{'=' * 50}")
        print(f"ALERT SUMMARY: {len(self.alerts_sent)} alerts sent")
        print(f"{'=' * 50}")
        for a in self.alerts_sent:
            print(f"  [{a['timestamp']}] {a['type']} {a.get('subject', a.get('message', ''))[:60]}")

alerts = AlertManager()

print("=== PRODUCTION ALERTING PATTERNS ===\n")

alerts.send_slack("data-alerts", "ETL job 'daily_sales' started (Run #4521)", "info")
alerts.send_slack("data-alerts", "DQ Check FAILED: null_rate > threshold (8.2% > 5%)", "critical")
alerts.send_pagerduty("Data Pipelines", "DQ failure in daily_sales: null_rate exceeded threshold")
alerts.send_email("ETL Job Failed: daily_sales", "DQ check failed. See logs.", ["data-eng@company.com"])
alerts.send_slack("data-alerts", "ETL job 'daily_sales' completed (after retry)", "info")

alerts.summary()


 ### Key Takeaway â€” Concept 86
 - **Retries** with exponential backoff handle transient failures gracefully
 - **run_if** conditions enable sophisticated pipeline branching
 - **Task Values** pass data between tasks (demonstrated in Section 83.4)
 - **Alerting** integrates with Email, Slack, PagerDuty, Webhooks
 - CE can simulate all of these patterns in Python


 ## Concept 87: Git Integration & Repos

 ### Production Behaviour
 Databricks Repos provides first-class Git integration for version-controlled notebook development:

 | Feature | Description |
 |---------|-------------|
 | **Git Providers** | GitHub, GitLab, Bitbucket, Azure DevOps, AWS CodeCommit |
 | **Repo Structure** | Notebooks, Python files, YAML configs all in one repo |
 | **Branching** | Main, feature branches, PR-based workflow |
 | **Sync** | Pull latest, create branch from repo, merge |
 | **CI/CD** | Trigger jobs on PR, run tests on push |

 ```
 REPOS DIRECTORY LAYOUT:

 databricks-project/
 |-- src/ingest/ingest_raw.py
 |-- src/transform/transform_sales.py
 |-- src/transform/transform_inventory.py
 |-- src/aggregate/kpi_aggregates.py
 |-- tests/test_ingest.py
 |-- tests/test_transform.py
 |-- config/dev_config.yaml
 |-- config/prod_config.yaml
 |-- databricks.yml          (DAB definition)
 |-- .github/workflows/deploy.yml  (CI/CD)
 |-- README.md
 ```

 ### CE Limitation
 Community Edition has **limited Git support** (no Repos API). We demonstrate the patterns and best practices conceptually.


 #### 87.1 Branching Strategy for Notebook Development


In [ ]:
branching_strategy = """
GIT BRANCHING STRATEGY FOR DATABRICKS PROJECTS

  main ------*----------------------------*--- (v1.1.0) ---*-- (v1.2.0)
             |                            |                 |
             +-- develop -*--*--*------*------------------*--
                          |  |  |      |
                          |  |  |      +-- release/v1.1 --*-- merge back
                          |  |  |
                          |  |  +-- feature/add-dq   --*-- PR --> merge to develop
                          |  |
                          |  +-- feature/new-metrics --*-- PR --> merge to develop
                          |
                          +-- hotfix/critical-bug --*-- PR --> merge to main + develop

  WORKFLOW:
  1. Create feature branch from develop
  2. Develop in Databricks Repos (auto-syncs to feature branch)
  3. Open PR in GitHub -> trigger CI checks (lint, unit tests)
  4. Code review -> approve -> merge to develop
  5. Deploy from develop to staging workspace (DABs deploy)
  6. Create release branch -> deploy to production
  7. Merge release branch to main + back to develop

  PROTECTION RULES (main branch):
    - Require pull request reviews (>= 1 approval)
    - Require status checks to pass (CI)
    - Require branches to be up to date
    - No direct pushes to main
"""

print(branching_strategy)


 #### 87.2 CI/CD Pipeline Concept (GitHub Actions)


In [ ]:
github_actions_yaml = """
# .github/workflows/databricks-ci.yml
# PRODUCTION CI/CD PIPELINE FOR DATABRICKS NOTEBOOKS

name: Databricks CI/CD Pipeline

on:
  pull_request:
    branches: [main, develop]
  push:
    branches: [develop]

jobs:
  lint-and-test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - name: Lint with flake8
        run: |
          pip install flake8
          flake8 src/ tests/ --max-line-length=120
      - name: Run unit tests
        run: |
          pip install -r requirements.txt
          pytest tests/ -v --cov=src/ --cov-report=xml

  deploy-staging:
    needs: lint-and-test
    if: github.event_name == 'push' && github.ref == 'refs/heads/develop'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Install Databricks CLI
        uses: databricks/setup-cli@main
      - name: Deploy to Staging
        run: |
          databricks bundle validate -t staging
          databricks bundle deploy -t staging
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_STAGING_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_STAGING_TOKEN }}

  deploy-production:
    needs: lint-and-test
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    environment: production
    steps:
      - uses: actions/checkout@v4
      - name: Install Databricks CLI
        uses: databricks/setup-cli@main
      - name: Deploy to Production
        run: |
          databricks bundle validate -t production
          databricks bundle deploy -t production
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_PROD_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_PROD_TOKEN }}
"""

print("=== CI/CD PIPELINE (GitHub Actions) ===")
print(github_actions_yaml)


 #### 87.3 PR Review Checklist Simulation


In [ ]:
class PRReviewChecklist:
    """Simulates a pull request review process for notebook changes."""

    def __init__(self, pr_title, author, changed_files):
        self.pr_title = pr_title
        self.author = author
        self.changed_files = changed_files
        self.checks = self._run_automated_checks()

    def _run_automated_checks(self):
        checks = [
            ("No hardcoded secrets", self._check_no_secrets()),
            ("Widgets have defaults", (True, "OK")),
            ("Tests added for new logic", self._check_tests_exist()),
            ("Notebooks have Markdown descriptions", (True, "OK")),
            ("Cluster tags configured", (True, "OK")),
        ]
        return checks

    def _check_no_secrets(self):
        for f in self.changed_files:
            if "password" in f.get("content", "").lower() and "dbutils.secrets" not in f.get("content", ""):
                return False, f"Hardcoded password in {f['name']}"
        return True, "No hardcoded secrets found"

    def _check_tests_exist(self):
        has_test = any("test_" in f["name"] for f in self.changed_files)
        if not has_test:
            return False, "No test files added"
        return True, "Tests found"

    def print_report(self):
        print(f"\n{'=' * 60}")
        print(f"PR REVIEW: {self.pr_title}")
        print(f"Author: {self.author}  |  Files: {len(self.changed_files)}")
        print(f"{'=' * 60}")
        print(f"{'Check':<35} {'Status':<10} {'Details'}")
        print("-" * 60)
        all_pass = True
        for name, (passed, detail) in self.checks:
            icon = "PASS" if passed else "FAIL"
            if not passed:
                all_pass = False
            print(f"{name:<35} {icon:<10} {detail}")
        print("-" * 60)
        print(f"OVERALL: {'APPROVED' if all_pass else 'CHANGES REQUESTED'}")

pr = PRReviewChecklist(
    pr_title="feat: Add data quality checks to ETL pipeline",
    author="data-engineer-1",
    changed_files=[
        {"name": "src/transform/dq_checks.py", "content": "def check_nulls(df): return df.count()"},
        {"name": "tests/test_dq_checks.py", "content": "def test_check_nulls(): assert True"},
        {"name": "src/transform/legacy_script.py", "content": "password = 'hardcoded123'"},
        {"name": "config/dev_config.yaml", "content": "threshold: 0.05"},
    ]
)
pr.print_report()


 ### Key Takeaway â€” Concept 87
 - Use Databricks Repos for Git-backed notebook development (not /Users/ folders)
 - Structured directory layout: src/, tests/, config/
 - Branch strategy: Main (prod) <- Develop (integration) <- Feature branches
 - PR workflow: automated lint + tests -> code review -> merge -> deploy
 - CE has limited Git â€” practice the structure and workflows conceptually


 ## Concept 88: Monitoring, Alerting & SQL Alerts

 ### Production Behaviour
 Databricks provides several layers of monitoring:

 | Mechanism | Scope | Use Case |
 |-----------|-------|----------|
 | **SQL Alerts** | DBSQL queries | Threshold-based: e.g., 0 rows ingested |
 | **Job Run Notifications** | Per-job | Email on start/success/failure |
 | **Webhook Destinations** | Account-wide | Slack, PagerDuty, Teams, custom HTTP |
 | **System Tables** | Account-wide | Query job/cluster/audit history |
 | **Cluster Event Logs** | Per-cluster | Spark events, GC logs, executor metrics |
 | **Grafana Integration** | Workspace | Custom dashboards from system tables |

 ### CE Limitation
 No DBSQL Alerts, no webhook destinations. We build manual monitoring and alerting in Python.


 #### 88.1 Data Freshness Monitor (CE Implementation)


In [ ]:
import datetime
import random
from dataclasses import dataclass

@dataclass
class FreshnessCheck:
    table_name: str
    passed: bool
    details: str

class DataPipelineMonitor:
    """
    Production-equivalent monitoring system.
    In production, these checks would be SQL Alerts in DBSQL.
    """

    def __init__(self):
        self.checks_run = []

    def check_freshness(self, table_name, expected_latest, actual_latest, max_delay_hours=4):
        delay_hours = (expected_latest - actual_latest).total_seconds() / 3600
        check = FreshnessCheck(
            table_name=table_name,
            passed=delay_hours <= max_delay_hours,
            details=f"Delay: {delay_hours:.1f}h (max: {max_delay_hours}h)"
        )
        self.checks_run.append(check)
        return check

    def check_null_rate(self, table_name, column, null_rate, threshold=0.05):
        check = FreshnessCheck(
            table_name=table_name,
            passed=null_rate <= threshold,
            details=f"Null rate {null_rate:.2%} for {column} (threshold: {threshold:.2%})"
        )
        self.checks_run.append(check)
        return check

    def check_volume(self, table_name, actual_rows, expected_min, expected_max=None):
        passed = actual_rows >= expected_min
        if expected_max is not None:
            passed = passed and (actual_rows <= expected_max)
        rng = f"{expected_min:,}-{expected_max:,}" if expected_max else f">={expected_min:,}"
        check = FreshnessCheck(
            table_name=table_name,
            passed=passed,
            details=f"Row count {actual_rows:,} (expected: {rng})"
        )
        self.checks_run.append(check)
        return check

    def run_all_checks(self):
        now = datetime.datetime(2025, 1, 15, 8, 0, 0)
        self.check_freshness("gold.daily_sales", now, now - datetime.timedelta(hours=2), 4)
        self.check_freshness("gold.daily_inventory", now, now - datetime.timedelta(hours=6), 4)
        self.check_freshness("gold.customer_360", now, now - datetime.timedelta(hours=1), 4)
        self.check_null_rate("silver.transactions", "customer_id", 0.02, 0.05)
        self.check_null_rate("silver.transactions", "product_sku", 0.08, 0.05)
        self.check_null_rate("silver.orders", "delivery_date", 0.01, 0.05)
        self.check_volume("bronze.raw_events", 157_832, 100_000, 200_000)
        self.check_volume("gold.daily_sales", 12_457, 1_000)

monitor = DataPipelineMonitor()
monitor.run_all_checks()

print("=" * 60)
print("DATA PIPELINE HEALTH REPORT")
print("=" * 60)

print(f"\n{'Check':<45} {'Status':<8} {'Details'}")
print("-" * 80)
passed_count = 0
for check in monitor.checks_run:
    icon = "PASS" if check.passed else "FAIL"
    if check.passed:
        passed_count += 1
    print(f"{check.table_name:<45} {icon:<8} {check.details}")

print("-" * 80)
total = len(monitor.checks_run)
print(f"\nHEALTH SCORE: {passed_count}/{total} passed ({(passed_count/total*100):.1f}%)")
if passed_count < total:
    print(f"{total - passed_count} alerts generated - review failed checks above.")
else:
    print("All checks passed - pipeline is healthy.")


 #### 88.2 Simulated SQL Alert (What Production Looks Like)


In [ ]:
sql_alert_definition = {
    "name": "Stale Data Alert: gold.daily_sales",
    "description": "Triggers if daily_sales hasn't been updated in 4+ hours",
    "query": """
        SELECT
            CASE WHEN MAX(load_timestamp) < DATEADD(HOUR, -4, CURRENT_TIMESTAMP())
                 THEN 'STALE' ELSE 'FRESH'
            END AS data_status,
            MAX(load_timestamp) AS last_load,
            DATEDIFF(HOUR, MAX(load_timestamp), CURRENT_TIMESTAMP()) AS hours_since_update
        FROM gold.daily_sales
        WHERE load_date = CURRENT_DATE()
    """,
    "condition": {"column": "data_status", "op": "==", "value": "STALE"},
    "schedule": {"quartz_cron_expression": "0 0/15 * * * ?", "timezone_id": "UTC"},
    "notification": {
        "destinations": [
            {"type": "EMAIL", "addresses": ["data-eng@company.com"]},
            {"type": "SLACK", "channel": "#data-alerts"},
            {"type": "PAGERDUTY", "integration_key": "secret://pd-key"}
        ]
    },
    "state": "ENABLED"
}

print("=== SIMULATED SQL ALERT DEFINITION (DBSQL) ===\n")
print(json.dumps(sql_alert_definition, indent=2))
print("\nIn Community Edition, this exact alert cannot be created.")
print("Use the Python monitor above + cron/scheduler instead.")


 #### 88.3 Alert Severity Classification


In [ ]:
class AlertRouter:
    """Routes alerts to the right team based on severity and domain."""

    ROUTING_TABLE = {
        "data_freshness": {
            "P1_CRITICAL": ["pagerduty://data-oncall", "slack://#data-critical"],
            "P2_HIGH": ["slack://#data-alerts", "email://data-eng@company.com"],
            "P3_MEDIUM": ["email://data-eng@company.com"],
            "P4_LOW": [],
        },
        "pipeline_failure": {
            "P1_CRITICAL": ["pagerduty://data-oncall"],
            "P2_HIGH": ["slack://#data-alerts"],
            "P3_MEDIUM": ["email://data-eng@company.com"],
            "P4_LOW": [],
        },
        "cost_anomaly": {
            "P1_CRITICAL": ["pagerduty://platform-oncall"],
            "P2_HIGH": ["slack://#platform-alerts"],
            "P3_MEDIUM": ["email://platform@company.com"],
            "P4_LOW": [],
        },
    }

    @staticmethod
    def route(domain, severity, message):
        channels = AlertRouter.ROUTING_TABLE.get(domain, {}).get(severity, [])
        if channels:
            print(f"[{severity[:2]}] {domain}: {message}")
            for ch in channels:
                print(f"    -> {ch}")
        else:
            print(f"[{severity[:2]}] {domain}: {message} (logged only)")
        return channels

print("=== ALERT SEVERITY CLASSIFICATION ===\n")

print(f"  P1 CRITICAL â€” Service Down (immediate response, 24/7 on-call)")
print(f"  P2 HIGH     â€” Degraded (respond within 30 min)")
print(f"  P3 MEDIUM   â€” Minor Issue (respond within 4 hours)")
print(f"  P4 LOW      â€” Informational (no SLA)")

print(f"\n=== ROUTING EXAMPLES ===\n")
AlertRouter.route("data_freshness", "P1_CRITICAL", "gold.daily_sales last updated 6 hours ago")
print()
AlertRouter.route("pipeline_failure", "P2_HIGH", "Task 'transform_sales' failed after 3 retries")
print()
AlertRouter.route("cost_anomaly", "P3_MEDIUM", "DBU usage 30% above 7-day average")


 ### Key Takeaway â€” Concept 88
 - SQL Alerts are the primary threshold-based notification mechanism in production
 - Classify alerts: P1 (critical) to P4 (informational)
 - Route alerts by domain and severity to appropriate channels
 - CE: implement monitoring functions in Python; simulate alert routing
 - Production: SQL Alerts -> Webhook Destinations -> PagerDuty/Slack/Email


 ## Concept 89: System Tables for Operations

 ### Production Behaviour
 System tables provide operational data for monitoring, cost tracking, and auditing:

 | Schema | Key Tables | Purpose |
 |--------|-----------|---------|
 | **system.billing** | usage, list_prices | Cost tracking, DBU consumption |
 | **system.compute** | clusters, node_timeline | Cluster lifecycle, auto-scaling events |
 | **system.jobs** | job_runs, task_runs | Job execution history, durations, failures |
 | **system.access** | audit | Access logs for compliance |
 | **system.storage** | blob_usage | Storage consumption |

 ### CE Limitation
 System tables are not available in Community Edition. We build our own operational tracking.


 #### 89.1 Custom Operational Tracking Tables


In [ ]:
import datetime
import random
import uuid
from collections import defaultdict

class OperationalDB:
    """Simulates system tables for Community Edition."""

    def __init__(self):
        self.job_runs = []
        self.cost_records = []

    def record_job_run(self, job_name, task_name, status, duration_seconds, run_date=None):
        self.job_runs.append({
            "run_id": str(uuid.uuid4())[:8],
            "job_name": job_name,
            "task_name": task_name,
            "status": status,
            "start_time": (run_date or datetime.datetime.now()) - datetime.timedelta(seconds=duration_seconds),
            "end_time": run_date or datetime.datetime.now(),
            "duration_seconds": duration_seconds,
            "run_date": (run_date or datetime.datetime.now()).strftime("%Y-%m-%d"),
        })

    def record_cost(self, sku, dbus, cost_usd, usage_date):
        self.cost_records.append({
            "usage_date": usage_date,
            "sku_name": sku,
            "dbus_consumed": dbus,
            "cost_usd": cost_usd,
        })

# --- Generate Synthetic Data ---
ops_db = OperationalDB()
base_date = datetime.datetime(2025, 1, 15)

for day_offset in range(30, 0, -1):
    run_date = base_date - datetime.timedelta(days=day_offset)
    run_date_str = run_date.strftime("%Y-%m-%d")

    ingest_status = random.choices(["SUCCESS", "FAILED"], weights=[0.92, 0.08])[0]
    ops_db.record_job_run("Daily ETL", "ingest_raw", ingest_status, random.uniform(120, 600), run_date)

    if ingest_status == "SUCCESS":
        for task in ["transform_sales", "transform_inventory"]:
            ops_db.record_job_run("Daily ETL", task,
                random.choices(["SUCCESS", "FAILED"], weights=[0.95, 0.05])[0],
                random.uniform(180, 900), run_date)
        ops_db.record_job_run("Daily ETL", "aggregate_kpis", "SUCCESS", random.uniform(60, 300), run_date)

    ops_db.record_cost("JOBS_COMPUTE", random.uniform(10, 80), random.uniform(4, 32), run_date_str)
    ops_db.record_cost("ALL_PURPOSE_COMPUTE", random.uniform(5, 40), random.uniform(2.75, 22), run_date_str)

print(f"Generated: {len(ops_db.job_runs)} job runs, {len(ops_db.cost_records)} cost records")


 #### 89.2 Operational Dashboard Queries


In [ ]:
from collections import Counter

print("=" * 70)
print("OPERATIONAL DASHBOARD")
print("=" * 70)

# --- Widget 1: Job Success Rate ---
print("\nJOB EXECUTION SUMMARY (last 30 days)")
print("-" * 70)

total_runs = len(ops_db.job_runs)
success_runs = sum(1 for r in ops_db.job_runs if r["status"] == "SUCCESS")
failed_runs = sum(1 for r in ops_db.job_runs if r["status"] == "FAILED")
avg_duration = sum(r["duration_seconds"] for r in ops_db.job_runs) / total_runs if total_runs else 0

print(f"  Total runs  : {total_runs}")
print(f"  Successes   : {success_runs} ({success_runs/total_runs*100:.1f}%)")
print(f"  Failures    : {failed_runs} ({failed_runs/total_runs*100:.1f}%)")
print(f"  Avg duration: {avg_duration/60:.1f} minutes")

# --- Widget 2: Per-Task Performance ---
print(f"\nTASK-LEVEL PERFORMANCE")
print("-" * 70)

task_stats = defaultdict(lambda: {"runs": 0, "success": 0, "total_dur": 0})
for r in ops_db.job_runs:
    t = r["task_name"]
    task_stats[t]["runs"] += 1
    if r["status"] == "SUCCESS":
        task_stats[t]["success"] += 1
    task_stats[t]["total_dur"] += r["duration_seconds"]

print(f"{'Task':<25} {'Runs':>6} {'Success %':>10} {'Avg Dur (s)':>12}")
print("-" * 56)
for task, stats in sorted(task_stats.items()):
    pct = stats["success"] / stats["runs"] * 100
    avg = stats["total_dur"] / stats["runs"]
    print(f"{task:<25} {stats['runs']:>6} {pct:>9.1f}% {avg:>11.0f}")

# --- Widget 3: Daily Cost Trend ---
print(f"\nDAILY COST TREND (last 7 days)")
print("-" * 70)

daily_costs = defaultdict(lambda: defaultdict(float))
for rec in ops_db.cost_records:
    daily_costs[rec["usage_date"]][rec["sku_name"]] += rec["cost_usd"]

dates = sorted(daily_costs.keys())[-7:]
print(f"{'Date':<12} {'Jobs':>10} {'All-Purpose':>12} {'Total':>10}")
print("-" * 46)
for d in dates:
    jobs = daily_costs[d].get("JOBS_COMPUTE", 0)
    ap = daily_costs[d].get("ALL_PURPOSE_COMPUTE", 0)
    print(f"{d:<12} ${jobs:>9.2f} ${ap:>11.2f} ${jobs+ap:>9.2f}")

# --- Widget 4: Threshold Check ---
print(f"\nAUTOMATED THRESHOLD CHECK")
print("-" * 70)

failure_rate = failed_runs / total_runs * 100 if total_runs else 0
if failure_rate > 10.0:
    print(f"  WARNING: Failure rate ({failure_rate:.1f}%) exceeds threshold (10%)!")
else:
    print(f"  Failure rate ({failure_rate:.1f}%) is within acceptable range (10%)")


 #### 89.3 Cost Attribution & Showback


In [ ]:
print("=" * 60)
print("COST ATTRIBUTION SUMMARY (Monthly Estimate)")
print("=" * 60)

monthly_costs = [
    ("Production ETL Jobs", 1420, 568.00, 45.2),
    ("Data Science (All-Purpose)", 680, 374.00, 29.8),
    ("DBSQL Dashboards", 320, 176.00, 14.0),
    ("Dev/Test Environments", 250, 137.50, 11.0),
]

print(f"\n{'Category':<30} {'DBUs':>8} {'Cost (USD)':>12} {'Share':>8}")
print("-" * 60)
total_cost = 0
for name, dbus, cost, pct in monthly_costs:
    print(f"{name:<30} {dbus:>8,.0f} ${cost:>11,.2f} {pct:>7.1f}%")
    total_cost += cost
print("-" * 60)
print(f"{'TOTAL':<30} {'':>8} ${total_cost:>11,.2f} {'100.0%':>8}")

print(f"\nCost management tips:")
print(f"  1. Tag all clusters with cost_center and team")
print(f"  2. Query system.billing.usage daily for anomalies")
print(f"  3. Set per-user DBU budgets with alert thresholds")
print(f"  4. Use Serverless for variable workloads, Job Clusters for predictable")
print(f"  5. Auto-terminate idle all-purpose clusters (< 60 min)")


 ### Key Takeaway â€” Concept 89
 - System tables are the operational backbone of a production Databricks workspace
 - system.billing.usage -> track costs; system.compute.clusters -> cluster lifecycle; system.jobs.job_runs -> job history
 - Build operational dashboards in DBSQL querying these tables
 - CE: create your own operational tracking (even if in-memory or CSV)
 - Tag everything for cost attribution (team, project, environment)


 ## Concept 90: Databricks Asset Bundles (DABs)

 ### Production Behaviour
 Databricks Asset Bundles (DABs) bring infrastructure-as-code to the Databricks platform:

 | Concept | Description |
 |---------|-------------|
 | **databricks.yml** | Single file defining all resources (jobs, pipelines, models, schemas) |
 | **Bundle** | A collection of resources deployed together to a target |
 | **Targets** | Environment-specific overrides (dev, staging, production) |
 | **Variables** | Parameterisation across environments |
 | **bundle validate** | Validate YAML syntax and resource references |
 | **bundle deploy** | Deploy resources to a target workspace |
 | **bundle run** | Run a job or pipeline defined in the bundle |
 | **bundle destroy** | Remove deployed resources |

 ```
 DAB: INFRASTRUCTURE-AS-CODE

   databricks.yml                     Workspace
   +------------------+              +------------------+
   | resources:       |   deploy     | Jobs:            |
   |   jobs:          | ---------->  |   +- daily_etl   |
   |     daily_etl:   |              |   +- dq_monitor  |
   |       ...        |              | Pipelines:       |
   |   pipelines:     |              |   +- sales_dlt   |
   |     sales_dlt:   |              | Notifications:   |
   |       ...        |              |   +- webhooks    |
   | targets:         |              +------------------+
   |   dev/staging/prod  |            Same YAML, different targets
   +------------------+
 ```

 ### CE Limitation
 The Databricks CLI bundle command does not work with Community Edition. We show the YAML structure and explain the IaC philosophy.


 #### 90.1 Complete databricks.yml Example


In [ ]:
databricks_yml = """
# ============================================================
# databricks.yml  --  Full Bundle Definition
# ============================================================
# Deploy with:
#   databricks bundle validate -t dev
#   databricks bundle deploy -t dev
#   databricks bundle run daily_etl -t dev
# ============================================================

bundle:
  name: "sales_data_pipeline"

variables:
  run_date:
    description: "Processing date (YYYY-MM-DD)"
    default: "2025-01-15"
  environment:
    description: "Deployment environment"
    default: "dev"

resources:
  jobs:
    daily_etl:
      name: "[${bundle.target}] Daily ETL Pipeline"
      max_concurrent_runs: 1
      schedule:
        quartz_cron_expression: "0 0 6 * * ?"
        timezone_id: "UTC"
        pause_status: "UNPAUSED"

      email_notifications:
        on_failure:
          - "data-eng@company.com"
        no_alert_for_skipped_runs: false

      tasks:
        - task_key: ingest_raw
          job_cluster_key: etl_cluster
          notebook_task:
            notebook_path: ${workspace.root_path}/src/ingest/ingest_raw
            base_parameters:
              run_date: "{{run_date}}"
              environment: "${bundle.target}"
          max_retries: 3
          min_retry_interval_millis: 60000
          timeout_seconds: 1800

        - task_key: transform_sales
          depends_on:
            - task_key: ingest_raw
          notebook_task:
            notebook_path: ${workspace.root_path}/src/transform/transform_sales
          run_if: ALL_SUCCESS
          timeout_seconds: 3600

        - task_key: transform_inventory
          depends_on:
            - task_key: ingest_raw
          notebook_task:
            notebook_path: ${workspace.root_path}/src/transform/transform_inventory
          run_if: ALL_SUCCESS
          timeout_seconds: 3600

        - task_key: aggregate_kpis
          depends_on:
            - task_key: transform_sales
            - task_key: transform_inventory
          notebook_task:
            notebook_path: ${workspace.root_path}/src/aggregate/kpi_aggregates
          run_if: ALL_SUCCESS

        - task_key: dq_checks
          depends_on:
            - task_key: aggregate_kpis
          notebook_task:
            notebook_path: ${workspace.root_path}/src/quality/dq_checks
          run_if: ALL_SUCCESS

    # Job 2: Data Quality Monitoring
    dq_monitoring:
      name: "[${bundle.target}] Data Quality Monitor"
      schedule:
        quartz_cron_expression: "0 0/15 * * * ?"
        timezone_id: "UTC"
      tasks:
        - task_key: freshness_check
          sql_task:
            warehouse_id: "${var.sql_warehouse_id}"
            query:
              query_text: |
                SELECT
                  CASE WHEN MAX(load_timestamp) < DATEADD(HOUR, -4, CURRENT_TIMESTAMP())
                       THEN 'STALE' ELSE 'FRESH'
                  END AS data_status
                FROM gold.daily_sales
          timeout_seconds: 300

  # Shared job cluster definition
  job_clusters:
    - job_cluster_key: etl_cluster
      new_cluster:
        spark_version: "${var.spark_version}"
        node_type_id: "${var.node_type_id}"
        num_workers: "${var.num_workers}"
        autoscale:
          min_workers: "${var.min_workers}"
          max_workers: "${var.max_workers}"
        spark_conf:
          spark.databricks.delta.optimizeWrite.enabled: "true"
          spark.databricks.delta.autoCompact.enabled: "true"
        custom_tags:
          environment: "${bundle.target}"
          team: "data-engineering"
          project: "sales-data-pipeline"

targets:
  dev:
    mode: development
    default: true
    workspace:
      host: "https://dev-workspace.cloud.databricks.com"
      root_path: "/Workspace/Users/.bundle/sales-pipeline/dev"
    variables:
      spark_version: "14.3.x-scala2.12"
      node_type_id: "i3.xlarge"
      num_workers: 1
      min_workers: 1
      max_workers: 4
      sql_warehouse_id: "abc123dev"
      etl_schedule_paused: "PAUSED"

  staging:
    mode: production
    workspace:
      host: "https://staging-workspace.cloud.databricks.com"
      root_path: "/Workspace/Shared/.bundle/sales-pipeline/staging"
    variables:
      spark_version: "14.3.x-scala2.12"
      node_type_id: "i3.2xlarge"
      num_workers: 2
      min_workers: 2
      max_workers: 8
      sql_warehouse_id: "def456staging"
      etl_schedule_paused: "UNPAUSED"

  production:
    mode: production
    workspace:
      host: "https://prod-workspace.cloud.databricks.com"
      root_path: "/Workspace/Shared/.bundle/sales-pipeline/production"
    run_as:
      service_principal_name: "sp-sales-pipeline-prod"
    variables:
      spark_version: "14.3.x-scala2.12"
      node_type_id: "i3.2xlarge"
      num_workers: 4
      min_workers: 4
      max_workers: 20
      sql_warehouse_id: "ghi789prod"
      etl_schedule_paused: "UNPAUSED"
"""

print("=== COMPLETE databricks.yml (DABs Definition) ===\n")
print(databricks_yml)


 #### 90.2 DABs CLI Commands Cheat Sheet


In [ ]:
print("=== DABS CLI COMMAND REFERENCE ===\n")

commands = {
    "Project Initialization": [
        ("databricks bundle init", "Initialise a new bundle from a template"),
        ("databricks bundle init --template default-python", "Use the Python template"),
    ],
    "Validation & Preview": [
        ("databricks bundle validate", "Validate YAML syntax and resource references"),
        ("databricks bundle validate -t dev", "Validate for a specific target"),
    ],
    "Deployment": [
        ("databricks bundle deploy", "Deploy all resources to default target"),
        ("databricks bundle deploy -t staging", "Deploy to staging workspace"),
        ("databricks bundle deploy --force-lock", "Deploy even if bundle is locked"),
    ],
    "Execution": [
        ("databricks bundle run daily_etl", "Run a job defined in the bundle"),
        ("databricks bundle run daily_etl -t dev", "Run a job in a specific target"),
    ],
    "Lifecycle": [
        ("databricks bundle destroy", "Remove all deployed resources"),
        ("databricks bundle destroy -t staging", "Tear down staging resources"),
        ("databricks bundle summary", "Show deployed resources summary"),
    ],
    "Development": [
        ("databricks bundle sync", "Sync local files to workspace (dev mode)"),
    ],
}

for category, cmds in commands.items():
    print(f"--- {category} ---")
    for cmd, desc in cmds:
        print(f"  {cmd:<50}  {desc}")
    print()


 #### 90.3 Infrastructure-as-Code Philosophy


In [ ]:
iac_philosophy = """
INFRASTRUCTURE-AS-CODE PRINCIPLES WITH DABs
============================================

1. EVERYTHING AS CODE
   - Jobs, pipelines, clusters, queries all defined in YAML
   - Version-controlled alongside notebook code
   - No clicking in the UI to configure production resources

2. ENVIRONMENT PARITY
   - Same databricks.yml deploys to dev, staging, and production
   - Target-specific variables handle differences (cluster size, paths)
   - "Works on my machine" becomes "Works on every workspace"

3. IMMUTABLE DEPLOYMENTS
   - Each deploy is a complete, self-contained snapshot
   - Roll back by deploying a previous Git commit
   - No drift between Git and what's running

4. CI/CD INTEGRATION
   - GitHub Actions / Azure DevOps runs bundle validate on PR
   - Auto-deploys to staging on merge to develop
   - Manual approval gate for production deploys

5. SECURITY BY DEFAULT
   - Service principals run production jobs (not user accounts)
   - Secrets referenced via ${secrets/my-secret} never in YAML
   - Audit trail: every deploy is a Git commit

6. TESTABILITY
   - Bundle validate catches YAML errors before deploy
   - Dev target runs with smaller clusters, paused schedules
   - Integration tests can run via bundle run in CI

COMPARISON: DABs vs Manual Configuration
=========================================

| Aspect | Manual (UI) | DABs |
|--------|------------|------|
| Reproducibility | Click-by-click | Single command |
| Version Control | None | Full Git history |
| Rollback | Recreate from memory | git revert + deploy |
| Team Collaboration | Screenshots/Slack | PR review |
| Multi-workspace | Repeat N times | One command per target |
| Audit Trail | Manual notes | Git commits |
"""

print(iac_philosophy)


 ### Key Takeaway â€” Concept 90
 - DABs enable infrastructure-as-code for the entire Databricks platform
 - Single databricks.yml defines jobs, pipelines, clusters, queries
 - Targets (dev/staging/prod) manage environment-specific differences
 - Bundle CLI: validate -> deploy -> run -> destroy
 - CE cannot use bundles â€” understand the YAML structure and IaC philosophy
 - Production: DABs + GitHub Actions = full GitOps workflow


 # COMPREHENSIVE SUMMARY

 ## Concepts 81â€“90: Workflows, CI/CD & Operations

 ### What You've Learned

 | # | Concept | Level | Key Insight |
 |---|---------|-------|-------------|
 | 81 | Choosing Compute for Jobs | Easy | Job Clusters for prod, All-Purpose for dev only |
 | 82 | Secrets Management | Easy | dbutils.secrets.get() > hardcoded; CE: env vars |
 | 83 | Parameterization | Easy | Widgets + Job Params + Task Values; widgets work in CE |
 | 84 | Notebook Patterns | Easy | %run (shared ns), dbutils.notebook.run() (isolated) |
 | 85 | Multi-Task Workflows | Medium | DAG-based orchestration; CE: Python orchestrator |
 | 86 | Retries & Conditional Logic | Medium | run_if conditions + exponential backoff retries |
 | 87 | Git Integration & Repos | Medium | PR-based workflow for notebooks; main/develop/feature |
 | 88 | Monitoring & Alerting | Medium | SQL Alerts in production; Python monitors in CE |
 | 89 | System Tables | Medium | system.billing/compute/jobs; CE: custom tracking |
 | 90 | Asset Bundles (DABs) | Hard | Infrastructure-as-code for Databricks |

 ### Architecture Overview

 ```
 PRODUCTION OPERATIONAL ARCHITECTURE
 ====================================

   Developer                   CI/CD                     Databricks Workspace
   +-------+                  +-------+                  +-------------------+
   | Write |    git push      | GitHub|   bundle deploy  | Jobs + Pipelines  |
   | Code  | ---------------->|Actions| ---------------->|                   |
   | in    |                  |       |                  | +- daily_etl      |
   | Repos |                  | lint  |                  | +- dq_monitor     |
   +-------+                  | test  |                  | +- sales_dlt      |
                              | deploy|                  |                   |
                              +-------+                  +--------+----------+
                                                               |
                                                               | runs
                                                               v
                                                        +-------------------+
                                                        | Monitoring Stack  |
                                                        | +- SQL Alerts     |
                                                        | +- System Tables  |
                                                        | +- Webhooks       |
                                                        +--------+----------+
                                                                 |
                                                                 v
                                                          +-------------------+
                                                          | On-Call Team      |
                                                          | (PagerDuty/Slack) |
                                                          +-------------------+
 ```

 ### CE vs Production Feature Matrix

 | Feature | Community Edition | Production |
 |---------|------------------|-------------|
 | Widgets | Fully supported | Fully supported |
 | %run / dbutils.notebook.run() | Fully supported | Fully supported |
 | %sql / %fs / %sh / %pip | Fully supported | Fully supported |
 | Multi-task Jobs | Not available | Yes (core feature) |
 | Job Clusters | Not available | Yes |
 | Serverless Compute | Not available | Yes |
 | Secret Scopes | Limited / Not available | Yes |
 | Git / Repos | Limited | Yes (full integration) |
 | SQL Alerts (DBSQL) | Not available | Yes |
 | System Tables | Not available | Yes |
 | Webhook Destinations | Not available | Yes |
 | Asset Bundles (DABs) | Not available | Yes |

 ### Next Steps
 1. **Practice** widget parameterisation â€” it's fully available in CE
 2. **Build** the Python orchestrator for your own ETL tasks
 3. **Study** the databricks.yml structure for when you have production access
 4. **Apply** the retry decorator pattern to your existing code
 5. **Set up** Git-based workflow using Repos (when available)
 6. **Read** Databricks docs on Workflows, DABs, and System Tables


 ## SELF-ASSESSMENT QUIZ

 Test your understanding of Concepts 81â€“90:

 ### Easy (Concepts 81â€“84)

 **Q1:** Which compute type should NEVER be used for production jobs?
 - A) Job Cluster
 - B) Serverless Compute
 - C) All-Purpose Cluster
 - D) Instance Pool

 **Q2:** How do you retrieve a secret in a Databricks notebook?
 - A) `os.environ.get('SECRET')`
 - B) `dbutils.secrets.get(scope='x', key='y')`
 - C) `open('/etc/secrets/.env').read()`
 - D) `spark.conf.get('secret')`

 **Q3:** What's the resolution order for parameters?
 - A) Widgets > Code Defaults > Job Params
 - B) Job Params > Widgets > Code Defaults
 - C) Code Defaults > Job Params > Widgets
 - D) All are equal priority

 **Q4:** What's the difference between `%run` and `dbutils.notebook.run()`?
 - A) No difference â€” they're aliases
 - B) %run shares namespace; dbutils.notebook.run() is isolated
 - C) %run is isolated; dbutils.notebook.run() shares namespace
 - D) %run only works in production

 ### Medium (Concepts 85â€“89)

 **Q5:** Which run_if condition means "run only if no upstream task failed"?
 - A) ALL_SUCCESS
 - B) ALL_DONE
 - C) NONE_FAILED
 - D) AT_LEAST_ONE_SUCCESS

 **Q6:** What's the recommended Git branch strategy for Databricks projects?
 - A) Everyone commits directly to main
 - B) Main (prod) <- Develop (staging) <- Feature branches
 - C) One branch per workspace
 - D) No branches needed with notebooks

 **Q7:** Which system table tracks DBU consumption?
 - A) system.compute.clusters
 - B) system.jobs.job_runs
 - C) system.billing.usage
 - D) system.access.audit

 ### Hard (Concept 90)

 **Q8:** What command deploys a DAB to the staging target?
 - A) `databricks bundle run -t staging`
 - B) `databricks bundle deploy -t staging`
 - C) `databricks bundle push staging`
 - D) `databricks deploy --env staging`

 **Q9:** Which file defines ALL resources in a Databricks Asset Bundle?
 - A) `bundle.json`
 - B) `deployment.yaml`
 - C) `databricks.yml`
 - D) `workspace.tf`

 **Q10:** What's the primary benefit of DABs over manual UI configuration?
 - A) Faster UI response time
 - B) Infrastructure-as-code: versioned, reproducible, auditable
 - C) Cheaper DBU pricing
 - D) Automatic data quality checks

 ---
 ### Answers
 1=C, 2=B, 3=B, 4=B, 5=C, 6=B, 7=C, 8=B, 9=C, 10=B

 **Scoring:** 8-10 = Production Ready | 5-7 = On Track | <5 = Review Needed


---

# 10 Architecture Advanced Patterns



 # 10 — Architecture & Advanced Patterns

 **Concepts Covered:** #91–#100  
 **Environment:** Databricks Community Edition (single-node, free)  
 **Goal:** Master advanced architecture patterns, performance diagnostics, testing frameworks, and capstone integration of all 100 concepts.

 | # | Concept | Difficulty | Type |
 |---|---------|------------|------|
 | 91 | Shallow vs. Deep Clones | Easy | Hands-on |
 | 92 | Managed vs. External Tables: Architecture Tradeoffs | Medium | Hands-on |
 | 93 | Legacy: Partitioning & Z-ORDER | Medium | Hands-on |
 | 94 | Lakehouse Table Design Patterns | Medium | Hands-on |
 | 95 | Delta UniForm & Multi-Engine Interoperability | Medium | Conceptual |
 | 96 | Compute Policies & Cluster Governance | Medium | Conceptual |
 | 97 | AI Functions for Data Engineers | Hard | Conceptual |
 | 98 | Multi-Workspace Architecture | Hard | Conceptual |
 | 99 | Performance Troubleshooting Methodology | Hard | Hands-on |
 | 100 | Testing Patterns for Data Pipelines | Hard | Hands-on |

 ---

 ### GRAND FINALE

 This is the **capstone notebook** of the 100-Concept Databricks Professional Learning series. It ties together concepts from all previous notebooks (#1–#90) and pushes into production-grade architecture and engineering patterns.

 **References to prior notebooks:**
 - Notebook 01: Delta Lake Fundamentals (#1–#10)
 - Notebook 02: Spark Execution Model (#11–#20)
 - Notebook 03: SQL & DataFrames (#21–#30)
 - Notebook 04: Data Ingestion Patterns (#31–#40)
 - Notebook 05: Streaming & Incremental Patterns (#41–#50)
 - Notebook 06: Performance & Cost Optimization (#51–#60)
 - Notebook 07: Security & Governance (#61–#70)
 - Notebook 08: Orchestration & Workflows (#71–#80)
 - Notebook 09: MLflow & ML Engineering (#81–#90)

 ---


 ## Setup: Environment & Synthetic Data

 We create a dedicated working directory and generate synthetic sales, customer, and product datasets used throughout all 10 concepts.


In [ ]:
import os
import time
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

base_dir = "/tmp/architecture_advanced_patterns"
dbutils.fs.rm(base_dir, recurse=True)
print(f"Working directory: {base_dir}")

# ── Synthetic Sales Transactions ──────────────────────────────────
sales_data = []
for i in range(1, 5001):
    sales_data.append((
        i,                                           # transaction_id
        f"cust_{i % 200 + 1:04d}",                  # customer_id
        f"prod_{i % 50 + 1:03d}",                   # product_id
        round(abs(hash(f"s{i}") % 5000) / 100, 2),  # amount
        f"store_{(i % 10) + 1}",                     # store_id
        f"region_{(i % 4) + 1}",                     # region
        (2024 + (i % 2)),                             # year
        (i % 12) + 1,                                 # month
        "completed" if i % 10 != 0 else "returned"   # status
    ))

sales_df = spark.createDataFrame(sales_data, [
    "transaction_id", "customer_id", "product_id",
    "amount", "store_id", "region", "year", "month", "status"
])
sales_df.write.format("delta").mode("overwrite").option("mergeSchema", "true") \
    .save(f"{base_dir}/sales")

# ── Customer Dimension ────────────────────────────────────────────
customer_data = []
segments = ["Enterprise", "SMB", "Startup", "Government", "Education"]
tiers = ["Platinum", "Gold", "Silver", "Bronze"]
for i in range(1, 201):
    customer_data.append((
        f"cust_{i:04d}",
        f"Customer_{i}",
        segments[i % 5],
        tiers[i % 4],
        f"region_{(i % 4) + 1}",
        f"202{i % 4 + 1}-01-01" if i % 4 == 0 else None,
    ))

cust_df = spark.createDataFrame(customer_data, [
    "customer_id", "customer_name", "segment", "tier", "region", "acquisition_date"
])
cust_df.write.format("delta").mode("overwrite").option("mergeSchema", "true") \
    .save(f"{base_dir}/customers")

# ── Product Dimension ─────────────────────────────────────────────
product_data = []
categories = ["Electronics", "Clothing", "Food", "Hardware", "Software"]
for i in range(1, 51):
    product_data.append((
        f"prod_{i:03d}",
        f"Product_{i}",
        categories[i % 5],
        round(abs(hash(f"p{i}") % 5000) / 100, 2)
    ))

prod_df = spark.createDataFrame(product_data, [
    "product_id", "product_name", "category", "list_price"
])
prod_df.write.format("delta").mode("overwrite").option("mergeSchema", "true") \
    .save(f"{base_dir}/products")

print("Setup complete. Created:")
print(f"  sales:     {sales_df.count()} rows x {len(sales_df.columns)} cols")
print(f"  customers: {cust_df.count()} rows x {len(cust_df.columns)} cols")
print(f"  products:  {prod_df.count()} rows x {len(prod_df.columns)} cols")


 ---
 ## Concept 91 — Shallow vs. Deep Clones [Easy]

 ### What Problem It Solves

 You need a copy of a Delta table for development, testing, or migration — but copying terabytes of data is slow and expensive. Delta Lake provides two cloning strategies that let you choose between speed and independence.

 ### Key Concepts

 | Property | SHALLOW CLONE | DEEP CLONE |
 |---|---|---|
 | **Data copy?** | No — references source files | Yes — full independent copy |
 | **Speed** | O(1) — metadata-only operation | O(data size) |
 | **Storage cost** | Nearly zero (metadata only) | Full duplicate of data |
 | **Independence** | Depends on source files existing | Fully independent |
 | **VACUUM risk** | If source is VACUUMed, clone breaks | No risk |
 | **Use case** | Dev/test, short-lived experiments | Migration, snapshots, hand-offs |

 ### Real-World Use Cases
 - **Shallow Clone:** Create a QA copy of production data for testing — instant, costs nothing extra.
 - **Deep Clone:** Migrate a table to a new storage location or cloud provider; create an immutable snapshot for audit.


 ### Step 1: Create the Source Table


In [ ]:
print("=" * 60)
print("CONCEPT 91: SHALLOW vs DEEP CLONES")
print("=" * 60)

source_table = f"{base_dir}/clone_source"
sales_df.write.format("delta").mode("overwrite").save(source_table)

# Register as SQL table
spark.sql(f"CREATE OR REPLACE TABLE clone_source USING DELTA LOCATION '{source_table}'")

print("\nSource table details:")
spark.sql("DESCRIBE DETAIL clone_source").select(
    "name", "numFiles", "sizeInBytes", "location"
).show(truncate=False)


 ### Step 2: Create a SHALLOW CLONE (Zero-Copy)


In [ ]:
t0 = time.time()
spark.sql(f"CREATE OR REPLACE TABLE clone_shallow SHALLOW CLONE clone_source")
t_shallow = time.time() - t0

print(f"SHALLOW CLONE created in {t_shallow:.3f}s")
print("\nShallow clone details:")
spark.sql("DESCRIBE DETAIL clone_shallow").select(
    "name", "numFiles", "sizeInBytes", "location"
).show(truncate=False)

shallow_loc = spark.sql("DESCRIBE DETAIL clone_shallow").select("location").collect()[0][0]
source_loc = spark.sql("DESCRIBE DETAIL clone_source").select("location").collect()[0][0]
print(f"\nSource location:     {source_loc}")
print(f"Shallow clone loc:   {shallow_loc}")
print(f"\nNOTE: Shallow clone has its OWN _delta_log directory ({shallow_loc})")
print(f"      but the data files (.parquet) still live in the source location.")
print(f"      This is why it's near-instant: only metadata is duplicated.")


 ### Step 3: Create a DEEP CLONE (Full Independent Copy)


In [ ]:
t0 = time.time()
spark.sql(f"CREATE OR REPLACE TABLE clone_deep DEEP CLONE clone_source")
t_deep = time.time() - t0

print(f"DEEP CLONE created in {t_deep:.3f}s")
print("\nDeep clone details:")
spark.sql("DESCRIBE DETAIL clone_deep").select(
    "name", "numFiles", "sizeInBytes", "location"
).show(truncate=False)

deep_loc = spark.sql("DESCRIBE DETAIL clone_deep").select("location").collect()[0][0]
print(f"\nDeep clone location: {deep_loc}")
print(f"\nSpeed comparison:")
print(f"  Shallow clone: {t_shallow:.3f}s (metadata only)")
print(f"  Deep clone:    {t_deep:.3f}s (full data copy + metadata)")


 ### Step 4: Verify Data Independence


In [ ]:
# Show all three tables have identical data
print("Row counts across all three tables:")
display(spark.sql("""
    SELECT 'source'  AS table_name, COUNT(*) AS row_count FROM clone_source
    UNION ALL
    SELECT 'shallow' AS table_name, COUNT(*) AS row_count FROM clone_shallow
    UNION ALL
    SELECT 'deep'    AS table_name, COUNT(*) AS row_count FROM clone_deep
"""))

# Now INSERT into the shallow clone — does it affect the source?
spark.sql("INSERT INTO clone_shallow SELECT 99999, 'cust_test', 'prod_001', 99.99, 'store_1', 'region_1', 2025, 1, 'pending'")

print("\nAfter inserting into SHALLOW clone:")
print(f"  Source rows:   {spark.sql('SELECT COUNT(*) FROM clone_source').collect()[0][0]}")
print(f"  Shallow rows:  {spark.sql('SELECT COUNT(*) FROM clone_shallow').collect()[0][0]}")
print(f"  Deep rows:     {spark.sql('SELECT COUNT(*) FROM clone_deep').collect()[0][0]}")
print("  → Shows clones are logically independent (their own transaction logs)")


 ### Step 5: Decision Tree


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                      CLONE STRATEGY DECISION TREE                        ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  Need a table copy?                                                      ║
║  ├─ Short-lived (dev/test, experiments)?                                 ║
║  │  └─ SHALLOW CLONE — instant, near-zero cost                           ║
║  │                                                                       ║
║  ├─ Long-lived and independent?                                          ║
║  │  └─ Will source ever be VACUUMed or deleted?                          ║
║  │     ├─ YES → DEEP CLONE — data lives independently                    ║
║  │     └─ NO  → SHALLOW CLONE may be fine (but risky)                    ║
║  │                                                                       ║
║  ├─ Migrating to new location/cloud?                                     ║
║  │  └─ DEEP CLONE — with LOCATION clause to specify new path             ║
║  │                                                                       ║
║  ├─ Immutable snapshot for audit?                                        ║
║  │  └─ DEEP CLONE — guarantee independence from source mutations         ║
║  │                                                                       ║
║  └─ PARTITION FILTER needed? (clone only specific partitions)            ║
║     └─ Use CLONE with WHERE clause                                       ║
║                                                                          ║
║  KEY INSIGHT: Shallow clone files are in SOURCE directory;               ║
║              if source is VACUUMed → shallow clone BREAKS.               ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


 ---
 ## Concept 92 — Managed vs. External Tables: Architecture Tradeoffs [Medium]

 ### What Problem It Solves

 When you create a Delta table, you choose who controls the data's lifecycle — Databricks (managed) or you (external). This decision has deep implications for governance, multi-engine access, disaster recovery, and cost.

 ### Key Concepts

 | Property | MANAGED TABLE | EXTERNAL TABLE |
 |---|---|---|
 | **Data location** | Databricks chooses (Unity Catalog managed storage) | You specify the path |
 | **DROP TABLE** | Deletes data AND metadata | Deletes metadata only; data survives |
 | **Lifecycle** | Databricks controls retention, VACUUM, cleanup | You manage everything |
 | **Multi-engine access** | Limited (only through Databricks) | Any engine that reads Delta (Spark, Trino, Dremio, etc.) |
 | **Predictive Optimization** | Supported | Not supported (requires managed tables in Unity Catalog) |
 | **Governance** | Full Unity Catalog integration | You manage permissions on cloud storage too |

 **NOTE:** Managed tables require Unity Catalog (full platform). In Community Edition we use `LOCATION` to control storage explicitly.

 ### When to Use Each

 **Choose Managed when:**
 - Databricks is the sole compute engine
 - You want Predictive Optimization
 - Governance and lifecycle are managed centrally
 - Simpler operational model

 **Choose External when:**
 - Multiple engines read the same data (Spark, Trino, Presto, Dremio, Snowflake, BigQuery)
 - You own the cloud storage lifecycle (regulatory/data residency requirements)
 - You need fine-grained control over file layout and retention policies
 - The data must survive DROP TABLE or workspace deletion


 ### Step 1: Create Managed-Style and External-Style Tables


In [ ]:
print("=" * 60)
print("CONCEPT 92: MANAGED vs EXTERNAL TABLES")
print("=" * 60)

# ── Managed-style table (no explicit LOCATION) ────────────────────
spark.sql("DROP TABLE IF EXISTS sales_managed")
spark.sql("""
    CREATE TABLE sales_managed (
        transaction_id INT,
        customer_id STRING,
        product_id STRING,
        amount DECIMAL(10,2),
        store_id STRING
    ) USING DELTA
""")
spark.sql("INSERT INTO sales_managed SELECT transaction_id, customer_id, product_id, amount, store_id FROM clone_source")
print("Managed-style table created (Databricks controls the data location).")

# ── External-style table (explicit LOCATION) ─────────────────────
spark.sql("DROP TABLE IF EXISTS sales_external")
external_data_loc = f"{base_dir}/external_data/sales"
spark.sql(f"""
    CREATE TABLE sales_external (
        transaction_id INT,
        customer_id STRING,
        product_id STRING,
        amount DECIMAL(10,2),
        store_id STRING
    ) USING DELTA
    LOCATION '{external_data_loc}'
""")
spark.sql("INSERT INTO sales_external SELECT transaction_id, customer_id, product_id, amount, store_id FROM clone_source")
print(f"External-style table created at: {external_data_loc}")


 ### Step 2: DESCRIBE EXTENDED — See the Table Type


In [ ]:
print("Managed table — DESCRIBE EXTENDED:")
desc_managed = spark.sql("DESCRIBE EXTENDED sales_managed").filter(
    col("col_name").isin("Location", "Type", "Provider")
)
desc_managed.show(truncate=False)

print("\nExternal table — DESCRIBE EXTENDED:")
desc_external = spark.sql("DESCRIBE EXTENDED sales_external").filter(
    col("col_name").isin("Location", "Type", "Provider")
)
desc_external.show(truncate=False)

# ── List files for external table location ───────────────────────
print(f"\nFiles at external data location ({external_data_loc}):")
for f in dbutils.fs.ls(external_data_loc):
    print(f"  {f.name}  ({f.size:,} bytes)")


 ### Step 3: DROP Behavior — The Critical Difference


In [ ]:
# ── Create two MORE tables specifically for the drop test ─────────
spark.sql("""
    CREATE TABLE IF NOT EXISTS drop_test_managed (
        transaction_id INT, amount DECIMAL(10,2)
    ) USING DELTA
""")
spark.sql("INSERT INTO drop_test_managed VALUES (1, 100.00), (2, 200.00)")

drop_test_external_loc = f"{base_dir}/drop_test_external"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS drop_test_external (
        transaction_id INT, amount DECIMAL(10,2)
    ) USING DELTA
    LOCATION '{drop_test_external_loc}'
""")
spark.sql("INSERT INTO drop_test_external VALUES (1, 100.00), (2, 200.00)")

# Verify data files exist
print("Before DROP:")
print(f"  External data files: {len([f for f in dbutils.fs.ls(drop_test_external_loc) if f.name.endswith('.parquet')])}")

# ── DROP BOTH TABLES ─────────────────────────────────────────────
spark.sql("DROP TABLE IF EXISTS drop_test_managed")
spark.sql("DROP TABLE IF EXISTS drop_test_external")

print("\nAfter DROP:")
try:
    files_after = [f for f in dbutils.fs.ls(drop_test_external_loc) if f.name.endswith('.parquet')]
    print(f"  External data files STILL EXIST: {len(files_after)}")
    print("  ✓ External table data SURVIVES DROP")
except Exception as e:
    print(f"  External data files NOT FOUND: {e}")
    print("  (Managed table data would be deleted — cannot verify in Community Edition)")

# Clean up
dbutils.fs.rm(drop_test_external_loc, recurse=True)


 ### Decision Matrix


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                MANAGED vs EXTERNAL — DECISION MATRIX                     ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  REQUIREMENT                        MANAGED    EXTERNAL                  ║
║  ──────────────────────────────────────────────────────                  ║
║  Databricks-only access               ✓          ✓                       ║
║  Multi-engine reads (Trino/Snowflake)  ✗          ✓                      ║
║  Predictive Optimization               ✓          ✗                      ║
║  Data survives DROP TABLE              ✗          ✓                      ║
║  Data survives workspace deletion      ✗          ✓                      ║
║  Centralized lifecycle management      ✓          ✗                      ║
║  Regulatory data residency              ✗*         ✓                     ║
║  Unity Catalog governance               ✓         ✓(registered)          ║
║  Liquid Clustering                      ✓          ✓                     ║
║  DELETE/UPDATE/MERGE                    ✓          ✓                     ║
║  Time Travel (DESCRIBE HISTORY)         ✓          ✓                     ║
║                                                                          ║
║  * Managed tables use Databricks-managed storage; you choose             ║
║    the region at workspace creation but have less granular control.      ║
║                                                                          ║
║  BEST PRACTICE: Start with MANAGED for operational simplicity;           ║
║                switch to EXTERNAL when you hit an explicit need.         ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


 ---
 ## Concept 93 — Legacy: Partitioning & Z-ORDER [Medium]

 ### What Problem It Solves

 Before Liquid Clustering (Delta 3.0+, full platform), the primary ways to organize data for fast queries were **Hive-style partitioning** and **Z-ORDER**. While Liquid Clustering has largely superseded these, they remain relevant for legacy systems, external readers, and specific low-cardinality use cases.

 ### Key Concepts

 **Partitioning:**
 - Divides data into folder hierarchies (e.g., `year=2024/month=01/`)
 - File-level pruning via directory listing — no metadata read needed
 - Works with ANY reader (even non-Delta!) — this is why it still matters
 - **Downside:** Too many partitions → small file problem; too few → no pruning benefit
 - **Rule of thumb:** Only partition on low-cardinality columns (year, month, region, category)

 **Z-ORDER:**
 - Co-locates related data within the same set of files
 - Speeds up queries with filters on the Z-ORDER columns
 - Works WITHIN partitions (not as a replacement)
 - **Downside:** Must be reapplied after writes; maintenance overhead

 **Why Liquid Clustering is better:**
 - No manual partition key selection
 - Adaptive — reclusters incrementally
 - No small-file problem from bad partition choices
 - BUT: Requires Delta 3.0+ and full Databricks platform


 ### Step 1: Create a Hive-Partitioned Table


In [ ]:
print("=" * 60)
print("CONCEPT 93: LEGACY PARTITIONING & Z-ORDER")
print("=" * 60)

# ── Create a larger dataset for meaningful partitioning ───────────
partitioned_path = f"{base_dir}/legacy_partitioned"

df_part = sales_df.withColumn("year_month", 
    concat(col("year").cast("string"), lit("-"), lpad(col("month").cast("string"), 2, "0"))
)

df_part.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .save(partitioned_path)

print("Partitioned table created with partitionBy('year', 'month')")
print(f"\nDirectory structure at {partitioned_path}:")
for item in sorted(dbutils.fs.ls(partitioned_path), key=lambda x: x.name):
    if item.name.startswith("year"):
        print(f"  {item.name}/")
        for sub in dbutils.fs.ls(item.path):
            if sub.name.startswith("month"):
                file_count = len([f for f in dbutils.fs.ls(sub.path) if f.name.endswith('.parquet')])
                print(f"    {sub.name}/  ({file_count} parquet files)")


 ### Step 2: Demonstrate Partition Pruning


In [ ]:
print("Query 1: Full table scan (no partition filter)")
t0 = time.time()
result_full = spark.sql(f"SELECT COUNT(*) FROM delta.`{partitioned_path}`").collect()[0][0]
t_full = time.time() - t0

print(f"  Full scan: {result_full} rows in {t_full:.3f}s")

print("\nQuery 2: Single partition filter (year=2024, month=6)")
t0 = time.time()
result_part = spark.sql(
    f"SELECT COUNT(*) FROM delta.`{partitioned_path}` WHERE year = 2024 AND month = 6"
).collect()[0][0]
t_part = time.time() - t0

print(f"  Partition filter: {result_part} rows in {t_part:.3f}s")

# Show the query plan — note the PartitionFilters
print("\nQuery plan WITH partition filter:")
spark.sql(
    f"EXPLAIN EXTENDED SELECT * FROM delta.`{partitioned_path}` WHERE year = 2024 AND month = 6"
).show(truncate=False)


 ### Step 3: Apply Z-ORDER and Compare


In [ ]:
zorder_path = f"{base_dir}/legacy_zordered"

# Create a non-partitioned version for Z-ORDER demo
sales_df.write.format("delta").mode("overwrite").save(zorder_path)

# Test query speed before Z-ORDER
print("Before Z-ORDER:")
spark.sql(f"DESCRIBE DETAIL delta.`{zorder_path}`").select("numFiles", "sizeInBytes").show()

t0 = time.time()
result_before = spark.sql(
    f"SELECT * FROM delta.`{zorder_path}` WHERE customer_id = 'cust_0050'"
).count()
t_before_z = time.time() - t0
print(f"Filter on customer_id (no Z-ORDER): {result_before} matching rows in {t_before_z:.3f}s")

# ── Apply Z-ORDER ────────────────────────────────────────────────
print("\nApplying Z-ORDER BY (customer_id)...")
t0 = time.time()
spark.sql(f"OPTIMIZE delta.`{zorder_path}` ZORDER BY (customer_id)")
t_zorder = time.time() - t0
print(f"Z-ORDER completed in {t_zorder:.2f}s")

print("\nAfter Z-ORDER:")
spark.sql(f"DESCRIBE DETAIL delta.`{zorder_path}`").select("numFiles", "sizeInBytes").show()

# Test query speed after Z-ORDER
t0 = time.time()
result_after = spark.sql(
    f"SELECT * FROM delta.`{zorder_path}` WHERE customer_id = 'cust_0050'"
).count()
t_after_z = time.time() - t0
print(f"Filter on customer_id (WITH Z-ORDER): {result_after} matching rows in {t_after_z:.3f}s")

print(f"\n  Improvement: {t_before_z:.3f}s → {t_after_z:.3f}s")


 ### Step 4: Migration Strategy — Legacy to Liquid Clustering


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║             LEGACY → LIQUID CLUSTERING MIGRATION STRATEGY                ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  1. AUDIT existing partitioning schemes                                  ║
║     └─ Identify tables with many small files (bad partition key)         ║
║     └─ Identify tables where partitioning adds little value              ║
║                                                                          ║
║  2. PRIORITIZE high-impact tables                                        ║
║     └─ Tables with > 10k partitions → migrate first                      ║
║     └─ Tables where OPTIMIZE runs frequently → migrate                   ║
║                                                                          ║
║  3. ENABLE Liquid Clustering (Delta 3.0+, full platform)                 ║
║     └─ ALTER TABLE t CLUSTER BY (column1, column2)                       ║
║     └─ Liquid Clustering works alongside existing partitions             ║
║                                                                          ║
║  4. GRADUALLY remove partitioning                                        ║
║     └─ Create CLONED table (DEEP CLONE) without partition columns        ║
║     └─ Apply CLUSTER BY on the new table                                 ║
║     └─ Validate query performance, then swap                             ║
║                                                                          ║
║  5. When to KEEP legacy partitioning:                                    ║
║     └─ External readers (Trino/Presto/Dremio) that benefit from          ║
║        directory pruning                                                 ║
║     └─ Very low cardinality (year, continent) — directory pruning        ║
║        is still fast                                                     ║
║     └─ Mixed workload where partition-awareness matters downstream       ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


 ---
 ## Concept 94 — Lakehouse Table Design Patterns [Medium]

 ### What Problem It Solves

 Should you store data in a wide denormalized table for fast queries, or in a normalized star schema for flexibility? The lakehouse architecture supports both — and the choice has significant performance, maintainability, and cost implications.

 ### Key Patterns

 | Pattern | Structure | Best For | Downside |
 |---|---|---|---|
 | **Wide Denormalized** | All columns in one table | BI dashboards, ad-hoc analytics | Redundant data, slower writes, complex updates |
 | **Star Schema** | Fact + dimension tables | ETL pipelines, data warehouse | JOIN required for every query |
 | **One Big Table (OBT)** | Pre-joined, pre-aggregated | Real-time dashboards | Extremely wide, expensive refreshes |
 | **Data Vault** | Hub/Link/Satellite tables | Enterprise data integration, audit trails | Complex to query |

 ### Design Principles for Lakehouse
 1. **Medallion Architecture**: Bronze (raw) → Silver (cleansed) → Gold (business-level)
 2. **Column ordering matters**: First 32 columns get statistics — put filter columns first
 3. **Clustering keys**: Align with most common query filters, not just any column
 4. **Right-size data types**: DECIMAL(10,2) not DOUBLE for currency; INT not BIGINT when possible


 ### Step 1: Design a Star Schema (Gold Layer)


In [ ]:
print("=" * 60)
print("CONCEPT 94: LAKEHOUSE TABLE DESIGN PATTERNS")
print("=" * 60)

# ── Fact Table: sales_facts (thin, many rows) ────────────────────
fact_sales = spark.sql(f"""
    CREATE OR REPLACE TABLE gold.fact_sales
    USING DELTA
    LOCATION '{base_dir}/gold/fact_sales'
    AS
    SELECT
        transaction_id,
        customer_id,
        product_id,
        store_id,
        amount,
        year,
        month,
        CASE WHEN status = 'returned' THEN 1 ELSE 0 END AS is_returned,
        CAST(amount AS DECIMAL(10,2)) AS net_amount
    FROM delta.`{base_dir}/sales`
""")
print("Fact table created: gold.fact_sales")

# ── Dimension: Customer ───────────────────────────────────────────
spark.sql(f"""
    CREATE OR REPLACE TABLE gold.dim_customer
    USING DELTA
    LOCATION '{base_dir}/gold/dim_customer'
    AS SELECT * FROM delta.`{base_dir}/customers`
""")
print("Dimension table created: gold.dim_customer")

# ── Dimension: Product ────────────────────────────────────────────
spark.sql(f"""
    CREATE OR REPLACE TABLE gold.dim_product
    USING DELTA
    LOCATION '{base_dir}/gold/dim_product'
    AS SELECT * FROM delta.`{base_dir}/products`
""")
print("Dimension table created: gold.dim_product")

# ── Dimension: Date (generated) ───────────────────────────────────
date_data = [(2024, m, f"2024-{m:02d}-01") for m in range(1, 13)] + \
            [(2025, m, f"2025-{m:02d}-01") for m in range(1, 13)]
date_df = spark.createDataFrame(date_data, ["year", "month", "month_start"])

date_df.write.format("delta").mode("overwrite").save(f"{base_dir}/gold/dim_date")
spark.sql(f"""
    CREATE OR REPLACE TABLE gold.dim_date
    USING DELTA
    LOCATION '{base_dir}/gold/dim_date'
    AS SELECT * FROM delta.`{base_dir}/gold/dim_date`
""")
print("Dimension table created: gold.dim_date")

print("\nStar Schema Summary:")
display(spark.sql("""
    SELECT 'fact_sales' AS table_name, COUNT(*) AS row_count FROM gold.fact_sales
    UNION ALL
    SELECT 'dim_customer', COUNT(*) FROM gold.dim_customer
    UNION ALL
    SELECT 'dim_product',  COUNT(*) FROM gold.dim_product
    UNION ALL
    SELECT 'dim_date',     COUNT(*) FROM gold.dim_date
"""))


 ### Step 2: Query Pattern — Star Schema JOIN


In [ ]:
print("Star Schema Query: Sales by customer segment and product category")
t0 = time.time()

star_query = spark.sql("""
    SELECT
        c.segment,
        p.category,
        SUM(f.net_amount) AS total_revenue,
        COUNT(DISTINCT f.transaction_id) AS transaction_count,
        ROUND(AVG(f.amount), 2) AS avg_transaction_value
    FROM gold.fact_sales f
    JOIN gold.dim_customer c ON f.customer_id = c.customer_id
    JOIN gold.dim_product  p ON f.product_id = p.product_id
    WHERE f.is_returned = 0
    GROUP BY c.segment, p.category
    ORDER BY c.segment, total_revenue DESC
""")

display(star_query)
print(f"Star schema query completed in {time.time() - t0:.3f}s")


 ### Step 3: Alternative — Wide Denormalized Table for Reporting


In [ ]:
# ── Pre-join everything into one wide table ───────────────────────
wide_reporting_path = f"{base_dir}/gold/wide_sales_reporting"

spark.sql(f"""
    CREATE OR REPLACE TABLE gold.wide_sales_reporting
    USING DELTA
    LOCATION '{wide_reporting_path}'
    AS
    SELECT
        f.transaction_id,
        f.amount,
        f.net_amount,
        f.year,
        f.month,
        f.store_id,
        f.is_returned,
        c.customer_name,
        c.segment AS customer_segment,
        c.tier AS customer_tier,
        c.region AS customer_region,
        p.product_name,
        p.category AS product_category,
        p.list_price
    FROM gold.fact_sales f
    LEFT JOIN gold.dim_customer c ON f.customer_id = c.customer_id
    LEFT JOIN gold.dim_product  p ON f.product_id = p.product_id
""")
print("Wide denormalized table created: gold.wide_sales_reporting")

# ── Same analysis, NO JOINS needed ───────────────────────────────
print("\nSame query on wide denormalized table (NO JOINS):")
t0 = time.time()

wide_query = spark.sql("""
    SELECT
        customer_segment,
        product_category,
        SUM(net_amount) AS total_revenue,
        COUNT(DISTINCT transaction_id) AS transaction_count,
        ROUND(AVG(amount), 2) AS avg_transaction_value
    FROM gold.wide_sales_reporting
    WHERE is_returned = 0
    GROUP BY customer_segment, product_category
    ORDER BY customer_segment, total_revenue DESC
""")

display(wide_query)
print(f"Wide table query completed in {time.time() - t0:.3f}s")


 ### Step 4: Design Tradeoffs Summary


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                    TABLE DESIGN PATTERN TRADEOFFS                        ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  STAR SCHEMA (normalized)                                                ║
║  ├─ ✓ Storage-efficient (no duplicate dimension data)                   ║
║  ├─ ✓ Easy to update dimensions independently                            ║
║  ├─ ✓ Flexible — add dimensions without changing facts                  ║
║  ├─ ✓ Best for ETL patterns, complex transformations                    ║
║  └─ ✗ Every query needs JOINs — can be slow without optimization        ║
║                                                                          ║
║  WIDE DENORMALIZED                                                       ║
║  ├─ ✓ Lightning-fast reads (no JOINs)                                   ║
║  ├─ ✓ Simple queries — BI tools love wide tables                        ║
║  ├─ ✓ Best for dashboards, ad-hoc analytics                             ║
║  └─ ✗ Redundant data (customer name in every row)                       ║
║  └─ ✗ Expensive to update dimensions (must rewrite entire table)        ║
║  └─ ✗ Wide schemas hit the 32-column stats limit                        ║
║                                                                          ║
║  HYBRID (recommended for most lakehouses)                                ║
║  ├─ Gold layer: Wide reporting tables for BI consumers                  ║
║  ├─ Silver layer: Normalized star schema for transformations             ║
║  ├─ Bronze layer: Raw ingest (no design — fidelity matters)             ║
║  └─ Rebuild wide tables nightly/weekly; keep star schema as source      ║
║                                                                          ║
║  DESIGN HEURISTICS:                                                      ║
║  ├─ Put most-filtered columns in the FIRST 32 positions                 ║
║  ├─ Cluster/partition on DATE columns (most queries filter by time)     ║
║  ├─ Use DECIMAL for currency (not FLOAT/DOUBLE)                         ║
║  ├─ Avoid deeply nested STRUCTs in frequently queried columns           ║
║  └─ Use MERGE for dimension updates, INSERT for fact appends            ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


 ---
 ## Concept 95 — Delta UniForm & Multi-Engine Interoperability [Medium]

 ### What Problem It Solves

 Your organization uses multiple engines — Spark for ETL, Snowflake for BI, BigQuery for ML, Trino for ad-hoc. Each engine wants its own table format (Delta, Iceberg, Hudi). Copying data between formats is expensive, slow, and creates consistency problems.

 **Delta UniForm** solves this: write once as Delta, and the table is automatically readable as Iceberg (and soon Hudi) — with no data copy.

 **NOTE:** UniForm requires Delta 3.0+ on full Databricks platform. This section explains the concept, configuration, and decision framework.

 ### Key Concepts

 | Approach | How It Works | Pros | Cons |
 |---|---|---|---|
 | **UniForm** | Single Delta table auto-generates Iceberg/Hudi metadata | One copy, all engines | Full platform only |
 | **Delta Sharing** | Databricks shares data via open protocol | No copy, governed access | Recipient needs Databricks or open connector |
 | **Federation** | Query external data in-place via Lakehouse Federation | No copy, no ingest | Query performance depends on remote system |
 | **Manual Export** | Periodically copy data to another format | Works everywhere | Duplicate data, stale, expensive |

 ### How UniForm Works (Under the Hood)

 1. You enable UniForm on a Delta table: `ENABLE UNIFORM(ICEBERG_COMPAT_VERSION=2)`
 2. On each Delta commit, Databricks automatically generates Iceberg metadata files alongside the Delta log
 3. Iceberg readers (Trino, Snowflake, BigQuery, Athena) see the same data files — they just read a different metadata path
 4. No data duplication — the Parquet files are the same


 ### Step 1: Simulate UniForm Metadata Structure (Conceptual)


In [ ]:
print("=" * 60)
print("CONCEPT 95: DELTA UNIFORM & MULTI-ENGINE INTEROPERABILITY")
print("=" * 60)

# Show what a UniForm-enabled table's directory looks like
print("""
 Delta Table with UniForm enabled:

   my_table/
   ├── _delta_log/           ← Delta transaction log (native)
   │   ├── 000000.json
   │   ├── 000001.json
   │   └── ...
   ├── metadata/             ← Iceberg metadata (auto-generated by UniForm)
   │   ├── v1.metadata.json
   │   ├── v2.metadata.json
   │   └── snap-*.avro
   ├── part-00000-xxx.parquet  ← Same data files for ALL engines
   ├── part-00001-xxx.parquet
   └── ...

 KEY INSIGHT: The .parquet data files are SHARED.
              Only the METADATA layer differs per format.
""")


 ### Step 2: Decision Tree — UniForm vs Sharing vs Federation


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║          MULTI-ENGINE ACCESS: DECISION FRAMEWORK                         ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  START HERE:  Does the consumer use Databricks or an open connector?     ║
║                                                                          ║
║  ├─ YES → DELTA SHARING                                                  ║
║  │    └─ Recipient gets governed, live access. No data copy.             ║
║  │    └─ Best when: you control both sides, want centralized governance  ║
║  │                                                                       ║
║  ├─ Consumer uses ICEBERG-compatible engine (Trino, Snowflake, etc.)?    ║
║  │  ├─ YES → UNIFORM                                                     ║
║  │  │    └─ Single Delta table, readable as Iceberg                      ║
║  │  │    └─ Best when: one producer (Databricks), many consumers         ║
║  │  │    └─ Example: Write ETL in Databricks; BI team reads via Trino    ║
║  │  │                                                                    ║
║  │  └─ Consumer uses HUDI?                                               ║
║  │     └─ UniForm Hudi support (roadmap)                                 ║
║  │                                                                       ║
║  └─ Consumer needs to query data in ANOTHER system's native storage?     ║
║     └─ LAKEHOUSE FEDERATION                                              ║
║        └─ Query Snowflake/Postgres/MySQL tables directly from Databricks ║
║        └─ Best when: data lives elsewhere, occasional access needed      ║
║                                                                          ║
║  COST COMPARISON:                                                        ║
║  ├─ UniForm: Metadata generation cost (tiny). Storage = 1x data.         ║
║  ├─ Delta Sharing: No extra storage. Egress costs may apply.             ║
║  ├─ Federation: Query pushdown to source system. Source bears query cost.║
║  └─ Manual Export: 2x+ storage. Stale data. High maintenance.            ║
║                                                                          ║
║  COMPATIBILITY MATRIX:                                                   ║
║  Engine               Delta   Iceberg   Hudi      Notes                  ║
║  ──────────────────────────────────────────────────                      ║
║  Databricks             ✓       ✗*       ✗*      *Unless via UniForm     ║
║  Apache Spark           ✓       ✓         ✓       Depends on config      ║
║  Snowflake               ✗       ✓         ✗      Iceberg tables         ║
║  BigQuery                ✗       ✓         ✗      Iceberg via BigLake    ║
║  Trino/Presto            ✗       ✓         ✓      Plugins required       ║
║  Athena                  ✗       ✓         ✗      Iceberg via Glue       ║
║  Dremio                  ✓       ✓         ✗      Direct Delta + Iceberg ║
║  Redshift Spectrum       ✗       ✓         ✗      Iceberg only           ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


 ### Step 3: Conceptual UniForm Configuration


In [ ]:
# ── This is conceptual code — UniForm requires full platform ─────
print("""
── Configuring UniForm (Full Platform Only) ──

-- Enable UniForm on an existing table:
ALTER TABLE sales_facts
SET TBLPROPERTIES (
    'delta.universalFormat.enabledFormats' = 'iceberg'
);

-- Check UniForm status:
DESCRIBE EXTENDED sales_facts;
-- Look for: delta.universalFormat.enabledFormats

-- External engines connect to the Iceberg catalog path:
--   s3://bucket/table/metadata/

-- On Trino, for example:
--   CREATE TABLE trino_sales (...)
--   WITH (format = 'ICEBERG', location = 's3://bucket/table/');

── Considerations ──
• UniForm adds small metadata write overhead on each Delta commit
• Iceberg metadata is auto-generated — no manual sync needed
• Column mapping mode must be 'name' (not 'id') for Iceberg compat
• Partition evolution and schema evolution work natively
• VACUUM must respect both Delta and Iceberg retention requirements
""")


 ---
 ## Concept 96 — Compute Policies & Cluster Governance [Medium]

 ### What Problem It Solves

 Developers create clusters with 100 nodes and forget to terminate them. Costs spiral. You need guardrails that balance developer productivity with budget discipline — without requiring every cluster to go through an approval process.

 **Compute policies** define what cluster configurations are allowed. They act as templates with fixed values, allowed ranges, and mandatory tags.

 **NOTE:** Policies are managed at the admin level (full platform). This section shows the policy structure and governance concepts applicable in any Databricks environment.

 ### Key Concepts

 | Policy Element | Purpose | Example |
 |---|---|---|
 | **Fixed values** | Lock a setting (cannot be changed) | `autotermination_minutes: 30` |
 | **Allowed ranges** | Limit a setting to a range | `num_workers: {min: 1, max: 10}` |
 | **Default values** | Pre-fill a setting (can be changed) | `spark_version: "13.3.x-scala2.12"` |
 | **Required tags** | Force tagging for cost tracking | `"cost_center": {"type": "fixed", "value": "required"}` |
 | **Hidden fields** | Remove advanced options from UI | Spot instance config |

 ### Policy Types
 - **Personal Compute (default):** Per-user, auto-terminates, limited size
 - **Power User:** More workers, spot instances allowed, longer auto-termination
 - **Job Compute:** Optimized for jobs, no interactive features, aggressive auto-termination
 - **Restricted:** Only specific instance types and versions


 ### Step 1: Sample Compute Policy JSON


In [ ]:
print("=" * 60)
print("CONCEPT 96: COMPUTE POLICIES & CLUSTER GOVERNANCE")
print("=" * 60)

# Define sample policies as Python dicts for analysis
general_policy = {
    "name": "Standard Developer",
    "description": "Balanced policy for daily development work",
    "definition": json.dumps({
        "autotermination_minutes": {
            "type": "range",
            "minValue": 15,
            "maxValue": 120,
            "defaultValue": 30,
            "isOptional": False
        },
        "num_workers": {
            "type": "range",
            "minValue": 2,
            "maxValue": 20,
            "defaultValue": 4,
            "isOptional": False
        },
        "node_type_id": {
            "type": "allowlist",
            "values": [
                "i3.xlarge",
                "i3.2xlarge",
                "m5d.xlarge",
                "m5d.2xlarge"
            ],
            "defaultValue": "i3.xlarge"
        },
        "custom_tags.cost_center": {
            "type": "regex",
            "pattern": "^CC-[0-9]{4}$",
            "defaultValue": "CC-0000"
        },
        "custom_tags.environment": {
            "type": "fixed",
            "value": "development"
        },
        "spark_conf.spark.sql.shuffle.partitions": {
            "type": "fixed",
            "value": "auto"
        },
        "driver_node_type_id": {
            "type": "fixed",
            "value": "i3.xlarge"
        }
    }, indent=2)
}

print("── Sample Policy: Standard Developer ──")
print(json.dumps(json.loads(general_policy["definition"]), indent=2))


 ### Step 2: Cost-Effective Cluster Policy (Job Compute)


In [ ]:
job_policy = {
    "name": "Production Jobs",
    "description": "Optimized for scheduled jobs — spot instances, tight auto-termination",
    "definition": json.dumps({
        "autotermination_minutes": {
            "type": "fixed",
            "value": 15,
            "isOptional": False
        },
        "num_workers": {
            "type": "range",
            "minValue": 2,
            "maxValue": 50,
            "defaultValue": 8,
            "isOptional": False
        },
        "node_type_id": {
            "type": "allowlist",
            "values": [
                "i3.xlarge",
                "i3.2xlarge",
                "i3.4xlarge",
                "i3en.xlarge",
                "i3en.2xlarge"
            ],
            "defaultValue": "i3.xlarge"
        },
        "enable_elastic_disk": {
            "type": "fixed",
            "value": True
        },
        "spark_conf.spark.sql.adaptive.enabled": {
            "type": "fixed",
            "value": "true"
        },
        "spark_conf.spark.sql.adaptive.coalescePartitions.enabled": {
            "type": "fixed",
            "value": "true"
        },
        "spark_conf.spark.databricks.delta.optimizeWrite.enabled": {
            "type": "fixed",
            "value": "true"
        },
        "custom_tags.cost_center": {
            "type": "regex",
            "pattern": "^CC-[0-9]{4}$",
            "defaultValue": "CC-0000"
        },
        "custom_tags.workload_type": {
            "type": "fixed",
            "value": "production_job"
        }
    }, indent=2)
}

print("── Sample Policy: Production Jobs ──")
print(json.dumps(json.loads(job_policy["definition"]), indent=2))


 ### Step 3: Governance Framework Summary


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                  COMPUTE GOVERNANCE FRAMEWORK                             ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  THREE PILLARS OF COST CONTROL:                                          ║
║                                                                          ║
║  1. PREVENT (Policies)                                                   ║
║     ├─ Limit max workers (e.g., 20 for dev, 50 for prod)                 ║
║     ├─ Restrict instance types to cost-effective families (i3, m5d)      ║
║     ├─ Enforce auto-termination (15-30 min)                               ║
║     ├─ Require cost_center and environment tags                           ║
║     └─ Disable expensive features for dev (Photon, GPU)                   ║
║                                                                          ║
║  2. MONITOR (Usage Tracking)                                             ║
║     ├─ system.billing.usage table (full platform)                        ║
║     ├─ Tag-based cost allocation by team/department                       ║
║     ├─ Set up budgets and alerts                                         ║
║     └─ Weekly cost review with platform team                             ║
║                                                                          ║
║  3. OPTIMIZE (Right-Sizing)                                              ║
║     ├─ Review cluster utilization: < 50% util? Downsize.                 ║
║     ├─ Use Jobs Compute for pipelines (cheaper than All-Purpose)         ║
║     ├─ Serverless for bursty SQL workloads                               ║
║     ├─ Spot instances for batch jobs (30-50% cheaper)                    ║
║     └─ Instance pools for warm start (reduces start-up time, saves DBU) ║
║                                                                          ║
║  POLICY ESCALATION PATTERN:                                              ║
║                                                                          ║
║  Default (Personal Compute)                                              ║
║    ├─ Max 8 workers, 2 hr timeout, no spot                               ║
║    │                                                                      ║
║    ├─ Need more power? → Request "Power User" policy                     ║
║    │   └─ Max 20 workers, spot allowed, longer timeout                   ║
║    │                                                                      ║
║    ├─ Running production jobs? → Use "Job Compute"                       ║
║    │   └─ Auto-configured, spot-friendly, tight timeout                  ║
║    │                                                                      ║
║    └─ Special requirements? → Exception request (temporary, reviewed)    ║
║        └─ GPU, Photon, > 50 workers, specialized instance type           ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


 ---
 ## Concept 97 — AI Functions for Data Engineers [Medium]

 ### What Problem It Solves

 You need to classify thousands of product reviews by sentiment, extract entities from contracts, or summarize support tickets — tasks that traditionally require custom ML models, regex spaghetti, or manual review. AI Functions bring LLM capabilities directly into SQL, making these tasks as simple as calling `ai_classify()`.

 **NOTE:** AI Functions require full Databricks platform with Foundation Model APIs and AI features enabled. Community Edition does not support them. This section demonstrates the concepts, syntax, and decision framework.

 ### Key Functions

 | Function | Purpose | Example Use |
 |---|---|---|
 | `ai_query()` | Send a prompt to a model | "Summarize this ticket" |
 | `ai_classify()` | Classify text into categories | Sentiment, priority, department |
 | `ai_extract()` | Extract structured data from text | Entities, dates, amounts |
 | `ai_generate()` | Generate text content | Product descriptions, email drafts |
 | `ai_mask()` | Mask PII in text | Redact names, emails, SSNs |

 ### When AI Functions Replace Traditional Logic

 | Task | Traditional Approach | AI Function Approach |
 |---|---|---|
 | **Sentiment analysis** | Custom ML model, training data, deployment | `ai_classify(text, ['positive','negative','neutral'])` |
 | **Entity extraction** | Regex patterns, NER models, multiple passes | `ai_extract(text, ['company', 'date', 'amount'])` |
 | **Ticket routing** | Keyword matching, fragile rules | `ai_classify(ticket, departments)` |
 | **Data cleansing** | Complex UDFs, lookup tables | `ai_generate("Standardize: " + raw_address)` |
 | **Text summarization** | Extractive summarization models | `ai_query("Summarize: " + long_text)` |


 ### Step 1: Traditional Regex Approach vs AI Function (Conceptual)


In [ ]:
print("=" * 60)
print("CONCEPT 97: AI FUNCTIONS FOR DATA ENGINEERS")
print("=" * 60)

# ── Create sample support ticket data ─────────────────────────────
tickets_data = [
    (1, "I cannot log into my account. The password reset email never arrives.", "open"),
    (2, "Your product is amazing! We increased sales by 40% since switching.", "resolved"),
    (3, "Billing charged me twice this month. Need immediate refund of $299.99.", "open"),
    (4, "The API returns 503 errors after the latest update. Our production is down.", "open"),
    (5, "Just wanted to say thank you for the great support. Everything works perfectly.", "closed"),
    (6, "I need to update our company address to 123 Main St, Springfield, IL 62701.", "open"),
    (7, "Can you send the Q4 invoice to john.doe@example.com? Purchase order #PO-8823.", "open"),
    (8, "Data pipeline has been failing since 2024-11-15. Error: Spark memory exceeded.", "open"),
    (9, "Feature request: please add dark mode to your dashboard. Many users requested this.", "open"),
    (10, "Urgent: Security vulnerability found in version 2.4.1. Please patch immediately.", "open"),
]

tickets_df = spark.createDataFrame(tickets_data, ["ticket_id", "description", "status"])
tickets_df.createOrReplaceTempView("support_tickets")

print("Support tickets created:")
display(tickets_df)


 ### Step 2: Traditional Approach — Regex + UDF for Classification


In [ ]:
# ── Traditional: manually classify with regex patterns ─────────────
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import re

def classify_ticket_regex(description):
    """Classify ticket using fragile regex patterns."""
    description = description.lower()
    
    if any(w in description for w in ["billing", "charge", "refund", "invoice", "payment"]):
        return "Billing"
    elif any(w in description for w in ["login", "password", "account", "cannot log"]):
        return "Account Access"
    elif any(w in description for w in ["error", "bug", "fail", "down", "503", "500", "broken"]):
        return "Technical Issue"
    elif any(w in description for w in ["security", "vulnerability", "patch", "breach"]):
        return "Security"
    elif any(w in description for w in ["feature", "request", "add", "enhancement"]):
        return "Feature Request"
    elif any(w in description for w in ["thank", "great", "amazing", "love"]):
        return "Positive Feedback"
    else:
        return "General Inquiry"

classify_udf = udf(classify_ticket_regex, StringType())

print("── Traditional Regex Classification ──")
tickets_df \
    .withColumn("classification", classify_udf(col("description"))) \
    .select("ticket_id", "classification", "description") \
    .show(truncate=80)


 ### Step 3: AI Function Approach (Conceptual — Full Platform Only)


In [ ]:
print("""
── AI Function Approach (Full Platform Only) ──

-- Replace 30+ lines of regex with ONE SQL call:

SELECT
    ticket_id,
    description,
    ai_classify(
        description,
        ARRAY('Billing', 'Account Access', 'Technical Issue',
              'Security', 'Feature Request', 'Positive Feedback',
              'General Inquiry')
    ) AS classification
FROM support_tickets;

-- Additional AI-powered enrichment:

SELECT
    ticket_id,
    description,
    ai_classify(description,
        ARRAY('urgent', 'high', 'medium', 'low')) AS priority,
    ai_extract(description,
        ARRAY('email', 'amount', 'date', 'version_number')) AS extracted_entities,
    ai_generate(
        'Write a 1-sentence summary: ' || description
    ) AS summary
FROM support_tickets;

── Key Differences from Regex ──

╔══════════════════════════════════════════════════════════════════╗
║  ASPECT            REGEX/UDF              AI FUNCTION            ║
╠══════════════════════════════════════════════════════════════════╣
║  Accuracy          70-80% (misses nuances) 90-95%+               ║
║  Maintenance       Fragile rules to update  Prompt engineering    ║
║  New categories    Rewrite pattern files   Add to category array  ║
║  Languages         One per language        Multilingual by default║
║  Edge cases        Handles known patterns  Handles novel patterns ║
║  Latency           Milliseconds            1-3 seconds per row    ║
║  Cost per row      Free (compute only)     ~$0.01-0.05 per 1k    ║
║  Infrastructure    Just Spark              Foundation Model API   ║
╚══════════════════════════════════════════════════════════════════╝

── When to Use Which ──

USE AI FUNCTIONS when:
• Text is unstructured / free-form (tickets, reviews, emails)
• Patterns are too varied for regex
• You need semantic understanding (sentiment, intent)
• Multi-language support needed
• Accuracy matters more than latency

USE REGEX/UDF when:
• Patterns are predictable (log parsing, error codes, URLs)
• Latency must be < 100ms per row
• Processing 100M+ rows (AI costs add up)
• Deterministic output required (same input → same output always)
• No LLM dependency desired
""")


 ### Step 4: Cost Estimation Framework


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                    AI FUNCTION COST ESTIMATION                           ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  TOKEN PRICING (approximate, varies by model):                           ║
║  ├─ Input:  ~$0.50 per million tokens                                    ║
║  └─ Output: ~$1.50 per million tokens                                    ║
║                                                                          ║
║  ESTIMATION FORMULA:                                                     ║
║  Cost = num_rows * (avg_input_tokens * input_rate +                      ║
║                     avg_output_tokens * output_rate)                     ║
║                                                                          ║
║  EXAMPLE CALCULATION:                                                    ║
║  ├─ 1 million support tickets                                            ║
║  ├─ avg 200 input tokens per ticket (description)                        ║
║  ├─ avg 10 output tokens (category label)                                ║
║  ├─ Input cost:  1M * 200/1M * $0.50 = $100                             ║
║  ├─ Output cost: 1M * 10/1M  * $1.50 = $15                              ║
║  └─ Total: ~$115 for 1M classifications                                  ║
║                                                                          ║
║  COMPARISON with manual review:                                          ║
║  ├─ AI classify 1M tickets: $115, ~1 hour compute time                   ║
║  └─ Manual review 1M tickets: ~$10,000+, weeks of effort                 ║
║                                                                          ║
║  OPTIMIZATION STRATEGIES:                                                ║
║  ├─ Batch processing during off-peak hours (lower token pricing)         ║
║  ├─ Pre-filter: only send ambiguous cases to AI, use regex for obvious   ║
║  ├─ Cache results: re-process only new/updated rows                      ║
║  └─ Use smaller/faster models for simple classifications                 ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


 ---
 ## Concept 98 — Multi-Workspace Architecture [Hard]

 ### What Problem It Solves

 A single Databricks workspace works for small teams, but enterprises face challenges: different teams need isolation, data must stay in specific regions, compliance mandates separation of duties. When do you split into multiple workspaces, and how do you maintain governance across them?

 ### Key Drivers for Multi-Workspace

 | Driver | Example | Impact |
 |---|---|---|
 | **Team isolation** | Data Engineering vs Data Science teams shouldn't impact each other's compute | Separate workspaces per team |
 | **Environment separation** | Dev/Staging/Prod | Separate workspaces per environment |
 | **Regional deployment** | EU data must stay in EU; US data in US | Workspace per region |
 | **Compliance boundary** | PCI-DSS workloads vs general analytics | Separate workspace with strict controls |
 | **Cost allocation** | Each business unit manages its own budget | Workspace per BU with per-workspace budgets |
 | **Blast radius** | Limit impact of misconfiguration or runaway jobs | Smaller workspaces = smaller blast radius |

 ### When One Workspace Suffices
 - Small-to-medium team (< 50 users)
 - Single region, single compliance regime
 - Unity Catalog provides sufficient isolation (schemas/catalogs)
 - No regulatory requirement for physical separation


 ### Step 1: Multi-Workspace Architecture Diagram


In [ ]:
print("=" * 60)
print("CONCEPT 98: MULTI-WORKSPACE ARCHITECTURE")
print("=" * 60)

print("""
╔══════════════════════════════════════════════════════════════════════════════════════════════╗
║                           ENTERPRISE MULTI-WORKSPACE ARCHITECTURE                            ║
╠══════════════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                              ║
║  ┌─────────────────────────────────────────────┐                                             ║
║  │        ACCOUNT LEVEL (Databricks Account)    │                                             ║
║  │  ┌───────────────────────────────────────┐   │                                             ║
║  │  │     Unity Catalog (Metastore)         │   │ ← Shared across ALL workspaces             ║
║  │  │  ┌──────────┐ ┌──────────┐ ┌───────┐ │   │                                             ║
║  │  │  │ dev_cat  │ │ stg_cat  │ │prd_cat│ │   │ ← Catalogs per environment                 ║
║  │  │  └──────────┘ └──────────┘ └───────┘ │   │                                             ║
║  │  └───────────────────────────────────────┘   │                                             ║
║  │  ┌─────────────┐  ┌─────────────┐  ┌───────┐ │                                             ║
║  │  │ Account     │  │ Account     │  │Account│ │ ← Admin: Users, Groups, Service Principals  ║
║  │  │ Users/Groups│  │ Policies    │  │ Budgets│ │                                             ║
║  │  └─────────────┘  └─────────────┘  └───────┘ │                                             ║
║  └─────────────────────────────────────────────┘                                             ║
║                         │                        │                        │                   ║
║            ┌────────────┼────────────┬───────────┼────────────┬───────────┼───────────┐       ║
║            ▼            ▼            ▼           ▼            ▼           ▼           ▼       ║
║  ┌─────────────────┐ ┌─────────────────┐ ┌──────────────┐ ┌─────────────────┐ ┌─────────────┐ ║
║  │   WORKSPACE     │ │   WORKSPACE     │ │  WORKSPACE   │ │   WORKSPACE     │ │  WORKSPACE  │ ║
║  │   US-East       │ │   US-East       │ │  US-East     │ │   EU-West       │ │  EU-West    │ ║
║  │   Development   │ │   Staging       │ │  Production  │ │   Production    │ │  Analytics  │ ║
║  │                 │ │                 │ │              │ │                 │ │             │ ║
║  │ [DE Team]      │ │ [Integration]   │ │ [Prod Jobs]  │ │ [EU Data Team] │ │ [BI Users]  │ ║
║  │ [DS Team]      │ │ [QA Testing]    │ │ [Dashboards] │ │ [Compliance]   │ │ [Ad-hoc]    │ ║
║  │ [Sandboxes]    │ │                 │ │ [Monitoring] │ │                │ │             │ ║
║  │                 │ │                 │ │              │ │                │ │             │ ║
║  │ Compute:       │ │ Compute:        │ │ Compute:     │ │ Compute:       │ │ Compute:    │ ║
║  │ All-Purpose    │ │ Jobs + SQL WH  │ │ Jobs Compute │ │ All-Purpose    │ │ SQL WH      │ ║
║  │                 │ │                 │ │              │ │                │ │             │ ║
║  │ Tags: env=dev  │ │ Tags: env=stg  │ │ Tags: env=prd│ │ Tags: env=prd  │ │ Tags: env=… │ ║
║  │       team=de  │ │       team=qa  │ │       team=de│ │       team=eu  │ │       team=bi│ ║
║  └─────────────────┘ └─────────────────┘ └──────────────┘ └─────────────────┘ └─────────────┘ ║
║                                                                                              ║
║  ─────────────────────────────────────────────────────────────────────────────────────────  ║
║                                                                                              ║
║  CROSS-WORKSPACE ORCHESTRATION:                                                              ║
║  ┌────────────────────────────────────────────────────────────────┐                         ║
║  │ Workspace A (Dev) ──deploy──▶ Workspace B (Staging)             │                         ║
║  │   DABs validate + deploy     │ tests pass?                     │                         ║
║  │                              │   ──promote──▶ Workspace C (Prod)│                         ║
║  │                                                               │                         ║
║  │ Cross-workspace job triggers via:                              │                         ║
║  │ • Databricks Workflows (cross-workspace trigger)               │                         ║
║  │ • External orchestrator (Airflow, Dagster, Prefect)            │                         ║
║  │ • CI/CD pipeline (GitHub Actions, Jenkins, Azure DevOps)       │                         ║
║  │ • Terraform/Pulumi for infra-as-code                          │                         ║
║  └────────────────────────────────────────────────────────────────┘                         ║
║                                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════════════════════╝
""")


 ### Step 2: Splitting Decision Framework


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║              WHEN TO SPLIT: DECISION FRAMEWORK                           ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  SPLIT BY TEAM when:                                                    ║
║  ├─ Teams have different compute needs (DE needs many small clusters;    ║
║  │  DS needs GPU)                                                        ║
║  ├─ Teams manage their own budgets                                       ║
║  ├─ Risk of one team's jobs impacting another's                          ║
║  └─ Teams use different cloud accounts/regions                           ║
║                                                                          ║
║  SPLIT BY ENVIRONMENT when:                                              ║
║  ├─ Strict change control required (prod changes need approval)          ║
║  ├─ Prod data must be isolated from dev access                           ║
║  ├─ Different security policies per environment                          ║
║  └─ You need environment-specific cluster policies                       ║
║                                                                          ║
║  SPLIT BY REGION when:                                                   ║
║  ├─ Data residency requirements (GDPR, CCPA, local laws)                 ║
║  ├─ Latency-sensitive workloads (compute near data)                      ║
║  ├─ Different cloud providers per region                                 ║
║  └─ Disaster recovery across regions                                     ║
║                                                                          ║
║  SPLIT BY COMPLIANCE when:                                               ║
║  ├─ PCI-DSS, HIPAA, or SOX workloads need isolation                      ║
║  ├─ Audit requires clear boundaries between regulated/unregulated data   ║
║  └─ Different retention/deletion policies                                ║
║                                                                          ║
║  KEEP SINGLE WORKSPACE when:                                             ║
║  ├─ Team < 50 users, < 3 distinct team functions                         ║
║  ├─ Unity Catalog provides sufficient isolation                          ║
║  ├─ No regulatory need for physical separation                           ║
║  └─ Simplicity wins: fewer workspaces = less operational overhead        ║
║                                                                          ║
║  ANTI-PATTERNS:                                                          ║
║  ├─ One workspace per person (way too many!)                              ║
║  ├─ Splitting early before you have a governance model                   ║
║  ├─ Different workspaces for the SAME purpose (fragmented data)          ║
║  └─ Creating workspaces manually (use Terraform/Pulumi!)                 ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


 ---
 ## Concept 99 — Performance Troubleshooting Methodology [Hard]

 ### What Problem It Solves

 A pipeline that ran in 10 minutes now takes 2 hours. A dashboard query times out. Without a structured diagnostic approach, you waste hours guessing — adding partitions, throwing more hardware at it, or rewriting code blindly.

 This concept provides a **repeatable methodology** for diagnosing and fixing slow Spark/Delta workloads, backed by the tools and techniques from Concepts #1–#98.

 ### The Diagnostic Checklist

 1. **Check the Spark UI** — Stage breakdown, task durations, skew
 2. **Identify shuffle/spill bottlenecks** — Spill to disk, exchange sizes
 3. **Examine the query plan** — Full table scans, broadcast vs sort-merge join
 4. **Verify clustering/statistics** — Are files being pruned?
 5. **Check cluster utilization** — CPU idle, GC pressure, disk I/O


 ### Step 1: Create a Deliberately Slow Query


In [ ]:
print("=" * 60)
print("CONCEPT 99: PERFORMANCE TROUBLESHOOTING METHODOLOGY")
print("=" * 60)

# Create a larger dataset to make performance issues visible
large_sales = spark.range(0, 200000).select(
    col("id").alias("transaction_id"),
    concat(lit("cust_"), lpad((col("id") % 1000).cast("string"), 4, "0")).alias("customer_id"),
    concat(lit("prod_"), lpad((col("id") % 200).cast("string"), 3, "0")).alias("product_id"),
    (rand(42) * 5000).cast(DecimalType(10, 2)).alias("amount"),
    concat(lit("store_"), ((col("id") % 20) + 1).cast("string")).alias("store_id"),
    col("id").cast("timestamp").alias("transaction_date")
).repartition(50)

large_sales_path = f"{base_dir}/perf_diag/large_sales"
large_sales.write.format("delta").mode("overwrite").save(large_sales_path)
spark.sql(f"""
    CREATE OR REPLACE TABLE perf_sales
    USING DELTA
    LOCATION '{large_sales_path}'
""")

# Create a large dimension table (simulating a poorly designed join)
large_dim = spark.range(0, 50000).select(
    concat(lit("cust_"), lpad((col("id") % 1000).cast("string"), 4, "0")).alias("customer_id"),
    concat(lit("Customer_"), (col("id") % 1000).cast("string")).alias("customer_name"),
    array("Enterprise", "SMB", "Startup", "Government", "Education")
        .getItem((col("id") % 5).cast("int")).alias("segment"),
    (rand(99) * 100000).cast(DecimalType(12, 2)).alias("lifetime_value"),
    col("id").cast("string").alias("filler_1"),
    col("id").cast("string").alias("filler_2"),
    col("id").cast("string").alias("filler_3")
).repartition(30)

large_dim_path = f"{base_dir}/perf_diag/large_dim"
large_dim.write.format("delta").mode("overwrite").save(large_dim_path)
spark.sql(f"""
    CREATE OR REPLACE TABLE perf_customers
    USING DELTA
    LOCATION '{large_dim_path}'
""")

print(f"perf_sales: {spark.table('perf_sales').count():,} rows")
print(f"perf_customers: {spark.table('perf_customers').count():,} rows")


 ### Step 2: Diagnostic Check #1 — Examine the Query Plan


In [ ]:
print("── CHECKLIST ITEM 1: Examine the Query Plan ──\n")

# The "slow" query: join with no optimization
slow_query = """
    SELECT
        c.segment,
        COUNT(*) AS transaction_count,
        SUM(s.amount) AS total_revenue
    FROM perf_sales s
    JOIN perf_customers c ON s.customer_id = c.customer_id
    GROUP BY c.segment
    ORDER BY total_revenue DESC
"""

# Show the EXPLAIN plan
print("Physical Plan for the 'slow' query:")
spark.sql(f"EXPLAIN EXTENDED {slow_query}").show(truncate=False)


 ### Step 3: Diagnostic Check #2 — Run the Query and Time It


In [ ]:
print("── CHECKLIST ITEM 2: Run and Time the Query ──\n")

t0 = time.time()
result_slow = spark.sql(slow_query)
row_count = result_slow.count()
t_slow = time.time() - t0

display(result_slow)
print(f"\nSLOW query completed in {t_slow:.2f}s")
print(f"Result rows: {row_count}")


 ### Step 4: Diagnostic Check #3 — Check Shuffle and Spill


In [ ]:
print("── CHECKLIST ITEM 3: Shuffle & Spill Analysis ──\n")

shuffle_partitions = spark.conf.get("spark.sql.shuffle.partitions")
print(f"spark.sql.shuffle.partitions = {shuffle_partitions}")

aqe_enabled = spark.conf.get("spark.sql.adaptive.enabled")
print(f"spark.sql.adaptive.enabled = {aqe_enabled}")

broadcast_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(f"spark.sql.autoBroadcastJoinThreshold = {broadcast_threshold}")

print("\n── Analyzing JOIN strategy ──")
join_plan = spark.sql(f"""
    EXPLAIN EXTENDED
    SELECT s.*, c.segment
    FROM perf_sales s
    JOIN perf_customers c ON s.customer_id = c.customer_id
    LIMIT 100
""").collect()

for row in join_plan:
    line = row[0]
    if any(kw in line for kw in ["Join", "Broadcast", "SortMerge", "Shuffle"]):
        print(f"  {line}")


 ### Step 5: Diagnostic Check #4 — Apply Fixes Step by Step


In [ ]:
print("── CHECKLIST ITEM 4: Apply Fixes ──\n")

# FIX 1: Broadcast hint for the smaller dimension table
print("FIX 1: Adding BROADCAST hint for dimension table\n")

optimized_query_1 = """
    SELECT /*+ BROADCAST(c) */
        c.segment,
        COUNT(*) AS transaction_count,
        SUM(s.amount) AS total_revenue
    FROM perf_sales s
    JOIN perf_customers c ON s.customer_id = c.customer_id
    GROUP BY c.segment
    ORDER BY total_revenue DESC
"""

t0 = time.time()
result_opt1 = spark.sql(optimized_query_1)
result_opt1.count()
t_opt1 = time.time() - t0
print(f"  SLOW query:  {t_slow:.2f}s")
print(f"  BROADCAST:   {t_opt1:.2f}s")
if t_slow > 0:
    print(f"  Improvement: {((t_slow - t_opt1) / t_slow * 100):.0f}% faster")

# FIX 2: ANALYZE TABLE for statistics
print(f"\nFIX 2: ANALYZE TABLE for column statistics\n")
spark.sql("ANALYZE TABLE perf_sales COMPUTE STATISTICS FOR ALL COLUMNS")
spark.sql("ANALYZE TABLE perf_customers COMPUTE STATISTICS FOR ALL COLUMNS")
print("  Statistics computed.")

stats = spark.sql("DESCRIBE EXTENDED perf_sales").filter(
    col("col_name").rlike("Statistics|numRows|sizeInBytes")
)
print("\n  Table statistics:")
stats.show(truncate=False)

# FIX 3: OPTIMIZE with ZORDER on join key
print(f"\nFIX 3: OPTIMIZE with ZORDER on join key\n")
spark.sql("OPTIMIZE perf_sales ZORDER BY (customer_id)")
spark.sql("OPTIMIZE perf_customers ZORDER BY (customer_id)")
print("  Z-ORDER applied on customer_id for both tables.")

# Re-run the optimized query
t0 = time.time()
result_opt3 = spark.sql(optimized_query_1)
result_opt3.count()
t_opt3 = time.time() - t0

print(f"\n  SLOW query:          {t_slow:.2f}s")
print(f"  + BROADCAST:         {t_opt1:.2f}s")
print(f"  + STATS + Z-ORDER:   {t_opt3:.2f}s")


 ### Step 6: Reusable Diagnostic Function


In [ ]:
def diagnose_query(spark, query_name, sql_query):
    """Reusable diagnostic function for query troubleshooting.
    
    Prints: execution plan summary, timing, row count, and optimization hints.
    Reference: Run this on any slow query to get a structured diagnosis.
    """
    print(f"\n{'='*70}")
    print(f" DIAGNOSTIC REPORT: {query_name}")
    print(f"{'='*70}")
    
    plan_rows = spark.sql(f"EXPLAIN EXTENDED {sql_query}").collect()
    plan_text = "\n".join([r[0] for r in plan_rows])
    
    has_broadcast = "BroadcastHashJoin" in plan_text or "BroadcastNestedLoopJoin" in plan_text
    has_sortmerge = "SortMergeJoin" in plan_text
    has_cartesian = "CartesianProduct" in plan_text
    has_full_scan = "Scan parquet" in plan_text
    
    print(f"\n  JOIN STRATEGY:")
    if has_broadcast:
        print(f"    ✓ Broadcast Join (efficient for small tables)")
    elif has_sortmerge:
        print(f"    ⚠ Sort-Merge Join (shuffle-heavy — consider BROADCAST hint)")
    elif has_cartesian:
        print(f"    ✗ Cartesian Product (CRITICAL — add join condition!)")
    else:
        print(f"    ? Unknown join type")
    
    if has_full_scan:
        print(f"    ⚠ Full table scan detected — check partition/ZORDER/pruning")
    
    shuffle_parts = spark.conf.get("spark.sql.shuffle.partitions")
    aqe = spark.conf.get("spark.sql.adaptive.enabled")
    
    print(f"\n  CONFIGURATION CHECK:")
    print(f"    shuffle.partitions:    {shuffle_parts}")
    print(f"    adaptive.enabled:      {aqe}")
    
    t0 = time.time()
    result = spark.sql(sql_query)
    num_rows = result.count()
    elapsed = time.time() - t0
    
    print(f"\n  QUERY STATS:")
    print(f"    Rows returned: {num_rows:,}")
    print(f"    Time elapsed:  {elapsed:.2f}s")
    
    print(f"\n  RECOMMENDATIONS:")
    if has_sortmerge and spark.conf.get("spark.sql.autoBroadcastJoinThreshold") != "-1":
        print(f"    → Try /*+ BROADCAST(small_table) */ hint")
    if has_full_scan:
        print(f"    → Run ANALYZE TABLE ... COMPUTE STATISTICS FOR ALL COLUMNS")
        print(f"    → Consider ZORDER on filter columns")
    if elapsed > 10:
        print(f"    → Check for data skew in the Spark UI")
        print(f"    → Verify no unnecessary columns in SELECT *")
    
    print(f"\n{'='*70}")
    
    return {
        "query_name": query_name,
        "rows": num_rows,
        "time_seconds": elapsed,
        "join_type": "broadcast" if has_broadcast else "sort_merge" if has_sortmerge else "unknown",
        "has_full_scan": has_full_scan,
        "recommendations": [
            r for r in [
                "Try BROADCAST hint" if has_sortmerge else None,
                "Run ANALYZE TABLE" if has_full_scan else None,
                "Check data skew" if elapsed > 10 else None
            ] if r
        ]
    }

# ── Test the diagnostic function ──────────────────────────────────
diagnosis = diagnose_query(spark, "Segment Revenue Analysis",
    """
    SELECT c.segment,
           COUNT(*) AS cnt,
           SUM(s.amount) AS total
    FROM perf_sales s
    JOIN perf_customers c ON s.customer_id = c.customer_id
    GROUP BY c.segment
    ORDER BY total DESC
    """
)

print("\nDiagnostic report object:")
for k, v in diagnosis.items():
    print(f"  {k}: {v}")


 ### Step 7: Troubleshooting Decision Tree


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║              PERFORMANCE TROUBLESHOOTING DECISION TREE                   ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  QUERY IS SLOW ──▶                                                       ║
║  │                                                                       ║
║  ├─ Check Spark UI: Are tasks skewed?                                    ║
║  │  ├─ YES → Data skew?                                                  ║
║  │  │  ├─ Salted join (add random prefix to keys)                        ║
║  │  │  └─ AQE skew join (spark.sql.adaptive.skewJoin.enabled=true)       ║
║  │  └─ NO → Continue                                                     ║
║  │                                                                       ║
║  ├─ Check Spark UI: Spill to disk?                                       ║
║  │  ├─ YES → Memory pressure                                             ║
║  │  │  ├─ Increase spark.sql.shuffle.partitions                          ║
║  │  │  ├─ Increase executor memory                                       ║
║  │  │  └─ Reduce data per task (increase parallelism)                    ║
║  │  └─ NO → Continue                                                     ║
║  │                                                                       ║
║  ├─ Check EXPLAIN: SortMergeJoin on large tables?                        ║
║  │  ├─ YES → Can small table be broadcast?                               ║
║  │  │  ├─ YES → Use /*+ BROADCAST(small_table) */ hint                   ║
║  │  │  └─ NO  → Ensure join keys are Z-ORDERed                           ║
║  │  │          → Check if Liquid Clustering can help (full platform)      ║
║  │  └─ NO → Continue                                                     ║
║  │                                                                       ║
║  ├─ Check EXPLAIN: Full table scan?                                      ║
║  │  ├─ YES → Are files being pruned?                                     ║
║  │  │  ├─ NO  → Run ANALYZE TABLE (stats for first 32 cols)              ║
║  │  │  ├─ NO  → Reorder columns (filter cols in first 32 positions)     ║
║  │  │  └─ NO  → Apply Z-ORDER or Liquid Clustering                       ║
║  │  └─ YES → Good (files are being pruned)                               ║
║  │                                                                       ║
║  ├─ Check Cluster Utilization: Low CPU?                                   ║
║  │  ├─ YES → I/O bound → Check file sizes (too many small files?)        ║
║  │  │  └─ Run OPTIMIZE to compact small files                            ║
║  │  └─ NO → CPU bound → Add workers or use larger instances              ║
║  │                                                                       ║
║  └─ CHECK WRITES: Is MERGE slow? (Concepts #59-#60)                      ║
║     ├─ low_shuffle_merge enabled?                                         ║
║     ├─ ZORDER on merge key?                                              ║
║     ├─ Too many small files before merge?                                 ║
║     └─ optimizeWrite enabled?                                            ║
║                                                                          ║
║  REMEMBER: Performance tuning is a CYCLE, not a checklist:               ║
║  1. Measure baseline                                                      ║
║  2. Form hypothesis                                                       ║
║  3. Apply ONE change                                                      ║
║  4. Measure again                                                         ║
║  5. Keep or revert                                                        ║
║  6. Repeat until performance target met                                   ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


 ---
 ## Concept 100 — Testing Patterns for Data Pipelines [Hard]

 ### What Problem It Solves

 "The pipeline succeeded but the data is wrong." Without testing, data quality issues propagate downstream silently. Testing data pipelines is harder than testing application code — data changes over time, schemas evolve, and edge cases (nulls, duplicates, late arrivals) are the norm, not the exception.

 This concept establishes a **comprehensive testing framework** including unit tests for transformations, integration tests for pipelines, data quality assertions, and schema evolution tests.

 ### Testing Pyramid for Data Pipelines

 ```
         ╱  E2E Tests ╲           (Full pipeline, real data)
        ╱──────────────╲
       ╱ Integration     ╲         (Component interactions, subset data)
      ╱────────────────────╲
     ╱   Unit Tests          ╲       (Individual transforms, sample data)
    ╱──────────────────────────╲
 ```

 ### What We Test

 | Level | What | Example |
 |---|---|---|
 | **Unit** | Single transformation function | Does `clean_amount()` handle negative values? |
 | **Unit** | Schema validation | Does the output have the expected columns? |
 | **Integration** | Join correctness | Do fact-dimension joins produce correct results? |
 | **Integration** | Aggregation accuracy | Are SUM/COUNT/AVG correct? |
 | **Assertions** | Business rules | No transaction amount > $1M? |
 | **Assertions** | Data quality | No nulls in required fields? |
 | **Edge cases** | Robustness | Empty input, all nulls, schema changes |


 ### Step 1: Create Test Fixtures


In [ ]:
print("=" * 60)
print("CONCEPT 100: TESTING PATTERNS FOR DATA PIPELINES")
print("=" * 60)

# ── Test Fixture: Small, controlled DataFrames ────────────────────

# Fixture 1: Normal sales data (happy path)
normal_sales = spark.createDataFrame([
    (1, "cust_001", "prod_001", 100.00, "store_1", "completed"),
    (2, "cust_002", "prod_002", 200.00, "store_2", "completed"),
    (3, "cust_001", "prod_001", 150.00, "store_1", "completed"),
], ["transaction_id", "customer_id", "product_id", "amount", "store_id", "status"])

# Fixture 2: Edge cases
edge_sales = spark.createDataFrame([
    (4, None, "prod_001", 0.00, None, "completed"),       # null IDs, zero amount
    (5, "cust_003", None, -50.00, "store_1", "returned"),  # null product, negative
    (6, "cust_001", "prod_001", 100.00, "store_1", "completed"),  # duplicate!
], ["transaction_id", "customer_id", "product_id", "amount", "store_id", "status"])

# Fixture 3: Empty data
empty_sales = spark.createDataFrame([], normal_sales.schema)

# Fixture 4: Schema evolution scenario
evolved_sales = spark.createDataFrame([
    (7, "cust_004", "prod_003", 300.00, "store_3", "pending", "web", 5),
    (8, "cust_005", "prod_004", 400.00, "store_4", "completed", "mobile", 3),
], ["transaction_id", "customer_id", "product_id", "amount", "store_id", "status", "channel", "discount_pct"])

print("Test fixtures created:")
print(f"  normal_sales:  {normal_sales.count()} rows, {len(normal_sales.columns)} cols")
print(f"  edge_sales:    {edge_sales.count()} rows (nulls, negatives, duplicates)")
print(f"  empty_sales:   {empty_sales.count()} rows (empty)")
print(f"  evolved_sales: {evolved_sales.count()} rows (new columns: channel, discount_pct)")


 ### Step 2: Define Transformation Functions to Test


In [ ]:
# ── Transformation: Clean and validate sales data ──────────────────
from pyspark.sql.functions import when

def transform_sales(df):
    """Clean and enrich raw sales data. Pure function — no side effects."""
    return df \
        .withColumn("amount",
            when(col("amount").isNull(), lit(0.0))
            .otherwise(col("amount"))
        ) \
        .withColumn("amount",
            when(col("amount") < 0, lit(0.0))
            .otherwise(col("amount"))
        ) \
        .withColumn("customer_id",
            when(col("customer_id").isNull(), lit("UNKNOWN"))
            .otherwise(col("customer_id"))
        ) \
        .withColumn("product_id",
            when(col("product_id").isNull(), lit("UNKNOWN"))
            .otherwise(col("product_id"))
        ) \
        .withColumn("sales_channel",
            when(col("amount") > 500, lit("high_value"))
            .when(col("amount") > 100, lit("medium"))
            .otherwise(lit("low"))
        )

# ── Transformation: Aggregate by customer ─────────────────────────
def aggregate_by_customer(sales_df):
    """Aggregate sales by customer. Returns summary DataFrame."""
    return sales_df \
        .groupBy("customer_id") \
        .agg(
            count("*").alias("transaction_count"),
            sum("amount").alias("total_amount"),
            round(avg("amount"), 2).alias("avg_transaction_value"),
            max("amount").alias("max_transaction")
        )

# ── Transformation: Calculate revenue by segment ──────────────────
def revenue_by_segment(sales_df, customer_df):
    """Join sales with customer dimension, calculate segment revenue."""
    return sales_df \
        .join(customer_df, "customer_id", "left") \
        .groupBy("segment") \
        .agg(
            count("*").alias("transaction_count"),
            sum("amount").alias("total_revenue")
        )

print("Transformation functions defined.")
print("  transform_sales(df)         → Clean & enrich")
print("  aggregate_by_customer(df)   → Customer-level summary")
print("  revenue_by_segment(s, c)    → Segment-level revenue")


 ### Step 3: UNIT TESTS — Transformations


In [ ]:
print("=" * 60)
print("UNIT TESTS: Transformation Functions")
print("=" * 60)

test_results = []

# ── Test 1: Happy Path ─────────────────────────────────────────────
def test_transform_sales_normal():
    """Normal data should be enriched with sales_channel column."""
    result = transform_sales(normal_sales)
    
    assert "sales_channel" in result.columns, "Missing sales_channel column"
    assert result.count() == 3, f"Expected 3 rows, got {result.count()}"
    
    channels = [r.sales_channel for r in result.select("sales_channel").distinct().collect()]
    assert "low" in channels, "Missing 'low' channel"
    assert "medium" in channels, "Missing 'medium' channel"
    
    return True, "Normal data transforms correctly"

# ── Test 2: Edge Cases (nulls, negatives) ──────────────────────────
def test_transform_sales_edge_cases():
    """Null IDs should become 'UNKNOWN', negative amounts should become 0."""
    result = transform_sales(edge_sales)
    
    assert result.filter(col("customer_id") == "UNKNOWN").count() > 0, \
        "Null customer_id not replaced with UNKNOWN"
    assert result.filter(col("product_id") == "UNKNOWN").count() > 0, \
        "Null product_id not replaced with UNKNOWN"
    
    negative_count = result.filter(col("amount") < 0).count()
    assert negative_count == 0, f"Found {negative_count} negative amounts (should be 0)"
    
    zero_count = result.filter(col("amount") == 0.0).count()
    assert zero_count >= 1, f"Expected at least 1 zero amount, got {zero_count}"
    
    return True, "Edge cases handled correctly"

# ── Test 3: Empty Input ────────────────────────────────────────────
def test_transform_sales_empty():
    """Empty input should produce empty output (not error)."""
    result = transform_sales(empty_sales)
    assert result.count() == 0, f"Expected 0 rows from empty input, got {result.count()}"
    return True, "Empty input handled gracefully"

# ── Test 4: Idempotency ────────────────────────────────────────────
def test_transform_sales_idempotent():
    """Running transform twice should produce identical results."""
    first = transform_sales(normal_sales)
    second = transform_sales(first)
    assert first.count() == second.count(), "Row counts differ"
    assert set(first.columns) == set(second.columns), "Column sets differ"
    return True, "Transform is idempotent"

# ── Run all unit tests ─────────────────────────────────────────────
tests = [
    ("Happy Path", test_transform_sales_normal),
    ("Edge Cases", test_transform_sales_edge_cases),
    ("Empty Input", test_transform_sales_empty),
    ("Idempotency", test_transform_sales_idempotent),
]

for test_name, test_fn in tests:
    try:
        passed, msg = test_fn()
        status = "✓ PASS"
        test_results.append((test_name, "PASS", msg))
    except AssertionError as e:
        status = "✗ FAIL"
        test_results.append((test_name, "FAIL", str(e)))
    except Exception as e:
        status = "✗ ERROR"
        test_results.append((test_name, "ERROR", str(e)))
    print(f"  {status}  {test_name:<20}  {test_results[-1][2]}")

passed = sum(1 for r in test_results if r[1] == "PASS")
total = len(test_results)
print(f"\n  Unit Tests: {passed}/{total} passed")


 ### Step 4: INTEGRATION TESTS — Aggregation & Join Correctness


In [ ]:
print("=" * 60)
print("INTEGRATION TESTS: Aggregations & Joins")
print("=" * 60)

integration_results = []

# ── Test 5: Aggregation Accuracy ───────────────────────────────────
def test_aggregate_by_customer():
    """Customer aggregation should produce correct totals."""
    clean = transform_sales(normal_sales)
    result = aggregate_by_customer(clean)
    
    cust_001 = result.filter(col("customer_id") == "cust_001").collect()[0]
    assert cust_001.transaction_count == 2, \
        f"Expected 2 transactions, got {cust_001.transaction_count}"
    assert abs(float(cust_001.total_amount) - 250.00) < 0.01, \
        f"Expected 250.00, got {cust_001.total_amount}"
    
    cust_002 = result.filter(col("customer_id") == "cust_002").collect()[0]
    assert cust_002.transaction_count == 1
    assert abs(float(cust_002.total_amount) - 200.00) < 0.01
    
    return True, "Aggregation totals verified"

# ── Test 6: Column Types After Transform ──────────────────────────
def test_column_types():
    """Verify expected column types after transformation."""
    clean = transform_sales(normal_sales)
    schema = {f.name: f.dataType.simpleString() for f in clean.schema.fields}
    
    assert schema["transaction_id"] == "bigint", f"Expected bigint, got {schema['transaction_id']}"
    assert schema["customer_id"] == "string", f"Expected string, got {schema['customer_id']}"
    assert "sales_channel" in schema, "Missing sales_channel column"
    
    return True, "Column types verified"

# ── Test 7: No Duplicate Keys ──────────────────────────────────────
def test_no_duplicate_transaction_ids():
    """Transaction IDs should be unique after processing."""
    clean = transform_sales(normal_sales)
    total = clean.count()
    distinct_ids = clean.select("transaction_id").distinct().count()
    assert total == distinct_ids, \
        f"Duplicate transaction_ids: {total} rows but {distinct_ids} distinct"
    return True, "No duplicate transaction IDs"

# ── Test 8: Business Rule — Amount Bounds ──────────────────────────
def test_amount_bounds():
    """Amounts should be non-negative and within expected range."""
    clean = transform_sales(normal_sales)
    
    negative = clean.filter(col("amount") < 0).count()
    assert negative == 0, f"Found {negative} negative amounts"
    
    out_of_range = clean.filter((col("amount") < 0) | (col("amount") > 1000)).count()
    assert out_of_range == 0, f"Found {out_of_range} amounts outside [0, 1000]"
    
    return True, "Amount bounds valid"

# ── Run all integration tests ──────────────────────────────────────
integration_tests = [
    ("Aggregation Accuracy", test_aggregate_by_customer),
    ("Column Types", test_column_types),
    ("No Duplicate Keys", test_no_duplicate_transaction_ids),
    ("Amount Bounds", test_amount_bounds),
]

for test_name, test_fn in integration_tests:
    try:
        passed, msg = test_fn()
        status = "✓ PASS"
        integration_results.append((test_name, "PASS", msg))
    except AssertionError as e:
        status = "✗ FAIL"
        integration_results.append((test_name, "FAIL", str(e)))
    except Exception as e:
        status = "✗ ERROR"
        integration_results.append((test_name, "ERROR", str(e)))
    print(f"  {status}  {test_name:<25}  {integration_results[-1][2]}")

passed_int = sum(1 for r in integration_results if r[1] == "PASS")
total_int = len(integration_results)
print(f"\n  Integration Tests: {passed_int}/{total_int} passed")


 ### Step 5: SCHEMA EVOLUTION TESTS


In [ ]:
print("=" * 60)
print("SCHEMA EVOLUTION TESTS")
print("=" * 60)

evolution_results = []

# ── Test 9: Handle New Columns ─────────────────────────────────────
def test_handle_new_columns():
    """Transformation should handle input with additional columns."""
    result = transform_sales(evolved_sales)
    assert result.count() == 2, f"Expected 2 rows, got {result.count()}"
    assert "sales_channel" in result.columns, "Missing sales_channel"
    return True, "Evolved schema handled (extra columns)"

# ── Test 10: Missing Columns Detection ────────────────────────────
def test_detect_missing_columns():
    """Should detect when required columns are missing."""
    incomplete = spark.createDataFrame([
        (9, "cust_006", "prod_005", "store_5", "completed"),
    ], ["transaction_id", "customer_id", "product_id", "store_id", "status"])
    
    required_cols = {"transaction_id", "customer_id", "product_id", "amount", "store_id", "status"}
    actual_cols = set(incomplete.columns)
    missing = required_cols - actual_cols
    
    assert missing == {"amount"}, f"Expected missing {{'amount'}}, got {missing}"
    return True, f"Missing columns detected: {missing}"

# ── Test 11: NULL-Heavy Data ──────────────────────────────────────
def test_null_heavy_data():
    """Pipeline should handle data where most fields are NULL."""
    null_heavy = spark.createDataFrame([
        (10, None, None, None, None, None),
        (11, "cust_007", None, 50.00, None, "completed"),
    ], ["transaction_id", "customer_id", "product_id", "amount", "store_id", "status"])
    
    result = transform_sales(null_heavy)
    assert result.count() == 2, "Row count changed"
    
    unknown_count = result.filter(col("customer_id") == "UNKNOWN").count()
    assert unknown_count == 1, f"Expected 1 UNKNOWN customer, got {unknown_count}"
    
    zero_amount_count = result.filter(col("amount") == 0.0).count()
    assert zero_amount_count >= 1, "Expected zero amount, got none"
    
    return True, "NULL-heavy data handled correctly"

# ── Run schema evolution tests ─────────────────────────────────────
evol_tests = [
    ("New Columns", test_handle_new_columns),
    ("Missing Columns Detection", test_detect_missing_columns),
    ("NULL-Heavy Data", test_null_heavy_data),
]

for test_name, test_fn in evol_tests:
    try:
        passed, msg = test_fn()
        status = "✓ PASS"
        evolution_results.append((test_name, "PASS", msg))
    except AssertionError as e:
        status = "✗ FAIL"
        evolution_results.append((test_name, "FAIL", str(e)))
    except Exception as e:
        status = "✗ ERROR"
        evolution_results.append((test_name, "ERROR", str(e)))
    print(f"  {status}  {test_name:<25}  {evolution_results[-1][2]}")

passed_evol = sum(1 for r in evolution_results if r[1] == "PASS")
total_evol = len(evolution_results)
print(f"\n  Schema Evolution Tests: {passed_evol}/{total_evol} passed")


 ### Step 6: Comprehensive Test Runner


In [ ]:
class DataPipelineTestRunner:
    """Reusable test runner for data pipeline testing.
    
    Usage:
        runner = DataPipelineTestRunner("Sales Pipeline Tests")
        runner.add_test("Happy Path", test_fn_1)
        runner.add_test("Edge Cases", test_fn_2)
        runner.run()
        runner.summary()
    """
    
    def __init__(self, suite_name):
        self.suite_name = suite_name
        self.tests = []
        self.results = []
    
    def add_test(self, name, test_function, category="General"):
        self.tests.append({"name": name, "fn": test_function, "category": category})
    
    def run(self):
        print(f"\n{'='*70}")
        print(f" TEST SUITE: {self.suite_name}")
        print(f"{'='*70}")
        
        for test in self.tests:
            try:
                passed, msg = test["fn"]()
                self.results.append({
                    "name": test["name"],
                    "category": test["category"],
                    "status": "PASS",
                    "message": msg
                })
                print(f"  ✓ PASS  [{test['category']}] {test['name']:<30} {msg}")
            except AssertionError as e:
                self.results.append({
                    "name": test["name"],
                    "category": test["category"],
                    "status": "FAIL",
                    "message": str(e)
                })
                print(f"  ✗ FAIL  [{test['category']}] {test['name']:<30} {str(e)}")
            except Exception as e:
                self.results.append({
                    "name": test["name"],
                    "category": test["category"],
                    "status": "ERROR",
                    "message": str(e)
                })
                print(f"  ✗ ERROR [{test['category']}] {test['name']:<30} {str(e)}")
        
        self._print_summary()
    
    def _print_summary(self):
        passed = sum(1 for r in self.results if r["status"] == "PASS")
        failed = sum(1 for r in self.results if r["status"] == "FAIL")
        errors = sum(1 for r in self.results if r["status"] == "ERROR")
        total = len(self.results)
        
        print(f"\n{'─'*70}")
        print(f" SUMMARY: {passed}/{total} PASSED")
        if failed > 0:
            print(f"          {failed} FAILED")
        if errors > 0:
            print(f"          {errors} ERRORS")
        print(f"{'─'*70}")
        
        rating = "✦✦✦✦✦" if passed == total else ("✦✦✦✧✧" if passed >= total * 0.8 else "✦✧✧✧✧")
        print(f" QUALITY RATING: {rating}")
    
    def get_failures(self):
        return [r for r in self.results if r["status"] != "PASS"]
    
    def assert_all_passed(self):
        """Fail if any test didn't pass. Raises AssertionError."""
        failures = self.get_failures()
        if failures:
            msg = f"{len(failures)} test(s) failed:\n"
            for f in failures:
                msg += f"  - {f['name']}: {f['message']}\n"
            raise AssertionError(msg)
        return True

# ── Create and run the full test suite ─────────────────────────────
full_runner = DataPipelineTestRunner("Full Pipeline Test Suite")

full_runner.add_test("Happy Path Transform", test_transform_sales_normal, "Unit")
full_runner.add_test("Edge Cases Transform", test_transform_sales_edge_cases, "Unit")
full_runner.add_test("Empty Input", test_transform_sales_empty, "Unit")
full_runner.add_test("Idempotency", test_transform_sales_idempotent, "Unit")
full_runner.add_test("Aggregation Accuracy", test_aggregate_by_customer, "Integration")
full_runner.add_test("Column Types", test_column_types, "Integration")
full_runner.add_test("No Duplicates", test_no_duplicate_transaction_ids, "Integration")
full_runner.add_test("Amount Bounds", test_amount_bounds, "Integration")
full_runner.add_test("New Columns", test_handle_new_columns, "Schema Evolution")
full_runner.add_test("Missing Columns", test_detect_missing_columns, "Schema Evolution")
full_runner.add_test("NULL-Heavy Data", test_null_heavy_data, "Schema Evolution")

full_runner.run()


 ### Step 7: CI/CD Integration Pattern (Conceptual)


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                    CI/CD FOR DATA PIPELINES (DABs)                       ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  TESTING IN CI/CD:                                                      ║
║                                                                          ║
║  ┌──────────────────────────────────────────────────────────────┐       ║
║  │  PR Opened ──▶ CI Pipeline                                    │       ║
║  │  ├─ 1. Lint: ruff/black (Python), sqlfluff (SQL)              │       ║
║  │  ├─ 2. Unit Tests: test transforms with sample data           │       ║
║  │  ├─ 3. Integration Tests: run on dev workspace with subset    │       ║
║  │  ├─ 4. DABs Validate: databricks bundle validate              │       ║
║  │  └─ 5. Plan: terraform plan / bundle plan                     │       ║
║  └──────────────────────────────────────────────────────────────┘       ║
║                         │                                                ║
║                         ▼                                                ║
║  ┌──────────────────────────────────────────────────────────────┐       ║
║  │  Merge to main ──▶ CD Pipeline                                │       ║
║  │  ├─ 1. Deploy to Staging: databricks bundle deploy -t stg     │       ║
║  │  ├─ 2. Smoke Tests: run on staging with prod-like data        │       ║
║  │  ├─ 3. Data Quality Checks: run expectation suite             │       ║
║  │  ├─ 4. Approval Gate: manual or automated                     │       ║
║  │  └─ 5. Deploy to Prod: databricks bundle deploy -t prod       │       ║
║  └──────────────────────────────────────────────────────────────┘       ║
║                                                                          ║
║  TEST DATA STRATEGY:                                                    ║
║  ├─ Unit tests: hardcoded fixtures (like this notebook)                 ║
║  ├─ Integration: 1% sample of production data (anonymized)              ║
║  ├─ E2E: synthetic data matching production distributions               ║
║  └─ NEVER use real production PII in test suites!                       ║
║                                                                          ║
║  ASSERTION PATTERNS:                                                     ║
║  ├─ Row counts: expected = actual (after filter/join)                   ║
║  ├─ Column types: verify schema matches expected                        ║
║  ├─ Value ranges: min/max within bounds                                 ║
║  ├─ Uniqueness: no duplicate keys                                       ║
║  ├─ Referential integrity: FKs exist in dimension tables                ║
║  ├─ Business rules: revenue > 0, dates in range, statuses valid         ║
║  └─ No data loss: source count = target count (accounting for filters)  ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


 ---
 ---
 # GRAND FINALE

 ## Congratulations! You've Completed All 100 Concepts

 If you've worked through all 10 notebooks in this series, you've covered the full spectrum of Databricks and Delta Lake engineering — from the fundamentals of the transaction log to multi-workspace enterprise architecture.

 ### What You've Built

 Through these notebooks, you've created and worked with:

 - **Delta Lake tables** with ACID transactions, time travel, schema evolution, and deletion vectors
 - **Streaming pipelines** with Auto Loader, Structured Streaming, and Change Data Feed
 - **Performance-optimized** workloads with Z-ORDER, Liquid Clustering, Photon, and Predictive Optimization
 - **Secure and governed** data with Unity Catalog, row filters, column masks, and data lineage
 - **Orchestrated workflows** with Delta Live Tables, Databricks Workflows, and DABs
 - **ML-integrated pipelines** with Feature Store, Model Registry, and MLflow
 - **Production-grade testing** frameworks with unit, integration, and data quality assertions
 - **Enterprise architecture** patterns for multi-team, multi-region deployments

 ### Skills You Can Now Claim

 - Delta Lake Architecture & Optimization
 - Apache Spark Performance Tuning
 - Data Pipeline Testing & CI/CD
 - Lakehouse Table Design (Star Schema, OBT, Data Vault)
 - Enterprise Security & Governance
 - Multi-Engine Interoperability
 - Cost Optimization & Cluster Governance
 - Production Operations & Troubleshooting


 ## 100-Concept Self-Assessment Scorecard

 Rate your confidence on each concept (1–5) to identify areas for further study.


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════════════════╗
║                          100-CONCEPT SELF-ASSESSMENT SCORECARD                           ║
╠══════════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                          ║
║  RATING GUIDE:                                                                           ║
║  1 — Heard of it                  3 — Understand, can use           5 — Can teach it      ║
║  2 — Basic understanding          4 — Can debug & optimize                                ║
║                                                                                          ║
║  ┌─────┬─────────────────────────────────────────┬───┐  ┌─────┬────────────────────────┬───┐
║  │  #  │ Concept                                  │ ★ │  │  #  │ Concept                 │ ★ │
║  ├─────┼─────────────────────────────────────────┼───┤  ├─────┼────────────────────────┼───┤
║  │  1  │ ACID Transactions                        │   │  │  51 │ Serverless Economics    │   │
║  │  2  │ Transaction Log (_delta_log)             │   │  │  52 │ Predicate Pushdown      │   │
║  │  3  │ Time Travel & RESTORE                    │   │  │  53 │ Predictive Optimization │   │
║  │  4  │ Schema Enforcement vs Evolution          │   │  │  54 │ Column Statistics       │   │
║  │  5  │ Liquid Clustering                        │   │  │  55 │ Write Optimization      │   │
║  │  6  │ OPTIMIZE & File Compaction               │   │  │  56 │ Cluster Sizing          │   │
║  │  7  │ VACUUM & Storage Lifecycle               │   │  │  57 │ DBU Cost Model          │   │
║  │  8  │ Change Data Feed (CDF)                   │   │  │  58 │ Query Profile           │   │
║  │  9  │ Deletion Vectors                         │   │  │  59 │ Join Performance        │   │
║  │ 10  │ Table Properties & Config               │   │  │  60 │ MERGE Amplification     │   │
║  ├─────┼─────────────────────────────────────────┼───┤  ├─────┼────────────────────────┼───┤
║  │ 11  │ Spark Architecture Overview             │   │  │  61 │ Unity Catalog Metastore │   │
║  │ 12  │ DAG Execution & Lazy Evaluation         │   │  │  62 │ Table ACLs              │   │
║  │ 13  │ Shuffle & Partition Mechanics           │   │  │  63 │ Row Filters             │   │
║  │ 14  │ Data Skew Diagnosis                     │   │  │  64 │ Column Masks            │   │
║  │ 15  │ Tungsten & Whole-Stage CodeGen          │   │  │  65 │ Data Lineage            │   │
║  │ 16  │ Broadcast vs Sort-Merge Joins           │   │  │  66 │ External Locations      │   │
║  │ 17  │ Adaptive Query Execution (AQE)          │   │  │  67 │ Credential Passthrough  │   │
║  │ 18  │ Memory Management (on-heap/off-heap)    │   │  │  68 │ Information Schema      │   │
║  │ 19  │ Partition Coalescing                    │   │  │  69 │ Audit Logging           │   │
║  │ 20  │ Stage-Level Scheduling                  │   │  │  70 │ Data Classification     │   │
║  ├─────┼─────────────────────────────────────────┼───┤  ├─────┼────────────────────────┼───┤
║  │ 21  │ DataFrame vs SQL: When Each Wins        │   │  │  71 │ Delta Live Tables (DLT) │   │
║  │ 22  │ Temporary Views & CTEs                  │   │  │  72 │ DLT CDC/SCD Type 2      │   │
║  │ 23  │ Window Functions for Analytics          │   │  │  73 │ Databricks Workflows    │   │
║  │ 24  │ Pivot/Unpivot Operations                │   │  │  74 │ Job Clusters vs Pools   │   │
║  │ 25  │ Subqueries & Correlated Queries         │   │  │  75 │ Trigger Types           │   │
║  │ 26  │ UDFs vs Built-in Functions              │   │  │  76 │ Job Parameters          │   │
║  │ 27  │ Spark Catalyst Optimizer                │   │  │  77 │ Repair & Retry Logic    │   │
║  │ 28  │ Higher-Order Functions                  │   │  │  78 │ DABs (CI/CD Bundles)    │   │
║  │ 29  │ Photon Engine                           │   │  │  79 │ DABs with Terraform     │   │
║  │ 30  │ Query Federation                        │   │  │  80 │ Monitoring & SLA Alerts │   │
║  ├─────┼─────────────────────────────────────────┼───┤  ├─────┼────────────────────────┼───┤
║  │ 31  │ Auto Loader (file notification)         │   │  │  81 │ MLflow Tracking         │   │
║  │ 32  │ COPY INTO for batch ingestion           │   │  │  82 │ MLflow Model Registry   │   │
║  │ 33  │ Incremental Ingestion Patterns          │   │  │  83 │ Feature Store Eng.      │   │
║  │ 34  │ Bronze Table Design (Raw Zone)          │   │  │  84 │ Feature Lookup          │   │
║  │ 35  │ Schema Inference & Evolution            │   │  │  85 │ Online vs Offline Store │   │
║  │ 36  │ File Formats: Parquet, Avro, CSV        │   │  │  86 │ Hyperopt Tuning         │   │
║  │ 37  │ Ingestion from Kafka/Event Hubs         │   │  │  87 │ Autologging             │   │
║  │ 38  │ Slowly Changing Dimensions (SCD)        │   │  │  88 │ Serving Endpoints       │   │
║  │ 39  │ Data Quality: Lakeflow Expectations     │   │  │  89 │ Lakehouse Monitoring    │   │
║  │ 40  │ Streaming ForeachBatch Patterns         │   │  │  90 │ Model Lifecycle Mgmt    │   │
║  ├─────┼─────────────────────────────────────────┼───┤  ├─────┼────────────────────────┼───┤
║  │ 41  │ Structured Streaming Overview           │   │  │  91 │ Shallow vs Deep Clones  │   │
║  │ 42  │ Watermarking & State Management         │   │  │  92 │ Managed vs External     │   │
║  │ 43  │ Output Modes & Triggers                 │   │  │  93 │ Legacy Partition/Z-ORDER│   │
║  │ 44  │ Streaming Aggregations                  │   │  │  94 │ Lakehouse Design Patt.  │   │
║  │ 45  │ Streaming Joins (Stream-Stream)         │   │  │  95 │ Delta UniForm & Interop │   │
║  │ 46  │ Change Data Feed for Streaming          │   │  │  96 │ Compute Policies        │   │
║  │ 47  │ Streaming Checkpoint Recovery           │   │  │  97 │ AI Functions for DE     │   │
║  │ 48  │ Idempotent Writes for Streaming         │   │  │  98 │ Multi-Workspace Arch    │   │
║  │ 49  │ Rate Limiting & Back-Pressure           │   │  │  99 │ Performance Troubleshoot│   │
║  │ 50  │ Delta Live Tables Streaming             │   │  │ 100 │ Testing Patterns        │   │
║  └─────┴─────────────────────────────────────────┴───┘  └─────┴────────────────────────┴───┘
║                                                                                          ║
║  ┌─────────────────────────────────────────────────────────────────────────────────┐    ║
║  │  SCORING:                                                                         │    ║
║  │  ├─ 400-500  ✦✦✦✦✦  Expert — ready for certification and production leadership   │    ║
║  │  ├─ 300-399  ✦✦✦✦✧  Advanced — strong practitioner, fill a few gaps               │    ║
║  │  ├─ 200-299  ✦✦✦✧✧  Intermediate — solid foundation, continue building            │    ║
║  │  ├─ 100-199  ✦✦✧✧✧  Beginner — keep learning, revisit Notebooks 01-03            │    ║
║  │  └─ <100     ✦✧✧✧✧  Just starting — welcome! Work through sequentially            │    ║
║  └─────────────────────────────────────────────────────────────────────────────────┘    ║
║                                                                                          ║
╚══════════════════════════════════════════════════════════════════════════════════════════╝
""")


 ---
 ## Next Steps: Where to Go From Here

 ### Certifications

 | Certification | Level | Focus | Recommended After |
 |---|---|---|---|
 | **Databricks Data Engineer Associate** | Entry | Delta, Spark, ETL basics | Completing notebooks 1–3 |
 | **Databricks Data Engineer Professional** | Advanced | DLT, streaming, optimization, DABs | All 10 notebooks |
 | **Databricks ML Engineer Associate** | Entry | MLflow, Feature Store, model serving | Notebooks 1–3 + 9 |
 | **Databricks SQL Analyst** | Entry | SQL warehouses, dashboards, BI | Notebooks 1–3 |
 | **Databricks Platform Administrator** | Advanced | Workspaces, policies, Unity Catalog | Notebooks 6–8, 10 |

 ### Hands-On Projects

 | Project | Concepts Applied | Difficulty |
 |---|---|---|
 | **Build a Medallion Pipeline** | #1-#10, #31-#40 | Intermediate |
 | **Real-Time Fraud Detection** | #41-#50, #81-#90 | Advanced |
 | **Enterprise Data Platform** | #61-#70, #91-#100 | Expert |
 | **Cost Optimization Audit** | #51-#60, #96 | Intermediate |
 | **Multi-Engine Lakehouse** | #92, #95, #98 | Expert |

 ### Community & Resources

 | Resource | URL | Purpose |
 |---|---|---|
 | **Databricks Documentation** | docs.databricks.com | Official reference |
 | **Delta Lake OSS** | delta.io | Open-source Delta |
 | **Apache Spark Docs** | spark.apache.org/docs | Spark internals |
 | **Databricks Community** | community.databricks.com | Q&A, discussions |
 | **Databricks Blog** | databricks.com/blog | Best practices, releases |
 | **Databricks Academy** | partner-academy.databricks.com | Free courses |
 | **Delta Lake GitHub** | github.com/delta-io/delta | Source code, examples |

 ### Learning Path Continuation

 ```
 100-Concept Series (You Are Here)
       │
       ├─▶ Databricks Associate Certification
       │
       ├─▶ Databricks Professional Certification
       │
       ├─▶ Real-World Project Portfolio (3-5 projects)
       │
       └─▶ Community Contribution (blog posts, talks, open source)
 ```


In [ ]:
# CLEANUP: Drop all tables created in this notebook
print("Cleaning up...")
tables_to_drop = [
    "clone_source", "clone_shallow", "clone_deep",
    "sales_managed", "sales_external",
    "drop_test_managed", "drop_test_external",
    "gold.fact_sales", "gold.dim_customer", "gold.dim_product",
    "gold.dim_date", "gold.wide_sales_reporting",
    "perf_sales", "perf_customers"
]
for t in tables_to_drop:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    except:
        pass
dbutils.fs.rm(base_dir, recurse=True)
print("Cleanup complete.")


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                                                                          ║
║                         END OF NOTEBOOK SERIES                           ║
║                                                                          ║
║                   10 Architecture & Advanced Patterns                    ║
║                                                                          ║
║          Concepts #91-#100  |  Capstone  |  Grand Finale                 ║
║                                                                          ║
║   "The best time to start learning was yesterday.                         ║
║    The second best time is now."                                          ║
║                                                                          ║
║   Thank you for completing the 100-Concept Databricks                    ║
║   Professional Learning series. Go build something amazing.              ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


In [ ]:
print("End of 10_Architecture_Advanced_Patterns notebook. All 10 concepts (#91-#100) covered. GRAND FINALE complete.")
